In [ ]:
import pandas as pd

# File paths (updated for Extension_4.0)
base_path = '<LOCAL_PATH>'

# Define the datasets
datasets = {
    '(Train)': 'CWD_negative_training_with_seasons2.csv',
    '(Test)': 'CWD_positive_testing_with_seasons2.csv'
}

# Define function to print unique IDs, status, and fix counts
def print_unique_ids_and_status(df, label):
    ids = sorted(df['id'].unique())
    print(f"\n📄 {label} — {len(ids)} unique IDs:")
    print(ids)

    # CWD Positive
    positive_ids = sorted(df.loc[df['cwd_status'] == 'positive', 'id'].unique())
    print(f"  ➡ CWD Positive IDs ({len(positive_ids)}): {positive_ids}")

    # CWD Negative
    negative_ids = sorted(df.loc[df['cwd_status'] == 'negative', 'id'].unique())
    print(f"  ➡ CWD Negative IDs ({len(negative_ids)}): {negative_ids}")

    # Fix counts per ID
    fix_counts = df['id'].value_counts().sort_index()
    print("\n📊 Number of GPS fixes per ID:")
    print(fix_counts.to_string())

    # Optional: summary stats
    print(f"\n   ➤ Fix count summary: min={fix_counts.min()}, max={fix_counts.max()}, median={fix_counts.median()}, mean={fix_counts.mean():.1f}")

def run_datetime_eda(df, label):
    print(f"\n🧪 EDA on `date` for {label}")

    # Check for missing
    null_count = df['date'].isnull().sum()
    print(f"  ➤ Null values in `date`: {null_count}")

    # Convert to datetime
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    coercion_count = df['date'].isnull().sum() - null_count
    if coercion_count > 0:
        print(f"  ⚠️ {coercion_count} additional nulls after conversion (possibly bad formats)")

    # Check range
    min_time = df['date'].min()
    max_time = df['date'].max()
    print(f"  ➤ Overall date range: {min_time} to {max_time}")

    # Duration per ID
    id_group = df.groupby('id')['date']
    durations = (id_group.max() - id_group.min()).dt.total_seconds() / (60*60*24)
    print(f"  ➤ Duration (days) across IDs: min={durations.min():.2f}, max={durations.max():.2f}, median={durations.median():.2f}")

    # Continuity: time between rows per ID
    discontinuous_ids = []
    for _id, group in df.sort_values('date').groupby('id'):
        times = group['date'].dropna()
        deltas = times.diff().dt.total_seconds().dropna()
        if (deltas == 0).all():
            discontinuous_ids.append(_id)

    if discontinuous_ids:
        print(f"  ⚠️ {len(discontinuous_ids)} IDs have no time variation (i.e., all timestamps are identical):")
        print(f"     {discontinuous_ids[:10]}{' ...' if len(discontinuous_ids) > 10 else ''}")
    else:
        print(f"  ✅ All IDs have time variation.")

    return df

# Load and process each dataset
for label, file_name in datasets.items():
    path = f"{base_path}/{file_name}"
    df = pd.read_csv(path)
    print_unique_ids_and_status(df, label)
    df = run_datetime_eda(df, label)


In [ ]:
import pandas as pd

# File paths (updated for Extension_4.0)
base_path = '<LOCAL_PATH>'

# Define the datasets
files = {
    '(Train)': 'CWD_negative_training_with_seasons2.csv',
    '(Test)': 'CWD_positive_testing_with_seasons2.csv'
}

dfs = []
for split, fname in files.items():
    df = pd.read_csv(f"{base_path}/{fname}")
    df['split'] = split
    dfs.append(df)

full = pd.concat(dfs, ignore_index=True)

# ---- Animal-level summary ----
animal_status = (
    full
    .groupby('id')
    .agg(
        CWD_Status=('cwd_status', 'first'),
        n_fixes=('id', 'size'),
        start_date=('date', 'min'),
        end_date=('date', 'max')
    )
    .reset_index()
)

animal_status['duration_days'] = (
    pd.to_datetime(animal_status['end_date']) -
    pd.to_datetime(animal_status['start_date'])
).dt.days

# ---- Counts ----
n_animals = animal_status['id'].nunique()
n_pos = (animal_status['CWD_Status'] == 'positive').sum()
n_neg = (animal_status['CWD_Status'] == 'negative').sum()

cwd_prev = 100 * n_pos / n_animals

print(f"\nTotal animals: {n_animals}")
print(f"CWD+ animals: {n_pos}")
print(f"CWD- animals: {n_neg}")
print(f"Overall CWD prevalence: {cwd_prev:.1f}%")

print("\nTracking duration (days):")
print(animal_status['duration_days'].describe(percentiles=[.05,.25,.5,.75,.95]))

print("\nFix count per animal:")
print(animal_status['n_fixes'].describe(percentiles=[.05,.25,.5,.75,.95]))


In [ ]:
import pandas as pd

# Ensure datetime
full['date'] = pd.to_datetime(full['date'])

# Sort so diffs are correct
full = full.sort_values(['id', 'date'])

# Compute time difference between consecutive fixes per animal
full['time_diff_hours'] = (
    full.groupby('id')['date']
    .diff()
    .dt.total_seconds() / 3600
)

# Remove first fix per animal (NaN)
diffs = full['time_diff_hours'].dropna()

print("\nGPS Fix Interval Summary (hours):")
print(diffs.describe(percentiles=[.05, .25, .5, .75, .95]))

print("\nMost common intervals:")
print(diffs.round(2).value_counts().head(10))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 100% experiment: leak-free preprocessing with ID-disjoint splits + train-only fit
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

# ── REPRODUCIBILITY ───────────────────────────────────────────────────────────
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13)
np.random.seed(13)

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base       = Path("<LOCAL_PATH>")

# inputs: training pool is CWD-free; test is CWD-positive
train_raw  = base / "CWD_negative_training_with_seasons2.csv"   # negatives only (still guard below)
test_raw   = base / "CWD_positive_testing_with_seasons2.csv"     # positives (mixed OK; we only transform)

# outputs
out_train  = base / "train_processed_100.csv"
out_val    = base / "val_processed_100.csv"
out_test   = base / "test_processed_100.csv"

enc_path   = base / "encoder_100.pkl"
sca_path   = base / "scaler_100.pkl"
feat_path  = base / "feature_columns_100.pkl"
split_path = base / "split_ids_100.pkl"

to_drop = [
    'date_std', 'dx','dy','dt','burst','seg_key'
]

num_feats = [
    'velo','abs_rel_angle','sinuosity',
    'rolling_abs_rel_angle_mean','rolling_abs_rel_angle_sd',
    'rolling_abs_rel_angle_skew','rolling_abs_rel_angle_kurt',
    'rolling_velo_mean','rolling_velo_sd','rolling_velo_skew','rolling_velo_kurt',
    'akde_low','akde_est','akde_high'
]

cat_feats  = ['season']
meta_feats = ['id','date']

# ── UTILITIES ──────────────────────────────────────────────────────────────────
def basic_cleanup(df: pd.DataFrame) -> pd.DataFrame:
    """Drop unused cols, enforce numeric types, keep pkey intact, preserve order."""
    df = df.drop(columns=to_drop, errors='ignore').copy()
    
    # Preserve original datetime string AND a parsed version
    df['datetime_raw'] = df['date']   # keep raw untouched
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    
    for c in num_feats:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    
    df = df.dropna(subset=num_feats + cat_feats).copy()
    
    # Sort by id + parsed datetime (if valid), fallback to pkey for stability
    df = df.sort_values(['id', 'date']).reset_index(drop=True)
    return df

def transform_with_artifacts(df, encoder, scaler, feature_columns):
    """Apply fitted encoder/scaler; return DataFrame with pkey + meta + features."""
    cat_mat = encoder.transform(df[cat_feats])
    num_mat = scaler.transform(df[num_feats])
    cat_names = encoder.get_feature_names_out(cat_feats).tolist()
    X = np.hstack([num_mat, cat_mat])
    feat_df = pd.DataFrame(X, columns=num_feats + cat_names)

    feat_df = feat_df.reindex(columns=feature_columns, fill_value=0.0)
    out = pd.concat([df[meta_feats].reset_index(drop=True),
                     feat_df.reset_index(drop=True)], axis=1)
    return out

# ── 1) LOAD TRAIN POOL ────────────────────────────────────────────────────────
df_train_pool = pd.read_csv(train_raw)
df_train_pool = df_train_pool[df_train_pool['cwd_status'] == "negative"].copy()
df_train_pool = basic_cleanup(df_train_pool)

# ── 2) ID-DISJOINT SPLIT ──────────────────────────────────────────────────────
ids = df_train_pool['id'].unique()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=13)
train_idx, val_idx = next(gss.split(df_train_pool, groups=df_train_pool['id']))

train_ids = df_train_pool.iloc[train_idx]['id'].unique()
val_ids   = df_train_pool.iloc[val_idx]['id'].unique()

df_train = df_train_pool[df_train_pool['id'].isin(train_ids)].copy()
df_val   = df_train_pool[df_train_pool['id'].isin(val_ids)].copy()

# ── 3) FIT ENCODER & SCALER ON TRAIN ONLY ─────────────────────────────────────
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
scaler  = MinMaxScaler()

encoder.fit(df_train[cat_feats])
scaler.fit(df_train[num_feats])

cat_names = encoder.get_feature_names_out(cat_feats).tolist()
feature_columns = num_feats + cat_names

# ── 4) TRANSFORM SPLITS ──────────────────────────────────────────────────────
df_train_out = transform_with_artifacts(df_train, encoder, scaler, feature_columns)
df_val_out   = transform_with_artifacts(df_val,   encoder, scaler, feature_columns)

# ── 5) TEST ───────────────────────────────────────────────────────────────────
df_test_raw = pd.read_csv(test_raw)
df_test_raw = basic_cleanup(df_test_raw)
df_test_out = transform_with_artifacts(df_test_raw, encoder, scaler, feature_columns)

# ── 6) SAVE ARTIFACTS ─────────────────────────────────────────────────────────
joblib.dump(encoder, enc_path)
joblib.dump(scaler,  sca_path)
joblib.dump(feature_columns, feat_path)

# always keep IDs as strings
train_ids_str = sorted({str(x).strip() for x in train_ids})
val_ids_str   = sorted({str(x).strip() for x in val_ids})
test_ids_str  = sorted({str(x).strip() for x in df_test_out['id'].astype(str).unique()})

joblib.dump(
    {"train_ids": train_ids_str,
     "val_ids":   val_ids_str,
     "test_ids":  test_ids_str},
    split_path
)

df_train_out.to_csv(out_train, index=False)
df_val_out.to_csv(out_val, index=False)
df_test_out.to_csv(out_test, index=False)

print("✅ Preprocessing complete with pkey preserved.")
print("Train IDs:", train_ids_str)
print("Val IDs:",   val_ids_str)
print("Test IDs:",  test_ids_str)




In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Autoencoder training USING CUSTOM LOSS + flexible thresholding
# ──────────────────────────────────────────────────────────────────────────────
import os, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ===== Reproducibility =====
os.environ["PYTHONHASHSEED"] = "0"
random.seed(13); np.random.seed(13)
tf.keras.utils.set_random_seed(13)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

# ===== Config / Paths =====
base = "<LOCAL_PATH>"
train_path = f"{base}/train_processed_100.csv"
val_path   = f"{base}/val_processed_100.csv"   # negatives-only val (from preprocessing)
test_path  = f"{base}/test_processed_100.csv"

feat_cols_path = f"{base}/feature_columns_100.pkl"
model_path     = f"{base}/autoencoder_model.h5"
threshold_path = f"{base}/reconstruction_threshold.pkl"
history_path   = f"{base}/training_history.npy"
ckpt_path      = f"{base}/ae_ckpt.weights.h5"

# ===== Threshold policy (choose one) =====
THRESHOLD_POLICY = "train_lowest_p95"   # "train_lowest_p95" | "train_quantile" | "val_row_fpr" | "val_id_flag_rate" | "mad"

# Params for each policy
LOWEST_FRAC   = 0.75      # for "train_lowest_p95" 
LOWEST_Q      = 90        # percentile on the lowest fraction
TRAIN_Q       = 0.90      # for "train_quantile": e.g., 90th percentile of all train errors
VAL_ROW_FPR   = 0.20      # for "val_row_fpr": allow 20% of NEG val rows to flag → lower θ, more flags
VAL_ID_RATE   = 0.20      # for "val_id_flag_rate": allow 20% of NEG val IDs to flag at least 1 row
MIN_ROWS_PER_ID = 1       # ignore super-short IDs when computing ID-level rate
MAD_K         = 3.0       # for "mad"


# ===== Load features with frozen column order =====
feature_columns = joblib.load(feat_cols_path)

def load_X(path):
    df = pd.read_csv(path)
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    X = df[feature_columns].values.astype(np.float32)
    meta = df[["id"]].copy()
    if "datetime" in df.columns:
        meta["datetime"] = df["datetime"].values
    return X, meta, df

X_train, meta_train, df_train_raw = load_X(train_path)
X_val,   meta_val,   df_val_raw   = load_X(val_path)
X_test,  meta_test,  df_test_raw  = load_X(test_path)

# ===== tf.data pipelines =====
batch_size = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, X))
    if training:
        ds = ds.shuffle(min(10000, len(X)), seed=13, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False).cache().prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, training=True)
val_ds   = make_ds(X_val,   training=False)

# ===== Model =====
input_dim = X_train.shape[1]
reg = regularizers.l2(1e-4)

inp = layers.Input(shape=(input_dim,))
x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(inp)
x = layers.Dense(16, activation='relu', kernel_regularizer=reg)(x)
z = layers.Dense(4,  activation='relu', kernel_regularizer=reg, name="bottleneck")(x)
y = layers.Dense(16, activation='relu', kernel_regularizer=reg)(z)
y = layers.Dense(32, activation='relu', kernel_regularizer=reg)(y)
out = layers.Dense(input_dim, activation='linear')(y)
autoencoder = Model(inp, out, name="autoencoder")

# === Custom loss ===
LAMBDA_DEV = 0.5                      # ← your choice
LOSS_SCALE = 1.0 + LAMBDA_DEV*input_dim  # α = 1 + λ d

def get_custom_loss(loss_scale=LOSS_SCALE):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)  # mean over features
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(LOSS_SCALE)
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(1e-6), loss=loss_fn)

# ===== Train =====
early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
ckpt = ModelCheckpoint(filepath=ckpt_path, save_weights_only=True, monitor="val_loss", save_best_only=True)

history = autoencoder.fit(
    train_ds, epochs=100, validation_data=val_ds, callbacks=[early_stop, ckpt], verbose=1
)

autoencoder.save(model_path)
np.save(history_path, history.history, allow_pickle=True)
print(f"✅ Saved model → {model_path}")

# ===== Errors on train/val (row-level, using SAME loss) =====
recon_train = autoencoder.predict(X_train, batch_size=batch_size, verbose=0)
train_errors = loss_fn(X_train, recon_train).numpy().astype(np.float64)

recon_val = autoencoder.predict(X_val, batch_size=batch_size, verbose=0)
val_errors = loss_fn(X_val, recon_val).numpy().astype(np.float64)

# ===== Threshold policies =====
def choose_threshold(policy: str):
    info = {"policy": policy}   # <— standard key

    if policy == "train_lowest_p95":
        n_low = int(len(train_errors) * LOWEST_FRAC)
        base = np.sort(train_errors)[:n_low]
        theta = float(np.percentile(base, LOWEST_Q))
        info.update({"lowest_frac": LOWEST_FRAC, "percentile": LOWEST_Q, "n_low": n_low})

    elif policy == "train_quantile":
        theta = float(np.quantile(train_errors, TRAIN_Q))
        info.update({"train_quantile": TRAIN_Q})

    elif policy == "val_row_fpr":
        q = 1.0 - float(VAL_ROW_FPR)
        theta = float(np.quantile(val_errors, q))
        info.update({"val_row_fpr": VAL_ROW_FPR, "val_quantile": q})

    elif policy == "val_id_flag_rate":
        q_grid = np.linspace(0.50, 0.999, 200)
        ids = df_val_raw["id"].values
        best_theta, best_rate, best_q = None, None, None
        for q in q_grid:
            th = np.quantile(val_errors, q)
            row_flag = val_errors > th
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            rate = float(id_any.reindex(eligible).fillna(False).mean())
            if rate <= VAL_ID_RATE:
                best_theta, best_rate, best_q = float(th), rate, float(q)
                break
        if best_theta is None:
            best_q = float(q_grid[-1])
            best_theta = float(np.quantile(val_errors, best_q))
            row_flag = val_errors > best_theta
            id_any = pd.Series(row_flag).groupby(ids, sort=False).any()
            n_rows = pd.Series(1, index=ids).groupby(ids).sum()
            eligible = n_rows[n_rows >= MIN_ROWS_PER_ID].index
            best_rate = float(id_any.reindex(eligible).fillna(False).mean())
        theta = best_theta
        info.update({"val_id_rate_target": VAL_ID_RATE, "achieved_rate": best_rate, "val_quantile": best_q})

    elif policy == "mad":
        med = float(np.median(train_errors))
        mad = float(np.median(np.abs(train_errors - med)) + 1e-12)
        theta = med + MAD_K * 1.4826 * mad
        info.update({"median": med, "mad": mad, "k": MAD_K})

    else:
        raise ValueError(f"Unknown THRESHOLD_POLICY: {policy}")

    return theta, info

theta, th_info = choose_threshold(THRESHOLD_POLICY)

# Persist policy hyperparameters for reproducibility
if THRESHOLD_POLICY == "train_lowest_p95":
    th_info.update({"LOWEST_FRAC": float(LOWEST_FRAC), "LOWEST_Q": float(LOWEST_Q)})
elif THRESHOLD_POLICY == "train_quantile":
    th_info.update({"TRAIN_Q": float(TRAIN_Q)})
elif THRESHOLD_POLICY == "val_row_fpr":
    th_info.update({"VAL_ROW_FPR": float(VAL_ROW_FPR)})
elif THRESHOLD_POLICY == "val_id_flag_rate":
    th_info.update({"VAL_ID_RATE": float(VAL_ID_RATE), "MIN_ROWS_PER_ID": int(MIN_ROWS_PER_ID)})
elif THRESHOLD_POLICY == "mad":
    th_info.update({"MAD_K": float(MAD_K)})

th_info["effective_multiplier"] = float(LOSS_SCALE)  # alias for reviewers
print(f"[loss] λ={LAMBDA_DEV} | d={input_dim} | α=1+λ·d={LOSS_SCALE:.3f} (scaled-MSE)")


# ===== Diagnostics: how aggressive is θ? =====
def percent_rows_flagged(errors, theta):
    return float(np.mean(errors > theta))

train_row_rate = percent_rows_flagged(train_errors, theta)
val_row_rate   = percent_rows_flagged(val_errors, theta)

# ID-level on val (negatives)
val_ids = df_val_raw["id"].values
val_row_flag = val_errors > theta
val_id_any = pd.Series(val_row_flag).groupby(val_ids, sort=False).any()
val_n_rows = pd.Series(1, index=val_ids).groupby(val_ids).sum()
val_eligible = val_n_rows[val_n_rows >= MIN_ROWS_PER_ID].index
val_id_rate = float(val_id_any.reindex(val_eligible).fillna(False).mean())

print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")
print(f"   Train rows flagged: {100*train_row_rate:.1f}%")
print(f"   Val   rows flagged: {100*val_row_rate:.1f}%")
print(f"   Val     IDs flagged (≥1 row): {100*val_id_rate:.1f}% (min_rows={MIN_ROWS_PER_ID})")

# ===== Persist threshold + bookkeeping =====
# Diagnostics
# (unchanged, but print 'policy')
print(f"📏 θ = {theta:.6g} | policy = {THRESHOLD_POLICY}")

# Persist threshold + bookkeeping (use 'policy')
th_info.update({
    "policy": THRESHOLD_POLICY,
    "theta": float(theta),
    "lambda_dev": float(LAMBDA_DEV),
    "input_dim": int(input_dim),
    "loss_scale": float(LOSS_SCALE),  # α
    "loss_name": "scaled_mse_equivalent_to_mse_plus_lambda_sse",
    "train_rows_flagged": train_row_rate,
    "val_rows_flagged": val_row_rate,
    "val_ids_flagged": val_id_rate,
    "min_rows_per_id": int(MIN_ROWS_PER_ID)
})
joblib.dump(th_info, threshold_path)
print(f"✅ Threshold saved → {threshold_path}")


In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── CONFIG ─────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
val_csv         = base / "val_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
threshold_path  = base / "reconstruction_threshold.pkl"   # contains θ + loss params from training
feat_cols_path  = base / "feature_columns_100.pkl"      # enforce column order
param_out_path  = base / "id_rule_params.pkl"

USE_MIXED_VAL = False  # your val is negatives-only per preprocessing

# preregistered operating point (choose & freeze before test)
FROZEN_COUNT     = 550
FROZEN_FRAC      = 0.75
MIN_ROWS_PER_ID  = 1          # consider 100 if short IDs are noisy
AGGREGATION_RULE = "OR"       # freeze: "OR", "AND", "FRAC_ONLY", or "COUNT_ONLY"

# if you ever use mixed val with labels, these grids are considered a priori
COUNTS = [50, 100, 150, 200]
FRACS  = [0.80, 0.90, 0.925]
POS_IDS_VAL = []   # only used when USE_MIXED_VAL=True

# ── LOAD MODEL / FEATURES / TRAIN ARTIFACT ────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)
autoencoder = load_model(model_path, compile=False)

# load training artifact with θ + loss params
th_info   = joblib.load(threshold_path)
theta     = float(th_info["theta"])
policy    = th_info.get("policy", "unknown")

# reconstruct scaled-MSE loss consistently
lambda_dev = float(th_info.get("lambda_dev", 0.5))
input_dim  = int(th_info.get("input_dim", len(feature_columns)))
loss_scale = float(th_info.get("loss_scale", 1.0 + lambda_dev * input_dim))

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
print(f"Using TRAIN-derived θ = {theta:.6f} (policy={policy}) | λ={lambda_dev} | d={input_dim} | α={loss_scale:.6g}")

# ── LOAD VAL AND COMPUTE ROW ERRORS (FROZEN FEATURE ORDER) ────────────────────
df_val = pd.read_csv(val_csv)
if "datetime" in df_val.columns:
    df_val["datetime"] = pd.to_datetime(df_val["datetime"], errors="coerce")

X_val     = df_val[feature_columns].values.astype(np.float32)
recon_val = autoencoder.predict(X_val, batch_size=128, verbose=0)
val_err   = loss_fn(X_val, recon_val).numpy().astype(np.float64)

rows_val = pd.DataFrame({
    "id": df_val["id"].values,
    "datetime": df_val["datetime"] if "datetime" in df_val.columns else pd.NaT,
    "reconstruction_error": val_err
})
rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

# ── AGGREGATE BY ID ───────────────────────────────────────────────────────────
g         = rows_val.groupby("id", sort=False)
n_rows    = g.size()
n_flags   = g["row_flag"].sum()
flag_frac = n_flags / n_rows

# consistent eligible set (evaluation/selection must use the same gate)
eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_rows    = n_rows.reindex(eligible_idx).astype(int)
n_flags   = n_flags.reindex(eligible_idx).astype(int)
flag_frac = flag_frac.reindex(eligible_idx).astype(float)

# ── SELECT (COUNT, FRAC) IF MIXED VAL; OTHERWISE FREEZE A PRIORI ─────────────
if USE_MIXED_VAL:
    pos_val = set(POS_IDS_VAL) & set(eligible_idx)
    if len(pos_val) == 0:
        raise ValueError("No positive IDs present in eligible validation set.")

    best = None; best_key = None
    for c in COUNTS:
        for f in FRACS:
            idx_count = set(n_flags[n_flags >= c].index)
            idx_frac  = set(flag_frac[flag_frac >= f].index)

            if   AGGREGATION_RULE == "AND":       pred = idx_count & idx_frac
            elif AGGREGATION_RULE == "FRAC_ONLY": pred = idx_frac
            elif AGGREGATION_RULE == "COUNT_ONLY":pred = idx_count
            else:                                 pred = idx_count | idx_frac  # OR

            recall = len(pred & pos_val) / len(pos_val)
            key = (recall, c, f)  # maximize recall; tie-break to stricter (higher c,f)
            if (best is None) or (key > best_key):
                best = {"count": int(c), "frac": float(f), "val_recall": float(recall)}
                best_key = key

    COUNT, FRAC = best["count"], best["frac"]
    print(f"Frozen ID rule from validation (positives-only, {AGGREGATION_RULE}): "
          f"COUNT={COUNT} | FRAC={FRAC} | recall={best['val_recall']:.3f}")
else:
    COUNT, FRAC = int(FROZEN_COUNT), float(FROZEN_FRAC)
    print(f"Using pre-registered ID rule ({AGGREGATION_RULE}; negatives-only val): "
          f"COUNT={COUNT} | FRAC={FRAC} | min_rows={MIN_ROWS_PER_ID}")

# ── SAVE FROZEN PARAMS FOR TEST ───────────────────────────────────────────────
frozen = {
    "theta": float(theta),
    "count": int(COUNT),
    "frac": float(FRAC),
    "lambda_dev": float(lambda_dev),
    "input_dim": int(input_dim),
    "loss_scale": float(loss_scale),
    "policy": policy,
    "aggregation_rule": AGGREGATION_RULE,  # ← frozen & explicit
    "val_meta": {
        "min_rows_per_id": int(MIN_ROWS_PER_ID),
        "use_mixed_val": bool(USE_MIXED_VAL)
    }
}
joblib.dump(frozen, param_out_path)
print(f"✅ Saved frozen params to: {param_out_path}")


In [ ]:

rows_val_out = str(val_csv).replace(
    ".csv",
    "_rows_with_flags_FROZEN.csv"
)

rows_val["row_flag"] = rows_val["reconstruction_error"] > theta

rows_val.to_csv(rows_val_out, index=False)
print(f"✅ Saved validation row flags → {rows_val_out}")

In [ ]:
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import load_model
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────────
base = Path("<LOCAL_PATH>")
test_feat_path  = base / "test_processed_100.csv"
model_path      = base / "autoencoder_model.h5"
frozen_param_p  = base / "id_rule_params.pkl"           # produced in validation (frozen params)
feat_cols_path  = base / "feature_columns_100.pkl"      # frozen feature order

rows_out_csv    = str(test_feat_path).replace(".csv", "_rows_with_flags_FROZEN.csv")
ids_out_csv     = str(test_feat_path).replace(".csv", "_id_summary_FROZEN.csv")
residuals_out   = str(test_feat_path).replace(".csv", "_rows_with_recon_and_residuals.csv")

# (Optional) test labels for evaluation
actual_cwd_instances_test     = ['A41', 'A43', 'A52', 'A56', 'A60', 'F106', 'F107', 'F26', 'F27', 'F30', 'F48', 'F62', 'F66', 'F73', 'F78', 'F79', 'F83', 'F87', 'F94']
unaffected_cwd_instances_test = []

# ── LOAD ARTIFACTS ────────────────────────────────────────────────────────────
feature_columns = joblib.load(feat_cols_path)           # list[str]
frozen = joblib.load(frozen_param_p)                    # dict from validation

theta  = float(frozen["theta"])
COUNT  = int(frozen["count"])
FRAC   = float(frozen["frac"])
policy = frozen.get("policy", "unknown")

# loss consistency (scaled-MSE, α = 1 + λ·d)
lambda_dev = float(frozen.get("lambda_dev", 0.5))
input_dim  = int(frozen.get("input_dim", len(feature_columns)))
loss_scale = float(frozen.get("loss_scale", 1.0 + lambda_dev * input_dim))

AGGREGATION_RULE = str(frozen.get("aggregation_rule", "OR")).upper()
MIN_ROWS_PER_ID  = int(frozen.get("val_meta", {}).get("min_rows_per_id", 1))

print(f"Using frozen params → θ={theta:.6f} | COUNT={COUNT} | FRAC={FRAC} | "
      f"λ={lambda_dev} | d={input_dim} | α={loss_scale:.3f} | MIN_ROWS={MIN_ROWS_PER_ID} | "
      f"policy={policy} | agg_rule={AGGREGATION_RULE}")

def get_custom_loss(loss_scale=loss_scale):
    def custom_loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred), axis=-1)
        return loss_scale * mse
    return custom_loss

loss_fn = get_custom_loss(loss_scale)
autoencoder = load_model(model_path, compile=False)

# ── LOAD TEST + COMPUTE RECONS + RESIDUALS ────────────────────────────────────
df = pd.read_csv(test_feat_path)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

X_test = df[feature_columns].values.astype(np.float32)
recon  = autoencoder.predict(X_test, batch_size=128, verbose=0)

row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta

# absolute residuals per feature
residuals = np.abs(X_test - recon)
df_resid = pd.DataFrame(residuals, columns=[f"{c}_resid" for c in feature_columns])
df_recon = pd.DataFrame(recon, columns=[f"{c}_recon" for c in feature_columns])

rows_full = pd.concat([
    df.reset_index(drop=True)[["id", "date"]],
    pd.DataFrame({"reconstruction_error": row_err, "row_flag": row_flag}),
    df_recon, df_resid
], axis=1)

rows_full.to_csv(residuals_out, index=False)
print(f"✅ Saved per-row reconstructions + residuals → {residuals_out}")

# ── CONTINUE WITH AGGREGATION / IDS SUMMARY (unchanged) ───────────────────────
g        = pd.DataFrame({"row_flag": row_flag, "err": row_err, "id": df["id"]}).groupby("id", sort=False)
n_rows   = g.size()
n_flags  = g["row_flag"].sum()
flag_fr  = n_flags / n_rows
id_max   = g["err"].max()
id_mean  = g["err"].mean()

eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
n_flags_elig = n_flags.reindex(eligible_idx)
flag_fr_elig = flag_fr.reindex(eligible_idx)

if AGGREGATION_RULE == "AND":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "FRAC_ONLY":
    pred_ids = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
elif AGGREGATION_RULE == "COUNT_ONLY":
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index)
else:  # OR
    pred_ids = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)

pred_ids = sorted(pred_ids)
print("Predicted positive IDs:", pred_ids)

rows = pd.DataFrame({
    "id": df["id"].values,
    "date": df["date"] if "date" in df.columns else pd.NaT,
    "reconstruction_error": row_err,
    "row_flag": row_flag
})
rows.to_csv(rows_out_csv, index=False)


# Rule-aligned and component scores (for plots & AUC)
eps = 1e-12
count_norm_all = n_flags / max(COUNT, eps)
frac_norm_all  = flag_fr / max(FRAC, eps)

if AGGREGATION_RULE == "AND":
    score_all = np.minimum(count_norm_all, frac_norm_all)
elif AGGREGATION_RULE == "FRAC_ONLY":
    score_all = frac_norm_all
elif AGGREGATION_RULE == "COUNT_ONLY":
    score_all = count_norm_all
else:  # OR
    score_all = np.maximum(count_norm_all, frac_norm_all)

ids_df = pd.DataFrame({
    "id": n_rows.index,
    "n_rows": n_rows.values,
    "n_flags": n_flags.reindex(n_rows.index).values,
    "flag_frac": flag_fr.reindex(n_rows.index).values,
    "rule_aligned_score": score_all.reindex(n_rows.index).values,
    "frac_norm_score": frac_norm_all.reindex(n_rows.index).values,
    "count_norm_score": count_norm_all.reindex(n_rows.index).values,
    "id_max_error": id_max.reindex(n_rows.index).values,
    "id_mean_error": id_mean.reindex(n_rows.index).values,
    "eligible": [1 if i in set(eligible_idx) else 0 for i in n_rows.index],
    "is_positive_id": [1 if i in pred_ids else 0 for i in n_rows.index]
})
ids_df.to_csv(ids_out_csv, index=False)
print(f"✅ Wrote: {rows_out_csv}\n✅ Wrote: {ids_out_csv}")

# ── (OPTIONAL) TEST EVALUATION — eligible set only, rule-aligned score ────────
pos_set = set(actual_cwd_instances_test)
neg_set = set(unaffected_cwd_instances_test)

eval_ids   = sorted((pos_set | neg_set) & set(eligible_idx))
n_pos_eval = sum(i in pos_set for i in eval_ids)
n_neg_eval = sum(i in neg_set for i in eval_ids)
print(f"Eval on eligible IDs only → n_eval={len(eval_ids)} (pos={n_pos_eval}, neg={n_neg_eval}; min_rows={MIN_ROWS_PER_ID})")

if eval_ids and (n_pos_eval > 0 or n_neg_eval > 0):
    y_true = np.array([1 if i in pos_set else 0 for i in eval_ids])
    y_pred = np.array([1 if i in pred_ids else 0 for i in eval_ids])

    if (y_true == 1).all():
        TP = np.sum(y_pred == 1)
        FN = np.sum(y_pred == 0)
        FP = 0
        TN = 0

        sensitivity = TP / (TP + FN) if (TP + FN) else 0.0
        precision   = TP / (TP + FP) if (TP + FP) else 0.0
        accuracy    = (TP + TN) / (TP + FN + TN + FP) if (TP + FN + TN + FP) else 0.0
        f1_score    = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) else 0.0
        prevalence  = 1.0
        tss         = np.nan

        print(f"TP = {TP}, FN = {FN}")
        print(f"Sensitivity     (TPR) = {sensitivity:.3f}")
        print(f"Precision              = {precision:.3f}")
        print(f"Accuracy               = {accuracy:.3f}")
        print(f"F1 Score               = {f1_score:.3f}")
        print(f"Prevalence Identified  = {prevalence:.3f}")
        print(f"TSS                   = N/A (no negatives)")


['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']

['A41', '', '', '', 'A60', '', 'F107', '', '', '', '', '', '', '', '', '', '', '', '']


A41 is the one that is questionaibly CWD early stage

In [ ]:
# === Overfitting sanity check: loss curves ===
import numpy as np
import matplotlib.pyplot as plt

hist = np.load(history_path, allow_pickle=True).item()  # {'loss': [...], 'val_loss': [...]}
train_loss = np.array(hist["loss"])
val_loss   = np.array(hist["val_loss"])

best_epoch = int(np.argmin(val_loss))
print("\n=== Loss history summary ===")
print(f"  Best val_loss epoch: {best_epoch} "
      f"(train_loss={train_loss[best_epoch]:.6g}, val_loss={val_loss[best_epoch]:.6g})")
print(f"  Final epoch train_loss: {train_loss[-1]:.6g}, val_loss: {val_loss[-1]:.6g}")
print(f"  Min train_loss overall: {train_loss.min():.6g}")
print(f"  Min val_loss   overall: {val_loss.min():.6g}")

plt.figure()
plt.plot(train_loss, label="train_loss")
plt.plot(val_loss,   label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (scaled MSE)")
plt.legend()
plt.title("Autoencoder train vs val loss")
plt.tight_layout()
plt.show()


In [ ]:
row_err = loss_fn(X_test, recon).numpy().astype(np.float64)
row_flag = row_err > theta
# Save a processed test file that includes row_flag (so contrib script can use it)
df_test_for_contrib = df.copy()
df_test_for_contrib["row_flag"] = row_flag.astype(bool)
df_test_for_contrib["reconstruction_error"] = row_err.astype(float)

PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS2.csv")
df_test_for_contrib.to_csv(PROC_TEST_WITH_FLAGS, index=False)
print("✅ PROC_TEST_WITH_FLAGS2 =", PROC_TEST_WITH_FLAGS)


In [ ]:
# ===== AE Feature Contributions — holdout-friendly (manuscript-ready, with TP/FN) =====
# UPDATED to plug directly into your AE testing run that already computed:
#   recon, pred_ids, MIN_ROWS_PER_ID, AGGREGATION_RULE, COUNT, FRAC,
#   plus your test id lists: actual_cwd_instances_test / unaffected_cwd_instances_test
#
# Key updates vs your original:
#   1) base/paths point to your Genotype folder (edit if needed)
#   2) PROC_TEST points to *_WITH_FLAGS.csv (written right after row_flag creation)
#   3) TRY_PREDICT_RECON = False (uses recon you already computed)
#   4) pos_set/neg_set + pred_ids injected so TP/FN partitions match frozen-rule outputs
#   5) removed duplicate function definitions near the bottom

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
import joblib

# ----------------------------
# Config (point to your AE test run)
# ----------------------------
base   = Path("<LOCAL_PATH>")
outdir = base / "ae_holdout_results"
outdir.mkdir(parents=True, exist_ok=True)

# Processed test file MUST include row_flag (recommended: write *_WITH_FLAGS.csv after AE testing)
# If you followed the earlier step, you created:
#   PROC_TEST_WITH_FLAGS = str(test_feat_path).replace(".csv", "_WITH_FLAGS.csv")
# If you already have that variable in-session, this will use it; otherwise it falls back.
try:
    PROC_TEST = Path(PROC_TEST_WITH_FLAGS)  # created in AE testing step
except NameError:
    PROC_TEST = base / "test_processed_100_WITH_FLAGS.csv"  # rename if yours differs

SCA_PATH  = base / "scaler_100.pkl"             # optional; falls back to scaled units if missing
FEAT_PATH = base / "feature_columns_100.pkl"    # frozen feature order

# Use recon from upstream (AE testing)
TRY_PREDICT_RECON = False
BATCH = 256  # only used if TRY_PREDICT_RECON=True

# Row-level flagging (only used if PROC_TEST has no 'row_flag' column)
ROW_FLAG_TAU  = None
ROW_FLAG_PCTL = 95.0

# ID aggregation params (use frozen values from AE testing if present)
try:
    MIN_ROWS_PER_ID = int(MIN_ROWS_PER_ID)
    AGGREGATION_RULE = str(AGGREGATION_RULE).upper()
    COUNT = int(COUNT)
    FRAC  = float(FRAC)
except NameError:
    MIN_ROWS_PER_ID = 10
    AGGREGATION_RULE = "OR"
    COUNT = 5
    FRAC  = 0.05

# Ground-truth on holdout (prefer explicit lists from your AE testing script)
try:
    pos_set = set(actual_cwd_instances_test)
    neg_set = set(unaffected_cwd_instances_test)
    ASSUME_TEST_ALL_POS = (len(neg_set) == 0)
except NameError:
    ASSUME_TEST_ALL_POS = True
    pos_set, neg_set = None, None

# Prefer pred_ids computed from frozen rule (so partitions are identical)
try:
    pred_ids = sorted(set(pred_ids))
except NameError:
    pred_ids = None

# ----------------------------
# Load processed test & feature list
# ----------------------------
df_test    = pd.read_csv(PROC_TEST)
feat_names = joblib.load(FEAT_PATH)  # exact order used for training

missing_feats = [c for c in feat_names if c not in df_test.columns]
if missing_feats:
    raise KeyError(
        f"Processed test missing {len(missing_feats)} features. Example: {missing_feats[:10]}"
    )

X_test   = df_test[feat_names].to_numpy(dtype=np.float32)
ids_rows = df_test["id"].to_numpy()

has_row_flag = "row_flag" in df_test.columns

# ----------------------------
# Reconstructions & residuals
# ----------------------------
try:
    _ = recon  # use precomputed recon if provided
except NameError:
    if not TRY_PREDICT_RECON:
        raise ValueError("No 'recon' provided and TRY_PREDICT_RECON is False.")
    try:
        _ = autoencoder
    except NameError:
        raise ValueError("To predict 'recon' on the fly, define 'autoencoder' in the environment.")
    recon = autoencoder.predict(X_test, batch_size=BATCH, verbose=0)

recon = np.asarray(recon, float)
if X_test.shape != recon.shape:
    raise ValueError(f"Shape mismatch: X_test {X_test.shape} vs recon {recon.shape}")

residual_sq = (X_test - recon) ** 2     # (N_rows, n_features)
resid_total = residual_sq.sum(axis=1)   # per-row total residual (τ)

# ----------------------------
# Build the 'rows' frame for masks
# ----------------------------
rows = pd.DataFrame({"id": ids_rows})
if has_row_flag:
    rows["row_flag"] = df_test["row_flag"].astype(bool).to_numpy()
else:
    if ROW_FLAG_TAU is not None:
        rows["row_flag"] = resid_total > ROW_FLAG_TAU
    else:
        thr = np.nanpercentile(resid_total, ROW_FLAG_PCTL)
        rows["row_flag"] = resid_total >= thr

mask_all  = np.ones(len(rows), dtype=bool)
mask_flag = rows["row_flag"].to_numpy(bool)
mask_eval = mask_all.copy()  # whole holdout

# ----------------------------
# Per-feature contribution helper (→ %)
# ----------------------------
def contrib_pct(rsq, mask):
    if rsq.size == 0 or not np.any(mask):
        return np.full(rsq.shape[1], np.nan, dtype=float)
    m = np.nanmean(rsq[mask], axis=0)
    s = np.nansum(m)
    return (m / s) * 100.0 if s > 0 else np.full(rsq.shape[1], np.nan, dtype=float)

contrib_all   = contrib_pct(residual_sq, mask_all)
contrib_flag  = contrib_pct(residual_sq, mask_flag)
contrib_eval  = contrib_pct(residual_sq, mask_eval)

# ----------------------------
# Erraticness across IDs (median per-feature SD in %-points)
# ----------------------------
id_to_rows = defaultdict(list)
for r, i in enumerate(rows["id"].to_numpy()):
    id_to_rows[i].append(r)

def per_id_contrib_stats(mask):
    per_id = []
    for i, idxs in id_to_rows.items():
        idxs = np.asarray(idxs, dtype=int)
        idxs = idxs[mask[idxs]]
        if idxs.size == 0:
            continue
        m = np.nanmean(residual_sq[idxs], axis=0)
        s = np.nansum(m)
        per_id.append((i, (m / s * 100.0) if s > 0 else np.full(residual_sq.shape[1], np.nan)))
    if not per_id:
        return dict(median_feature_sd_pp=np.nan, n_ids=0)
    M = np.vstack([v for _, v in per_id])  # (n_ids, n_features)
    std_per_feat_pp = np.nanstd(M, axis=0)
    return dict(median_feature_sd_pp=float(np.nanmedian(std_per_feat_pp)), n_ids=len(per_id))

err_all  = per_id_contrib_stats(mask_all)
err_flag = per_id_contrib_stats(mask_flag)
err_eval = per_id_contrib_stats(mask_eval)

# ----------------------------
# Simple concentration proxies
# ----------------------------
def topk_share_pct(arr, k=3):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return np.nan
    a_sorted = np.sort(a)[::-1]
    return float(np.sum(a_sorted[:min(k, a_sorted.size)]))

top3_all,  top5_all  = topk_share_pct(contrib_all, 3), topk_share_pct(contrib_all, 5)
top3_flag, top5_flag = topk_share_pct(contrib_flag, 3), topk_share_pct(contrib_flag, 5)
top3_eval, top5_eval = topk_share_pct(contrib_eval, 3), topk_share_pct(contrib_eval, 5)

# ----------------------------
# Per-feature table
# ----------------------------
df_contrib = pd.DataFrame({
    "feature":    feat_names,
    "All_%":      contrib_all,
    "Flagged_%":  contrib_flag,
    "Eval_%":     contrib_eval,
})
df_contrib["Delta_Flagged_minus_All_pp"] = df_contrib["Flagged_%"] - df_contrib["All_%"]

print("\n=== AE Feature Contributions (% of reconstruction error) — Holdout ===")
print(df_contrib.sort_values("Flagged_%", ascending=False).head(15).to_string(
    index=False, float_format=lambda x: f"{x:.2f}"
))

summary_tbl = pd.DataFrame([
    {"Group":"All rows",     "Top3_share_%": top3_all,  "Top5_share_%": top5_all,
     "Median_feat_SD_pp": err_all["median_feature_sd_pp"],  "n_ids": err_all["n_ids"]},
    {"Group":"Flagged rows", "Top3_share_%": top3_flag, "Top5_share_%": top5_flag,
     "Median_feat_SD_pp": err_flag["median_feature_sd_pp"], "n_ids": err_flag["n_ids"]},
    {"Group":"Eval IDs",     "Top3_share_%": top3_eval, "Top5_share_%": top5_eval,
     "Median_feat_SD_pp": err_eval["median_feature_sd_pp"], "n_ids": err_eval["n_ids"]},
])
summary_tbl["ΔFlagged−All_Top3_pp"] = summary_tbl.loc[1, "Top3_share_%"] - summary_tbl.loc[0, "Top3_share_%"]
summary_tbl["ΔFlagged−All_Top5_pp"] = summary_tbl.loc[1, "Top5_share_%"] - summary_tbl.loc[0, "Top5_share_%"]
summary_tbl["ΔFlagged−All_SD_pp"]   = summary_tbl.loc[1, "Median_feat_SD_pp"] - summary_tbl.loc[0, "Median_feat_SD_pp"]

print("\n=== Simple concentration & variability summary (holdout) ===")
print(summary_tbl.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# ----------------------------
# Category roll-up (movement story)
# ----------------------------
def categorize(fname: str) -> str:
    f = fname.lower()
    if "velo" in f: return "velocity"
    if "angle" in f: return "turning-angle"
    if "sinu" in f:  return "sinuosity"
    if "skew" in f or "kurt" in f: return "distributional-moment"
    if "rolling" in f or "mean" in f or "sd" in f: return "rolling-statistic"
    if f in {"low", "est", "high"} or "model" in f: return "model-derived"
    if "season" in f: return "season"
    return "other"

cat = pd.DataFrame({
    "feature":   feat_names,
    "category":  [categorize(f) for f in feat_names],
    "All_%":     contrib_all,
    "Flagged_%": contrib_flag,
    "Eval_%":    contrib_eval
})
cat_roll = (cat.groupby("category")[["All_%", "Flagged_%", "Eval_%"]]
              .sum()
              .sort_values("Flagged_%", ascending=False))
cat_roll["Delta_Flagged_minus_All_pp"] = cat_roll["Flagged_%"] - cat_roll["All_%"]

print("\n=== Category roll-up (% of recon error) — Holdout ===")
print(cat_roll.to_string(float_format=lambda x: f"{x:.1f}"))

# ----------------------------
# Build TP / FP / FN / TN partitions (ID-level)
# ----------------------------
# 1) Ground-truth ID sets (prefer injected pos_set/neg_set)
if (pos_set is None) or (neg_set is None):
    if (not ASSUME_TEST_ALL_POS) and ("CWD_label" in df_test.columns):
        id_lab = df_test.groupby("id")["CWD_label"].max()
        pos_set = set(id_lab.index[id_lab == 1])
        neg_set = set(id_lab.index[id_lab == 0])
    else:
        pos_set = set(df_test["id"].unique())
        neg_set = set()

# 2) Aggregate row_flags → predicted positive IDs (unless pred_ids already provided)
def aggregate_ids_from_rowflags(row_flags_bool, ids_series,
                                MIN_ROWS_PER_ID=MIN_ROWS_PER_ID,
                                AGGREGATION_RULE=AGGREGATION_RULE,
                                COUNT=COUNT, FRAC=FRAC):
    tmp = pd.DataFrame({"id": np.asarray(ids_series), "row_flag": row_flags_bool.astype(int)})
    g = tmp.groupby("id", sort=False)
    n_rows = g.size()
    n_flags = g["row_flag"].sum()
    flag_fr = n_flags / n_rows

    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index
    n_flags_elig = n_flags.reindex(eligible_idx)
    flag_fr_elig = flag_fr.reindex(eligible_idx)

    if AGGREGATION_RULE == "AND":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) & set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "FRAC_ONLY":
        pred = set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    elif AGGREGATION_RULE == "COUNT_ONLY":
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index)
    else:  # OR
        pred = set(n_flags_elig[n_flags_elig >= COUNT].index) | set(flag_fr_elig[flag_fr_elig >= FRAC].index)
    return sorted(pred), eligible_idx

if pred_ids is None:
    pred_ids, eligible_idx = aggregate_ids_from_rowflags(rows["row_flag"].to_numpy(bool), rows["id"].to_numpy())
else:
    # eligible IDs should follow the same MIN_ROWS rule; recompute eligibility only
    tmp = pd.DataFrame({"id": rows["id"].to_numpy()})
    n_rows = tmp.groupby("id", sort=False).size()
    eligible_idx = n_rows[n_rows >= MIN_ROWS_PER_ID].index

# 3) Confusion partitions on eligible IDs
eval_ids = sorted((pos_set | neg_set) & set(eligible_idx))
TP_ids = [i for i in eval_ids if (i in pos_set) and (i in pred_ids)]
FP_ids = [i for i in eval_ids if (i in neg_set) and (i in pred_ids)]
FN_ids = [i for i in eval_ids if (i in pos_set) and (i not in pred_ids)]
TN_ids = [i for i in eval_ids if (i in neg_set) and (i not in pred_ids)]

rid = rows["id"]
row_flag_arr = rows["row_flag"].to_numpy(bool)

m_tp = rid.isin(TP_ids).to_numpy()
m_fp = rid.isin(FP_ids).to_numpy()
m_fn = rid.isin(FN_ids).to_numpy()
m_tn = rid.isin(TN_ids).to_numpy()

m_tp_flagged = m_tp & row_flag_arr
m_fp_flagged = m_fp & row_flag_arr
m_fn_unflagged = m_fn & (~row_flag_arr)
m_tn_unflagged = m_tn & (~row_flag_arr)

# ----------------------------
# Per-class contribution tables
# ----------------------------
def table_from_mask(mask, label):
    pct = contrib_pct(residual_sq, mask)
    return (pd.DataFrame({"feature": feat_names, f"{label}_%": pct})
              .sort_values(f"{label}_%", ascending=False)
              .reset_index(drop=True))

tp_tbl = table_from_mask(m_tp, "TP")
fp_tbl = table_from_mask(m_fp, "FP")
fn_tbl = table_from_mask(m_fn, "FN")
tn_tbl = table_from_mask(m_tn, "TN")

all_series = pd.Series(contrib_all, index=feat_names, name="All_%")

def top_with_delta(df_, label, k=10):
    col = f"{label}_%"
    if df_[col].notna().any():
        out = (df_.head(k)
                 .merge(all_series.reset_index().rename(columns={"index":"feature"}),
                        on="feature", how="left"))
        out[f"Delta_{label}_minus_All_pp"] = out[col] - out["All_%"]
        print(f"\n=== Top {k} features — {label} ===")
        print(out.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
    else:
        print(f"\n=== Top {k} features — {label} ===\n(no rows)")

top_with_delta(tp_tbl, "TP")
top_with_delta(fp_tbl, "FP")
top_with_delta(fn_tbl, "FN")
top_with_delta(tn_tbl, "TN")

# Save per-class CSVs
tp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tp.csv", index=False)
fp_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fp.csv", index=False)
fn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_fn.csv", index=False)
tn_tbl.to_csv(outdir / "ae_holdout_feat_contrib_tn.csv", index=False)

# Save ALL per-class features (merge All_% + delta for every feature)
for label, df_ in {"TP": tp_tbl, "FP": fp_tbl, "FN": fn_tbl, "TN": tn_tbl}.items():
    if df_ is not None and not df_.empty:
        merged = df_.merge(all_series.reset_index().rename(columns={"index": "feature"}),
                           on="feature", how="left")
        merged[f"Delta_{label}_minus_All_pp"] = merged[f"{label}_%"] - merged["All_%"]
        merged.to_csv(outdir / f"ae_all_features_{label}.csv", index=False)
        print(f"✅ Saved full feature list for {label}: {outdir}/ae_all_features_{label}.csv")

# ----------------------------
# Movement–residual linkage (inverse-scaling numeric features only)
# ----------------------------
try:
    scaler = joblib.load(SCA_PATH)
    if not hasattr(scaler, "inverse_transform"):
        raise TypeError("Loaded scaler has no inverse_transform")

    # Prefer the scaler's known feature names if present (skips one-hot/cats not seen by scaler)
    scaler_feats = list(getattr(scaler, "feature_names_in_", []))
    num_cols = [c for c in feat_names if c in scaler_feats] if scaler_feats else feat_names
    cat_cols = [c for c in feat_names if c not in num_cols]

    X_num = df_test[num_cols].to_numpy()
    X_cat = df_test[cat_cols].to_numpy() if cat_cols else np.empty((len(df_test), 0))
    X_num_raw = scaler.inverse_transform(X_num)

    df_raw = pd.DataFrame(np.hstack([X_num_raw, X_cat]), columns=num_cols + cat_cols)
    units_label = "raw units"
except Exception as e:
    print(f"[warn] Using scaled units — {e}")
    df_raw = df_test[feat_names].copy()
    units_label = "scaled units"

# Add IDs + row flags
df_raw["id"] = rows["id"].values
df_raw["row_flag"] = rows["row_flag"].values

# Collapsed movement metrics
df_raw["speed"]   = df_raw.filter(like="velo").mean(axis=1)
df_raw["turning"] = df_raw.filter(like="angle").mean(axis=1)
df_raw["sinu"]    = df_raw.filter(like="sinu").mean(axis=1)

def group_summary(mask, label):
    tmp = df_raw.loc[mask]
    return pd.Series({
        "n_rows":       int(len(tmp)),
        "mean_speed":   float(tmp["speed"].mean())   if len(tmp) else np.nan,
        "mean_turning": float(tmp["turning"].mean()) if len(tmp) else np.nan,
        "mean_sinu":    float(tmp["sinu"].mean())    if len(tmp) else np.nan
    }, name=label)

summ_pieces = [
    group_summary(mask_all, "All"),
    group_summary(mask_flag, "Flagged"),
]
if m_tp.any(): summ_pieces.append(group_summary(m_tp, "TP"))
if m_fn.any(): summ_pieces.append(group_summary(m_fn, "FN"))
if m_fp.any(): summ_pieces.append(group_summary(m_fp, "FP"))
if m_tn.any(): summ_pieces.append(group_summary(m_tn, "TN"))

summaries = pd.concat(summ_pieces, axis=1).T

# %diff vs TP (if TP exists)
if ("TP" in summaries.index) and (summaries.loc["TP", "n_rows"] > 0):
    for col in ["mean_speed", "mean_turning", "mean_sinu"]:
        base_val = summaries.loc["TP", col]
        for grp in summaries.index:
            if grp == "TP":
                continue
            summaries.loc[grp, f"{col}_vsTP_%diff"] = (
                (summaries.loc[grp, col] - base_val) / base_val * 100
                if (pd.notna(base_val) and base_val != 0) else np.nan
            )

print(f"\n=== Movement summaries by class ({units_label}) — Holdout ===")
print(summaries.round(3))
summaries.to_csv(outdir / f"ae_holdout_movement_summaries_by_class_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                 index=True)

# Direct per-variable summaries
vars_of_interest = [
    "abs_rel_angle", "est", "low", "high",
    "rolling_abs_rel_angle_kurt", "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd", "rolling_abs_rel_angle_skew",
    "rolling_velo_kurt", "rolling_velo_mean", "rolling_velo_sd",
    "rolling_velo_skew", "sinuosity_28pt", "velo"
]
present = [v for v in vars_of_interest if v in df_raw.columns]

def group_summary_vars(mask, label):
    tmp = df_raw.loc[mask, present]
    stats = {f"{v}_mean": tmp[v].mean() for v in present}
    stats.update({f"{v}_sd": tmp[v].std() for v in present})
    stats["n_rows"] = len(tmp)
    return pd.Series(stats, name=label)

summ_vars_pieces = [
    group_summary_vars(mask_all,  "All"),
    group_summary_vars(mask_flag, "Flagged"),
]
if m_tp.any(): summ_vars_pieces.append(group_summary_vars(m_tp, "TP"))
if m_fn.any(): summ_vars_pieces.append(group_summary_vars(m_fn, "FN"))
if m_fp.any(): summ_vars_pieces.append(group_summary_vars(m_fp, "FP"))
if m_tn.any(): summ_vars_pieces.append(group_summary_vars(m_tn, "TN"))

summaries_vars = pd.concat(summ_vars_pieces, axis=1).T

# diffs vs TP (only if TP exists)
if "TP" in summaries_vars.index and summaries_vars.loc["TP", "n_rows"] > 0:
    for v in present:
        base = summaries_vars.loc["TP", f"{v}_mean"]
        for grp in summaries_vars.index:
            if grp == "TP":
                continue
            summaries_vars.loc[grp, f"{v}_diff_vsTP"] = summaries_vars.loc[grp, f"{v}_mean"] - base
            summaries_vars.loc[grp, f"{v}_pctdiff_vsTP"] = (
                (summaries_vars.loc[grp, f"{v}_mean"] - base) / base * 100
                if (pd.notna(base) and base != 0) else np.nan
            )

print("\n=== Per-variable movement summaries — Holdout ===")
print(summaries_vars.round(3))
summaries_vars.to_csv(outdir / f"ae_holdout_movement_summaries_by_variable_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# Residual correlation with movement metrics
movement_links = pd.DataFrame({
    "resid_total": resid_total,
    "speed":       df_raw["speed"],
    "turning":     df_raw["turning"],
    "sinu":        df_raw["sinu"],
}).corr()

print("\n=== Correlation of movement variables with total reconstruction error ===")
print(movement_links.round(3))
movement_links.to_csv(outdir / f"ae_holdout_movement_residual_correlations_{'raw' if units_label=='raw units' else 'scaled'}.csv",
                      index=True)

# ----------------------------
# Save the main contribution tables
# ----------------------------
(df_contrib.sort_values("Flagged_%", ascending=False)
          .to_csv(outdir / "ae_holdout_feature_contributions_by_feature.csv", index=False))
cat_roll.to_csv(outdir / "ae_holdout_feature_contributions_by_category.csv")
summary_tbl.to_csv(outdir / "ae_holdout_feature_contrib_summary_simple.csv", index=False)

print(f"\n💾 Saved tables to: {outdir}")
print(f"IDs — TP:{len(TP_ids)}  FN:{len(FN_ids)}  FP:{len(FP_ids)}  TN:{len(TN_ids)}  | Eligible:{len(set(eval_ids))}")


In [ ]:
# make_CWDpos_CWDneg_movement_pct_heatmap_RAW.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
import joblib


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path("<LOCAL_PATH>")

TRAIN_PATH = BASE / "train_processed_100.csv"
VAL_PATH = BASE / "val_processed_100.csv"
TEST_PATH = BASE / "test_processed_100.csv"
SCALER_PATH = BASE / "scaler_100.pkl"

OUT_DIR = BASE / "figures_cwd"
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = OUT_DIR / "CWD+_vs_CWD-__pct_heatmap_RAW"


# =============================================================================
# CWD status ID lists
# =============================================================================

CWD_NEG_IDS = [
    "A29", "A36", "A38", "A39", "A40", "A44", "A45", "A46",
    "A47", "A48", "A49", "A50", "A51", "A53", "A54", "A55",
    "A57", "A58", "A59", "A61", "A62", "A63", "F04", "F100",
    "F103", "F109", "F31", "F33", "F34", "F42", "F47", "F67",
    "F74", "F76", "F77", "F82", "F91", "F93", "F97", "F99"
]

CWD_POS_IDS = [
    "A41", "A43", "A52", "A56", "A60", "F106", "F107", "F26",
    "F27", "F30", "F48", "F62", "F66", "F73", "F78", "F79",
    "F83", "F87", "F94"
]


# =============================================================================
# Comparison selection
# =============================================================================

ONLY_POS_VS_NEG = True

comp_order_full = [
    "CWD$^{-}$ vs. NegBaseline",
    "CWD$^{+}$ vs. NegBaseline",
    "CWD$^{+}$ vs. CWD$^{-}$"
]

comp_order_posneg = [
    "CWD$^{+}$ vs. CWD$^{-}$"
]


# =============================================================================
# Helper functions
# =============================================================================

def standardize_id(value):
    """
    Standardize deer IDs without removing the A/F prefix.
    """

    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value == "":
        return None

    return value


def keep_cols(df, features):
    """
    Keep only deer ID and model-feature columns.
    """

    required_columns = {"id", *features}

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            "Dataset is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    return df[
        ["id"] + features
    ].copy()


def inverse_transform_features(
    df,
    scaler,
    features
):
    """
    Convert scaled model features back to raw units.
    """

    transformed_values = scaler.inverse_transform(
        df[features].to_numpy()
    )

    output = df[["id"]].copy()

    output[features] = transformed_values

    return output


def group_mean(
    df,
    features,
    ids
):
    """
    Calculate observation-level feature means
    for the selected deer IDs.
    """

    subset = df[
        df["id_clean"].isin(ids)
    ]

    if subset.empty:
        raise ValueError(
            f"No observations were found for IDs: {ids}"
        )

    return subset[features].mean(
        numeric_only=True
    )


BASE_FLOOR = 1e-8


def pct_diff(
    A,
    B,
    floor=BASE_FLOOR
):
    """
    Calculate percentage difference:

        100 × (A − B) / max(|B|, floor)
    """

    A = np.asarray(
        A,
        dtype=float
    )

    B = np.asarray(
        B,
        dtype=float
    )

    denominator = np.maximum(
        np.abs(B),
        floor
    )

    return 100.0 * (
        A - B
    ) / denominator


# =============================================================================
# Load scaled datasets
# =============================================================================

train = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)

test = pd.read_csv(
    TEST_PATH
)

print(
    f"Training rows:   {len(train):,}"
)

print(
    f"Validation rows: {len(val):,}"
)

print(
    f"Testing rows:    {len(test):,}"
)


# =============================================================================
# Load scaler and obtain exact feature order
# =============================================================================

scaler = joblib.load(
    SCALER_PATH
)

if not hasattr(
    scaler,
    "feature_names_in_"
):
    raise AttributeError(
        "The loaded scaler does not contain feature_names_in_. "
        "Confirm that the scaler was fitted using a pandas DataFrame."
    )

FEATURES = list(
    scaler.feature_names_in_
)

print(
    f"\nNumber of model features: "
    f"{len(FEATURES)}"
)

print(
    "Features:"
)

for feature in FEATURES:
    print(
        f"  - {feature}"
    )


# =============================================================================
# Keep only ID and feature columns
# =============================================================================

train = keep_cols(
    train,
    FEATURES
)

val = keep_cols(
    val,
    FEATURES
)

test = keep_cols(
    test,
    FEATURES
)


# =============================================================================
# Inverse-transform features back to raw units
# =============================================================================

train_raw = inverse_transform_features(
    train,
    scaler,
    FEATURES
)

val_raw = inverse_transform_features(
    val,
    scaler,
    FEATURES
)

test_raw = inverse_transform_features(
    test,
    scaler,
    FEATURES
)


# =============================================================================
# Standardize deer IDs
# =============================================================================

for df in (
    train_raw,
    val_raw,
    test_raw
):
    df["id_clean"] = df["id"].map(
        standardize_id
    )


CWD_NEG_IDS_CLEAN = sorted({
    standardize_id(deer_id)
    for deer_id in CWD_NEG_IDS
    if standardize_id(deer_id) is not None
})


CWD_POS_IDS_CLEAN = sorted({
    standardize_id(deer_id)
    for deer_id in CWD_POS_IDS
    if standardize_id(deer_id) is not None
})


# =============================================================================
# Source pools
# =============================================================================

all_raw = pd.concat(
    [
        train_raw,
        val_raw,
        test_raw
    ],
    ignore_index=True
)


ids_present = set(
    all_raw[
        "id_clean"
    ]
    .dropna()
    .unique()
)


neg_ids_present = sorted(
    set(CWD_NEG_IDS_CLEAN) &
    ids_present
)


pos_ids_present = sorted(
    set(CWD_POS_IDS_CLEAN) &
    ids_present
)


print(
    f"\nCWD- IDs present: "
    f"{len(neg_ids_present)}"
)

print(
    f"CWD+ IDs present: "
    f"{len(pos_ids_present)}"
)

print(
    f"\nCWD- IDs use<LOCAL_PATH>"
    f"{neg_ids_present}"
)

print(
    f"\nCWD+ IDs use<LOCAL_PATH>"
    f"{pos_ids_present}"
)


missing_neg_ids = sorted(
    set(CWD_NEG_IDS_CLEAN) -
    ids_present
)


missing_pos_ids = sorted(
    set(CWD_POS_IDS_CLEAN) -
    ids_present
)


if missing_neg_ids:
    print(
        f"\nCWD- IDs not foun<LOCAL_PATH>"
        f"{missing_neg_ids}"
    )


if missing_pos_ids:
    print(
        f"\nCWD+ IDs not foun<LOCAL_PATH>"
        f"{missing_pos_ids}"
    )


# =============================================================================
# Negative baseline
# =============================================================================

neg_baseline_raw = pd.concat(
    [
        train_raw,
        val_raw
    ],
    ignore_index=True
)


baseline_ids_present = sorted(
    set(
        neg_baseline_raw[
            "id_clean"
        ]
        .dropna()
        .unique()
    ) &
    set(
        CWD_NEG_IDS_CLEAN
    )
)


if not neg_ids_present:
    raise ValueError(
        "No CWD- deer IDs were found in the combined datasets."
    )


if not pos_ids_present:
    raise ValueError(
        "No CWD+ deer IDs were found in the combined datasets."
    )


if not baseline_ids_present:
    raise ValueError(
        "No CWD- baseline IDs were found in training and validation."
    )


# =============================================================================
# Calculate group means
# =============================================================================

mu_neg = group_mean(
    all_raw,
    FEATURES,
    neg_ids_present
)


mu_pos = group_mean(
    all_raw,
    FEATURES,
    pos_ids_present
)


mu_base = group_mean(
    neg_baseline_raw,
    FEATURES,
    baseline_ids_present
)


# =============================================================================
# Build percentage-difference table
# =============================================================================

comparison_columns = {
    "feature": FEATURES
}


if not ONLY_POS_VS_NEG:

    comparison_columns[
        "CWD$^{-}$ vs. NegBaseline"
    ] = pct_diff(
        mu_neg.values,
        mu_base.values
    )

    comparison_columns[
        "CWD$^{+}$ vs. NegBaseline"
    ] = pct_diff(
        mu_pos.values,
        mu_base.values
    )


comparison_columns[
    "CWD$^{+}$ vs. CWD$^{-}$"
] = pct_diff(
    mu_pos.values,
    mu_neg.values
)


all_df = pd.DataFrame(
    comparison_columns
)


comp_order = (
    comp_order_posneg
    if ONLY_POS_VS_NEG
    else comp_order_full
)


heatmap_df = (
    all_df
    .set_index(
        "feature"
    )
    .loc[
        :,
        comp_order
    ]
)


# =============================================================================
# Feature ordering and display labels
# =============================================================================

pretty_order = [

    # Velocity metrics
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction and turning-angle metrics
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path-shape metric
    "sinuosity",
    "sinuosity_28pt",
    "sinuosity_56pt",

    # Space-use metrics
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "sinuosity_28pt":
        "Sinuosity",

    "sinuosity_56pt":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [
    feature
    for feature in pretty_order
    if feature in heatmap_df.index
]


heatmap_df = heatmap_df.reindex(
    available_pretty_order
)


heatmap_df.index = [
    pretty_map.get(
        feature,
        feature
    )
    for feature in heatmap_df.index
]


# =============================================================================
# Print numerical values for verification
# =============================================================================

print(
    "\nPercentage-difference table:"
)

print(
    heatmap_df
    .round(1)
    .to_string()
)


# =============================================================================
# Determine color scale
# =============================================================================

data_values = heatmap_df.to_numpy(
    dtype=float
)


if not np.isfinite(
    data_values
).any():
    raise ValueError(
        "The heatmap contains no finite values."
    )


data_min = float(
    np.nanmin(
        data_values
    )
)


data_max = float(
    np.nanmax(
        data_values
    )
)


print(
    f"\nData range: "
    f"{data_min:.1f}% to "
    f"{data_max:.1f}%"
)


vmin = float(
    np.floor(
        data_min
    )
)


vmax = float(
    np.ceil(
        data_max
    )
)


# Ensure both negative and positive sides appear
if vmin >= 0:
    vmin = -5.0


if vmax <= 0:
    vmax = 5.0


print(
    f"Rounded scale range: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Build custom discrete color boundaries
# =============================================================================

candidate_bounds = [
    vmin,
    -30,
    -15,
    -10,
    -5,
    0,
    5,
    10,
    15,
    30,
    vmax
]


bounds = [
    float(bound)
    for bound in candidate_bounds
    if vmin <= bound <= vmax
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(bounds)
)


if len(bounds) < 2:
    bounds = [
        vmin - 1,
        vmax + 1
    ]


print(
    f"Color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Build discrete colors
# =============================================================================

base_cmap = mpl.colormaps[
    "RdBu_r"
]


bin_midpoints = np.array(
    [
        (
            lower +
            upper
        ) / 2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


negative_extent = max(
    abs(vmin),
    1.0
)


positive_extent = max(
    abs(vmax),
    1.0
)


color_positions = []


for midpoint in bin_midpoints:

    if midpoint < 0:

        relative_strength = min(
            abs(midpoint) /
            negative_extent,
            1.0
        )

        cmap_position = (
            0.5 -
            0.5 *
            relative_strength
        )

    elif midpoint > 0:

        relative_strength = min(
            midpoint /
            positive_extent,
            1.0
        )

        cmap_position = (
            0.5 +
            0.5 *
            relative_strength
        )

    else:

        cmap_position = 0.5


    color_positions.append(
        cmap_position
    )


color_positions = np.asarray(
    color_positions,
    dtype=float
)


# Force outer intervals to darkest colors
if len(color_positions) > 0:

    if bin_midpoints[0] < 0:
        color_positions[0] = 0.0

    if bin_midpoints[-1] > 0:
        color_positions[-1] = 1.0


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_RdBu"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


print(
    "\nColor intervals:"
)


for index, (
    lower,
    upper
) in enumerate(
    zip(
        bounds[:-1],
        bounds[1:]
    )
):

    print(
        f"  {lower:>6.1f}% to "
        f"{upper:>6.1f}% "
        f"-> colormap position "
        f"{color_positions[index]:.3f}"
    )


# =============================================================================
# Figure dimensions
#
# IMPORTANT:
# The figure is kept wide enough for the long feature labels,
# but the actual colored heatmap column is deliberately narrow.
# =============================================================================

n_rows, n_cols = heatmap_df.shape


FIG_WIDTH = 8.0

ROW_HEIGHT = 0.62

FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.0,
    8.5,
    12.0
)


# =============================================================================
# Font sizes
# =============================================================================

AX_LABEL_FS = 26

X_TICK_FS = 20

Y_TICK_FS = 20

ANNOT_FS = 20

CBAR_LABEL_FS = 24

CBAR_TICK_FS = 18


# =============================================================================
# Create figure
#
# Do NOT use constrained_layout here.
# Fixed margins give much better control over the width
# of a one-column heatmap.
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# =============================================================================
# Main heatmap axis
#
# [left, bottom, width, height]
#
# This keeps the numerical column compact without turning it
# into the extremely thin strip shown in the previous version.
# =============================================================================

ax = fig.add_axes(
    [
        0.52,   # left
        0.12,   # bottom
        0.18,   # width
        0.80    # height
    ]
)


# =============================================================================
# Colorbar axis
#
# Kept narrow and separated from the heatmap.
# =============================================================================

cax = fig.add_axes(
    [
        0.76,   # left
        0.12,   # bottom
        0.035,  # width
        0.80    # height
    ]
)



# =============================================================================
# Create heatmap
# =============================================================================

hm = sns.heatmap(
    heatmap_df,
    cmap=discrete_cmap,
    norm=norm,

    annot=True,
    fmt=".1f",

    annot_kws={
        "fontsize": ANNOT_FS,
        "fontfamily": "Times New Roman",
        "fontweight": "bold"
    },

    linewidths=1.0,
    linecolor="white",

    ax=ax,

    cbar=True,
    cbar_ax=cax
)


# =============================================================================
# Force ALL cell annotations to use exactly the same bold font
# =============================================================================

flat_values = (
    heatmap_df
    .to_numpy()
    .flatten()
)


for text, value in zip(
    ax.texts,
    flat_values
):

    text.set_fontfamily(
        "Times New Roman"
    )

    text.set_fontsize(
        ANNOT_FS
    )

    text.set_fontweight(
        "bold"
    )


    if not np.isfinite(
        value
    ):
        continue


    # White only for darkest color intervals
    if (
        value <= bounds[1]
        or
        value >= bounds[-2]
    ):

        text.set_color(
            "white"
        )

    else:

        text.set_color(
            "black"
        )


# =============================================================================
# Axis labels
# =============================================================================

ax.set_xlabel(
    "Comparison",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=14
)


ax.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Tick positions
# =============================================================================

ax.tick_params(
    axis="x",
    rotation=0,
    length=0
)


ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)


# =============================================================================
# Force x-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_xticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        X_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )

    label.set_horizontalalignment(
        "center"
    )


# =============================================================================
# Force y-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        Y_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )


# =============================================================================
# Colorbar formatting
# =============================================================================

cbar = hm.collections[
    0
].colorbar


cbar.set_label(
    "% Difference (Raw Units)",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


candidate_ticks = np.array(
    [
        vmin,
        -30,
        -15,
        -10,
        -5,
        0,
        5,
        10,
        15,
        30,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [
        f"{tick:.0f}%"
        for tick in nice_ticks
    ]
)


# =============================================================================
# Force ALL colorbar tick labels to same font and weight
# =============================================================================

for label in cbar.ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        CBAR_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cax.yaxis.set_label_position(
    "right"
)


cax.yaxis.set_ticks_position(
    "right"
)


# =============================================================================
# Make colorbar outline slightly stronger
# =============================================================================

cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Save and display
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG to: "
    f"{png_path}"
)


print(
    f"Saved PDF to: "
    f"{pdf_path}"
)


plt.show()

plt.close(
    fig
)

In [ ]:
# =============================================================================
# make_CWDpos_CWDneg_movement_pct_heatmap_RAW_BY_SEX.py
#
# Creates TWO heatmaps:
#
#   1. Female CWD+ vs. Female CWD-
#   2. Male   CWD+ vs. Male   CWD-
#
# Sex information is obtained from:
#
#   <LOCAL_PATH>
#
# Both heatmaps use the SAME color scale so that colors are directly
# comparable between females and males.
# =============================================================================

from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    "<LOCAL_PATH>"
)

TRAIN_PATH = BASE / "train_processed_100.csv"
VAL_PATH = BASE / "val_processed_100.csv"
TEST_PATH = BASE / "test_processed_100.csv"
SCALER_PATH = BASE / "scaler_100.pkl"


# -----------------------------------------------------------------------------
# Metadata containing deer_id and sex
# -----------------------------------------------------------------------------

META_PATH = Path(
    "<LOCAL_PATH>"
    "Genotype_Plotting_and_Simulation/"
    "WTD_Meta.txt"
)


# -----------------------------------------------------------------------------
# Output directory
# -----------------------------------------------------------------------------

OUT_DIR = BASE / "figures_cwd_by_sex"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = (
    OUT_DIR /
    "CWD+_vs_CWD-_pct_heatmap_RAW_Female_Male"
)


# =============================================================================
# CWD status ID lists
# =============================================================================

CWD_NEG_IDS = [
    "A29", "A36", "A38", "A39", "A40", "A44", "A45", "A46",
    "A47", "A48", "A49", "A50", "A51", "A53", "A54", "A55",
    "A57", "A58", "A59", "A61", "A62", "A63", "F04", "F100",
    "F103", "F109", "F31", "F33", "F34", "F42", "F47", "F67",
    "F74", "F76", "F77", "F82", "F91", "F93", "F97", "F99"
]


CWD_POS_IDS = [
    "A41", "A43", "A52", "A56", "A60", "F106", "F107", "F26",
    "F27", "F30", "F48", "F62", "F66", "F73", "F78", "F79",
    "F83", "F87", "F94"
]


# =============================================================================
# Helper functions
# =============================================================================

def standardize_id(value):
    """
    Standardize deer IDs without removing the A/F prefix.
    """

    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value == "":
        return None

    return value


def standardize_sex(value):
    """
    Convert common sex labels to either 'Female' or 'Male'.

    Accepted examples:
        F
        Female
        FEMALE

        M
        Male
        MALE
    """

    if pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value == "":
        return None

    female_values = {
        "F",
        "FEMALE"
    }

    male_values = {
        "M",
        "MALE"
    }

    if value in female_values:
        return "Female"

    if value in male_values:
        return "Male"

    # Unknown coding
    return None


def keep_cols(df, features):
    """
    Keep only deer ID and model-feature columns.
    """

    required_columns = {
        "id",
        *features
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            "Dataset is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    return df[
        ["id"] + features
    ].copy()


def inverse_transform_features(
    df,
    scaler,
    features
):
    """
    Convert scaled model features back to raw units.
    """

    transformed_values = scaler.inverse_transform(
        df[features].to_numpy()
    )

    output = df[["id"]].copy()

    output[features] = transformed_values

    return output


def group_mean(
    df,
    features,
    ids
):
    """
    Calculate observation-level feature means for the selected deer IDs.

    This preserves the calculation used in the original script.
    """

    subset = df[
        df["id_clean"].isin(ids)
    ]

    if subset.empty:
        raise ValueError(
            "No observations were found for ID<LOCAL_PATH>"
            f"{ids}"
        )

    return subset[
        features
    ].mean(
        numeric_only=True
    )


BASE_FLOOR = 1e-8


def pct_diff(
    A,
    B,
    floor=BASE_FLOOR
):
    """
    Calculate percentage difference:

        100 × (A - B) / max(|B|, floor)

    Here:
        A = CWD+ mean
        B = CWD- mean
    """

    A = np.asarray(
        A,
        dtype=float
    )

    B = np.asarray(
        B,
        dtype=float
    )

    denominator = np.maximum(
        np.abs(B),
        floor
    )

    return (
        100.0 *
        (A - B) /
        denominator
    )


# =============================================================================
# Load scaled datasets
# =============================================================================

train = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)

test = pd.read_csv(
    TEST_PATH
)


print(
    f"Training rows:   {len(train):,}"
)

print(
    f"Validation rows: {len(val):,}"
)

print(
    f"Testing rows:    {len(test):,}"
)


# =============================================================================
# Load metadata
# =============================================================================

meta = pd.read_csv(
    META_PATH,
    sep="\t",
    dtype=str
)


print(
    f"\nMetadata rows: {len(meta):,}"
)

print(
    "\nMetadata columns:"
)

for column in meta.columns:
    print(
        f"  - {column}"
    )


# =============================================================================
# Verify required metadata columns
# =============================================================================

required_meta_columns = {
    "deer_id",
    "sex"
}


missing_meta_columns = (
    required_meta_columns.difference(
        meta.columns
    )
)


if missing_meta_columns:

    raise ValueError(
        "Metadata file is missing required columns: "
        f"{sorted(missing_meta_columns)}"
    )


# =============================================================================
# Standardize metadata IDs and sex
# =============================================================================

meta["id_clean"] = meta[
    "deer_id"
].map(
    standardize_id
)


meta["sex_clean"] = meta[
    "sex"
].map(
    standardize_sex
)


# =============================================================================
# Check sex coding
# =============================================================================

print(
    "\nOriginal sex values:"
)

print(
    meta["sex"]
    .value_counts(
        dropna=False
    )
)


print(
    "\nStandardized sex values:"
)

print(
    meta["sex_clean"]
    .value_counts(
        dropna=False
    )
)


# =============================================================================
# Check for conflicting sex assignments
# =============================================================================

sex_assignment_check = (
    meta
    .dropna(
        subset=[
            "id_clean",
            "sex_clean"
        ]
    )
    .groupby(
        "id_clean"
    )["sex_clean"]
    .nunique()
)


conflicting_sex_ids = (
    sex_assignment_check[
        sex_assignment_check > 1
    ]
    .index
    .tolist()
)


if conflicting_sex_ids:

    raise ValueError(
        "\nThe following deer have conflicting sex "
        "assignments in WTD_Meta.tx<LOCAL_PATH>"
        f"{conflicting_sex_ids}"
    )


# =============================================================================
# Create one metadata row per deer
# =============================================================================

meta_unique = (
    meta[
        [
            "id_clean",
            "sex_clean"
        ]
    ]
    .dropna()
    .drop_duplicates(
        subset="id_clean"
    )
    .copy()
)


# =============================================================================
# Build female and male ID sets
# =============================================================================

female_meta_ids = set(
    meta_unique.loc[
        meta_unique["sex_clean"] == "Female",
        "id_clean"
    ]
)


male_meta_ids = set(
    meta_unique.loc[
        meta_unique["sex_clean"] == "Male",
        "id_clean"
    ]
)


print(
    f"\nFemale IDs in metadata: "
    f"{len(female_meta_ids)}"
)

print(
    f"Male IDs in metadata: "
    f"{len(male_meta_ids)}"
)


# =============================================================================
# Load scaler and obtain exact model feature order
# =============================================================================

scaler = joblib.load(
    SCALER_PATH
)


if not hasattr(
    scaler,
    "feature_names_in_"
):

    raise AttributeError(
        "The loaded scaler does not contain "
        "feature_names_in_. Confirm that the scaler "
        "was fitted using a pandas DataFrame."
    )


FEATURES = list(
    scaler.feature_names_in_
)


print(
    f"\nNumber of model features: "
    f"{len(FEATURES)}"
)


print(
    "\nFeatures:"
)


for feature in FEATURES:

    print(
        f"  - {feature}"
    )


# =============================================================================
# Keep only ID and model features
# =============================================================================

train = keep_cols(
    train,
    FEATURES
)

val = keep_cols(
    val,
    FEATURES
)

test = keep_cols(
    test,
    FEATURES
)


# =============================================================================
# Inverse-transform features back to raw units
# =============================================================================

train_raw = inverse_transform_features(
    train,
    scaler,
    FEATURES
)


val_raw = inverse_transform_features(
    val,
    scaler,
    FEATURES
)


test_raw = inverse_transform_features(
    test,
    scaler,
    FEATURES
)


# =============================================================================
# Standardize deer IDs
# =============================================================================

for df in (
    train_raw,
    val_raw,
    test_raw
):

    df["id_clean"] = df[
        "id"
    ].map(
        standardize_id
    )


# =============================================================================
# Standardize CWD ID lists
# =============================================================================

CWD_NEG_IDS_CLEAN = sorted({
    standardize_id(deer_id)

    for deer_id in CWD_NEG_IDS

    if standardize_id(deer_id) is not None
})


CWD_POS_IDS_CLEAN = sorted({
    standardize_id(deer_id)

    for deer_id in CWD_POS_IDS

    if standardize_id(deer_id) is not None
})


# =============================================================================
# Combine movement datasets
# =============================================================================

all_raw = pd.concat(
    [
        train_raw,
        val_raw,
        test_raw
    ],
    ignore_index=True
)


ids_present = set(
    all_raw[
        "id_clean"
    ]
    .dropna()
    .unique()
)


# =============================================================================
# CWD IDs actually present in movement data
# =============================================================================

neg_ids_present = set(
    CWD_NEG_IDS_CLEAN
) & ids_present


pos_ids_present = set(
    CWD_POS_IDS_CLEAN
) & ids_present


# =============================================================================
# Split CWD groups by sex
# =============================================================================

female_neg_ids = sorted(
    neg_ids_present &
    female_meta_ids
)


female_pos_ids = sorted(
    pos_ids_present &
    female_meta_ids
)


male_neg_ids = sorted(
    neg_ids_present &
    male_meta_ids
)


male_pos_ids = sorted(
    pos_ids_present &
    male_meta_ids
)


# =============================================================================
# Print sex-specific samples
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "SEX-SPECIFIC CWD GROUPS"
)

print(
    "=" * 80
)


print(
    f"\nFemale CWD- deer: "
    f"{len(female_neg_ids)}"
)

print(
    female_neg_ids
)


print(
    f"\nFemale CWD+ deer: "
    f"{len(female_pos_ids)}"
)

print(
    female_pos_ids
)


print(
    f"\nMale CWD- deer: "
    f"{len(male_neg_ids)}"
)

print(
    male_neg_ids
)


print(
    f"\nMale CWD+ deer: "
    f"{len(male_pos_ids)}"
)

print(
    male_pos_ids
)


# =============================================================================
# Check that every comparison has data
# =============================================================================

sex_groups = {

    "Female CWD-":
        female_neg_ids,

    "Female CWD+":
        female_pos_ids,

    "Male CWD-":
        male_neg_ids,

    "Male CWD+":
        male_pos_ids
}


for group_name, ids in sex_groups.items():

    if len(ids) == 0:

        raise ValueError(
            f"No IDs were found for {group_name}. "
            "Check the sex coding in WTD_Meta.txt "
            "and the CWD ID lists."
        )


# =============================================================================
# Calculate sex-specific feature means
# =============================================================================

mu_female_neg = group_mean(
    all_raw,
    FEATURES,
    female_neg_ids
)


mu_female_pos = group_mean(
    all_raw,
    FEATURES,
    female_pos_ids
)


mu_male_neg = group_mean(
    all_raw,
    FEATURES,
    male_neg_ids
)


mu_male_pos = group_mean(
    all_raw,
    FEATURES,
    male_pos_ids
)


# =============================================================================
# Calculate percentage differences
#
# Female:
#       Female CWD+ versus Female CWD-
#
# Male:
#       Male CWD+ versus Male CWD-
# =============================================================================

female_pct = pct_diff(
    mu_female_pos.values,
    mu_female_neg.values
)


male_pct = pct_diff(
    mu_male_pos.values,
    mu_male_neg.values
)


# =============================================================================
# Build heatmap DataFrames
# =============================================================================

comparison_label = (
    "CWD$^{+}$ vs. CWD$^{-}$"
)


female_df = pd.DataFrame(
    {
        "feature":
            FEATURES,

        comparison_label:
            female_pct
    }
).set_index(
    "feature"
)


male_df = pd.DataFrame(
    {
        "feature":
            FEATURES,

        comparison_label:
            male_pct
    }
).set_index(
    "feature"
)


# =============================================================================
# Feature ordering and labels
# =============================================================================

pretty_order = [

    # Velocity
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction / turning angle
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path shape
    "sinuosity",
    "sinuosity_28pt",
    "sinuosity_56pt",

    # Space use
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "sinuosity_28pt":
        "Sinuosity",

    "sinuosity_56pt":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [

    feature

    for feature in pretty_order

    if (
        feature in female_df.index
        and
        feature in male_df.index
    )
]


female_df = female_df.reindex(
    available_pretty_order
)


male_df = male_df.reindex(
    available_pretty_order
)


# -----------------------------------------------------------------------------
# Give both heatmaps identical display labels
# -----------------------------------------------------------------------------

pretty_labels = [

    pretty_map.get(
        feature,
        feature
    )

    for feature in available_pretty_order
]


female_df.index = pretty_labels
male_df.index = pretty_labels


# =============================================================================
# Print numerical results
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FEMALE: CWD+ VS. CWD-"
)

print(
    "=" * 80
)

print(
    female_df
    .round(1)
    .to_string()
)


print(
    "\n"
    + "=" * 80
)

print(
    "MALE: CWD+ VS. CWD-"
)

print(
    "=" * 80
)

print(
    male_df
    .round(1)
    .to_string()
)


# =============================================================================
# Optional combined numerical table
# =============================================================================

combined_results = pd.DataFrame(
    {
        "Feature":
            pretty_labels,

        "Female_pct_difference":
            female_df[
                comparison_label
            ].values,

        "Male_pct_difference":
            male_df[
                comparison_label
            ].values
    }
)


print(
    "\n"
    + "=" * 80
)

print(
    "FEMALE AND MALE COMPARISON TABLE"
)

print(
    "=" * 80
)


print(
    combined_results
    .round(1)
    .to_string(
        index=False
    )
)


# =============================================================================
# Save numerical results
# =============================================================================

csv_path = (
    OUT_DIR /
    "CWD+_vs_CWD-_RAW_Female_Male_values.csv"
)


combined_results.to_csv(
    csv_path,
    index=False
)


print(
    f"\nSaved numerical results t<LOCAL_PATH>"
    f"{csv_path}"
)


# =============================================================================
# Determine SHARED color scale
#
# IMPORTANT:
# Both female and male panels use the same scale.
# =============================================================================

combined_values = np.concatenate(
    [
        female_df.to_numpy(
            dtype=float
        ).ravel(),

        male_df.to_numpy(
            dtype=float
        ).ravel()
    ]
)


finite_values = combined_values[
    np.isfinite(
        combined_values
    )
]


if len(finite_values) == 0:

    raise ValueError(
        "Neither heatmap contains finite values."
    )


data_min = float(
    np.min(
        finite_values
    )
)


data_max = float(
    np.max(
        finite_values
    )
)


print(
    f"\nCombined data range: "
    f"{data_min:.1f}% to "
    f"{data_max:.1f}%"
)


vmin = float(
    np.floor(
        data_min
    )
)


vmax = float(
    np.ceil(
        data_max
    )
)


# -----------------------------------------------------------------------------
# Ensure zero and both directions are represented
# -----------------------------------------------------------------------------

if vmin >= 0:
    vmin = -5.0


if vmax <= 0:
    vmax = 5.0


print(
    f"Shared rounded scale: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Discrete color boundaries
# =============================================================================

candidate_bounds = [
    vmin,
    -30,
    -15,
    -10,
    -5,
    0,
    5,
    10,
    15,
    30,
    vmax
]


bounds = [

    float(bound)

    for bound in candidate_bounds

    if (
        vmin <= bound <= vmax
    )
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(
        bounds
    )
)


if len(bounds) < 2:

    bounds = [
        vmin - 1,
        vmax + 1
    ]


print(
    f"Shared color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Build discrete shared colormap
# =============================================================================

base_cmap = mpl.colormaps[
    "RdBu_r"
]


bin_midpoints = np.array(
    [

        (
            lower +
            upper
        ) / 2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


negative_extent = max(
    abs(vmin),
    1.0
)


positive_extent = max(
    abs(vmax),
    1.0
)


color_positions = []


for midpoint in bin_midpoints:

    if midpoint < 0:

        relative_strength = min(
            abs(midpoint) /
            negative_extent,
            1.0
        )

        cmap_position = (
            0.5 -
            0.5 *
            relative_strength
        )

    elif midpoint > 0:

        relative_strength = min(
            midpoint /
            positive_extent,
            1.0
        )

        cmap_position = (
            0.5 +
            0.5 *
            relative_strength
        )

    else:

        cmap_position = 0.5


    color_positions.append(
        cmap_position
    )


color_positions = np.asarray(
    color_positions,
    dtype=float
)


# -----------------------------------------------------------------------------
# Force outer intervals to darkest colors
# -----------------------------------------------------------------------------

if len(color_positions) > 0:

    if bin_midpoints[0] < 0:

        color_positions[0] = 0.0


    if bin_midpoints[-1] > 0:

        color_positions[-1] = 1.0


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_RdBu"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


# =============================================================================
# Figure dimensions
# =============================================================================

n_rows = female_df.shape[0]


FIG_WIDTH = 12.5

ROW_HEIGHT = 0.62


FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.5,
    8.5,
    12.5
)


# =============================================================================
# Font sizes
# =============================================================================

TITLE_FS = 25

AX_LABEL_FS = 24

X_TICK_FS = 18

Y_TICK_FS = 19

ANNOT_FS = 18

CBAR_LABEL_FS = 22

CBAR_TICK_FS = 17


# =============================================================================
# Create figure
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# -----------------------------------------------------------------------------
# Grid:
#
#   female heatmap | male heatmap | shared colorbar
# -----------------------------------------------------------------------------

grid = fig.add_gridspec(
    nrows=1,
    ncols=3,

    width_ratios=[
        1.0,
        1.0,
        0.10
    ],

    left=0.35,
    right=0.90,
    bottom=0.13,
    top=0.88,

    wspace=0.55
)


ax_female = fig.add_subplot(
    grid[0, 0]
)


ax_male = fig.add_subplot(
    grid[0, 1]
)


cax = fig.add_subplot(
    grid[0, 2]
)


# =============================================================================
# Function for plotting each heatmap
# =============================================================================

def draw_heatmap(
    dataframe,
    ax,
    title
):

    hm = sns.heatmap(

        dataframe,

        cmap=discrete_cmap,

        norm=norm,

        annot=True,

        fmt=".1f",

        annot_kws={
            "fontsize":
                ANNOT_FS,

            "fontfamily":
                "Times New Roman",

            "fontweight":
                "bold"
        },

        linewidths=1.0,

        linecolor="white",

        ax=ax,

        cbar=False
    )


    # -------------------------------------------------------------------------
    # Annotation formatting
    # -------------------------------------------------------------------------

    flat_values = (
        dataframe
        .to_numpy()
        .flatten()
    )


    for text, value in zip(
        ax.texts,
        flat_values
    ):

        text.set_fontfamily(
            "Times New Roman"
        )

        text.set_fontsize(
            ANNOT_FS
        )

        text.set_fontweight(
            "bold"
        )


        if not np.isfinite(
            value
        ):
            continue


        # White text only on darkest intervals
        if (
            value <= bounds[1]
            or
            value >= bounds[-2]
        ):

            text.set_color(
                "white"
            )

        else:

            text.set_color(
                "black"
            )


    # -------------------------------------------------------------------------
    # Panel title
    # -------------------------------------------------------------------------

    ax.set_title(
        title,
        fontsize=TITLE_FS,
        fontfamily="Times New Roman",
        fontweight="bold",
        pad=18
    )


    # -------------------------------------------------------------------------
    # Axis labels
    # -------------------------------------------------------------------------

    ax.set_xlabel(
        "Comparison",
        fontsize=AX_LABEL_FS,
        fontfamily="Times New Roman",
        fontweight="bold",
        labelpad=14
    )


    ax.set_ylabel(
        "",
        fontsize=AX_LABEL_FS,
        fontfamily="Times New Roman",
        fontweight="bold"
    )


    ax.tick_params(
        axis="x",
        rotation=0,
        length=0
    )


    ax.tick_params(
        axis="y",
        rotation=0,
        length=0
    )


    # -------------------------------------------------------------------------
    # X tick formatting
    # -------------------------------------------------------------------------

    for label in ax.get_xticklabels():

        label.set_fontfamily(
            "Times New Roman"
        )

        label.set_fontsize(
            X_TICK_FS
        )

        label.set_fontweight(
            "bold"
        )

        label.set_rotation(
            0
        )


    # -------------------------------------------------------------------------
    # Y tick formatting
    # -------------------------------------------------------------------------

    for label in ax.get_yticklabels():

        label.set_fontfamily(
            "Times New Roman"
        )

        label.set_fontsize(
            Y_TICK_FS
        )

        label.set_fontweight(
            "bold"
        )

        label.set_rotation(
            0
        )


    return hm


# =============================================================================
# Female heatmap
# =============================================================================

draw_heatmap(
    female_df,
    ax_female,
    "Female"
)


# =============================================================================
# Male heatmap
# =============================================================================

draw_heatmap(
    male_df,
    ax_male,
    "Male"
)


# =============================================================================
# Feature axis label
# =============================================================================

ax_female.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Hide duplicate feature labels on male panel
#
# Rows align exactly with the female panel, so displaying the labels only
# once makes the figure much cleaner.
# =============================================================================

ax_male.set_yticklabels(
    []
)


ax_male.tick_params(
    axis="y",
    left=False
)


# =============================================================================
# Shared colorbar
# =============================================================================

sm = mpl.cm.ScalarMappable(
    norm=norm,
    cmap=discrete_cmap
)


sm.set_array(
    []
)


cbar = fig.colorbar(
    sm,
    cax=cax
)


cbar.set_label(
    "% Difference (Raw Units)",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


# =============================================================================
# Colorbar ticks
# =============================================================================

candidate_ticks = np.array(
    [
        vmin,
        -30,
        -15,
        -10,
        -5,
        0,
        5,
        10,
        15,
        30,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [

        f"{tick:.0f}%"

        for tick in nice_ticks
    ]
)


for label in cbar.ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        CBAR_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Overall title
# =============================================================================

fig.suptitle(
    "Movement Differences Between CWD$^{+}$ and CWD$^{-}$ Deer by Sex",
    fontsize=27,
    fontfamily="Times New Roman",
    fontweight="bold",
    y=0.96
)


# =============================================================================
# Save
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG t<LOCAL_PATH>"
    f"{png_path}"
)


print(
    f"\nSaved PDF t<LOCAL_PATH>"
    f"{pdf_path}"
)


# =============================================================================
# Display
# =============================================================================

plt.show()


plt.close(
    fig
)

In [ ]:
# =============================================================================
# DIAGNOSTIC:
# Why does CWD+ appear to have greater space use than CWD-?
#
# This does NOT change the heatmap.
#
# It compares:
#
#   1. Current observation-weighted mean
#   2. Deer-level mean (each deer gets equal weight)
#   3. Deer-level median
#
# for:
#   - akde_low
#   - akde_est
#   - akde_high
#
# separately for females and males.
# =============================================================================


AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high"
]


# =============================================================================
# Check which AKDE features actually exist
# =============================================================================

AKDE_FEATURES = [
    feature
    for feature in AKDE_FEATURES
    if feature in all_raw.columns
]


if not AKDE_FEATURES:

    raise ValueError(
        "No AKDE features were found in all_raw."
    )


print(
    "\nAKDE features being investigated:"
)

for feature in AKDE_FEATURES:

    print(
        f"  - {feature}"
    )


# =============================================================================
# Helper: percentage difference
# =============================================================================

def percent_difference(
    cwd_pos,
    cwd_neg,
    floor=1e-8
):
    """
    Percentage difference of CWD+ relative to CWD-.

        100 * (CWD+ - CWD-) / |CWD-|

    Positive:
        CWD+ is larger.

    Negative:
        CWD+ is smaller.
    """

    denominator = max(
        abs(float(cwd_neg)),
        floor
    )

    return (
        100.0 *
        (
            float(cwd_pos) -
            float(cwd_neg)
        ) /
        denominator
    )


# =============================================================================
# Helper: create one AKDE value per deer
# =============================================================================

def get_deer_level_akde(
    df,
    ids,
    feature
):
    """
    Return one row per deer.

    For each deer:
        - calculate mean AKDE value
        - calculate median AKDE value
        - count observations
        - count unique AKDE values

    If AKDE is repeated across GPS rows, the deer-level mean and median
    should generally be identical or nearly identical.
    """

    subset = df[
        df["id_clean"].isin(ids)
    ][
        [
            "id_clean",
            feature
        ]
    ].copy()


    subset[feature] = pd.to_numeric(
        subset[feature],
        errors="coerce"
    )


    subset = subset.dropna(
        subset=[
            "id_clean",
            feature
        ]
    )


    if subset.empty:

        raise ValueError(
            f"No observations found for {feature} "
            f"using ID<LOCAL_PATH>"
        )


    deer_summary = (
        subset
        .groupby(
            "id_clean",
            as_index=False
        )
        .agg(
            akde_mean=(
                feature,
                "mean"
            ),

            akde_median=(
                feature,
                "median"
            ),

            n_observations=(
                feature,
                "size"
            ),

            n_unique_values=(
                feature,
                "nunique"
            )
        )
    )


    return deer_summary


# =============================================================================
# Main diagnostic function
# =============================================================================

def diagnose_akde(
    df,
    sex_name,
    cwd_neg_ids,
    cwd_pos_ids
):

    print(
        "\n\n"
        + "=" * 90
    )

    print(
        f"{sex_name.upper()} AKDE DIAGNOSTIC"
    )

    print(
        "=" * 90
    )


    all_summary_rows = []


    for feature in AKDE_FEATURES:

        print(
            "\n\n"
            + "-" * 90
        )

        print(
            f"{sex_name.upper()} — {feature.upper()}"
        )

        print(
            "-" * 90
        )


        # =====================================================================
        # Get one summary row per deer
        # =====================================================================

        neg_deer = get_deer_level_akde(
            df,
            cwd_neg_ids,
            feature
        )


        pos_deer = get_deer_level_akde(
            df,
            cwd_pos_ids,
            feature
        )


        neg_deer["cwd_status"] = "CWD-"

        pos_deer["cwd_status"] = "CWD+"


        # =====================================================================
        # Print individual CWD- deer
        # =====================================================================

        print(
            "\nCWD- INDIVIDUAL DEER"
        )

        print(
            neg_deer
            .sort_values(
                "akde_mean"
            )
            .round(3)
            .to_string(
                index=False
            )
        )


        # =====================================================================
        # Print individual CWD+ deer
        # =====================================================================

        print(
            "\nCWD+ INDIVIDUAL DEER"
        )

        print(
            pos_deer
            .sort_values(
                "akde_mean"
            )
            .round(3)
            .to_string(
                index=False
            )
        )


        # =====================================================================
        # CURRENT METHOD:
        # mean across every observation
        # =====================================================================

        neg_obs_values = (
            df.loc[
                df["id_clean"].isin(
                    cwd_neg_ids
                ),
                feature
            ]
            .dropna()
            .astype(float)
        )


        pos_obs_values = (
            df.loc[
                df["id_clean"].isin(
                    cwd_pos_ids
                ),
                feature
            ]
            .dropna()
            .astype(float)
        )


        neg_observation_mean = (
            neg_obs_values.mean()
        )


        pos_observation_mean = (
            pos_obs_values.mean()
        )


        observation_pct = percent_difference(
            pos_observation_mean,
            neg_observation_mean
        )


        # =====================================================================
        # DEER-LEVEL MEAN:
        # each deer contributes equally
        # =====================================================================

        neg_deer_mean = (
            neg_deer[
                "akde_mean"
            ].mean()
        )


        pos_deer_mean = (
            pos_deer[
                "akde_mean"
            ].mean()
        )


        deer_mean_pct = percent_difference(
            pos_deer_mean,
            neg_deer_mean
        )


        # =====================================================================
        # DEER-LEVEL MEDIAN
        # =====================================================================

        neg_deer_median = (
            neg_deer[
                "akde_median"
            ].median()
        )


        pos_deer_median = (
            pos_deer[
                "akde_median"
            ].median()
        )


        deer_median_pct = percent_difference(
            pos_deer_median,
            neg_deer_median
        )


        # =====================================================================
        # Sample sizes
        # =====================================================================

        n_neg_deer = len(
            neg_deer
        )


        n_pos_deer = len(
            pos_deer
        )


        n_neg_obs = len(
            neg_obs_values
        )


        n_pos_obs = len(
            pos_obs_values
        )


        # =====================================================================
        # Print comparison
        # =====================================================================

        print(
            "\n"
            + "=" * 90
        )

        print(
            "GROUP COMPARISON"
        )

        print(
            "=" * 90
        )


        print(
            f"\nCWD- deer: {n_neg_deer}"
        )

        print(
            f"CWD+ deer: {n_pos_deer}"
        )


        print(
            f"\nCWD- observations: "
            f"{n_neg_obs:,}"
        )

        print(
            f"CWD+ observations: "
            f"{n_pos_obs:,}"
        )


        print(
            "\nCURRENT OBSERVATION-WEIGHTED METHOD"
        )

        print(
            f"  CWD- mean: "
            f"{neg_observation_mean:.3f}"
        )

        print(
            f"  CWD+ mean: "
            f"{pos_observation_mean:.3f}"
        )

        print(
            f"  Difference: "
            f"{observation_pct:+.2f}%"
        )


        print(
            "\nDEER-LEVEL MEAN "
            "(EACH DEER EQUALLY WEIGHTED)"
        )

        print(
            f"  CWD- mean: "
            f"{neg_deer_mean:.3f}"
        )

        print(
            f"  CWD+ mean: "
            f"{pos_deer_mean:.3f}"
        )

        print(
            f"  Difference: "
            f"{deer_mean_pct:+.2f}%"
        )


        print(
            "\nDEER-LEVEL MEDIAN"
        )

        print(
            f"  CWD- median: "
            f"{neg_deer_median:.3f}"
        )

        print(
            f"  CWD+ median: "
            f"{pos_deer_median:.3f}"
        )

        print(
            f"  Difference: "
            f"{deer_median_pct:+.2f}%"
        )


        # =====================================================================
        # Determine whether weighting materially changes result
        # =====================================================================

        weighting_change = (
            deer_mean_pct -
            observation_pct
        )


        print(
            "\nEFFECT OF OBSERVATION WEIGHTING"
        )

        print(
            f"  Deer-level minus observation-level: "
            f"{weighting_change:+.2f} percentage points"
        )


        if (
            np.sign(
                observation_pct
            )
            !=
            np.sign(
                deer_mean_pct
            )
        ):

            print(
                "\n  *** IMPORTANT: "
                "The direction of the effect CHANGES "
                "when each deer is weighted equally. ***"
            )

        else:

            print(
                "\n  Direction is unchanged when "
                "each deer is weighted equally."
            )


        # =====================================================================
        # Check repeated AKDE values
        # =====================================================================

        combined_deer = pd.concat(
            [
                neg_deer,
                pos_deer
            ],
            ignore_index=True
        )


        deer_with_multiple_akde = (
            combined_deer[
                combined_deer[
                    "n_unique_values"
                ] > 1
            ]
        )


        print(
            "\nAKDE REPETITION CHECK"
        )


        if deer_with_multiple_akde.empty:

            print(
                "  Every deer has only one unique "
                f"{feature} value repeated across rows."
            )

        else:

            print(
                "  Some deer have multiple unique "
                f"{feature} values:"
            )

            print(
                deer_with_multiple_akde[
                    [
                        "id_clean",
                        "cwd_status",
                        "n_unique_values",
                        "n_observations"
                    ]
                ]
                .to_string(
                    index=False
                )
            )


        # =====================================================================
        # Store summary
        # =====================================================================

        all_summary_rows.append(
            {
                "sex":
                    sex_name,

                "feature":
                    feature,

                "cwd_neg_deer":
                    n_neg_deer,

                "cwd_pos_deer":
                    n_pos_deer,

                "cwd_neg_observations":
                    n_neg_obs,

                "cwd_pos_observations":
                    n_pos_obs,

                "cwd_neg_observation_mean":
                    neg_observation_mean,

                "cwd_pos_observation_mean":
                    pos_observation_mean,

                "observation_weighted_pct":
                    observation_pct,

                "cwd_neg_deer_mean":
                    neg_deer_mean,

                "cwd_pos_deer_mean":
                    pos_deer_mean,

                "deer_level_mean_pct":
                    deer_mean_pct,

                "cwd_neg_deer_median":
                    neg_deer_median,

                "cwd_pos_deer_median":
                    pos_deer_median,

                "deer_level_median_pct":
                    deer_median_pct,

                "weighting_difference_pp":
                    weighting_change
            }
        )


    return pd.DataFrame(
        all_summary_rows
    )


# =============================================================================
# Run FEMALE diagnostic
# =============================================================================

female_akde_diagnostic = diagnose_akde(
    df=all_raw,
    sex_name="Female",
    cwd_neg_ids=female_neg_ids,
    cwd_pos_ids=female_pos_ids
)


# =============================================================================
# Run MALE diagnostic
# =============================================================================

male_akde_diagnostic = diagnose_akde(
    df=all_raw,
    sex_name="Male",
    cwd_neg_ids=male_neg_ids,
    cwd_pos_ids=male_pos_ids
)


# =============================================================================
# Combine diagnostic summaries
# =============================================================================

akde_diagnostic_summary = pd.concat(
    [
        female_akde_diagnostic,
        male_akde_diagnostic
    ],
    ignore_index=True
)


# =============================================================================
# Print final concise summary
# =============================================================================

print(
    "\n\n"
    + "=" * 100
)

print(
    "FINAL AKDE DIAGNOSTIC SUMMARY"
)

print(
    "=" * 100
)


summary_columns = [
    "sex",
    "feature",
    "cwd_neg_deer",
    "cwd_pos_deer",
    "observation_weighted_pct",
    "deer_level_mean_pct",
    "deer_level_median_pct",
    "weighting_difference_pp"
]


print(
    akde_diagnostic_summary[
        summary_columns
    ]
    .round(2)
    .to_string(
        index=False
    )
)


# =============================================================================
# Save diagnostic results
# =============================================================================

diagnostic_path = (
    OUT_DIR /
    "AKDE_CWD_by_sex_diagnostic.csv"
)


akde_diagnostic_summary.to_csv(
    diagnostic_path,
    index=False
)


print(
    "\nDiagnostic CSV saved to:"
)

print(
    diagnostic_path
)

In [ ]:
# =============================================================================
# DEER-LEVEL CWD+ VS CWD- COMPARISON
#
# No sex separation.
# No subsampling.
# No forced equal sample sizes.
#
# Every deer contributes exactly ONE value per feature.
# =============================================================================


# =============================================================================
# Create one feature vector per deer
# =============================================================================

def make_deer_level_data(
    df,
    features,
    ids
):
    """
    Collapse all GPS observations to one mean value per deer.

    Every deer receives equal weight regardless of the number
    of GPS observations.
    """

    subset = df[
        df["id_clean"].isin(ids)
    ][
        ["id_clean"] + features
    ].copy()


    if subset.empty:
        raise ValueError(
            "No observations were found for the requested deer IDs."
        )


    deer_level = (
        subset
        .groupby(
            "id_clean",
            as_index=False
        )
        [features]
        .mean()
    )


    return deer_level


# =============================================================================
# Create deer-level datasets
# =============================================================================

neg_deer = make_deer_level_data(
    all_raw,
    FEATURES,
    neg_ids_present
)


pos_deer = make_deer_level_data(
    all_raw,
    FEATURES,
    pos_ids_present
)


print(
    "\n"
    + "=" * 80
)

print(
    "DEER-LEVEL CWD+ VS CWD- COMPARISON"
)

print(
    "=" * 80
)


print(
    f"\nCWD- deer: {len(neg_deer)}"
)

print(
    f"CWD+ deer: {len(pos_deer)}"
)


# =============================================================================
# Group means
#
# IMPORTANT:
# These are means of DEER, not means of GPS observations.
# =============================================================================

mu_neg_mean = (
    neg_deer[
        FEATURES
    ]
    .mean()
)


mu_pos_mean = (
    pos_deer[
        FEATURES
    ]
    .mean()
)


# =============================================================================
# Group medians
#
# Useful because AKDE/home-range estimates can be highly right-skewed.
# =============================================================================

mu_neg_median = (
    neg_deer[
        FEATURES
    ]
    .median()
)


mu_pos_median = (
    pos_deer[
        FEATURES
    ]
    .median()
)


# =============================================================================
# Percentage difference using deer-level MEANS
# =============================================================================

mean_pct_difference = pct_diff(
    mu_pos_mean.values,
    mu_neg_mean.values
)


# =============================================================================
# Percentage difference using deer-level MEDIANS
# =============================================================================

median_pct_difference = pct_diff(
    mu_pos_median.values,
    mu_neg_median.values
)


# =============================================================================
# Results table
# =============================================================================

comparison_results = pd.DataFrame(
    {
        "feature":
            FEATURES,

        "CWD-_deer_mean":
            mu_neg_mean.values,

        "CWD+_deer_mean":
            mu_pos_mean.values,

        "mean_pct_difference":
            mean_pct_difference,

        "CWD-_deer_median":
            mu_neg_median.values,

        "CWD+_deer_median":
            mu_pos_median.values,

        "median_pct_difference":
            median_pct_difference
    }
)


# =============================================================================
# Print all results
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DEER-LEVEL RESULTS"
)

print(
    "=" * 100
)


print(
    comparison_results
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# Specifically inspect AKDE
# =============================================================================

AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high"
]


akde_results = comparison_results[
    comparison_results[
        "feature"
    ].isin(
        AKDE_FEATURES
    )
].copy()


print(
    "\n"
    + "=" * 100
)

print(
    "AKDE / HOME-RANGE RESULTS"
)

print(
    "=" * 100
)


print(
    akde_results
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# Print INDIVIDUAL AKDE values
#
# This is extremely important for determining whether a few CWD+ deer
# are driving the apparent difference.
# =============================================================================

available_akde = [
    feature
    for feature in AKDE_FEATURES
    if feature in FEATURES
]


print(
    "\n"
    + "=" * 100
)

print(
    "INDIVIDUAL CWD- AKDE VALUES"
)

print(
    "=" * 100
)


print(
    neg_deer[
        [
            "id_clean",
            *available_akde
        ]
    ]
    .sort_values(
        "akde_est"
        if "akde_est" in available_akde
        else available_akde[0]
    )
    .round(3)
    .to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "INDIVIDUAL CWD+ AKDE VALUES"
)

print(
    "=" * 100
)


print(
    pos_deer[
        [
            "id_clean",
            *available_akde
        ]
    ]
    .sort_values(
        "akde_est"
        if "akde_est" in available_akde
        else available_akde[0]
    )
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# BUILD HEATMAP
#
# I recommend using DEER-LEVEL MEDIANS for the AKDE-skew issue.
#
# But if you want the entire heatmap to remain mean-based,
# change median_pct_difference to mean_pct_difference below.
# =============================================================================

comparison_columns = {

    "feature":
        FEATURES,

    "CWD$^{+}$ vs. CWD$^{-}$":
        median_pct_difference
}


all_df = pd.DataFrame(
    comparison_columns
)


comp_order = [
    "CWD$^{+}$ vs. CWD$^{-}$"
]


heatmap_df = (
    all_df
    .set_index(
        "feature"
    )
    .loc[
        :,
        comp_order
    ]
)


# =============================================================================
# Save numerical results
# =============================================================================

output_path = (
    OUT_DIR /
    "CWD+_vs_CWD-_DEER_LEVEL_RESULTS.csv"
)


comparison_results.to_csv(
    output_path,
    index=False
)


print(
    "\nResults saved to:"
)

print(
    output_path
)

In [ ]:
# =============================================================================
# LONGITUDINAL CWD SPACE-USE ANALYSIS
# =============================================================================
#
# PRIMARY QUESTION
# -----------------------------------------------------------------------------
# Among deer that were confirmed CWD- and subsequently tested CWD+:
#
#     Did space use increase prior to first detected positivity,
#     then decrease after the deer tested positive?
#
#
# IMPORTANT INTERPRETATION
# -----------------------------------------------------------------------------
# "First CWD+" = first DETECTED positive test.
#
# This is NOT necessarily the biological infection date.
#
# Therefore:
#
#     PRE  = movement before first detected positive test
#     POST = movement after first detected positive test
#
#
# PRIMARY ANALYSIS
# -----------------------------------------------------------------------------
# 1. Load:
#
#       CWD_negative_training_with_seasons2.csv
#       CWD_positive_testing_with_seasons2.csv
#
# 2. Read annual CWD testing records from 2006-2010.
#
# 3. Determine:
#
#       first positive test
#       last negative test before first positive
#
# 4. Identify confirmed converters:
#
#       CWD-  --->  CWD+
#
# 5. Compare each converter TO ITSELF:
#
#       PRE vs POST AKDE
#
# 6. Construct event-time summaries relative to the first positive test.
#
# 7. If season exists, perform a same-season diagnostic.
#
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, binomtest


# =============================================================================
# FILE PATHS
# =============================================================================

BASE_DIR = Path(
    "<LOCAL_PATH>"
)


NEG_FILE = (
    BASE_DIR /
    "CWD_negative_training_with_seasons2.csv"
)


POS_FILE = (
    BASE_DIR /
    "CWD_positive_testing_with_seasons2.csv"
)


TEST_FILES = [

    Path(
        "<LOCAL_PATH>"
        "2006 Test Results.xls"
    ),

    Path(
        "<LOCAL_PATH>"
        "2007 Test Results.xls"
    ),

    Path(
        "<LOCAL_PATH>"
        "2008 Test Results.xls"
    ),

    Path(
        "<LOCAL_PATH>"
        "2009 Test Results.xlsx"
    ),

    Path(
        "<LOCAL_PATH>"
        "2010 Test Results.xlsx"
    ),
]


# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================

OUT_DIR = (
    BASE_DIR /
    "CWD_space_use_longitudinal"
)


OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# ANALYSIS SETTINGS
# =============================================================================

DATETIME_COL = "datetime"


AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high"
]


# -----------------------------------------------------------------------------
# Exclude observations immediately around the first positive test.
#
# Because your AKDE is based on a rolling ~7-day window, this prevents a
# rolling window from straddling the PRE / POST boundary.
# -----------------------------------------------------------------------------

BUFFER_DAYS = 7


# -----------------------------------------------------------------------------
# Minimum telemetry coverage required in BOTH the PRE and POST periods.
# -----------------------------------------------------------------------------

MIN_DAYS_PER_PHASE = 20


# -----------------------------------------------------------------------------
# PRE_MODE
#
# Recommended:
#
#     "after_last_negative"
#
# Only movement occurring between the last confirmed CWD- test and the first
# CWD+ test is considered PRE.
#
#
# Alternative:
#
#     "all_before_positive"
#
# Uses all available movement before the first CWD+ test.
# -----------------------------------------------------------------------------

PRE_MODE = "after_last_negative"


# -----------------------------------------------------------------------------
# Primary analysis uses confirmed converters only.
#
# Deer that were already CWD+ at their first recorded test are excluded from
# the paired longitudinal analysis.
# -----------------------------------------------------------------------------

CONVERTERS_ONLY = True


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================


def normalize_column_name(name):
    """
    Normalize a column name for matching.
    """

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).lower()
    )


def clean_deer_id(value):
    """
    Standardize deer IDs for matching between telemetry and test files.
    """

    if pd.isna(value):
        return np.nan

    value = str(
        value
    ).strip().upper()


    # Excel sometimes converts numeric IDs:
    #
    # 1234 -> 1234.0
    #
    value = re.sub(
        r"\.0$",
        "",
        value
    )


    return value


def find_column(
    df,
    candidates,
    contains=None
):
    """
    Find a likely column using exact normalized matching first,
    followed by partial matching.
    """

    if contains is None:
        contains = []


    normalized_columns = {

        normalize_column_name(col):
            col

        for col in df.columns
    }


    # -------------------------------------------------------------------------
    # Exact normalized match
    # -------------------------------------------------------------------------

    for candidate in candidates:

        candidate_norm = normalize_column_name(
            candidate
        )

        if candidate_norm in normalized_columns:

            return normalized_columns[
                candidate_norm
            ]


    # -------------------------------------------------------------------------
    # Partial match
    # -------------------------------------------------------------------------

    for col in df.columns:

        col_norm = normalize_column_name(
            col
        )


        for token in contains:

            token_norm = normalize_column_name(
                token
            )

            if token_norm in col_norm:

                return col


    return None


def parse_datetime_column(series):
    """
    Convert a Series to pandas datetime and remove timezone information
    if present.

    This avoids tz-aware vs tz-naive comparison errors.
    """

    result = pd.to_datetime(
        series,
        errors="coerce"
    )


    try:

        if isinstance(
            result.dtype,
            pd.DatetimeTZDtype
        ):

            result = (
                result
                .dt
                .tz_localize(None)
            )

    except Exception:

        pass


    return result


def classify_cwd_result(value):
    """
    Standardize CWD result labels to:

        POSITIVE
        NEGATIVE
        NaN
    """

    if pd.isna(value):
        return np.nan


    text = (
        str(value)
        .strip()
        .lower()
    )


    # -------------------------------------------------------------------------
    # Negative FIRST
    #
    # Important because "not detected" contains "detected".
    # -------------------------------------------------------------------------

    negative_values = {

        "negative",
        "neg",
        "n",
        "no",

        "cwd-",
        "cwd -",

        "-",

        "0",
        "0.0",

        "not detected",
        "non-detect",
        "nondetect",
        "non detected"
    }


    if text in negative_values:
        return "NEGATIVE"


    if "negative" in text:
        return "NEGATIVE"


    if "not detected" in text:
        return "NEGATIVE"


    # -------------------------------------------------------------------------
    # Positive
    # -------------------------------------------------------------------------

    positive_values = {

        "positive",
        "pos",
        "p",
        "yes",

        "cwd+",
        "cwd +",

        "+",

        "1",
        "1.0",

        "detected"
    }


    if text in positive_values:
        return "POSITIVE"


    if "positive" in text:
        return "POSITIVE"


    return np.nan


# =============================================================================
# LOAD MOVEMENT DATA
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "LOADING MOVEMENT DATA"
)

print(
    "=" * 100
)


neg_raw = pd.read_csv(
    NEG_FILE,
    low_memory=False
)


pos_raw = pd.read_csv(
    POS_FILE,
    low_memory=False
)


# -----------------------------------------------------------------------------
# Label these as COHORTS.
#
# Do not interpret every row in the positive cohort as biologically CWD+
# because some observations may precede the first detected positive test.
# -----------------------------------------------------------------------------

neg_raw[
    "cohort"
] = "CWD_negative_cohort"


pos_raw[
    "cohort"
] = "eventually_CWD_positive_cohort"


print(
    f"\nCWD-negative cohort rows: {len(neg_raw):,}"
)

print(
    f"CWD-positive cohort rows: {len(pos_raw):,}"
)


# =============================================================================
# CHECK DATETIME COLUMN
# =============================================================================

if DATETIME_COL not in neg_raw.columns:

    raise ValueError(
        f"\n'{DATETIME_COL}' is missing fro<LOCAL_PATH>"
        f"{NEG_FILE}\n\n"
        f"Column<LOCAL_PATH>)}"
    )


if DATETIME_COL not in pos_raw.columns:

    raise ValueError(
        f"\n'{DATETIME_COL}' is missing fro<LOCAL_PATH>"
        f"{POS_FILE}\n\n"
        f"Column<LOCAL_PATH>)}"
    )


# =============================================================================
# FIND ID COLUMNS
# =============================================================================

ID_CANDIDATES = [

    "id_clean",
    "id",
    "ID",

    "deer_id",
    "deer id",

    "animal_id",
    "animal id",

    "collar_id",
    "collar id"
]


neg_id_col = find_column(
    neg_raw,
    candidates=ID_CANDIDATES
)


pos_id_col = find_column(
    pos_raw,
    candidates=ID_CANDIDATES
)


if neg_id_col is None:

    raise ValueError(
        "\nCould not determine the deer ID column "
        "in the CWD-negative dataset."
    )


if pos_id_col is None:

    raise ValueError(
        "\nCould not determine the deer ID column "
        "in the CWD-positive dataset."
    )


print(
    "\nCWD-negative ID column:",
    neg_id_col
)

print(
    "CWD-positive ID column:",
    pos_id_col
)


# =============================================================================
# STANDARDIZE ID + DATETIME
# =============================================================================

neg_raw[
    "id_clean"
] = neg_raw[
    neg_id_col
].apply(
    clean_deer_id
)


pos_raw[
    "id_clean"
] = pos_raw[
    pos_id_col
].apply(
    clean_deer_id
)


neg_raw[
    "gps_datetime"
] = parse_datetime_column(
    neg_raw[
        DATETIME_COL
    ]
)


pos_raw[
    "gps_datetime"
] = parse_datetime_column(
    pos_raw[
        DATETIME_COL
    ]
)


# =============================================================================
# CHECK AKDE COLUMNS
# =============================================================================

for feature in AKDE_FEATURES:

    if feature not in neg_raw.columns:

        raise ValueError(
            f"\n{feature} is missing from the negative dataset."
        )


    if feature not in pos_raw.columns:

        raise ValueError(
            f"\n{feature} is missing from the positive dataset."
        )


# =============================================================================
# DATETIME DIAGNOSTICS
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "MOVEMENT DATETIME CHECK"
)

print(
    "=" * 100
)


print(
    "\nCWD-negative cohort:"
)

print(
    "Valid datetime rows:",
    neg_raw[
        "gps_datetime"
    ].notna().sum()
)

print(
    "Missing datetime rows:",
    neg_raw[
        "gps_datetime"
    ].isna().sum()
)

print(
    "First datetime:",
    neg_raw[
        "gps_datetime"
    ].min()
)

print(
    "Last datetime:",
    neg_raw[
        "gps_datetime"
    ].max()
)


print(
    "\nEventually-CWD-positive cohort:"
)

print(
    "Valid datetime rows:",
    pos_raw[
        "gps_datetime"
    ].notna().sum()
)

print(
    "Missing datetime rows:",
    pos_raw[
        "gps_datetime"
    ].isna().sum()
)

print(
    "First datetime:",
    pos_raw[
        "gps_datetime"
    ].min()
)

print(
    "Last datetime:",
    pos_raw[
        "gps_datetime"
    ].max()
)


# =============================================================================
# CHECK WHETHER ANY DEER APPEAR IN BOTH DATASETS
# =============================================================================

neg_ids = set(
    neg_raw[
        "id_clean"
    ].dropna()
)


pos_ids = set(
    pos_raw[
        "id_clean"
    ].dropna()
)


overlap_ids = (
    neg_ids
    &
    pos_ids
)


print(
    "\n"
    + "=" * 100
)

print(
    "COHORT ID CHECK"
)

print(
    "=" * 100
)


print(
    "\nUnique CWD-negative cohort deer:",
    len(
        neg_ids
    )
)

print(
    "Unique eventually-positive cohort deer:",
    len(
        pos_ids
    )
)

print(
    "Deer appearing in BOTH files:",
    len(
        overlap_ids
    )
)


if overlap_ids:

    print(
        "\nOverlapping IDs:"
    )

    print(
        sorted(
            overlap_ids
        )
    )


# =============================================================================
# COMBINE MOVEMENT DATA
# =============================================================================

all_raw = pd.concat(

    [
        neg_raw,
        pos_raw
    ],

    ignore_index=True,
    sort=False
)


all_raw = all_raw.dropna(
    subset=[
        "id_clean",
        "gps_datetime"
    ]
).copy()


all_raw[
    "gps_date"
] = (
    all_raw[
        "gps_datetime"
    ]
    .dt
    .normalize()
)


# =============================================================================
# CHECK DUPLICATE DEER/TIMESTAMP RECORDS
# =============================================================================

duplicate_key_mask = all_raw.duplicated(

    subset=[
        "id_clean",
        "gps_datetime"
    ],

    keep=False
)


n_duplicate_rows = int(
    duplicate_key_mask.sum()
)


print(
    "\nDuplicate deer/datetime rows:",
    n_duplicate_rows
)


if n_duplicate_rows > 0:

    print(
        "\nRemoving duplicated deer/datetime records."
    )

    print(
        "If a record occurs in both datasets, the positive-cohort "
        "version is retained."
    )


    # Negative first, positive second.
    # keep="last" therefore gives priority to positive-cohort record.

    all_raw = (

        all_raw

        .drop_duplicates(
            subset=[
                "id_clean",
                "gps_datetime"
            ],
            keep="last"
        )

        .copy()
    )


# =============================================================================
# FINAL MOVEMENT SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "COMBINED MOVEMENT DATA"
)

print(
    "=" * 100
)


print(
    f"\nRows: {len(all_raw):,}"
)

print(
    "Unique deer:",
    all_raw[
        "id_clean"
    ].nunique()
)

print(
    "Date range:",
    all_raw[
        "gps_datetime"
    ].min(),
    "to",
    all_raw[
        "gps_datetime"
    ].max()
)


# =============================================================================
# READ CWD TEST FILES
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "READING CWD TEST RECORDS"
)

print(
    "=" * 100
)


TEST_ID_CANDIDATES = [

    "id_clean",
    "id",
    "ID",

    "deer_id",
    "deer id",
    "Deer ID",

    "animal_id",
    "animal id",
    "Animal ID",

    "collar id",
    "collar_id"
]


TEST_DATE_CANDIDATES = [

    "test date",
    "test_date",

    "date tested",

    "sample date",
    "sample_date",

    "collection date",
    "collection_date",

    "capture date",
    "capture_date",

    "Date Capture",

    "Date",
    "date"
]


TEST_RESULT_CANDIDATES = [

    "cwd status",
    "cwd_status",

    "cwd result",
    "cwd_result",

    "CWD",

    "test result",
    "test_result",

    "ihc result",
    "ihc_result",

    "IHC",

    "result",
    "Result",

    "status",
    "Status"
]


test_records = []


# =============================================================================
# READ EACH WORKBOOK
#
# We try several possible header rows because older Excel files are often
# formatted with titles above the actual column header.
# =============================================================================

for file_path in TEST_FILES:

    print(
        "\n"
        + "-" * 100
    )

    print(
        "FILE:",
        file_path.name
    )


    if not file_path.exists():

        print(
            "WARNING: FILE DOES NOT EXIST"
        )

        continue


    try:

        workbook = pd.ExcelFile(
            file_path
        )

    except ImportError as exc:

        raise ImportError(
            "\nReading .xls files requires xlrd.\n\n"
            "Install it wit<LOCAL_PATH>"
            "    conda install xlrd\n\n"
        ) from exc


    print(
        "Sheets:",
        workbook.sheet_names
    )


    for sheet_name in workbook.sheet_names:

        usable_sheet = None

        selected_id_col = None
        selected_date_col = None
        selected_result_col = None
        selected_header = None


        # ---------------------------------------------------------------------
        # Try header rows 0 through 5
        # ---------------------------------------------------------------------

        for header_row in range(
            0,
            6
        ):

            try:

                temp = pd.read_excel(

                    file_path,

                    sheet_name=sheet_name,

                    header=header_row
                )

            except Exception:

                continue


            if temp.empty:

                continue


            id_col = find_column(

                temp,

                candidates=TEST_ID_CANDIDATES,

                contains=[
                    "deerid",
                    "animalid",
                    "collarid"
                ]
            )


            date_col = find_column(

                temp,

                candidates=TEST_DATE_CANDIDATES,

                contains=[
                    "testdate",
                    "capturedate",
                    "sampledate",
                    "collectiondate"
                ]
            )


            result_col = find_column(

                temp,

                candidates=TEST_RESULT_CANDIDATES,

                contains=[
                    "cwd",
                    "ihc",
                    "result"
                ]
            )


            if (
                id_col is not None
                and
                date_col is not None
                and
                result_col is not None
            ):

                usable_sheet = temp

                selected_id_col = id_col

                selected_date_col = date_col

                selected_result_col = result_col

                selected_header = header_row

                break


        # ---------------------------------------------------------------------
        # Skip non-data sheets
        # ---------------------------------------------------------------------

        if usable_sheet is None:

            print(
                f"\nSheet: {sheet_name}"
            )

            print(
                "  Could not identify required columns."
            )

            continue


        print(
            f"\nSheet: {sheet_name}"
        )

        print(
            "  Header row:",
            selected_header
        )

        print(
            "  ID column:",
            selected_id_col
        )

        print(
            "  Date column:",
            selected_date_col
        )

        print(
            "  Result column:",
            selected_result_col
        )


        standardized = pd.DataFrame(
            {
                "id_test":
                    usable_sheet[
                        selected_id_col
                    ],

                "test_date":
                    usable_sheet[
                        selected_date_col
                    ],

                "cwd_result_raw":
                    usable_sheet[
                        selected_result_col
                    ]
            }
        )


        standardized[
            "source_file"
        ] = file_path.name


        standardized[
            "source_sheet"
        ] = sheet_name


        test_records.append(
            standardized
        )


# =============================================================================
# MAKE SURE TEST DATA WERE FOUND
# =============================================================================

if len(test_records) == 0:

    raise ValueError(
        "\nNo usable CWD test records were found.\n"
        "Check the printed workbook diagnostics above."
    )


cwd_tests = pd.concat(

    test_records,

    ignore_index=True
)


# =============================================================================
# CLEAN TEST RECORDS
# =============================================================================

cwd_tests[
    "id_clean"
] = cwd_tests[
    "id_test"
].apply(
    clean_deer_id
)


cwd_tests[
    "test_date"
] = parse_datetime_column(
    cwd_tests[
        "test_date"
    ]
)


cwd_tests[
    "cwd_status"
] = cwd_tests[
    "cwd_result_raw"
].apply(
    classify_cwd_result
)


# =============================================================================
# IMPORTANT DIAGNOSTIC:
# CHECK HOW THE ORIGINAL TEST RESULTS WERE INTERPRETED
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "RAW TEST RESULT INTERPRETATION"
)

print(
    "=" * 100
)


raw_result_check = (

    cwd_tests[
        [
            "cwd_result_raw",
            "cwd_status"
        ]
    ]

    .drop_duplicates()

    .sort_values(
        by="cwd_result_raw",
        key=lambda x:
            x.astype(str)
    )
)


print(
    raw_result_check.to_string(
        index=False
    )
)


# =============================================================================
# REMOVE UNUSABLE TEST RECORDS
# =============================================================================

cwd_tests = cwd_tests.dropna(
    subset=[
        "id_clean",
        "test_date",
        "cwd_status"
    ]
).copy()


cwd_tests = (

    cwd_tests

    .sort_values(
        [
            "id_clean",
            "test_date"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\nUsable CWD test records:",
    len(
        cwd_tests
    )
)

print(
    "Unique tested deer:",
    cwd_tests[
        "id_clean"
    ].nunique()
)


# =============================================================================
# PRINT FULL TEST HISTORY
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ALL CLEANED CWD TEST RECORDS"
)

print(
    "=" * 100
)


print(
    cwd_tests[
        [
            "id_clean",
            "test_date",
            "cwd_status",
            "cwd_result_raw",
            "source_file"
        ]
    ]
    .to_string(
        index=False
    )
)


# =============================================================================
# DETERMINE FIRST POSITIVE TEST FOR EACH DEER
# =============================================================================

history_rows = []


for deer_id, deer_tests in cwd_tests.groupby(
    "id_clean"
):

    deer_tests = deer_tests.sort_values(
        "test_date"
    ).copy()


    positive_tests = deer_tests[
        deer_tests[
            "cwd_status"
        ] == "POSITIVE"
    ]


    # Deer never tested positive
    if positive_tests.empty:

        continue


    first_positive_date = (
        positive_tests[
            "test_date"
        ]
        .min()
    )


    negative_before = deer_tests[
        (
            deer_tests[
                "cwd_status"
            ] == "NEGATIVE"
        )
        &
        (
            deer_tests[
                "test_date"
            ]
            <
            first_positive_date
        )
    ]


    if negative_before.empty:

        last_negative_date = pd.NaT

    else:

        last_negative_date = (
            negative_before[
                "test_date"
            ]
            .max()
        )


    n_negative_before = len(
        negative_before
    )


    n_positive_total = len(
        positive_tests
    )


    history_rows.append(
        {
            "id_clean":
                deer_id,

            "last_negative_date":
                last_negative_date,

            "first_positive_date":
                first_positive_date,

            "n_negative_tests_before_positive":
                n_negative_before,

            "n_positive_tests":
                n_positive_total,

            "confirmed_converter":
                pd.notna(
                    last_negative_date
                )
        }
    )


cwd_history = pd.DataFrame(
    history_rows
)


cwd_history[
    "neg_to_pos_interval_days"
] = (

    cwd_history[
        "first_positive_date"
    ]

    -

    cwd_history[
        "last_negative_date"
    ]

).dt.days


# =============================================================================
# PRINT TEST HISTORIES
# =============================================================================

print(
    "\n"
    + "=" * 110
)

print(
    "CWD+ DEER TEST HISTORIES"
)

print(
    "=" * 110
)


print(
    cwd_history
    .sort_values(
        "first_positive_date"
    )
    .to_string(
        index=False
    )
)


print(
    "\nConfirmed CWD- -> CWD+ converters:",
    int(
        cwd_history[
            "confirmed_converter"
        ].sum()
    )
)


print(
    "Positive without prior documented negative:",
    int(
        (
            ~cwd_history[
                "confirmed_converter"
            ]
        ).sum()
    )
)


# =============================================================================
# MATCH CWD TEST IDS TO MOVEMENT IDS
# =============================================================================

movement_ids = set(
    all_raw[
        "id_clean"
    ].dropna()
)


positive_test_ids = set(
    cwd_history[
        "id_clean"
    ].dropna()
)


matched_ids = (
    movement_ids
    &
    positive_test_ids
)


unmatched_ids = (
    positive_test_ids
    -
    movement_ids
)


print(
    "\n"
    + "=" * 100
)

print(
    "MOVEMENT / TEST ID MATCHING"
)

print(
    "=" * 100
)


print(
    "\nPositive-test deer:",
    len(
        positive_test_ids
    )
)

print(
    "Matched to movement data:",
    len(
        matched_ids
    )
)

print(
    "Not matched:",
    len(
        unmatched_ids
    )
)


if unmatched_ids:

    print(
        "\nUnmatched test IDs:"
    )

    print(
        sorted(
            unmatched_ids
        )
    )


# =============================================================================
# MERGE TEST HISTORY INTO MOVEMENT DATA
# =============================================================================

gps_pos = all_raw.merge(

    cwd_history,

    on="id_clean",

    how="inner"
)


# =============================================================================
# RETAIN CONFIRMED CONVERTERS
# =============================================================================

if CONVERTERS_ONLY:

    gps_pos = gps_pos[
        gps_pos[
            "confirmed_converter"
        ]
    ].copy()


print(
    "\n"
    + "=" * 100
)

print(
    "LONGITUDINAL POSITIVE COHORT"
)

print(
    "=" * 100
)


print(
    "\nMovement rows:",
    f"{len(gps_pos):,}"
)

print(
    "Deer:",
    gps_pos[
        "id_clean"
    ].nunique()
)


# =============================================================================
# DAYS RELATIVE TO FIRST DETECTED POSITIVE TEST
# =============================================================================

gps_pos[
    "days_from_first_positive"
] = (

    gps_pos[
        "gps_datetime"
    ]

    -

    gps_pos[
        "first_positive_date"
    ]

).dt.total_seconds() / 86400.0


# =============================================================================
# ASSIGN PRE / POST PHASE
# =============================================================================

gps_pos[
    "phase"
] = pd.NA


# -----------------------------------------------------------------------------
# PRE PERIOD
# -----------------------------------------------------------------------------

if PRE_MODE == "after_last_negative":

    pre_mask = (

        gps_pos[
            "last_negative_date"
        ].notna()

        &

        gps_pos[
            "gps_datetime"
        ].ge(
            gps_pos[
                "last_negative_date"
            ]
        )

        &

        gps_pos[
            "gps_datetime"
        ].lt(

            gps_pos[
                "first_positive_date"
            ]

            -

            pd.to_timedelta(
                BUFFER_DAYS,
                unit="D"
            )
        )
    )


elif PRE_MODE == "all_before_positive":

    pre_mask = (

        gps_pos[
            "gps_datetime"
        ].lt(

            gps_pos[
                "first_positive_date"
            ]

            -

            pd.to_timedelta(
                BUFFER_DAYS,
                unit="D"
            )
        )
    )


else:

    raise ValueError(
        "\nPRE_MODE must be eithe<LOCAL_PATH>"
        "    'after_last_negative'\n"
        "or\n"
        "    'all_before_positive'"
    )


gps_pos.loc[
    pre_mask,
    "phase"
] = "PRE"


# -----------------------------------------------------------------------------
# POST PERIOD
# -----------------------------------------------------------------------------

post_mask = (

    gps_pos[
        "gps_datetime"
    ].gt(

        gps_pos[
            "first_positive_date"
        ]

        +

        pd.to_timedelta(
            BUFFER_DAYS,
            unit="D"
        )
    )
)


gps_pos.loc[
    post_mask,
    "phase"
] = "POST"


# =============================================================================
# KEEP PRE / POST RECORDS
# =============================================================================

phase_data = gps_pos[
    gps_pos[
        "phase"
    ].isin(
        [
            "PRE",
            "POST"
        ]
    )
].copy()


# =============================================================================
# TELEMETRY COVERAGE BY DEER AND PHASE
# =============================================================================

coverage = (

    phase_data

    .groupby(
        [
            "id_clean",
            "phase"
        ]
    )

    .agg(

        n_rows=(
            "gps_datetime",
            "size"
        ),

        n_days=(
            "gps_date",
            "nunique"
        ),

        first_datetime=(
            "gps_datetime",
            "min"
        ),

        last_datetime=(
            "gps_datetime",
            "max"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 110
)

print(
    "PRE / POST TELEMETRY COVERAGE"
)

print(
    "=" * 110
)


print(
    coverage
    .sort_values(
        [
            "id_clean",
            "phase"
        ]
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# REQUIRE MINIMUM COVERAGE
# =============================================================================

valid_phases = coverage[
    coverage[
        "n_days"
    ] >= MIN_DAYS_PER_PHASE
][
    [
        "id_clean",
        "phase"
    ]
].copy()


phase_data = phase_data.merge(

    valid_phases,

    on=[
        "id_clean",
        "phase"
    ],

    how="inner"
)


# =============================================================================
# RETAIN ONLY DEER WITH BOTH PRE AND POST DATA
# =============================================================================

n_phases_per_deer = (

    phase_data[
        [
            "id_clean",
            "phase"
        ]
    ]

    .drop_duplicates()

    .groupby(
        "id_clean"
    )

    .size()
)


paired_ids = n_phases_per_deer[
    n_phases_per_deer == 2
].index


phase_data = phase_data[
    phase_data[
        "id_clean"
    ].isin(
        paired_ids
    )
].copy()


print(
    "\n"
    + "=" * 100
)

print(
    "FINAL PAIRED SAMPLE"
)

print(
    "=" * 100
)


print(
    "\nDeer with sufficient PRE and POST movement data:",
    len(
        paired_ids
    )
)


print(
    "\nIDs:"
)

print(
    sorted(
        paired_ids.tolist()
    )
)


# =============================================================================
# DEER-LEVEL PRE / POST AKDE VALUES
#
# ONE VALUE PER DEER PER PHASE.
#
# We use the MEDIAN because AKDE estimates can be strongly right-skewed.
# =============================================================================

deer_phase = (

    phase_data

    .groupby(
        [
            "id_clean",
            "phase"
        ],
        as_index=False
    )

    .agg(

        akde_low=(
            "akde_low",
            "median"
        ),

        akde_est=(
            "akde_est",
            "median"
        ),

        akde_high=(
            "akde_high",
            "median"
        ),

        n_rows=(
            "gps_datetime",
            "size"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )
)


deer_phase = deer_phase.merge(

    cwd_history[
        [
            "id_clean",
            "last_negative_date",
            "first_positive_date",
            "neg_to_pos_interval_days"
        ]
    ],

    on="id_clean",

    how="left"
)


print(
    "\n"
    + "=" * 110
)

print(
    "DEER-LEVEL PRE / POST AKDE VALUES"
)

print(
    "=" * 110
)


print(
    deer_phase
    .sort_values(
        [
            "id_clean",
            "phase"
        ]
    )
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# CREATE PAIRED PRE / POST TABLE
# =============================================================================

paired_results = []


for feature in AKDE_FEATURES:

    temp = (

        deer_phase[
            [
                "id_clean",
                "phase",
                feature
            ]
        ]

        .pivot(
            index="id_clean",
            columns="phase",
            values=feature
        )

        .reset_index()
    )


    if (
        "PRE" not in temp.columns
        or
        "POST" not in temp.columns
    ):

        continue


    temp = temp.dropna(
        subset=[
            "PRE",
            "POST"
        ]
    ).copy()


    temp[
        "feature"
    ] = feature


    temp[
        "absolute_change"
    ] = (

        temp[
            "POST"
        ]

        -

        temp[
            "PRE"
        ]
    )


    temp[
        "percent_change"
    ] = np.where(

        temp[
            "PRE"
        ] != 0,

        (

            temp[
                "POST"
            ]

            -

            temp[
                "PRE"
            ]

        )

        /

        temp[
            "PRE"
        ]

        *

        100,

        np.nan
    )


    temp[
        "direction"
    ] = np.select(

        [
            temp[
                "POST"
            ]
            <
            temp[
                "PRE"
            ],

            temp[
                "POST"
            ]
            >
            temp[
                "PRE"
            ]
        ],

        [
            "DECREASED",
            "INCREASED"
        ],

        default="NO CHANGE"
    )


    paired_results.append(
        temp
    )


if len(paired_results) == 0:

    raise ValueError(
        "\nNo paired PRE/POST AKDE comparisons could be created."
    )


paired_changes = pd.concat(

    paired_results,

    ignore_index=True
)


paired_changes = paired_changes.merge(

    cwd_history[
        [
            "id_clean",
            "last_negative_date",
            "first_positive_date",
            "neg_to_pos_interval_days"
        ]
    ],

    on="id_clean",

    how="left"
)


# =============================================================================
# INDIVIDUAL AKDE_EST RESULTS
# =============================================================================

akde_est_individual = paired_changes[
    paired_changes[
        "feature"
    ] == "akde_est"
].copy()


print(
    "\n"
    + "=" * 120
)

print(
    "INDIVIDUAL CHANGE IN MEDIAN AKDE ESTIMATE"
)

print(
    "=" * 120
)


print(
    akde_est_individual[
        [
            "id_clean",
            "last_negative_date",
            "first_positive_date",
            "PRE",
            "POST",
            "absolute_change",
            "percent_change",
            "direction"
        ]
    ]

    .sort_values(
        "percent_change"
    )

    .round(
        3
    )

    .to_string(
        index=False
    )
)


# =============================================================================
# STATISTICAL SUMMARY
# =============================================================================

summary_rows = []


for feature in AKDE_FEATURES:

    temp = paired_changes[
        paired_changes[
            "feature"
        ] == feature
    ].copy()


    pre = temp[
        "PRE"
    ].to_numpy(
        dtype=float
    )


    post = temp[
        "POST"
    ].to_numpy(
        dtype=float
    )


    differences = (
        post
        -
        pre
    )


    n_deer = len(
        temp
    )


    n_decreased = int(
        (
            post
            <
            pre
        ).sum()
    )


    n_increased = int(
        (
            post
            >
            pre
        ).sum()
    )


    n_ties = int(
        (
            post
            ==
            pre
        ).sum()
    )


    # -------------------------------------------------------------------------
    # Paired Wilcoxon signed-rank test
    # -------------------------------------------------------------------------

    if (
        n_deer >= 2
        and
        not np.allclose(
            differences,
            0
        )
    ):

        wilcox = wilcoxon(

            post,
            pre,

            alternative="two-sided"
        )


        wilcoxon_W = (
            wilcox.statistic
        )


        wilcoxon_p = (
            wilcox.pvalue
        )

    else:

        wilcoxon_W = np.nan
        wilcoxon_p = np.nan


    # -------------------------------------------------------------------------
    # Directional sign test
    #
    # Among non-tied deer, is the number declining different from 50/50?
    # -------------------------------------------------------------------------

    n_non_ties = (
        n_decreased
        +
        n_increased
    )


    if n_non_ties > 0:

        sign_test = binomtest(

            k=n_decreased,

            n=n_non_ties,

            p=0.5,

            alternative="two-sided"
        )


        sign_test_p = (
            sign_test.pvalue
        )

    else:

        sign_test_p = np.nan


    summary_rows.append(
        {
            "feature":
                feature,

            "n_deer":
                n_deer,

            "PRE_group_median":
                np.nanmedian(
                    pre
                ),

            "POST_group_median":
                np.nanmedian(
                    post
                ),

            "group_median_pct_difference":
                (
                    (
                        np.nanmedian(
                            post
                        )
                        -
                        np.nanmedian(
                            pre
                        )
                    )
                    /
                    np.nanmedian(
                        pre
                    )
                    *
                    100
                )
                if np.nanmedian(
                    pre
                ) != 0
                else np.nan,

            "median_within_deer_pct_change":
                np.nanmedian(
                    temp[
                        "percent_change"
                    ]
                ),

            "n_decreased":
                n_decreased,

            "n_increased":
                n_increased,

            "n_ties":
                n_ties,

            "wilcoxon_W":
                wilcoxon_W,

            "wilcoxon_p":
                wilcoxon_p,

            "sign_test_p":
                sign_test_p
        }
    )


paired_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n"
    + "=" * 120
)

print(
    "PAIRED PRE-DETECTION VS POST-DETECTION SPACE-USE RESULTS"
)

print(
    "=" * 120
)


print(
    paired_summary
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# AUTOMATIC AKDE_EST INTERPRETATION
# =============================================================================

akde_est_summary = paired_summary[
    paired_summary[
        "feature"
    ] == "akde_est"
].iloc[0]


print(
    "\n"
    + "=" * 100
)

print(
    "AKDE_EST SUMMARY"
)

print(
    "=" * 100
)


print(
    "\nPaired deer:",
    int(
        akde_est_summary[
            "n_deer"
        ]
    )
)


print(
    "Median PRE AKDE:",
    round(
        akde_est_summary[
            "PRE_group_median"
        ],
        3
    )
)


print(
    "Median POST AKDE:",
    round(
        akde_est_summary[
            "POST_group_median"
        ],
        3
    )
)


print(
    "Median within-deer percent change:",
    round(
        akde_est_summary[
            "median_within_deer_pct_change"
        ],
        2
    ),
    "%"
)


print(
    "Decreased:",
    int(
        akde_est_summary[
            "n_decreased"
        ]
    )
)


print(
    "Increased:",
    int(
        akde_est_summary[
            "n_increased"
        ]
    )
)


print(
    "Wilcoxon p-value:",
    akde_est_summary[
        "wilcoxon_p"
    ]
)


print(
    "Direction/sign-test p-value:",
    akde_est_summary[
        "sign_test_p"
    ]
)


# =============================================================================
# FIGURE 1
#
# INDIVIDUAL PRE -> POST AKDE CHANGE
# =============================================================================

if len(
    akde_est_individual
) > 0:

    fig, ax = plt.subplots(
        figsize=(
            8,
            7
        )
    )


    for _, row in akde_est_individual.iterrows():

        ax.plot(

            [
                0,
                1
            ],

            [
                row[
                    "PRE"
                ],

                row[
                    "POST"
                ]
            ],

            marker="o",

            alpha=0.7
        )


    ax.set_xticks(
        [
            0,
            1
        ]
    )


    ax.set_xticklabels(
        [
            "Pre-detection",
            "Post-detection"
        ]
    )


    ax.set_ylabel(
        "Median 7-day AKDE estimate"
    )


    ax.set_title(
        "Within-deer change in space use\n"
        "relative to first detected CWD-positive test"
    )


    plt.tight_layout()


    paired_plot_path = (

        OUT_DIR /
        "AKDE_PRE_POST_INDIVIDUAL_CHANGE.png"
    )


    plt.savefig(

        paired_plot_path,

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# EVENT-TIME ANALYSIS
#
# This lets us examine the trajectory rather than collapsing everything into
# a single PRE and POST value.
# =============================================================================

EVENT_WINDOWS = [

    (
        -365,
        -181,
        "-365 to -181"
    ),

    (
        -180,
        -121,
        "-180 to -121"
    ),

    (
        -120,
        -61,
        "-120 to -61"
    ),

    (
        -60,
        -8,
        "-60 to -8"
    ),

    (
        8,
        60,
        "+8 to +60"
    ),

    (
        61,
        120,
        "+61 to +120"
    ),

    (
        121,
        180,
        "+121 to +180"
    ),

    (
        181,
        365,
        "+181 to +365"
    ),

    (
        366,
        730,
        "+366 to +730"
    )
]


gps_event = gps_pos.copy()


gps_event[
    "event_window"
] = pd.NA


for lower, upper, label in EVENT_WINDOWS:

    mask = (

        gps_event[
            "days_from_first_positive"
        ].ge(
            lower
        )

        &

        gps_event[
            "days_from_first_positive"
        ].le(
            upper
        )
    )


    gps_event.loc[
        mask,
        "event_window"
    ] = label


event_data = gps_event.dropna(
    subset=[
        "event_window",
        "akde_est"
    ]
).copy()


# =============================================================================
# ONE VALUE PER DEER PER EVENT WINDOW
# =============================================================================

deer_event = (

    event_data

    .groupby(
        [
            "id_clean",
            "event_window"
        ],
        as_index=False
    )

    .agg(

        median_akde_est=(
            "akde_est",
            "median"
        ),

        median_akde_low=(
            "akde_low",
            "median"
        ),

        median_akde_high=(
            "akde_high",
            "median"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )
)


# -----------------------------------------------------------------------------
# Require at least 7 days represented within a time bin.
# -----------------------------------------------------------------------------

deer_event = deer_event[
    deer_event[
        "n_days"
    ] >= 7
].copy()


# =============================================================================
# EVENT-TIME GROUP SUMMARY
# =============================================================================

event_summary = (

    deer_event

    .groupby(
        "event_window"
    )

    .agg(

        median_akde_est=(
            "median_akde_est",
            "median"
        ),

        q25=(
            "median_akde_est",
            lambda x:
                x.quantile(
                    0.25
                )
        ),

        q75=(
            "median_akde_est",
            lambda x:
                x.quantile(
                    0.75
                )
        ),

        mean_akde_est=(
            "median_akde_est",
            "mean"
        ),

        n_deer=(
            "id_clean",
            "nunique"
        )
    )

    .reset_index()
)


event_order = [

    label

    for _, _, label
    in EVENT_WINDOWS
]


event_summary[
    "event_window"
] = pd.Categorical(

    event_summary[
        "event_window"
    ],

    categories=event_order,

    ordered=True
)


event_summary = (

    event_summary

    .sort_values(
        "event_window"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 110
)

print(
    "AKDE THROUGH TIME RELATIVE TO FIRST DETECTED CWD+ TEST"
)

print(
    "=" * 110
)


print(
    event_summary
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# FIGURE 2
#
# EVENT-TIME TRAJECTORY
# =============================================================================

if len(
    event_summary
) > 0:

    x = np.arange(
        len(
            event_summary
        )
    )


    y = event_summary[
        "median_akde_est"
    ].to_numpy()


    q25 = event_summary[
        "q25"
    ].to_numpy()


    q75 = event_summary[
        "q75"
    ].to_numpy()


    lower_error = (
        y
        -
        q25
    )


    upper_error = (
        q75
        -
        y
    )


    fig, ax = plt.subplots(
        figsize=(
            12,
            7
        )
    )


    ax.errorbar(

        x,
        y,

        yerr=np.vstack(
            [
                lower_error,
                upper_error
            ]
        ),

        marker="o",

        capsize=5
    )


    # First positive test occurs between PRE and POST bins.

    ax.axvline(
        3.5,
        linestyle="--"
    )


    ax.set_xticks(
        x
    )


    ax.set_xticklabels(

        event_summary[
            "event_window"
        ].astype(str),

        rotation=45,

        ha="right"
    )


    ax.set_xlabel(
        "Days relative to first detected CWD-positive test"
    )


    ax.set_ylabel(
        "Median deer-level 7-day AKDE estimate"
    )


    ax.set_title(
        "Space-use trajectory around first detected CWD positivity"
    )


    plt.tight_layout()


    event_plot_path = (

        OUT_DIR /
        "AKDE_EVENT_TIME_FIRST_CWD_POSITIVE.png"
    )


    plt.savefig(

        event_plot_path,

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# OPTIONAL SAME-SEASON DIAGNOSTIC
#
# This is important because a PRE/POST difference could otherwise simply
# reflect seasonal changes in deer movement.
# =============================================================================

season_col = find_column(

    phase_data,

    candidates=[
        "season",
        "Season"
    ]
)


if season_col is not None:

    print(
        "\n"
        + "=" * 110
    )

    print(
        "SAME-SEASON PRE / POST DIAGNOSTIC"
    )

    print(
        "=" * 110
    )


    deer_season_phase = (

        phase_data

        .dropna(
            subset=[
                season_col
            ]
        )

        .groupby(
            [
                "id_clean",
                season_col,
                "phase"
            ],
            as_index=False
        )

        .agg(

            median_akde_est=(
                "akde_est",
                "median"
            ),

            n_days=(
                "gps_date",
                "nunique"
            )
        )
    )


    deer_season_phase = deer_season_phase[
        deer_season_phase[
            "n_days"
        ] >= 7
    ].copy()


    season_paired = (

        deer_season_phase

        .pivot_table(

            index=[
                "id_clean",
                season_col
            ],

            columns="phase",

            values="median_akde_est"
        )

        .reset_index()
    )


    if (
        "PRE" in season_paired.columns
        and
        "POST" in season_paired.columns
    ):

        season_paired = season_paired.dropna(
            subset=[
                "PRE",
                "POST"
            ]
        ).copy()


        season_paired[
            "absolute_change"
        ] = (

            season_paired[
                "POST"
            ]

            -

            season_paired[
                "PRE"
            ]
        )


        season_paired[
            "percent_change"
        ] = np.where(

            season_paired[
                "PRE"
            ] != 0,

            (

                season_paired[
                    "POST"
                ]

                -

                season_paired[
                    "PRE"
                ]

            )

            /

            season_paired[
                "PRE"
            ]

            *

            100,

            np.nan
        )


        season_paired[
            "direction"
        ] = np.select(

            [
                season_paired[
                    "POST"
                ]
                <
                season_paired[
                    "PRE"
                ],

                season_paired[
                    "POST"
                ]
                >
                season_paired[
                    "PRE"
                ]
            ],

            [
                "DECREASED",
                "INCREASED"
            ],

            default="NO CHANGE"
        )


        print(
            season_paired
            .sort_values(
                [
                    season_col,
                    "percent_change"
                ]
            )
            .round(
                3
            )
            .to_string(
                index=False
            )
        )


        print(
            "\nSame-season pairs:",
            len(
                season_paired
            )
        )


        print(
            "Same-season decreases:",
            int(
                (
                    season_paired[
                        "direction"
                    ]
                    ==
                    "DECREASED"
                ).sum()
            )
        )


        print(
            "Same-season increases:",
            int(
                (
                    season_paired[
                        "direction"
                    ]
                    ==
                    "INCREASED"
                ).sum()
            )
        )


        print(
            "Median same-season within-pair % change:",
            round(
                season_paired[
                    "percent_change"
                ].median(),
                2
            ),
            "%"
        )


    else:

        season_paired = pd.DataFrame()

        print(
            "\nNo same-season PRE/POST pairs were available."
        )


else:

    season_paired = pd.DataFrame()

    print(
        "\nNo season column was found. "
        "Skipping same-season diagnostic."
    )


# =============================================================================
# SAVE OUTPUTS
# =============================================================================

cwd_tests.to_csv(

    OUT_DIR /
    "01_CWD_TEST_RECORDS_CLEANED.csv",

    index=False
)


cwd_history.to_csv(

    OUT_DIR /
    "02_CWD_TEST_HISTORY_BY_DEER.csv",

    index=False
)


coverage.to_csv(

    OUT_DIR /
    "03_PRE_POST_TELEMETRY_COVERAGE.csv",

    index=False
)


deer_phase.to_csv(

    OUT_DIR /
    "04_PRE_POST_AKDE_DEER_LEVEL.csv",

    index=False
)


paired_changes.to_csv(

    OUT_DIR /
    "05_PRE_POST_AKDE_INDIVIDUAL_CHANGES.csv",

    index=False
)


paired_summary.to_csv(

    OUT_DIR /
    "06_PRE_POST_AKDE_SUMMARY.csv",

    index=False
)


deer_event.to_csv(

    OUT_DIR /
    "07_AKDE_EVENT_TIME_DEER_LEVEL.csv",

    index=False
)


event_summary.to_csv(

    OUT_DIR /
    "08_AKDE_EVENT_TIME_SUMMARY.csv",

    index=False
)


if len(
    season_paired
) > 0:

    season_paired.to_csv(

        OUT_DIR /
        "09_AKDE_SAME_SEASON_PRE_POST.csv",

        index=False
    )


# =============================================================================
# FINAL MESSAGE
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ANALYSIS COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nOutputs saved to:"
)

print(
    OUT_DIR
)


print(
    "\nThe three most important outputs to inspect are:"
)

print(
    "\n1. CWD+ DEER TEST HISTORIES"
)

print(
    "2. INDIVIDUAL CHANGE IN MEDIAN AKDE ESTIMATE"
)

print(
    "3. PAIRED PRE-DETECTION VS POST-DETECTION SPACE-USE RESULTS"
)


print(
    "\nAlso inspect:"
)

print(
    "4. AKDE THROUGH TIME RELATIVE TO FIRST DETECTED CWD+ TEST"
)

print(
    "5. SAME-SEASON PRE / POST DIAGNOSTIC"
)

In [ ]:
# =============================================================================
# HOME-RANGE TRAJECTORY FOR DEER THAT WERE CWD+ AT FIRST AVAILABLE TEST
# =============================================================================
#
# PURPOSE
# -----------------------------------------------------------------------------
# Some deer do NOT have a documented CWD- test before their first CWD+ test.
#
# These deer therefore cannot be included in the confirmed:
#
#       CWD-  ->  CWD+
#
# converter analysis.
#
# Instead, this analysis asks:
#
#   Among deer that were already CWD+ at their first recorded test,
#   how does home range / space use change through time?
#
#
# IMPORTANT:
#
# Telemetry BEFORE the first recorded positive test is NOT labeled CWD-.
#
# It is labeled:
#
#       PRE-POSITIVE TEST / STATUS UNKNOWN
#
# Telemetry AFTER the first positive test is known to occur after detection.
#
# =============================================================================


# =============================================================================
# IDENTIFY DEER THAT WERE POSITIVE AT FIRST AVAILABLE TEST
# =============================================================================

initial_positive_history = cwd_history[
    ~cwd_history[
        "confirmed_converter"
    ]
].copy()


initial_positive_ids = set(
    initial_positive_history[
        "id_clean"
    ]
)


print(
    "\n"
    + "=" * 100
)

print(
    "DEER POSITIVE AT FIRST AVAILABLE CWD TEST"
)

print(
    "=" * 100
)


print(
    f"\nNumber of initially CWD+ deer: "
    f"{len(initial_positive_ids)}"
)


print(
    "\nIDs:"
)

print(
    sorted(
        initial_positive_ids
    )
)


print(
    "\nTest history:"
)

print(
    initial_positive_history[
        [
            "id_clean",
            "first_positive_date",
            "n_positive_tests"
        ]
    ]
    .sort_values(
        "first_positive_date"
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# EXTRACT THEIR MOVEMENT DATA
# =============================================================================

initial_pos_gps = all_raw[
    all_raw[
        "id_clean"
    ].isin(
        initial_positive_ids
    )
].copy()


# Attach first positive test date
initial_pos_gps = initial_pos_gps.merge(

    initial_positive_history[
        [
            "id_clean",
            "first_positive_date"
        ]
    ],

    on="id_clean",

    how="left"
)


print(
    "\nMovement rows:",
    f"{len(initial_pos_gps):,}"
)


print(
    "Deer represented in movement data:",
    initial_pos_gps[
        "id_clean"
    ].nunique()
)


# =============================================================================
# TIME RELATIVE TO FIRST POSITIVE TEST
# =============================================================================

initial_pos_gps[
    "days_from_first_positive"
] = (

    initial_pos_gps[
        "gps_datetime"
    ]

    -

    initial_pos_gps[
        "first_positive_date"
    ]

).dt.total_seconds() / 86400.0


# =============================================================================
# BASIC COVERAGE CHECK
# =============================================================================

coverage_initial_pos = (

    initial_pos_gps

    .groupby(
        "id_clean"
    )

    .agg(

        first_gps=(
            "gps_datetime",
            "min"
        ),

        last_gps=(
            "gps_datetime",
            "max"
        ),

        first_positive_date=(
            "first_positive_date",
            "first"
        ),

        n_rows=(
            "gps_datetime",
            "size"
        ),

        n_days=(
            "gps_date",
            "nunique"
        ),

        min_days_relative=(
            "days_from_first_positive",
            "min"
        ),

        max_days_relative=(
            "days_from_first_positive",
            "max"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 110
)

print(
    "TELEMETRY COVERAGE FOR INITIALLY CWD+ DEER"
)

print(
    "=" * 110
)


print(
    coverage_initial_pos
    .round(
        1
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# CHECK WHETHER THESE DEER HAVE TELEMETRY BEFORE FIRST POSITIVE TEST
# =============================================================================

initial_pos_gps[
    "test_period"
] = np.where(

    initial_pos_gps[
        "days_from_first_positive"
    ] < 0,

    "BEFORE_FIRST_POSITIVE_STATUS_UNKNOWN",

    "AFTER_FIRST_POSITIVE"
)


period_counts = (

    initial_pos_gps

    .groupby(
        [
            "id_clean",
            "test_period"
        ]
    )

    .agg(

        n_rows=(
            "gps_datetime",
            "size"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 110
)

print(
    "MOVEMENT DATA BEFORE / AFTER FIRST POSITIVE TEST"
)

print(
    "=" * 110
)


print(
    period_counts
    .to_string(
        index=False
    )
)


# =============================================================================
# EVENT-TIME WINDOWS
#
# These let us follow home range through time.
#
# Negative windows:
#     movement occurred before first positive test,
#     but disease status is UNKNOWN.
#
# Positive windows:
#     movement occurred after the first documented positive test.
# =============================================================================

INITIAL_POS_WINDOWS = [

    (
        -365,
        -181,
        "UNKNOWN: -365 to -181"
    ),

    (
        -180,
        -91,
        "UNKNOWN: -180 to -91"
    ),

    (
        -90,
        -8,
        "UNKNOWN: -90 to -8"
    ),

    (
        8,
        60,
        "CWD+: +8 to +60"
    ),

    (
        61,
        120,
        "CWD+: +61 to +120"
    ),

    (
        121,
        180,
        "CWD+: +121 to +180"
    ),

    (
        181,
        365,
        "CWD+: +181 to +365"
    ),

    (
        366,
        730,
        "CWD+: +366 to +730"
    )
]


initial_pos_gps[
    "event_window"
] = pd.NA


for lower, upper, label in INITIAL_POS_WINDOWS:

    mask = (

        initial_pos_gps[
            "days_from_first_positive"
        ].ge(
            lower
        )

        &

        initial_pos_gps[
            "days_from_first_positive"
        ].le(
            upper
        )
    )


    initial_pos_gps.loc[
        mask,
        "event_window"
    ] = label


# =============================================================================
# RETAIN EVENT-WINDOW OBSERVATIONS
# =============================================================================

event_initial_pos = initial_pos_gps.dropna(
    subset=[
        "event_window",
        "akde_est"
    ]
).copy()


# =============================================================================
# ONE HOME-RANGE VALUE PER DEER PER TIME WINDOW
#
# Again, use MEDIAN AKDE so each deer gets equal weight.
# =============================================================================

deer_event_initial_pos = (

    event_initial_pos

    .groupby(
        [
            "id_clean",
            "event_window"
        ],
        as_index=False
    )

    .agg(

        median_akde_low=(
            "akde_low",
            "median"
        ),

        median_akde_est=(
            "akde_est",
            "median"
        ),

        median_akde_high=(
            "akde_high",
            "median"
        ),

        mean_akde_est=(
            "akde_est",
            "mean"
        ),

        n_rows=(
            "gps_datetime",
            "size"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )
)


# Require at least 7 days within an event bin
deer_event_initial_pos = deer_event_initial_pos[
    deer_event_initial_pos[
        "n_days"
    ] >= 7
].copy()


print(
    "\n"
    + "=" * 120
)

print(
    "INDIVIDUAL HOME-RANGE VALUES THROUGH TIME — INITIALLY CWD+ DEER"
)

print(
    "=" * 120
)


print(
    deer_event_initial_pos
    .sort_values(
        [
            "id_clean",
            "event_window"
        ]
    )
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# GROUP-LEVEL EVENT-TIME SUMMARY
#
# This gives:
#
#      median home range across deer
#
# at each point relative to first positive testing.
# =============================================================================

initial_pos_event_summary = (

    deer_event_initial_pos

    .groupby(
        "event_window"
    )

    .agg(

        n_deer=(
            "id_clean",
            "nunique"
        ),

        median_akde_est=(
            "median_akde_est",
            "median"
        ),

        mean_akde_est=(
            "median_akde_est",
            "mean"
        ),

        q25=(
            "median_akde_est",
            lambda x:
                x.quantile(
                    0.25
                )
        ),

        q75=(
            "median_akde_est",
            lambda x:
                x.quantile(
                    0.75
                )
        )
    )

    .reset_index()
)


window_order = [
    label
    for _, _, label
    in INITIAL_POS_WINDOWS
]


initial_pos_event_summary[
    "event_window"
] = pd.Categorical(

    initial_pos_event_summary[
        "event_window"
    ],

    categories=window_order,

    ordered=True
)


initial_pos_event_summary = (

    initial_pos_event_summary

    .sort_values(
        "event_window"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 120
)

print(
    "HOME-RANGE TRAJECTORY — INITIALLY CWD+ DEER"
)

print(
    "=" * 120
)


print(
    initial_pos_event_summary
    .round(
        3
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# POST-POSITIVE EARLY VS LATE ANALYSIS
#
# For these deer, this is the cleanest disease-status comparison.
#
# EARLY POST:
#
#       +8 to +90 days
#
# LATER POST:
#
#       +91 to +365 days
#
# Both periods are AFTER documented positivity.
# =============================================================================

initial_pos_gps[
    "post_stage"
] = pd.NA


early_post_mask = (

    initial_pos_gps[
        "days_from_first_positive"
    ].ge(
        8
    )

    &

    initial_pos_gps[
        "days_from_first_positive"
    ].le(
        90
    )
)


later_post_mask = (

    initial_pos_gps[
        "days_from_first_positive"
    ].ge(
        91
    )

    &

    initial_pos_gps[
        "days_from_first_positive"
    ].le(
        365
    )
)


initial_pos_gps.loc[
    early_post_mask,
    "post_stage"
] = "EARLY_POST_POSITIVE"


initial_pos_gps.loc[
    later_post_mask,
    "post_stage"
] = "LATER_POST_POSITIVE"


post_stage_data = initial_pos_gps.dropna(
    subset=[
        "post_stage",
        "akde_est"
    ]
).copy()


# =============================================================================
# COVERAGE FOR POST-POSITIVE STAGES
# =============================================================================

post_stage_coverage = (

    post_stage_data

    .groupby(
        [
            "id_clean",
            "post_stage"
        ]
    )

    .agg(

        n_days=(
            "gps_date",
            "nunique"
        ),

        n_rows=(
            "gps_datetime",
            "size"
        )
    )

    .reset_index()
)


# Require at least 20 days represented in each phase
valid_post_stages = post_stage_coverage[
    post_stage_coverage[
        "n_days"
    ] >= 20
][
    [
        "id_clean",
        "post_stage"
    ]
]


post_stage_data = post_stage_data.merge(

    valid_post_stages,

    on=[
        "id_clean",
        "post_stage"
    ],

    how="inner"
)


# =============================================================================
# ONE VALUE PER DEER PER POST-POSITIVE STAGE
# =============================================================================

deer_post_stage = (

    post_stage_data

    .groupby(
        [
            "id_clean",
            "post_stage"
        ],
        as_index=False
    )

    .agg(

        akde_low=(
            "akde_low",
            "median"
        ),

        akde_est=(
            "akde_est",
            "median"
        ),

        akde_high=(
            "akde_high",
            "median"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )
)


# =============================================================================
# CREATE PAIRED EARLY/LATE TABLE
# =============================================================================

post_paired = (

    deer_post_stage[
        [
            "id_clean",
            "post_stage",
            "akde_est"
        ]
    ]

    .pivot(
        index="id_clean",
        columns="post_stage",
        values="akde_est"
    )

    .reset_index()
)


if (
    "EARLY_POST_POSITIVE"
    in post_paired.columns

    and

    "LATER_POST_POSITIVE"
    in post_paired.columns
):

    post_paired = post_paired.dropna(
        subset=[
            "EARLY_POST_POSITIVE",
            "LATER_POST_POSITIVE"
        ]
    ).copy()


    post_paired[
        "absolute_change"
    ] = (

        post_paired[
            "LATER_POST_POSITIVE"
        ]

        -

        post_paired[
            "EARLY_POST_POSITIVE"
        ]
    )


    post_paired[
        "percent_change"
    ] = (

        (
            post_paired[
                "LATER_POST_POSITIVE"
            ]

            -

            post_paired[
                "EARLY_POST_POSITIVE"
            ]
        )

        /

        post_paired[
            "EARLY_POST_POSITIVE"
        ]

        *

        100
    )


    post_paired[
        "direction"
    ] = np.where(

        post_paired[
            "LATER_POST_POSITIVE"
        ]

        <

        post_paired[
            "EARLY_POST_POSITIVE"
        ],

        "DECREASED",

        "INCREASED"
    )


    print(
        "\n"
        + "=" * 120
    )

    print(
        "EARLY VS LATER POST-POSITIVE HOME RANGE"
    )

    print(
        "=" * 120
    )


    print(
        post_paired
        .sort_values(
            "percent_change"
        )
        .round(
            3
        )
        .to_string(
            index=False
        )
    )


    print(
        "\nDeer with paired early/later data:",
        len(
            post_paired
        )
    )


    print(
        "Decreased home range:",
        int(
            (
                post_paired[
                    "direction"
                ]
                ==
                "DECREASED"
            ).sum()
        )
    )


    print(
        "Increased home range:",
        int(
            (
                post_paired[
                    "direction"
                ]
                ==
                "INCREASED"
            ).sum()
        )
    )


    print(
        "Median within-deer percent change:",
        round(
            post_paired[
                "percent_change"
            ].median(),
            2
        ),
        "%"
    )


    # =========================================================================
    # PAIRED WILCOXON
    # =========================================================================

    if len(
        post_paired
    ) >= 2:

        early_values = post_paired[
            "EARLY_POST_POSITIVE"
        ].to_numpy()


        later_values = post_paired[
            "LATER_POST_POSITIVE"
        ].to_numpy()


        differences = (
            later_values
            -
            early_values
        )


        if not np.allclose(
            differences,
            0
        ):

            test = wilcoxon(

                later_values,
                early_values,

                alternative="two-sided"
            )


            print(
                "\nWilcoxon signed-rank test:"
            )

            print(
                "W =",
                round(
                    test.statistic,
                    3
                )
            )

            print(
                "p =",
                round(
                    test.pvalue,
                    6
                )
            )


else:

    post_paired = pd.DataFrame()

    print(
        "\nInsufficient paired early/later post-positive data."
    )


# =============================================================================
# INDIVIDUAL POST-POSITIVE TREND
#
# Fit a simple linear trend in AKDE over time AFTER positive testing.
#
# Negative slope:
#
#       home range decreases through time
#
# Positive slope:
#
#       home range increases through time
#
# We use deer-level 30-day bins first so individual deer with large numbers
# of fixes do not gain excessive influence.
# =============================================================================

post_only = initial_pos_gps[
    initial_pos_gps[
        "days_from_first_positive"
    ].ge(
        8
    )
].copy()


# Limit to first year after positivity for direct comparability
post_only = post_only[
    post_only[
        "days_from_first_positive"
    ].le(
        365
    )
].copy()


# =============================================================================
# CREATE 30-DAY BINS
# =============================================================================

post_only[
    "post_30day_bin"
] = (

    (
        post_only[
            "days_from_first_positive"
        ]

        //

        30
    )

    .astype(
        int
    )
)


deer_30day = (

    post_only

    .groupby(
        [
            "id_clean",
            "post_30day_bin"
        ],
        as_index=False
    )

    .agg(

        median_days=(
            "days_from_first_positive",
            "median"
        ),

        median_akde_est=(
            "akde_est",
            "median"
        ),

        n_days=(
            "gps_date",
            "nunique"
        )
    )
)


# Need reasonable representation inside bin
deer_30day = deer_30day[
    deer_30day[
        "n_days"
    ] >= 7
].copy()


# =============================================================================
# FIT SLOPE FOR EACH DEER
# =============================================================================

trend_rows = []


for deer_id, deer_data in deer_30day.groupby(
    "id_clean"
):

    deer_data = deer_data.sort_values(
        "median_days"
    )


    # Need at least 3 time bins
    if len(
        deer_data
    ) < 3:

        continue


    x = deer_data[
        "median_days"
    ].to_numpy()


    y = deer_data[
        "median_akde_est"
    ].to_numpy()


    slope, intercept = np.polyfit(
        x,
        y,
        1
    )


    # Convert daily slope to 30-day change
    slope_per_30_days = (
        slope
        *
        30
    )


    first_bin_akde = y[0]

    if first_bin_akde != 0:

        pct_change_per_30_days = (

            slope_per_30_days

            /

            first_bin_akde

            *

            100
        )

    else:

        pct_change_per_30_days = np.nan


    trend_rows.append(
        {
            "id_clean":
                deer_id,

            "n_30day_bins":
                len(
                    deer_data
                ),

            "slope_per_day":
                slope,

            "slope_per_30_days":
                slope_per_30_days,

            "approx_pct_change_per_30_days":
                pct_change_per_30_days,

            "trend":
                (
                    "DECREASING"
                    if slope < 0
                    else
                    "INCREASING"
                )
        }
    )


individual_trends = pd.DataFrame(
    trend_rows
)


print(
    "\n"
    + "=" * 120
)

print(
    "POST-POSITIVE HOME-RANGE TREND BY DEER"
)

print(
    "=" * 120
)


if len(
    individual_trends
) > 0:

    print(
        individual_trends
        .sort_values(
            "slope_per_30_days"
        )
        .round(
            4
        )
        .to_string(
            index=False
        )
    )


    print(
        "\nDecreasing:",
        int(
            (
                individual_trends[
                    "trend"
                ]
                ==
                "DECREASING"
            ).sum()
        )
    )


    print(
        "Increasing:",
        int(
            (
                individual_trends[
                    "trend"
                ]
                ==
                "INCREASING"
            ).sum()
        )
    )


    print(
        "\nMedian AKDE change per 30 days:"
    )

    print(
        round(
            individual_trends[
                "slope_per_30_days"
            ].median(),
            3
        )
    )


# =============================================================================
# FIGURE 1
#
# GROUP HOME-RANGE TRAJECTORY RELATIVE TO FIRST POSITIVE TEST
# =============================================================================

if len(
    initial_pos_event_summary
) > 0:

    x = np.arange(
        len(
            initial_pos_event_summary
        )
    )


    y = initial_pos_event_summary[
        "median_akde_est"
    ].to_numpy()


    lower_error = (

        y

        -

        initial_pos_event_summary[
            "q25"
        ].to_numpy()
    )


    upper_error = (

        initial_pos_event_summary[
            "q75"
        ].to_numpy()

        -

        y
    )


    fig, ax = plt.subplots(
        figsize=(
            13,
            7
        )
    )


    ax.errorbar(

        x,
        y,

        yerr=np.vstack(
            [
                lower_error,
                upper_error
            ]
        ),

        marker="o",

        capsize=5
    )


    # First positive test
    ax.axvline(
        2.5,
        linestyle="--"
    )


    ax.set_xticks(
        x
    )


    ax.set_xticklabels(

        initial_pos_event_summary[
            "event_window"
        ].astype(str),

        rotation=45,

        ha="right"
    )


    ax.set_xlabel(
        "Time relative to first documented CWD-positive test"
    )


    ax.set_ylabel(
        "Median deer-level 7-day AKDE estimate"
    )


    ax.set_title(
        "Home-range trajectory of deer positive at first available CWD test"
    )


    plt.tight_layout()


    plt.savefig(

        OUT_DIR /
        "INITIAL_POSITIVE_HOME_RANGE_TRAJECTORY.png",

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# FIGURE 2
#
# INDIVIDUAL 30-DAY HOME-RANGE TRAJECTORIES AFTER POSITIVITY
# =============================================================================

if len(
    deer_30day
) > 0:

    fig, ax = plt.subplots(
        figsize=(
            11,
            7
        )
    )


    for deer_id, deer_data in deer_30day.groupby(
        "id_clean"
    ):

        deer_data = deer_data.sort_values(
            "median_days"
        )


        ax.plot(

            deer_data[
                "median_days"
            ],

            deer_data[
                "median_akde_est"
            ],

            marker="o",

            alpha=0.7,

            label=str(
                deer_id
            )
        )


    ax.set_xlabel(
        "Days after first documented CWD-positive test"
    )


    ax.set_ylabel(
        "Median 7-day AKDE estimate"
    )


    ax.set_title(
        "Individual post-positive home-range trajectories"
    )


    ax.legend(
        title="Deer ID",
        bbox_to_anchor=(
            1.05,
            1
        ),
        loc="upper left"
    )


    plt.tight_layout()


    plt.savefig(

        OUT_DIR /
        "INITIAL_POSITIVE_INDIVIDUAL_HOME_RANGE_TRAJECTORIES.png",

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# SAVE RESULTS
# =============================================================================

coverage_initial_pos.to_csv(

    OUT_DIR /
    "INITIAL_POSITIVE_TELEMETRY_COVERAGE.csv",

    index=False
)


deer_event_initial_pos.to_csv(

    OUT_DIR /
    "INITIAL_POSITIVE_EVENT_TIME_DEER_LEVEL.csv",

    index=False
)


initial_pos_event_summary.to_csv(

    OUT_DIR /
    "INITIAL_POSITIVE_EVENT_TIME_SUMMARY.csv",

    index=False
)


deer_post_stage.to_csv(

    OUT_DIR /
    "INITIAL_POSITIVE_EARLY_LATE_HOME_RANGE.csv",

    index=False
)


if len(
    post_paired
) > 0:

    post_paired.to_csv(

        OUT_DIR /
        "INITIAL_POSITIVE_EARLY_LATE_PAIRED_CHANGES.csv",

        index=False
    )


deer_30day.to_csv(

    OUT_DIR /
    "INITIAL_POSITIVE_30DAY_HOME_RANGE.csv",

    index=False
)


if len(
    individual_trends
) > 0:

    individual_trends.to_csv(

        OUT_DIR /
        "INITIAL_POSITIVE_HOME_RANGE_TRENDS.csv",

        index=False
    )


print(
    "\n"
    + "=" * 100
)

print(
    "INITIALLY-POSITIVE HOME-RANGE ANALYSIS COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nResults saved to:"
)

print(
    OUT_DIR
)

In [ ]:
# =============================================================================
# DIAGNOSTIC: INSPECT THE THREE DEER WITH EXTREME POSITIVE AKDE TRENDS
# =============================================================================

CHECK_IDS = [
    "F30",
    "F41",
    "A43"
]


# =============================================================================
# 1. SHOW THEIR 30-DAY BIN VALUES
# =============================================================================

check_bins = (

    deer_30day[
        deer_30day[
            "id_clean"
        ].isin(
            CHECK_IDS
        )
    ]

    .sort_values(
        [
            "id_clean",
            "median_days"
        ]
    )

    .copy()
)


print(
    "\n"
    + "=" * 100
)

print(
    "30-DAY AKDE VALUES FOR EXTREME IDS"
)

print(
    "=" * 100
)


print(
    check_bins
    .round(6)
    .to_string(
        index=False
    )
)


# =============================================================================
# 2. SUMMARIZE FIRST, LAST, MINIMUM, MAXIMUM AKDE
# =============================================================================

diagnostic_rows = []


for deer_id, deer_data in check_bins.groupby(
    "id_clean"
):

    deer_data = deer_data.sort_values(
        "median_days"
    )


    first_row = deer_data.iloc[0]

    last_row = deer_data.iloc[-1]


    first_akde = first_row[
        "median_akde_est"
    ]


    last_akde = last_row[
        "median_akde_est"
    ]


    absolute_change = (
        last_akde
        -
        first_akde
    )


    if first_akde != 0:

        actual_first_to_last_pct = (
            absolute_change
            /
            first_akde
            *
            100
        )

    else:

        actual_first_to_last_pct = np.nan


    diagnostic_rows.append(
        {
            "id_clean":
                deer_id,

            "n_bins":
                len(
                    deer_data
                ),

            "first_days":
                first_row[
                    "median_days"
                ],

            "first_akde":
                first_akde,

            "last_days":
                last_row[
                    "median_days"
                ],

            "last_akde":
                last_akde,

            "minimum_akde":
                deer_data[
                    "median_akde_est"
                ].min(),

            "maximum_akde":
                deer_data[
                    "median_akde_est"
                ].max(),

            "absolute_first_to_last_change":
                absolute_change,

            "actual_first_to_last_pct":
                actual_first_to_last_pct
        }
    )


extreme_diagnostics = pd.DataFrame(
    diagnostic_rows
)


print(
    "\n"
    + "=" * 100
)

print(
    "EXTREME-ID AKDE DIAGNOSTICS"
)

print(
    "=" * 100
)


print(
    extreme_diagnostics
    .round(6)
    .to_string(
        index=False
    )
)


# =============================================================================
# 3. SHOW THE SLOPE CALCULATION THAT GENERATED THE HUGE PERCENTAGES
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TREND CALCULATION"
)

print(
    "=" * 100
)


print(
    individual_trends[
        individual_trends[
            "id_clean"
        ].isin(
            CHECK_IDS
        )
    ]
    .sort_values(
        "id_clean"
    )
    .round(6)
    .to_string(
        index=False
    )
)


# =============================================================================
# 4. INSPECT ORIGINAL ROW-LEVEL AKDE VALUES
# =============================================================================

raw_check = (

    initial_pos_gps[
        initial_pos_gps[
            "id_clean"
        ].isin(
            CHECK_IDS
        )
    ][
        [
            "id_clean",
            "gps_datetime",
            "days_from_first_positive",
            "akde_low",
            "akde_est",
            "akde_high"
        ]
    ]

    .sort_values(
        [
            "id_clean",
            "gps_datetime"
        ]
    )

    .copy()
)


print(
    "\n"
    + "=" * 100
)

print(
    "RAW AKDE SUMMARY BY ID"
)

print(
    "=" * 100
)


raw_summary = (

    raw_check

    .groupby(
        "id_clean"
    )

    .agg(

        n_rows=(
            "akde_est",
            "size"
        ),

        akde_min=(
            "akde_est",
            "min"
        ),

        akde_q01=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.01
                )
        ),

        akde_q05=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.05
                )
        ),

        akde_median=(
            "akde_est",
            "median"
        ),

        akde_q95=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.95
                )
        ),

        akde_q99=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.99
                )
        ),

        akde_max=(
            "akde_est",
            "max"
        )
    )

    .reset_index()
)


print(
    raw_summary
    .round(6)
    .to_string(
        index=False
    )
)


# =============================================================================
# 5. CHECK WHETHER LOW <= EST <= HIGH
#
# This should generally hold for valid AKDE confidence intervals.
# =============================================================================

raw_check[
    "ci_order_valid"
] = (

    (
        raw_check[
            "akde_low"
        ]
        <=
        raw_check[
            "akde_est"
        ]
    )

    &

    (
        raw_check[
            "akde_est"
        ]
        <=
        raw_check[
            "akde_high"
        ]
    )
)


ci_check = (

    raw_check

    .groupby(
        "id_clean"
    )

    .agg(

        rows=(
            "ci_order_valid",
            "size"
        ),

        valid_ci_rows=(
            "ci_order_valid",
            "sum"
        ),

        invalid_ci_rows=(
            "ci_order_valid",
            lambda x:
                (~x).sum()
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "AKDE CONFIDENCE-INTERVAL ORDER CHECK"
)

print(
    "=" * 100
)


print(
    ci_check.to_string(
        index=False
    )
)

In [ ]:
# =============================================================================
# DEEP AKDE DIAGNOSTIC FOR A43 AND F30
# =============================================================================

CHECK_IDS = [
    "A43",
    "F30"
]


check = (

    initial_pos_gps[
        initial_pos_gps[
            "id_clean"
        ].isin(
            CHECK_IDS
        )
    ]

    .sort_values(
        [
            "id_clean",
            "gps_datetime"
        ]
    )

    .copy()
)


# =============================================================================
# 1. DISTINGUISH MISSING CI VALUES FROM TRUE CI ORDERING FAILURES
# =============================================================================

check[
    "any_akde_missing"
] = (

    check[
        [
            "akde_low",
            "akde_est",
            "akde_high"
        ]
    ]

    .isna()

    .any(
        axis=1
    )
)


check[
    "true_ci_order_failure"
] = (

    ~check[
        "any_akde_missing"
    ]

    &

    (
        (
            check[
                "akde_low"
            ]
            >
            check[
                "akde_est"
            ]
        )

        |

        (
            check[
                "akde_est"
            ]
            >
            check[
                "akde_high"
            ]
        )
    )
)


ci_diagnostic = (

    check

    .groupby(
        "id_clean"
    )

    .agg(

        rows=(
            "gps_datetime",
            "size"
        ),

        missing_any_akde=(
            "any_akde_missing",
            "sum"
        ),

        true_ci_order_failures=(
            "true_ci_order_failure",
            "sum"
        ),

        complete_akde_rows=(
            "any_akde_missing",
            lambda x:
                (~x).sum()
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 100
)

print(
    "MISSING VALUES VS TRUE CI ORDER FAILURES"
)

print(
    "=" * 100
)


print(
    ci_diagnostic.to_string(
        index=False
    )
)


# =============================================================================
# 2. PRINT ANY TRUE LOW / EST / HIGH ORDERING FAILURES
# =============================================================================

true_failures = check[
    check[
        "true_ci_order_failure"
    ]
][
    [
        "id_clean",
        "gps_datetime",
        "days_from_first_positive",
        "akde_low",
        "akde_est",
        "akde_high"
    ]
].copy()


print(
    "\n"
    + "=" * 110
)

print(
    "TRUE AKDE CONFIDENCE-INTERVAL ORDER FAILURES"
)

print(
    "=" * 110
)


if len(
    true_failures
) == 0:

    print(
        "\nNone."
    )

else:

    print(
        true_failures
        .round(
            6
        )
        .to_string(
            index=False
        )
    )


# =============================================================================
# 3. SHOW AKDE DISTRIBUTION FOR EACH DEER
# =============================================================================

distribution = (

    check

    .groupby(
        "id_clean"
    )

    .agg(

        akde_min=(
            "akde_est",
            "min"
        ),

        akde_q01=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.01
                )
        ),

        akde_q05=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.05
                )
        ),

        akde_q25=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.25
                )
        ),

        akde_median=(
            "akde_est",
            "median"
        ),

        akde_q75=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.75
                )
        ),

        akde_q95=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.95
                )
        ),

        akde_q99=(
            "akde_est",
            lambda x:
                x.quantile(
                    0.99
                )
        ),

        akde_max=(
            "akde_est",
            "max"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 110
)

print(
    "AKDE DISTRIBUTION"
)

print(
    "=" * 110
)


print(
    distribution
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# 4. SHOW EVERY 30-DAY BIN
# =============================================================================

bins = (

    deer_30day[
        deer_30day[
            "id_clean"
        ].isin(
            CHECK_IDS
        )
    ]

    .sort_values(
        [
            "id_clean",
            "median_days"
        ]
    )

    .copy()
)


print(
    "\n"
    + "=" * 110
)

print(
    "ALL 30-DAY BINS"
)

print(
    "=" * 110
)


print(
    bins
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# 5. FIRST AND LAST 30-DAY BIN VALUES
# =============================================================================

first_last_rows = []


for deer_id, deer_data in bins.groupby(
    "id_clean"
):

    deer_data = deer_data.sort_values(
        "median_days"
    )


    first = deer_data.iloc[0]

    last = deer_data.iloc[-1]


    first_value = first[
        "median_akde_est"
    ]

    last_value = last[
        "median_akde_est"
    ]


    first_last_rows.append(
        {
            "id_clean":
                deer_id,

            "first_bin_days":
                first[
                    "median_days"
                ],

            "first_bin_AKDE":
                first_value,

            "last_bin_days":
                last[
                    "median_days"
                ],

            "last_bin_AKDE":
                last_value,

            "last_first_ratio":
                (
                    last_value
                    /
                    first_value
                )
                if first_value != 0
                else np.nan,

            "percent_change":
                (
                    (
                        last_value
                        -
                        first_value
                    )
                    /
                    first_value
                    *
                    100
                )
                if first_value != 0
                else np.nan
        }
    )


first_last = pd.DataFrame(
    first_last_rows
)


print(
    "\n"
    + "=" * 110
)

print(
    "FIRST VS LAST 30-DAY AKDE"
)

print(
    "=" * 110
)


print(
    first_last
    .round(
        6
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# 6. FIND THE BIGGEST JUMPS BETWEEN CONSECUTIVE 30-DAY BINS
# =============================================================================

jump_rows = []


for deer_id, deer_data in bins.groupby(
    "id_clean"
):

    deer_data = deer_data.sort_values(
        "median_days"
    ).copy()


    deer_data[
        "previous_akde"
    ] = deer_data[
        "median_akde_est"
    ].shift(
        1
    )


    deer_data[
        "absolute_jump"
    ] = (

        deer_data[
            "median_akde_est"
        ]

        -

        deer_data[
            "previous_akde"
        ]
    )


    deer_data[
        "pct_jump"
    ] = (

        deer_data[
            "absolute_jump"
        ]

        /

        deer_data[
            "previous_akde"
        ]

        *

        100
    )


    deer_data[
        "id_clean"
    ] = deer_id


    jump_rows.append(
        deer_data
    )


jumps = pd.concat(
    jump_rows,
    ignore_index=True
)


print(
    "\n"
    + "=" * 110
)

print(
    "LARGEST CONSECUTIVE 30-DAY AKDE JUMPS"
)

print(
    "=" * 110
)


print(
    jumps[
        [
            "id_clean",
            "median_days",
            "previous_akde",
            "median_akde_est",
            "absolute_jump",
            "pct_jump",
            "n_days"
        ]
    ]

    .sort_values(
        "pct_jump",
        ascending=False
    )

    .head(
        20
    )

    .round(
        6
    )

    .to_string(
        index=False
    )
)

====================================================================================================
MISSING VALUES VS TRUE CI ORDER FAILURES
====================================================================================================
id_clean  rows  missing_any_akde  true_ci_order_failures  complete_akde_rows
     A43  1119               207                       0                 912
     F30  2373               140                       0                2233

==============================================================================================================
TRUE AKDE CONFIDENCE-INTERVAL ORDER FAILURES
==============================================================================================================

None.

==============================================================================================================
AKDE DISTRIBUTION
==============================================================================================================
id_clean  akde_min  akde_q01  akde_q05  akde_q25  akde_median   akde_q75    akde_q95    akde_q99    akde_max
     A43                                                                                          7752.462382
     F30                                                                                          130.601111

==============================================================================================================
ALL 30-DAY BINS
==============================================================================================================
id_clean  post_30day_bin  median_days  median_akde_est  n_days
     A43               0    19.167350                       22
     A43               1    44.167593                       30
     A43               2    74.667604                       30
     A43               3   104.667529                       29
     A43               4   135.042448                       30
     A43               5   165.292488                       30
     A43               6   195.792587                       30
     A43               7   223.917350                       30
     A43               8   254.917350                       29
     A43               9   284.292350                       30
     A43              10   315.917390                       30
     A43              11   344.042535                       30
     F30               0    18.688154                       22
     F30               1    44.438252                       30
     F30               2    74.312963                       30
     F30               3   104.875683                       30
     F30               4   134.875880                       30
     F30               5   165.125556                       30
     F30               6   195.125613                       30
     F30               7   225.375787                       30
     F30               8   254.688252                       30
     F30               9   285.125764                       30
     F30              10   314.563275                       30

==============================================================================================================
FIRST VS LAST 30-DAY AKDE
==============================================================================================================
id_clean  first_bin_days  first_bin_AKDE  last_bin_days  last_bin_AKDE  last_first_ratio  percent_change
     A43       19.167350                                                      509.526855    50852.685493
     F30       18.688154                                                      22.709670     2170.966963

==============================================================================================================
LARGEST CONSECUTIVE 30-DAY AKDE JUMPS
==============================================================================================================
id_clean  median_days  previous_akde  median_akde_est  absolute_jump     pct_jump  n_days
     A43   195.792587                                     722.759038 16630.976136      30
     F30   195.125613                                      47.558049   791.285380      30
     F30   104.875683                                       5.194116   267.472287      30
     A43   284.292350                                    1939.956932   215.945456      30
     A43   344.042535                                     916.111181   181.628028      30
     A43   254.917350                                     448.853249    99.855715      29
     F30   314.563275                                      13.441225    49.807071      30
     A43   104.667529                                       0.891574    33.211766      29
     A43   165.292488                                       1.014317    30.445838      30
     F30   225.375787                                      12.802541    23.899483      30
     F30   165.125556                                       0.941240    18.568591      30
     A43    74.667604                                       0.326302    13.836841      30
     F30    74.312963                                       0.125890     6.932134      30
     F30    44.438252                                       0.035835     2.012945      30
     A43   135.042448                                      -0.244544    -6.838300      30
     A43    44.167593                                      -0.429669   -15.412025      30
     F30   134.875880                                      -2.067055   -28.966408      30
     F30   285.125764                                     -15.294720   -36.173723      30
     F30   254.688252                                     -24.089518   -36.295346      30
     A43   223.917350                                    -277.603087   -38.179235      30

In [ ]:
# =============================================================================
# TRACE EXTREME AKDE JUMPS BACK TO CALENDAR DATE / SEASON / PHASE
# =============================================================================

CHECK_IDS = [
    "A43",
    "F30"
]


# =============================================================================
# FIND OPTIONAL SEASON / PHASE COLUMNS
# =============================================================================

possible_context_cols = [

    "season",
    "Season",

    "phase",
    "phase_label",

    "migration_phase",
    "migratory_phase",

    "movement_phase"
]


context_cols = [

    col
    for col in possible_context_cols
    if col in initial_pos_gps.columns
]


print(
    "\nContext columns found:"
)

print(
    context_cols
)


# =============================================================================
# CREATE 30-DAY BIN NUMBER ON ORIGINAL ROW-LEVEL DATA
# =============================================================================

trace = initial_pos_gps[
    initial_pos_gps[
        "id_clean"
    ].isin(
        CHECK_IDS
    )
].copy()


trace = trace[
    trace[
        "days_from_first_positive"
    ].between(
        8,
        365
    )
].copy()


trace[
    "post_30day_bin"
] = (

    trace[
        "days_from_first_positive"
    ]

    //

    30

).astype(
    int
)


# =============================================================================
# SUMMARIZE EACH BIN
# =============================================================================

agg_dict = {

    "first_datetime":
        (
            "gps_datetime",
            "min"
        ),

    "last_datetime":
        (
            "gps_datetime",
            "max"
        ),

    "median_days":
        (
            "days_from_first_positive",
            "median"
        ),

    "median_akde":
        (
            "akde_est",
            "median"
        ),

    "min_akde":
        (
            "akde_est",
            "min"
        ),

    "max_akde":
        (
            "akde_est",
            "max"
        ),

    "n_rows":
        (
            "gps_datetime",
            "size"
        ),

    "n_days":
        (
            "gps_date",
            "nunique"
        )
}


bin_trace = (

    trace

    .groupby(
        [
            "id_clean",
            "post_30day_bin"
        ]
    )

    .agg(
        **agg_dict
    )

    .reset_index()

    .sort_values(
        [
            "id_clean",
            "post_30day_bin"
        ]
    )
)


# =============================================================================
# ADD SEASON / PHASE INFORMATION
#
# For each 30-day bin, show all distinct labels represented.
# =============================================================================

for col in context_cols:

    temp = (

        trace

        .groupby(
            [
                "id_clean",
                "post_30day_bin"
            ]
        )[col]

        .apply(
            lambda x:
                ", ".join(
                    sorted(
                        set(
                            x.dropna().astype(str)
                        )
                    )
                )
        )

        .reset_index(
            name=f"{col}_values"
        )
    )


    bin_trace = bin_trace.merge(

        temp,

        on=[
            "id_clean",
            "post_30day_bin"
        ],

        how="left"
    )


# =============================================================================
# CALCULATE CONSECUTIVE CHANGES
# =============================================================================

bin_trace[
    "previous_akde"
] = (

    bin_trace

    .groupby(
        "id_clean"
    )[
        "median_akde"
    ]

    .shift(
        1
    )
)


bin_trace[
    "absolute_change"
] = (

    bin_trace[
        "median_akde"
    ]

    -

    bin_trace[
        "previous_akde"
    ]
)


bin_trace[
    "pct_change"
] = (

    bin_trace[
        "absolute_change"
    ]

    /

    bin_trace[
        "previous_akde"
    ]

    *

    100
)


print(
    "\n"
    + "=" * 120
)

print(
    "CALENDAR / SEASON TRACE OF AKDE JUMPS"
)

print(
    "=" * 120
)


print(
    bin_trace
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# PRINT ONLY LARGE JUMPS
# =============================================================================

large_jumps = bin_trace[
    bin_trace[
        "pct_change"
    ].abs() >= 100
].copy()


print(
    "\n"
    + "=" * 120
)

print(
    "BINS WITH >100% CHANGE FROM PREVIOUS BIN"
)

print(
    "=" * 120
)


print(
    large_jumps
    .round(
        4
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# INSPECT ROW-LEVEL AKDE VALUES AROUND THE ~195-DAY JUMP
#
# We use a +/- 30 day neighborhood.
# =============================================================================

jump_neighborhood = trace[
    trace[
        "days_from_first_positive"
    ].between(
        150,
        230
    )
].copy()


print(
    "\n"
    + "=" * 120
)

print(
    "ROW-LEVEL VALUES AROUND ~195-DAY JUMP"
)

print(
    "=" * 120
)


display_cols = [

    "id_clean",
    "gps_datetime",
    "days_from_first_positive",

    *context_cols,

    "akde_low",
    "akde_est",
    "akde_high"
]


print(
    jump_neighborhood[
        display_cols
    ]

    .sort_values(
        [
            "id_clean",
            "gps_datetime"
        ]
    )

    .round(
        4
    )

    .to_string(
        index=False
    )
)

Context columns found:
['season', 'phase_label']

========================================================================================================================
CALENDAR / SEASON TRACE OF AKDE JUMPS
========================================================================================================================
id_clean  post_30day_bin      first_datetime       last_datetime  median_days  median_akde  min_akde  max_akde  n_rows  n_days  season_values phase_label_values  previous_akde  absolute_change  pct_change
     A43               0 2008-03-02 04:00:52 2008-03-23 22:01:08      19.1673                                       73      22         winter                               NaN              NaN         NaN
     A43               1 2008-03-24 04:01:08 2008-04-22 22:01:17      44.1676                                       87      30         winter                                            -0.4297    -15.4120
     A43               2 2008-04-23 04:00:59 2008-05-22 22:00:59      74.6676                                       93      30 summer, winter                                             0.3263     13.8368
     A43               3 2008-05-23 04:01:15 2008-06-21 22:01:08     104.6675                                       74      29         summer                                             0.8916     33.2118
     A43               4 2008-06-22 04:01:20 2008-07-21 22:01:06     135.0424                                       98      30         summer                                            -0.2445     -6.8383
     A43               5 2008-07-22 04:01:11 2008-08-20 22:01:20     165.2925                                      110      30         summer                                             1.0143     30.4458
     A43               6 2008-08-21 04:01:22 2008-09-19 22:01:19     195.7926                                       92      30 summer, winter                                           722.7590  16630.9761
     A43               7 2008-09-20 10:01:20 2008-10-19 22:00:56     223.9173                                       93      30         winter                                          -277.6031    -38.1792
     A43               8 2008-10-20 10:01:19 2008-11-18 16:01:20     254.9173                                       89      29         winter                                           448.8532     99.8557
     A43               9 2008-11-19 04:00:51 2008-12-18 16:01:17     284.2923                                       88      30         winter                                          1939.9569    215.9455
     A43              10 2008-12-19 04:00:52 2009-01-17 22:01:22     315.9174                                       88      30         winter                                         -2333.9233    -82.2293
     A43              11 2009-01-18 04:00:59 2009-02-16 22:01:23     344.0425                                       98      30         winter                                           916.1112    181.6280
     A43              12 2009-02-17 04:01:19 2009-02-21 04:00:52     362.0424          NaN       NaN       NaN      16       5         winter                                                NaN         NaN
     F30               0 2006-02-19 00:01:07 2006-03-12 21:00:54      18.6882                                      162      22         winter            Phase_1            NaN              NaN         NaN
     F30               1 2006-03-13 00:00:59 2006-04-11 21:00:59      44.4383                                      216      30         winter            Phase_1                          0.0358      2.0129
     F30               2 2006-04-12 00:00:55 2006-05-11 21:01:00      74.3130                                      210      30 summer, winter            Phase_1                          0.1259      6.9321
     F30               3 2006-05-12 00:00:53 2006-06-10 21:01:00     104.8757                                      221      30         summer   Phase_1, Phase_2                          5.1941    267.4723
     F30               4 2006-06-11 00:00:48 2006-07-10 21:00:59     134.8759                                      231      30         summer            Phase_2                         -2.0671    -28.9664
     F30               5 2006-07-11 00:00:51 2006-08-09 21:00:57     165.1256                                      223      30         summer            Phase_2                          0.9412     18.5686
     F30               6 2006-08-10 00:01:34 2006-09-08 21:00:59     195.1256                                      215      30 summer, winter   Phase_2, Phase_3                         47.5580    791.2854
     F30               7 2006-09-09 00:00:59 2006-10-08 21:01:09     225.3758                                      215      30         winter            Phase_3                         12.8025     23.8995
     F30               8 2006-10-09 00:01:00 2006-11-07 21:00:57     254.6883                                      222      30         winter            Phase_3                        -24.0895    -36.2953
     F30               9 2006-11-08 00:00:56 2006-12-07 21:00:51     285.1258                                      213      30         winter            Phase_3                        -15.2947    -36.1737
     F30              10 2006-12-08 00:01:29 2007-01-06 21:01:29     314.5633                                      220      30         winter            Phase_3                         13.4412     49.8071
     F30              11 2007-01-07 00:00:59 2007-01-09 09:01:21     330.5634          NaN       NaN       NaN      10       3         winter            Phase_3                             NaN         NaN

========================================================================================================================
BINS WITH >100% CHANGE FROM PREVIOUS BIN
========================================================================================================================
id_clean  post_30day_bin      first_datetime       last_datetime  median_days  median_akde  min_akde  max_akde  n_rows  n_days  season_values phase_label_values  previous_akde  absolute_change  pct_change
     A43               6 2008-08-21 04:01:22 2008-09-19 22:01:19     195.7926                                       92      30 summer, winter                                           722.7590  16630.9761
     A43               9 2008-11-19 04:00:51 2008-12-18 16:01:17     284.2923                                       88      30         winter                                          1939.9569    215.9455
     A43              11 2009-01-18 04:00:59 2009-02-16 22:01:23     344.0425                                       98      30         winter                                           916.1112    181.6280
     F30               3 2006-05-12 00:00:53 2006-06-10 21:01:00     104.8757                                      221      30         summer   Phase_1, Phase_2                          5.1941    267.4723
     F30               6 2006-08-10 00:01:34 2006-09-08 21:00:59     195.1256                                      215      30 summer, winter   Phase_2, Phase_3                         47.5580    791.2854

========================================================================================================================
ROW-LEVEL VALUES AROUND ~195-DAY JUMP
========================================================================================================================
id_clean        gps_datetime  days_from_first_positive season phase_label  akde_low  akde_est  akde_high
     A43 2008-07-22 04:01:11                  150.1675 summer         NaN    
     A43 2008-07-22 10:01:20                  150.4176 summer         NaN    
     A43 2008-07-22 22:00:59                  150.9173 summer         NaN    
     A43 2008-07-23 04:00:59                  151.1673 summer         NaN    
     A43 2008-07-23 10:01:20                  151.4176 summer         NaN    
     A43 2008-07-23 16:01:19                  151.6676 summer         NaN    
     A43 2008-07-23 22:01:08                  151.9175 summer         NaN    
     A43 2008-07-24 04:00:59                  152.1673 summer         NaN    
     A43 2008-07-24 10:01:21                  152.4176 summer         NaN    
     A43 2008-07-24 16:01:21                  152.6676 summer         NaN    
     A43 2008-07-24 22:00:55                  152.9173 summer         NaN    
     A43 2008-07-25 04:00:52                  153.1673 summer         NaN    
     A43 2008-07-25 10:01:59                  153.4180 summer         NaN    
     A43 2008-07-25 16:00:59                  153.6673 summer         NaN    
     A43 2008-07-25 22:00:57                  153.9173 summer         NaN    
     A43 2008-07-26 10:00:55                  154.4173 summer         NaN    
     A43 2008-07-26 16:01:14                  154.6675 summer         NaN    
     A43 2008-07-26 22:00:59                  154.9173 summer         NaN    
     A43 2008-07-27 04:00:59                  155.1673 summer         NaN    
     A43 2008-07-27 10:00:49                  155.4172 summer         NaN    
     A43 2008-07-27 16:01:19                  155.6676 summer         NaN    
     A43 2008-07-27 22:01:08                  155.9175 summer         NaN    
     A43 2008-07-28 04:01:06                  156.1674 summer         NaN       
     A43 2008-07-28 10:01:08                  156.4175 summer         NaN    
     A43 2008-07-28 16:01:08                  156.6675 summer         NaN    
     A43 2008-07-29 04:01:14                  157.1675 summer         NaN    
     A43 2008-07-29 10:00:49                  157.4172 summer         NaN   
     A43 2008-07-29 22:01:20                  157.9176 summer         NaN    
     A43 2008-07-30 04:01:08                  158.1675 summer         NaN    
     A43 2008-07-30 10:01:18                  158.4176 summer         NaN   
     A43 2008-07-30 16:01:17                  158.6676 summer         NaN    2.9017    5.6498     9.2982
     A43 2008-07-30 22:01:08                  158.9175 summer         NaN    2.9451    5.7910     9.5827
     A43 2008-07-31 04:00:59                  159.1673 summer         NaN    2.9221    5.7558     9.5338
     A43 2008-07-31 10:01:21                  159.4176 summer         NaN    2.8439    5.5244     9.0799
     A43 2008-07-31 16:01:19                  159.6676 summer         NaN    2.8263    5.6804     9.5138
     A43 2008-07-31 22:01:08                  159.9175 summer         NaN    2.8305    5.6230     9.3578
     A43 2008-08-01 04:00:50                  160.1672 summer         NaN    2.8185    5.7478     9.7033
     A43 2008-08-01 10:01:21                  160.4176 summer         NaN    2.5094    5.0453     8.4518
     A43 2008-08-01 16:01:08                  160.6675 summer         NaN    2.4951    5.1712     8.8060
     A43 2008-08-01 22:00:57                  160.9173 summer         NaN    2.6906    5.3984     9.0330
     A43 2008-08-02 04:01:21                  161.1676 summer         NaN    2.6141    5.0976     8.3966
     A43 2008-08-02 10:01:21                  161.4176 summer         NaN    2.2244    4.7730     8.2781
     A43 2008-08-02 16:01:21                  161.6676 summer         NaN    2.2060    4.7560     8.2690
     A43 2008-08-03 04:00:56                  162.1673 summer         NaN    2.5912    5.1160     8.4851
     A43 2008-08-03 10:01:13                  162.4175 summer         NaN    2.1836    4.2789     7.0671
     A43 2008-08-03 16:00:50                  162.6672 summer         NaN    2.6112    5.2201     8.7174
     A43 2008-08-03 22:00:59                  162.9173 summer         NaN    3.2121    6.4188    10.7167
     A43 2008-08-04 04:00:59                  163.1673 summer         NaN    3.3172    6.8390    11.6132
     A43 2008-08-04 10:01:09                  163.4175 summer         NaN    3.2227    6.4413    10.7556
     A43 2008-08-04 22:01:08                  163.9175 summer         NaN    3.2631    6.6149    11.1312
     A43 2008-08-05 04:00:54                  164.1673 summer         NaN    3.1488    5.7465     9.1126
     A43 2008-08-05 10:01:57                  164.4180 summer         NaN    2.7198    5.2426     8.5793
     A43 2008-08-05 16:00:59                  164.6673 summer         NaN    2.3494    4.3520     6.9615
     A43 2008-08-05 22:00:58                  164.9173 summer         NaN    2.5804    4.9021     7.9564
     A43 2008-08-06 04:01:14                  165.1675 summer         NaN    2.3045    3.9463     6.0224
     A43 2008-08-06 10:01:08                  165.4175 summer         NaN    2.4665    4.5962     7.3777
     A43 2008-08-06 16:00:59                  165.6673 summer         NaN    2.2679    3.8295     5.7936
     A43 2008-08-06 22:00:59                  165.9173 summer         NaN    2.2788    3.8006     5.7053
     A43 2008-08-07 04:00:59                  166.1673 summer         NaN    2.3909    4.3500     6.8855
     A43 2008-08-07 10:01:14                  166.4175 summer         NaN    2.3711    4.3218     6.8484
     A43 2008-08-07 16:01:20                  166.6676 summer         NaN    2.3598    4.2067     6.5786
     A43 2008-08-07 22:00:53                  166.9173 summer         NaN    2.2581    4.0354     6.3202
     A43 2008-08-08 04:01:20                  167.1676 summer         NaN    2.2331    3.9073     6.0420
     A43 2008-08-08 10:01:18                  167.4176 summer         NaN    2.1990    3.9253     6.1433
     A43 2008-08-08 16:01:20                  167.6676 summer         NaN    2.1912    3.8944     6.0793
     A43 2008-08-08 22:01:12                  167.9175 summer         NaN    2.2112    3.9570     6.2024
     A43 2008-08-09 04:01:20                  168.1676 summer         NaN    2.2566    4.0622     6.3898
     A43 2008-08-09 10:01:15                  168.4175 summer         NaN    2.3045    4.2138     6.6898
     A43 2008-08-09 16:00:57                  168.6673 summer         NaN    2.2985    4.2182     6.7111
     A43 2008-08-09 22:00:52                  168.9173 summer         NaN    2.2949    4.2288     6.7438
     A43 2008-08-10 10:01:08                  169.4175 summer         NaN    2.3023    4.3364     7.0037
     A43 2008-08-10 16:00:59                  169.6673 summer         NaN    2.3131    4.3418     6.9985
     A43 2008-08-10 22:01:06                  169.9174 summer         NaN    2.1246    3.9183     6.2517
     A43 2008-08-11 04:01:17                  170.1676 summer         NaN    2.0166    3.7640     6.0478
     A43 2008-08-11 10:01:08                  170.4175 summer         NaN    1.8984    3.3539     5.2167
     A43 2008-08-11 16:00:56                  170.6673 summer         NaN    1.6444    2.9508     4.6329
     A43 2008-08-11 22:00:58                  170.9173 summer         NaN    1.0125    1.7249     2.6241
     A43 2008-08-12 04:00:59                  171.1673 summer         NaN    1.1031    1.8530     2.7942
     A43 2008-08-12 10:01:41                  171.4178 summer         NaN    1.6519    3.0444     4.8556
     A43 2008-08-12 16:01:20                  171.6676 summer         NaN    1.8996    3.4843     5.5417
     A43 2008-08-12 22:01:20                  171.9176 summer         NaN    1.9968    3.7178     5.9647
     A43 2008-08-13 04:01:20                  172.1676 summer         NaN    2.0742    3.9045     6.3040
     A43 2008-08-13 10:01:08                  172.4175 summer         NaN    2.1157    4.0732     6.6612
     A43 2008-08-13 16:00:49                  172.6672 summer         NaN    2.0957    3.9704     6.4341
     A43 2008-08-13 22:00:54                  172.9173 summer         NaN    2.1389    4.1067     6.7056
     A43 2008-08-14 04:00:59                  173.1673 summer         NaN    2.0501    3.9969     6.5827
     A43 2008-08-14 10:01:19                  173.4176 summer         NaN    1.9178    3.7875     6.2827
     A43 2008-08-14 16:01:08                  173.6675 summer         NaN    1.9557    3.9809     6.7138
     A43 2008-08-14 22:01:08                  173.9175 summer         NaN    1.9189    3.7158     6.0966
     A43 2008-08-15 04:01:08                  174.1675 summer         NaN    1.9203    3.7249     6.1173
     A43 2008-08-15 10:01:19                  174.4176 summer         NaN    1.8930    3.6291     5.9207
     A43 2008-08-15 16:00:59                  174.6673 summer         NaN    1.9230    3.7442     6.1620
     A43 2008-08-15 22:01:20                  174.9176 summer         NaN    1.9022    3.6268     5.8986
     A43 2008-08-16 04:01:20                  175.1676 summer         NaN    1.9484    3.7256     6.0693
     A43 2008-08-16 10:02:22                  175.4183 summer         NaN    1.9629    3.7212     6.0326
     A43 2008-08-16 16:01:20                  175.6676 summer         NaN    1.9755    3.7363     6.0486
     A43 2008-08-16 22:01:17                  175.9176 summer         NaN    1.9604    3.7347     6.0714
     A43 2008-08-17 10:01:54                  176.4180 summer         NaN    1.9409    3.6856     5.9804
     A43 2008-08-17 16:00:59                  176.6673 summer         NaN    1.9611    3.7755     6.1742
     A43 2008-08-18 04:01:59                  177.1680 summer         NaN    1.8841    3.6004     5.8632
     A43 2008-08-18 10:01:08                  177.4175 summer         NaN    1.9059    3.6889     6.0508
     A43 2008-08-18 16:01:08                  177.6675 summer         NaN    1.8353    3.6706     6.1312
     A43 2008-08-18 22:00:59                  177.9173 summer         NaN    1.7577    3.8398     6.7216
     A43 2008-08-19 04:00:57                  178.1673 summer         NaN    1.5769    3.6700     6.6318
     A43 2008-08-19 10:01:19                  178.4176 summer         NaN    1.3777    2.9602     5.1377
     A43 2008-08-19 16:01:07                  178.6674 summer         NaN    1.2059    2.5805     4.4692
     A43 2008-08-19 22:00:59                  178.9173 summer         NaN    0.9335    1.9898     3.4389
     A43 2008-08-20 04:01:08                  179.1675 summer         NaN    0.7837    1.6124     2.7351
     A43 2008-08-20 10:01:22                  179.4176 summer         NaN   41.7284   79.2128   128.5103
     A43 2008-08-20 22:01:20                  179.9176 summer         NaN   39.3310   81.1304   137.8068
     A43 2008-08-21 04:01:22                  180.1676 summer         NaN   34.6120   89.4364   169.8300
     A43 2008-08-21 22:01:08                  180.9175 summer         NaN   37.1279   79.0191   136.4629
     A43 2008-08-22 04:00:59                  181.1673 summer         NaN   33.1738   64.0215   104.8405
     A43 2008-08-22 10:00:52                  181.4173 summer         NaN   30.7918   62.7863   105.9867
     A43 2008-08-22 16:01:37                  181.6678 summer         NaN   30.3956   60.1230    99.8190
     A43 2008-08-22 22:01:08                  181.9175 summer         NaN   15.5183   98.5481   256.7301
     A43 2008-08-23 04:00:50                  182.1672 summer         NaN       NaN       NaN        NaN
     A43 2008-08-23 10:00:59                  182.4173 summer         NaN       NaN       NaN        NaN
     A43 2008-08-23 16:00:55                  182.6673 summer         NaN       NaN       NaN        NaN
     A43 2008-08-23 22:00:54                  182.9173 summer         NaN 1579.4685 2623.4171  3927.9127
     A43 2008-08-24 10:01:18                  183.4176 summer         NaN 1573.3395 2575.7573  3821.2459
     A43 2008-08-24 16:01:14                  183.6675 summer         NaN 1566.2766 2601.5061  3895.1064
     A43 2008-08-24 22:01:20                  183.9176 summer         NaN       NaN       NaN        NaN
     A43 2008-08-25 04:01:20                  184.1676 summer         NaN       NaN       NaN        NaN
     A43 2008-08-25 10:01:20                  184.4176 summer         NaN 1903.6122 3161.8035  4734.0120
     A43 2008-08-25 16:01:20                  184.6676 summer         NaN 1834.0884 3046.3279  4561.1161
     A43 2008-08-26 10:01:41                  185.4178 summer         NaN 1795.9645 2983.0061  4466.3075
     A43 2008-08-27 04:01:41                  186.1678 summer         NaN 1956.2528 3202.6353  4751.2461
     A43 2008-08-27 10:01:21                  186.4176 summer         NaN 2011.2866 3340.6451  5001.7827
     A43 2008-08-27 22:01:59                  186.9180 summer         NaN 2059.9612 3421.4911  5122.8295
     A43 2008-08-28 04:01:21                  187.1676 summer         NaN 1597.4484 2653.2809  3972.6262
     A43 2008-08-28 10:00:49                  187.4172 summer         NaN       NaN       NaN        NaN
     A43 2008-08-28 22:01:19                  187.9176 summer         NaN 1389.0419 2307.1283  3454.3491
     A43 2008-08-29 10:01:19                  188.4176 summer         NaN 1392.7540 2249.9492  3309.3944
     A43 2008-08-29 16:00:57                  188.6673 summer         NaN 1409.8730 2308.1419  3424.2270
     A43 2008-08-30 04:00:53                  189.1673 summer         NaN 2315.5896 3740.7604  5502.1914
     A43 2008-08-30 10:01:08                  189.4175 summer         NaN 2246.9493 3629.8741  5339.0916
     A43 2008-08-30 16:01:08                  189.6675 summer         NaN 2331.9549 3817.7074  5663.7319
     A43 2008-08-30 22:00:53                  189.9173 summer         NaN 2342.2631 3834.5834  5688.7681
     A43 2008-08-31 04:00:49                  190.1672 winter         NaN 2026.3487 3317.3912  4921.4914
     A43 2008-08-31 10:01:08                  190.4175 winter         NaN 2047.1521 3351.4490  4972.0176
     A43 2008-08-31 22:01:21                  190.9176 winter         NaN 2029.0599 3321.8298  4928.0763
     A43 2008-09-01 04:01:21                  191.1676 winter         NaN 2284.5642 3740.1228  5548.6318
     A43 2008-09-01 10:01:07                  191.4174 winter         NaN 1082.4965 1797.9719  2692.0144
     A43 2008-09-01 16:00:49                  191.6672 winter         NaN 1074.5306 1784.7409  2672.2043
     A43 2008-09-02 10:01:22                  192.4176 winter         NaN 1158.1598 1870.9698  2751.9630
     A43 2008-09-02 16:01:19                  192.6676 winter         NaN 1162.9825 1903.9505  2824.5919
     A43 2008-09-02 22:01:08                  192.9175 winter         NaN  958.0005 1568.3689  2326.7422
     A43 2008-09-03 16:01:19                  193.6676 winter         NaN  934.4821 1474.1460  2134.8368
     A43 2008-09-04 04:01:20                  194.1676 winter         NaN  921.2953 1453.3438  2104.7113
     A43 2008-09-04 10:01:08                  194.4175 winter         NaN  905.3659 1428.2151  2068.3203
     A43 2008-09-04 16:01:20                  194.6676 winter         NaN  914.1582 1442.0850  2088.4065
     A43 2008-09-04 22:01:21                  194.9176 winter         NaN  930.8700 1468.4478  2126.5847
     A43 2008-09-05 04:01:21                  195.1676 winter         NaN  958.3683 1529.2428  2231.3550
     A43 2008-09-05 10:00:49                  195.4172 winter         NaN  984.4669 1590.3746  2339.2424
     A43 2008-09-05 16:01:21                  195.6676 winter         NaN  969.5851 1566.3335  2303.8809
     A43 2008-09-05 22:01:18                  195.9176 winter         NaN  585.2010  958.0487  1421.3061
     A43 2008-09-06 10:01:07                  196.4174 winter         NaN  574.5725  928.2033  1365.2711
     A43 2008-09-06 16:01:19                  196.6676 winter         NaN  569.0522  919.2854  1352.1541
     A43 2008-09-06 22:01:08                  196.9175 winter         NaN  566.4632  915.1031  1346.0024
     A43 2008-09-07 10:00:49                  197.4172 winter         NaN  564.7168  924.5135  1371.5552
     A43 2008-09-07 16:00:49                  197.6672 winter         NaN  461.3503  766.2795  1147.3123
     A43 2008-09-07 22:01:40                  197.9178 winter         NaN  450.2661  737.1430  1093.5832
     A43 2008-09-08 10:01:20                  198.4176 winter         NaN  443.0646  725.3532  1076.0924
     A43 2008-09-08 16:01:20                  198.6676 winter         NaN  445.6435  729.5752  1082.3559
     A43 2008-09-08 22:01:20                  198.9176 winter         NaN  445.2046  728.8566  1081.2899
     A43 2008-09-09 10:01:08                  199.4175 winter         NaN  398.1033  651.7458   966.8927
     A43 2008-09-09 16:01:49                  199.6679 winter         NaN  403.5823  651.9742   958.9726
     A43 2008-09-09 22:01:14                  199.9175 winter         NaN  327.8596  536.7480   796.2886
     A43 2008-09-10 04:01:08                  200.1675 winter         NaN  326.3207  534.2286   792.5509
     A43 2008-09-10 10:01:16                  200.4175 winter         NaN  321.7104  526.6809   781.3536
     A43 2008-09-10 16:01:19                  200.6676 winter         NaN  291.6548  471.1589   693.0159
     A43 2008-09-10 22:01:17                  200.9176 winter         NaN  291.8625  471.4944   693.5094
     A43 2008-09-11 10:00:59                  201.4173 winter         NaN  216.1659  349.2091   513.6429
     A43 2008-09-11 16:01:20                  201.6676 winter         NaN  271.0058  432.4367   630.9788
     A43 2008-09-11 22:01:08                  201.9175 winter         NaN  268.0676  427.7484   624.1380
     A43 2008-09-12 10:01:20                  202.4176 winter         NaN  223.2679  352.2053   510.0586
     A43 2008-09-12 22:01:20                  202.9176 winter         NaN  231.4272  361.1993   519.3940
     A43 2008-09-13 04:01:21                  203.1676 winter         NaN  234.7946  370.3887   536.3914
     A43 2008-09-13 16:01:19                  203.6676 winter         NaN  189.1400  295.1996   424.4884
     A43 2008-09-13 22:01:08                  203.9175 winter         NaN  187.8413  293.1727   421.5737
     A43 2008-09-14 16:01:28                  204.6677 winter         NaN  193.0362  295.5089   419.4514
     A43 2008-09-14 22:00:49                  204.9172 winter         NaN  197.1384  304.6268   435.1301
     A43 2008-09-15 04:01:19                  205.1676 winter         NaN  189.7075  290.4131   412.2182
     A43 2008-09-15 16:01:12                  205.6675 winter         NaN  156.6482  237.7037   335.3913
     A43 2008-09-15 22:01:12                  205.9175 winter         NaN  179.7518  272.7619   384.8572
     A43 2008-09-16 04:01:20                  206.1676 winter         NaN  174.5962  267.2801   379.3828
     A43 2008-09-16 10:01:20                  206.4176 winter         NaN  180.7466  276.6953   392.7470
     A43 2008-09-16 16:00:49                  206.6672 winter         NaN  180.2800  278.5765   397.9198
     A43 2008-09-16 22:01:20                  206.9176 winter         NaN  184.1641  281.9270   400.1730
     A43 2008-09-17 04:01:21                  207.1676 winter         NaN  182.7441  279.7532   397.0875
     A43 2008-09-17 10:01:21                  207.4176 winter         NaN  180.7568  276.7110   392.7693
     A43 2008-09-17 16:01:08                  207.6675 winter         NaN  185.8530  284.5125   403.8428
     A43 2008-09-17 22:01:08                  207.9175 winter         NaN  189.9832  293.5703   419.3370
     A43 2008-09-18 04:01:22                  208.1676 winter         NaN  189.1534  289.5649   411.0144
     A43 2008-09-18 10:01:16                  208.4175 winter         NaN  167.5736  256.5295   364.1232
     A43 2008-09-18 16:00:49                  208.6672 winter         NaN  171.4398  264.9162   378.4073
     A43 2008-09-18 22:00:59                  208.9173 winter         NaN  129.1604  199.5843   285.0870
     A43 2008-09-19 04:01:19                  209.1676 winter         NaN  157.8350  243.8935   348.3785
     A43 2008-09-19 10:00:58                  209.4173 winter         NaN  156.4804  241.8003   345.3885
     A43 2008-09-19 16:01:17                  209.6676 winter         NaN  156.2857  241.4994   344.9587
     A43 2008-09-19 22:01:19                  209.9176 winter         NaN  156.5640  241.9295   345.5730
     A43 2008-09-20 10:01:20                  210.4176 winter         NaN  147.4423  227.8343   325.4394
     A43 2008-09-20 16:00:50                  210.6672 winter         NaN  150.4454  234.8072   337.6458
     A43 2008-09-20 22:00:56                  210.9173 winter         NaN  169.8488  265.0911   381.1931
     A43 2008-09-21 04:01:20                  211.1676 winter         NaN  170.0499  265.4048   381.6443
     A43 2008-09-21 10:00:55                  211.4173 winter         NaN  166.8494  260.4097   374.4615
     A43 2008-09-21 16:01:08                  211.6675 winter         NaN  170.1959  268.4842   388.8149
     A43 2008-09-21 22:00:59                  211.9173 winter         NaN  173.8271  277.3713   404.7191
     A43 2008-09-22 04:00:59                  212.1673 winter         NaN  177.9619  283.9691   414.3462
     A43 2008-09-22 10:00:59                  212.4173 winter         NaN  180.7606  285.1499   412.9499
     A43 2008-09-22 16:01:08                  212.6675 winter         NaN  183.3830  292.6193   426.9679
     A43 2008-09-22 22:00:59                  212.9173 winter         NaN  190.7596  304.3900   444.1428
     A43 2008-09-23 10:00:59                  213.4173 winter         NaN  175.2765  276.4988   400.4215
     A43 2008-09-23 16:01:08                  213.6675 winter         NaN  172.7516  275.6551   402.2151
     A43 2008-09-23 22:01:20                  213.9176 winter         NaN  172.6508  275.4944   401.9805
     A43 2008-09-24 04:01:20                  214.1676 winter         NaN  169.3404  273.5640   402.3785
     A43 2008-09-24 10:00:50                  214.4172 winter         NaN  172.2148  281.9376   418.2665
     A43 2008-09-24 16:01:08                  214.6675 winter         NaN  173.0492  283.3035   420.2929
     A43 2008-09-24 22:01:19                  214.9176 winter         NaN  174.7784  286.1345   424.4927
     A43 2008-09-25 04:00:49                  215.1672 winter         NaN  175.8461  287.8825   427.0860
     A43 2008-09-25 10:00:54                  215.4173 winter         NaN  169.2366  273.3963   402.1319
     A43 2008-09-25 16:01:19                  215.6676 winter         NaN  158.3073  255.7403   376.1620
     A43 2008-09-26 04:00:59                  216.1673 winter         NaN  184.4577  297.9855   438.2994
     A43 2008-09-26 10:00:54                  216.4173 winter         NaN  189.6052  320.9738   486.3566
     A43 2008-09-26 16:01:12                  216.6675 winter         NaN  197.5758  315.2665   460.0130
     A43 2008-09-26 22:01:18                  216.9176 winter         NaN  200.3917  319.7597   466.5690
     A43 2008-09-27 10:01:19                  217.4176 winter         NaN  195.8586  312.5263   456.0147
     A43 2008-09-27 16:01:20                  217.6676 winter         NaN  245.1195  391.1306   570.7081
     A43 2008-09-27 22:01:20                  217.9176 winter         NaN  224.4302  358.1173   522.5377
     A43 2008-09-28 10:00:55                  218.4173 winter         NaN  234.2240  373.7450   545.3403
     A43 2008-09-28 22:01:22                  218.9176 winter         NaN  246.7992  385.1910   553.8934
     A43 2008-09-29 04:01:22                  219.1676 winter         NaN  244.8101  386.1881   559.2720
     A43 2008-09-29 10:00:59                  219.4173 winter         NaN  247.8373  395.4673   577.0359
     A43 2008-09-29 16:01:19                  219.6676 winter         NaN  259.2237  408.9255   592.1999
     A43 2008-09-29 22:01:19                  219.9176 winter         NaN  235.2577  371.1191   537.4493
     A43 2008-09-30 10:00:59                  220.4173 winter         NaN  300.3281  468.7361   674.0287
     A43 2008-09-30 16:01:19                  220.6676 winter         NaN  306.2776  478.0218   687.3813
     A43 2008-10-01 10:00:59                  221.4173 winter         NaN  297.6692  459.9714   657.0249
     A43 2008-10-01 16:01:22                  221.6676 winter         NaN  303.6137  469.1571   670.1458
     A43 2008-10-01 22:01:22                  221.9176 winter         NaN  299.4994  462.7995   661.0646
     A43 2008-10-02 04:01:38                  222.1678 winter         NaN  299.8318  463.3131   661.7983
     A43 2008-10-02 10:00:53                  222.4173 winter         NaN  304.5862  475.3819   683.5852
     A43 2008-10-02 16:00:49                  222.6672 winter         NaN  380.5423  593.9302   854.0541
     A43 2008-10-02 22:01:39                  222.9178 winter         NaN  369.9012  571.5873   816.4576
     A43 2008-10-03 04:01:20                  223.1676 winter         NaN  366.3240  571.7391   822.1439
     A43 2008-10-03 10:00:59                  223.4173 winter         NaN  347.1143  541.7576   779.0315
     A43 2008-10-03 16:01:06                  223.6674 winter         NaN  351.0769  553.8240   802.0398
     A43 2008-10-03 22:00:59                  223.9173 winter         NaN  360.8614  569.2590   824.3925
     A43 2008-10-04 10:00:59                  224.4173 winter         NaN  356.0639  561.6910   813.4327
     A43 2008-10-04 16:01:07                  224.6674 winter         NaN  363.2066  579.5590   845.6486
     A43 2008-10-04 22:01:08                  224.9175 winter         NaN  321.3247  519.0897   763.5161
     A43 2008-10-05 04:01:19                  225.1676 winter         NaN  323.8337  523.1428   769.4778
     A43 2008-10-05 10:01:08                  225.4175 winter         NaN  268.1723  439.0323   651.3231
     A43 2008-10-05 16:01:15                  225.6675 winter         NaN  250.4074  415.9140   622.7275
     A43 2008-10-05 22:00:55                  225.9173 winter         NaN  326.6545  534.7751   793.3617
     A43 2008-10-06 10:01:00                  226.4174 winter         NaN  328.6168  545.8159   817.2232
     A43 2008-10-06 16:00:53                  226.6673 winter         NaN  339.1415  563.2968   843.3964
     A43 2008-10-06 22:00:50                  226.9172 winter         NaN  401.3085  666.5530   997.9969
     A43 2008-10-07 04:00:51                  227.1673 winter         NaN  396.1002  657.9023   985.0446
     A43 2008-10-07 10:00:50                  227.4172 winter         NaN  402.2391  668.0987  1000.3111
     A43 2008-10-07 16:00:49                  227.6672 winter         NaN       NaN       NaN        NaN
     A43 2008-10-07 22:00:53                  227.9173 winter         NaN       NaN       NaN        NaN
     A43 2008-10-08 04:01:06                  228.1674 winter         NaN       NaN       NaN        NaN
     A43 2008-10-08 16:01:08                  228.6675 winter         NaN  702.4302 1149.9677  1706.0262
     A43 2008-10-08 22:00:57                  228.9173 winter         NaN  715.6952 1188.7335  1779.8318
     A43 2008-10-09 04:01:19                  229.1676 winter         NaN  720.4294 1196.5967  1791.6051
     A43 2008-10-09 10:01:08                  229.4175 winter         NaN  719.5024 1195.0570  1789.2997
     A43 2008-10-09 16:00:50                  229.6672 winter         NaN       NaN       NaN        NaN
     A43 2008-10-09 22:00:59                  229.9173 winter         NaN  542.2088  900.5814  1348.3960
     F30 2006-07-11 00:00:51                  150.0006 summer     Phase_2    2.6723    5.5094     9.3554
     F30 2006-07-11 03:00:49                  150.1256 summer     Phase_2    2.6863    5.8579    10.2447
     F30 2006-07-11 06:00:59                  150.2507 summer     Phase_2    2.6291    5.5427     9.5245
     F30 2006-07-11 09:00:42                  150.3755 summer     Phase_2    3.7121    5.9181     8.6302
     F30 2006-07-11 15:00:49                  150.6256 summer     Phase_2    3.9751    9.4800    17.3360
     F30 2006-07-11 18:01:18                  150.7509 summer     Phase_2    4.4610    7.1658    10.5012
     F30 2006-07-12 00:00:59                  151.0007 summer     Phase_2    4.5007    9.3126    15.8444
     F30 2006-07-12 03:00:48                  151.1256 summer     Phase_2    4.4582    9.1857    15.5931
     F30 2006-07-12 06:00:52                  151.2506 summer     Phase_2    4.4190    9.1213    15.4986
     F30 2006-07-12 09:00:56                  151.3756 summer     Phase_2    4.4385    9.1612    15.5660
     F30 2006-07-12 12:01:12                  151.5008 summer     Phase_2    4.8459    7.6320    11.0407
     F30 2006-07-12 15:00:54                  151.6256 summer     Phase_2    4.8380    7.6129    11.0066
     F30 2006-07-12 18:01:17                  151.7509 summer     Phase_2    4.8166    7.6670    11.1692
     F30 2006-07-12 21:00:59                  151.8757 summer     Phase_2    4.6777    7.5860    11.1861
     F30 2006-07-13 00:00:51                  152.0006 summer     Phase_2    4.6168    7.5011    11.0741
     F30 2006-07-13 03:01:00                  152.1257 summer     Phase_2    4.7042    7.5916    11.1589
     F30 2006-07-13 09:01:39                  152.3761 summer     Phase_2    4.6626    7.3902    10.7358
     F30 2006-07-13 12:01:09                  152.5008 summer     Phase_2    4.7020    7.4836    10.9012
     F30 2006-07-13 15:00:57                  152.6257 summer     Phase_2    4.6587    7.3621    10.6742
     F30 2006-07-13 18:01:10                  152.7508 summer     Phase_2    4.7181    7.5442    11.0229
     F30 2006-07-13 21:00:59                  152.8757 summer     Phase_2    4.6548    7.4068    10.7878
     F30 2006-07-14 00:00:48                  153.0006 summer     Phase_2    4.6587    7.4393    10.8601
     F30 2006-07-14 03:01:06                  153.1258 summer     Phase_2    4.5541    7.2512    10.5655
     F30 2006-07-14 06:01:00                  153.2507 summer     Phase_2    4.4431    7.0435    10.2333
     F30 2006-07-14 09:00:54                  153.3756 summer     Phase_2    4.4847    7.1460    10.4173
     F30 2006-07-14 12:01:09                  153.5008 summer     Phase_2    4.4244    6.9962    10.1479
     F30 2006-07-14 15:01:06                  153.6258 summer     Phase_2    4.3861    6.9207    10.0241
     F30 2006-07-14 18:00:59                  153.7507 summer     Phase_2    4.3815    6.9417    10.0816
     F30 2006-07-14 21:00:59                  153.8757 summer     Phase_2    4.3419    6.8668     9.9610
     F30 2006-07-15 00:00:48                  154.0006 summer     Phase_2    4.2837    6.7635     9.8005
     F30 2006-07-15 03:00:49                  154.1256 summer     Phase_2    4.2051    6.6513     9.6493
     F30 2006-07-15 06:00:59                  154.2507 summer     Phase_2    3.9660    6.2111     8.9516
     F30 2006-07-15 09:01:09                  154.3758 summer     Phase_2    3.9474    6.1809     8.9070
     F30 2006-07-15 12:00:49                  154.5006 summer     Phase_2    3.9311    6.1736     8.9140
     F30 2006-07-15 15:01:06                  154.6258 summer     Phase_2    3.8690    6.0319     8.6674
     F30 2006-07-15 18:01:00                  154.7507 summer     Phase_2    3.7961    6.0102     8.7248
     F30 2006-07-15 21:00:48                  154.8756 summer     Phase_2    3.7375    6.0120     8.8184
     F30 2006-07-16 00:00:48                  155.0006 summer     Phase_2    3.5942    5.7439     8.3895
     F30 2006-07-16 03:01:06                  155.1258 summer     Phase_2    3.4912    5.7401     8.5391
     F30 2006-07-16 06:01:12                  155.2508 summer     Phase_2    3.3518    5.4997     8.1707
     F30 2006-07-16 09:00:59                  155.3757 summer     Phase_2    3.3950    5.5962     8.3386
     F30 2006-07-16 12:00:51                  155.5006 summer     Phase_2    3.4521    5.6565     8.3964
     F30 2006-07-16 15:00:49                  155.6256 summer     Phase_2    3.4944    5.7532     8.5659
     F30 2006-07-16 18:00:50                  155.7506 summer     Phase_2    3.5870    5.8637     8.6909
     F30 2006-07-17 00:01:12                  156.0008 summer     Phase_2    3.5568    5.7962     8.5735
     F30 2006-07-17 06:01:00                  156.2507 summer     Phase_2    3.5739    5.8615     8.7058
     F30 2006-07-17 09:01:19                  156.3759 summer     Phase_2    3.5787    5.8584     8.6907
     F30 2006-07-17 12:01:04                  156.5007 summer     Phase_2    3.6202    5.9538     8.8586
     F30 2006-07-17 15:00:48                  156.6256 summer     Phase_2    3.6398    6.0051     8.9530
     F30 2006-07-17 18:00:43                  156.7505 summer     Phase_2    3.6440    6.0113     8.9615
     F30 2006-07-17 21:00:59                  156.8757 summer     Phase_2    3.9118    6.4465     9.6041
     F30 2006-07-18 00:00:49                  157.0006 summer     Phase_2    3.9547    6.5493     9.7877
     F30 2006-07-18 03:01:20                  157.1259 summer     Phase_2    4.0696    6.5159     9.5286
     F30 2006-07-18 06:01:00                  157.2507 summer     Phase_2    3.7941    8.4380    14.9060
     F30 2006-07-18 09:01:02                  157.3757 summer     Phase_2    3.9995    6.3405     9.2122
     F30 2006-07-18 12:01:09                  157.5008 summer     Phase_2    3.8167    6.0418     8.7697
     F30 2006-07-18 18:01:21                  157.7509 summer     Phase_2    3.2165    5.1247     7.4702
     F30 2006-07-18 21:00:58                  157.8757 summer     Phase_2    2.2875    4.0870     6.4000
     F30 2006-07-19 00:00:48                  158.0006 summer     Phase_2    2.2776    4.0392     6.2971
     F30 2006-07-19 03:00:59                  158.1257 summer     Phase_2    2.4767    3.7991     5.3999
     F30 2006-07-19 06:00:48                  158.2506 summer     Phase_2    2.4558    3.8008     5.4349
     F30 2006-07-19 09:01:13                  158.3758 summer     Phase_2    2.2819    3.8340     5.7821
     F30 2006-07-19 12:00:59                  158.5007 summer     Phase_2    2.2773    3.8916     5.9315
     F30 2006-07-19 15:00:54                  158.6256 summer     Phase_2    2.2767    3.8785     5.8999
     F30 2006-07-19 18:00:51                  158.7506 summer     Phase_2    2.2801    3.8600     5.8490
     F30 2006-07-19 21:01:00                  158.8757 summer     Phase_2    2.3168    4.0420     6.2392
     F30 2006-07-20 00:01:05                  159.0008 summer     Phase_2    2.3865    4.2908     6.7444
     F30 2006-07-20 03:00:57                  159.1257 summer     Phase_2    2.3742    4.2442     6.6484
     F30 2006-07-20 06:01:18                  159.2509 summer     Phase_2    2.2998    4.7235     8.0048
     F30 2006-07-20 09:00:59                  159.3757 summer     Phase_2    2.0136    3.7627     6.0495
     F30 2006-07-20 12:01:17                  159.5009 summer     Phase_2    2.0447    3.1638     4.5232
     F30 2006-07-20 15:00:49                  159.6256 summer     Phase_2    2.0458    3.1620     4.5172
     F30 2006-07-20 18:01:19                  159.7509 summer     Phase_2    2.0167    3.0979     4.4074
     F30 2006-07-20 21:00:59                  159.8757 summer     Phase_2    2.0251    3.1182     4.4434
     F30 2006-07-21 00:00:59                  160.0007 summer     Phase_2    2.0168    3.0974     4.4062
     F30 2006-07-21 03:00:48                  160.1256 summer     Phase_2    2.0113    3.0824     4.3783
     F30 2006-07-21 06:00:58                  160.2507 summer     Phase_2    2.0346    3.1343     4.4678
     F30 2006-07-21 12:00:59                  160.5007 summer     Phase_2    2.0401    3.1519     4.5015
     F30 2006-07-21 15:00:59                  160.6257 summer     Phase_2    2.0464    3.1463     4.4790
     F30 2006-07-21 18:01:00                  160.7507 summer     Phase_2    2.0475    3.1457     4.4760
     F30 2006-07-21 21:01:01                  160.8757 summer     Phase_2    2.1376    3.3043     4.7211
     F30 2006-07-22 00:00:54                  161.0006 summer     Phase_2    2.1453    3.2888     4.6726
     F30 2006-07-22 03:00:59                  161.1257 summer     Phase_2    2.1541    3.3120     4.7149
     F30 2006-07-22 06:00:44                  161.2505 summer     Phase_2    2.2354    3.4581     4.9434
     F30 2006-07-22 09:01:06                  161.3758 summer     Phase_2    2.1853    3.3085     4.6608
     F30 2006-07-22 15:00:59                  161.6257 summer     Phase_2    2.2119    3.3519     4.7251
     F30 2006-07-22 18:00:59                  161.7507 summer     Phase_2    2.2384    3.4037     4.8093
     F30 2006-07-22 21:00:48                  161.8756 summer     Phase_2    2.2683    3.4807     4.9485
     F30 2006-07-23 00:00:48                  162.0006 summer     Phase_2    2.3162    3.5652     5.0792
     F30 2006-07-23 03:01:02                  162.1257 summer     Phase_2    2.3416    3.6355     5.2093
     F30 2006-07-23 06:00:59                  162.2507 summer     Phase_2    2.3766    3.6965     5.3031
     F30 2006-07-23 09:01:18                  162.3759 summer     Phase_2    2.3756    3.6750     5.2533
     F30 2006-07-23 15:01:07                  162.6258 summer     Phase_2    2.3865    3.7257     5.3584
     F30 2006-07-23 18:00:49                  162.7506 summer     Phase_2    2.3827    3.8196     5.5901
     F30 2006-07-23 21:01:06                  162.8758 summer     Phase_2    2.4621    4.0420     6.0070
     F30 2006-07-24 00:00:59                  163.0007 summer     Phase_2    2.5405    4.0639     5.9394
     F30 2006-07-24 03:00:55                  163.1256 summer     Phase_2    2.5625    4.0835     5.9533
     F30 2006-07-24 06:00:59                  163.2507 summer     Phase_2    2.6042    4.1450     6.0381
     F30 2006-07-24 09:01:06                  163.3758 summer     Phase_2    2.6204    4.1876     6.1162
     F30 2006-07-24 12:01:14                  163.5009 summer     Phase_2    2.6599    4.1729     6.0210
     F30 2006-07-24 15:00:54                  163.6256 summer     Phase_2    2.6837    4.2206     6.0999
     F30 2006-07-24 18:00:58                  163.7507 summer     Phase_2    2.7115    4.3232     6.3047
     F30 2006-07-24 21:00:48                  163.8756 summer     Phase_2    2.7044    4.3163     6.2989
     F30 2006-07-25 00:00:59                  164.0007 summer     Phase_2    2.7259    4.3478     6.3421
     F30 2006-07-25 03:00:49                  164.1256 summer     Phase_2    2.7122    4.3738     6.4259
     F30 2006-07-25 06:01:18                  164.2509 summer     Phase_2    2.7558    4.4076     6.4409
     F30 2006-07-25 09:00:49                  164.3756 summer     Phase_2    2.7848    4.4745     6.5583
     F30 2006-07-25 12:01:09                  164.5008 summer     Phase_2    2.8470    4.5493     6.6441
     F30 2006-07-25 15:00:49                  164.6256 summer     Phase_2    2.8785    4.6212     6.7698
     F30 2006-07-25 18:00:57                  164.7507 summer     Phase_2    2.8805    4.5901     6.6917
     F30 2006-07-26 00:00:58                  165.0007 summer     Phase_2    2.8931    4.6297     6.7680
     F30 2006-07-26 03:00:48                  165.1256 summer     Phase_2    3.0048    4.9281     7.3193
     F30 2006-07-26 06:00:49                  165.2506 summer     Phase_2    2.9105    4.7009     6.9135
     F30 2006-07-26 09:00:49                  165.3756 summer     Phase_2    2.9272    4.7458     6.9967
     F30 2006-07-26 12:00:52                  165.5006 summer     Phase_2    3.9700    6.8241    10.4386
     F30 2006-07-26 15:01:07                  165.6258 summer     Phase_2    4.7094    8.1204    12.4453
     F30 2006-07-26 18:01:00                  165.7507 summer     Phase_2    4.9165    8.5221    13.1031
     F30 2006-07-26 21:01:05                  165.8758 summer     Phase_2    4.8610    8.2926    12.6257
     F30 2006-07-27 00:00:59                  166.0007 summer     Phase_2    4.7523    8.1136    12.3592
     F30 2006-07-27 03:01:06                  166.1258 summer     Phase_2    4.7587    8.0819    12.2707
     F30 2006-07-27 06:00:41                  166.2505 summer     Phase_2    4.6547    7.8534    11.8750
     F30 2006-07-27 09:01:06                  166.3758 summer     Phase_2    4.5939    7.6414    11.4518
     F30 2006-07-27 12:01:18                  166.5009 summer     Phase_2    4.5573    7.5737    11.3438
     F30 2006-07-27 15:00:59                  166.6257 summer     Phase_2    4.4892    7.4655    11.1865
     F30 2006-07-27 18:00:39                  166.7505 summer     Phase_2    4.3879    7.2530    10.8262
     F30 2006-07-27 21:01:06                  166.8758 summer     Phase_2    4.5136    7.4733    11.1670
     F30 2006-07-28 00:00:29                  167.0003 summer     Phase_2    4.5094    7.5025    11.2450
     F30 2006-07-28 03:00:48                  167.1256 summer     Phase_2    4.4414    7.2578    10.7545
     F30 2006-07-28 09:01:18                  167.3759 summer     Phase_2    4.4814    7.3026    10.8015
     F30 2006-07-28 12:01:18                  167.5009 summer     Phase_2    4.4808    7.2457    10.6644
     F30 2006-07-28 15:00:49                  167.6256 summer     Phase_2    4.4793    7.2426    10.6590
     F30 2006-07-28 18:00:59                  167.7507 summer     Phase_2    4.4882    7.2986    10.7813
     F30 2006-07-28 21:00:59                  167.8757 summer     Phase_2    4.4753    7.2670    10.7246
     F30 2006-07-29 00:00:38                  168.0004 summer     Phase_2    4.5780    7.4789    11.0802
     F30 2006-07-29 03:01:00                  168.1257 summer     Phase_2    4.6458    7.7356    11.6004
     F30 2006-07-29 06:01:01                  168.2507 summer     Phase_2    4.6119    7.5557    11.2144
     F30 2006-07-29 09:00:49                  168.3756 summer     Phase_2    4.6365    7.6683    11.4505
     F30 2006-07-29 12:01:07                  168.5008 summer     Phase_2    4.6608    7.6254    11.3080
     F30 2006-07-29 15:00:57                  168.6257 summer     Phase_2    4.6854    7.6651    11.3663
     F30 2006-07-29 18:00:49                  168.7506 summer     Phase_2    4.7164    7.7589    11.5465
     F30 2006-07-29 21:01:05                  168.8758 summer     Phase_2    4.7085    7.7372    11.5058
     F30 2006-07-30 00:00:55                  169.0006 summer     Phase_2    4.9338    8.2004    12.2833
     F30 2006-07-30 03:01:12                  169.1258 summer     Phase_2    4.8081    7.7351    11.3467
     F30 2006-07-30 06:00:59                  169.2507 summer     Phase_2    4.6688    7.4920    10.9720
     F30 2006-07-30 09:01:15                  169.3759 summer     Phase_2    4.6712    7.5143    11.0223
     F30 2006-07-30 18:00:56                  169.7506 summer     Phase_2    4.5730    7.3166    10.6946
     F30 2006-07-30 21:01:18                  169.8759 summer     Phase_2    4.7094    7.6457    11.2819
     F30 2006-07-31 00:00:57                  170.0007 summer     Phase_2    4.7895    7.8449    11.6419
     F30 2006-07-31 03:00:48                  170.1256 summer     Phase_2    4.7237    7.7274    11.4584
     F30 2006-07-31 06:00:48                  170.2506 summer     Phase_2    4.7001    7.7797    11.6227
     F30 2006-07-31 09:01:18                  170.3759 summer     Phase_2    4.4902    7.3449    10.8907
     F30 2006-07-31 12:00:57                  170.5007 summer     Phase_2    4.3515    7.0642    10.4234
     F30 2006-07-31 15:00:49                  170.6256 summer     Phase_2    4.2421    6.8605    10.0979
     F30 2006-07-31 18:01:00                  170.7507 summer     Phase_2    4.0689    6.5227     9.5462
     F30 2006-07-31 21:00:59                  170.8757 summer     Phase_2    4.0857    6.5806     9.6604
     F30 2006-08-01 00:00:52                  171.0006 summer     Phase_2    3.8961    6.1990     9.0281
     F30 2006-08-01 03:01:11                  171.1258 summer     Phase_2    3.7155    5.9178     8.6242
     F30 2006-08-01 06:01:04                  171.2507 summer     Phase_2    3.4160    5.4108     7.8569
     F30 2006-08-01 09:01:48                  171.3762 summer     Phase_2    3.3863    5.2885     7.6079
     F30 2006-08-01 15:00:56                  171.6256 summer     Phase_2    3.3446    5.2414     7.5576
     F30 2006-08-01 18:00:55                  171.7506 summer     Phase_2    3.3478    5.2484     7.5694
     F30 2006-08-01 21:01:06                  171.8758 summer     Phase_2    3.4572    5.4594     7.9115
     F30 2006-08-02 00:01:00                  172.0007 summer     Phase_2    3.5224    5.4479     7.7866
     F30 2006-08-02 03:01:06                  172.1258 summer     Phase_2    3.4921    5.4511     7.8393
     F30 2006-08-02 09:01:09                  172.3758 summer     Phase_2    3.2430    6.4115    10.6417
     F30 2006-08-02 12:01:23                  172.5010 summer     Phase_2    3.1870    4.8969     6.9682
     F30 2006-08-02 15:00:48                  172.6256 summer     Phase_2    2.9665    4.5292     6.4172
     F30 2006-08-02 18:00:59                  172.7507 summer     Phase_2    2.5672    3.8324     5.3470
     F30 2006-08-02 21:00:49                  172.8756 summer     Phase_2    2.2537    3.2991     4.5406
     F30 2006-08-03 00:00:56                  173.0006 summer     Phase_2    2.1529    3.1374     4.3044
     F30 2006-08-03 03:01:00                  173.1257 summer     Phase_2    2.1716    3.1665     4.3459
     F30 2006-08-03 06:01:12                  173.2508 summer     Phase_2    2.6470    3.9282     5.4582
     F30 2006-08-03 09:00:58                  173.3757 summer     Phase_2    2.6489    3.9388     5.4806
     F30 2006-08-03 12:00:49                  173.5006 summer     Phase_2    2.9736    4.4502     6.2197
     F30 2006-08-03 15:01:10                  173.6258 summer     Phase_2    3.3080    5.0078     7.0543
     F30 2006-08-03 18:01:22                  173.7509 summer     Phase_2    3.3638    5.1193     7.2376
     F30 2006-08-03 21:00:49                  173.8756 summer     Phase_2    3.3523    5.1281     7.2753
     F30 2006-08-04 00:00:49                  174.0006 summer     Phase_2    3.3448    5.1260     7.2814
     F30 2006-08-04 03:01:03                  174.1257 summer     Phase_2    3.3721    5.1637     7.3309
     F30 2006-08-04 06:00:59                  174.2507 summer     Phase_2    3.2976    5.0142     7.0847
     F30 2006-08-04 09:01:21                  174.3759 summer     Phase_2    3.3278    5.0798     7.1964
     F30 2006-08-04 12:01:36                  174.5011 summer     Phase_2    3.2972    4.9837     7.0130
     F30 2006-08-04 15:00:59                  174.6257 summer     Phase_2    3.3001    4.9915     7.0272
     F30 2006-08-04 18:01:18                  174.7509 summer     Phase_2    3.3461    5.1058     7.2315
     F30 2006-08-04 21:01:12                  174.8758 summer     Phase_2    3.4378    5.2057     7.3346
     F30 2006-08-05 03:01:01                  175.1257 summer     Phase_2    3.4019    5.1148     7.1714
     F30 2006-08-05 06:00:48                  175.2506 summer     Phase_2    3.2307    5.5126     8.3942
     F30 2006-08-05 09:01:09                  175.3758 summer     Phase_2    3.2452    5.4618     8.2460
     F30 2006-08-05 12:01:09                  175.5008 summer     Phase_2    3.3281    4.9861     6.9738
     F30 2006-08-05 15:01:06                  175.6258 summer     Phase_2    3.2946    4.9185     6.8626
     F30 2006-08-05 18:00:59                  175.7507 summer     Phase_2    3.2878    4.9427     6.9296
     F30 2006-08-05 21:01:18                  175.8759 summer     Phase_2    3.3316    5.0785     7.1877
     F30 2006-08-06 00:01:09                  176.0008 summer     Phase_2    3.1859    4.8009     6.7418
     F30 2006-08-06 03:00:58                  176.1257 summer     Phase_2    3.2352    5.5310     8.4323
     F30 2006-08-06 06:01:09                  176.2508 summer     Phase_2    3.9619    6.2520     9.0562
     F30 2006-08-06 09:01:20                  176.3759 summer     Phase_2    4.0045    6.3458     9.2173
     F30 2006-08-06 12:01:12                  176.5008 summer     Phase_2    4.3561    6.8660     9.9377
     F30 2006-08-06 15:00:59                  176.6257 summer     Phase_2    4.4050    6.9753    10.1269
     F30 2006-08-06 18:00:49                  176.7506 summer     Phase_2    4.6625   11.3174    20.8719
     F30 2006-08-06 21:00:56                  176.8756 summer     Phase_2    4.7654    9.6306    16.1784
     F30 2006-08-07 00:00:59                  177.0007 summer     Phase_2    4.7646    9.6493    16.2284
     F30 2006-08-07 06:02:10                  177.2515 summer     Phase_2    4.7271    9.1318    14.9624
     F30 2006-08-07 09:01:20                  177.3759 summer     Phase_2    4.7204    9.1138    14.9283
     F30 2006-08-07 12:00:59                  177.5007 summer     Phase_2    4.7615    9.2492    15.2019
     F30 2006-08-07 15:01:18                  177.6259 summer     Phase_2    4.7217    9.1151    14.9293
     F30 2006-08-07 18:00:59                  177.7507 summer     Phase_2    4.7592    9.2479    15.2026
     F30 2006-08-07 21:00:54                  177.8756 summer     Phase_2    4.7282    9.2852    15.3540
     F30 2006-08-08 00:00:49                  178.0006 summer     Phase_2    4.7444    9.3681    15.5382
     F30 2006-08-08 03:00:52                  178.1256 summer     Phase_2    4.7674    9.5170    15.8808
     F30 2006-08-08 06:01:23                  178.2510 summer     Phase_2    4.5559    8.9195    14.7244
     F30 2006-08-08 09:01:12                  178.3758 summer     Phase_2    4.5088    8.7887    14.4729
     F30 2006-08-08 12:01:19                  178.5009 summer     Phase_2    4.5119    8.8112    14.5252
     F30 2006-08-08 15:01:09                  178.6258 summer     Phase_2    4.5140    8.8226    14.5507
     F30 2006-08-08 18:01:22                  178.7509 summer     Phase_2    4.8190    7.8137    11.5204
     F30 2006-08-08 21:00:59                  178.8757 summer     Phase_2    4.9519    8.1780    12.2002
     F30 2006-08-09 00:00:48                  179.0006 summer     Phase_2    4.9138    8.1016    12.0735
     F30 2006-08-09 03:01:22                  179.1259 summer     Phase_2    4.8733    7.9929    11.8719
     F30 2006-08-09 06:00:53                  179.2506 summer     Phase_2    4.5902    9.5694    16.3467
     F30 2006-08-09 09:01:20                  179.3759 summer     Phase_2    4.6352    9.3314    15.6429
     F30 2006-08-09 12:01:22                  179.5009 summer     Phase_2    4.6459    9.3833    15.7577
     F30 2006-08-09 15:01:00                  179.6257 summer     Phase_2    4.6378    9.3586    15.7086
     F30 2006-08-09 18:00:59                  179.7507 summer     Phase_2    4.6292    9.3416    15.6804
     F30 2006-08-09 21:00:57                  179.8757 summer     Phase_2    4.6430    9.3940    15.7910
     F30 2006-08-10 00:01:34                  180.0011 summer     Phase_2    4.6903    9.5909    16.2147
     F30 2006-08-10 03:01:00                  180.1257 summer     Phase_2    4.9308    8.1683    12.2096
     F30 2006-08-10 06:00:59                  180.2507 summer     Phase_2    4.7427   11.1649    20.2890
     F30 2006-08-10 09:01:18                  180.3759 summer     Phase_2    4.6269   10.7517    19.4142
     F30 2006-08-10 12:01:06                  180.5008 summer     Phase_2    4.5038   10.1882    18.1530
     F30 2006-08-10 15:00:49                  180.6256 summer     Phase_2    4.6192    7.6146    11.3466
     F30 2006-08-10 18:01:00                  180.7507 summer     Phase_2    4.1390    8.7647    15.0964
     F30 2006-08-10 21:00:59                  180.8757 summer     Phase_2    4.0822    8.9514    15.6998
     F30 2006-08-11 00:00:57                  181.0007 summer     Phase_2    4.3061    7.0327    10.4171
     F30 2006-08-11 03:00:59                  181.1257 summer     Phase_2    4.3835    7.2804    10.9001
     F30 2006-08-11 06:00:48                  181.2506 summer     Phase_2    4.2909    7.1093    10.6275
     F30 2006-08-11 09:01:39                  181.3761 summer     Phase_2    4.0057    7.1138    11.0996
     F30 2006-08-11 12:01:09                  181.5008 summer     Phase_2    4.0091    7.1187    11.1064
     F30 2006-08-11 15:00:59                  181.6257 summer     Phase_2    4.0479    7.2214    11.2982
     F30 2006-08-11 18:00:55                  181.7506 summer     Phase_2    4.1048    7.4175    11.6940
     F30 2006-08-11 21:01:02                  181.8757 summer     Phase_2    4.0547    6.2075     8.8114
     F30 2006-08-12 00:00:59                  182.0007 summer     Phase_2    4.0554    6.2364     8.8791
     F30 2006-08-12 06:01:21                  182.2509 summer     Phase_2    3.9786    7.0666    11.0270
     F30 2006-08-12 09:01:23                  182.3760 summer     Phase_2    4.0009    7.1444    11.1843
     F30 2006-08-12 12:01:12                  182.5008 summer     Phase_2    4.0357    7.2572    11.4083
     F30 2006-08-12 15:00:59                  182.6257 summer     Phase_2    4.0825    7.4035    11.6966
     F30 2006-08-12 18:00:59                  182.7507 summer     Phase_2    4.0747    7.3731    11.6334
     F30 2006-08-12 21:00:59                  182.8757 summer     Phase_2    3.9216    7.2729    11.6421
     F30 2006-08-13 00:00:59                  183.0007 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 03:00:53                  183.1256 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 06:01:01                  183.2507 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 09:02:11                  183.3765 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 15:01:10                  183.6258 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 18:00:49                  183.7506 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-13 21:00:55                  183.8756 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 00:01:28                  184.0010 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 03:01:11                  184.1258 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 06:01:05                  184.2508 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 09:01:16                  184.3759 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 12:01:23                  184.5010 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 15:00:59                  184.6257 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 18:01:10                  184.7508 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-14 21:00:49                  184.8756 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 03:01:00                  185.1257 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 06:00:58                  185.2507 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 09:00:58                  185.3757 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 12:00:52                  185.5006 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 15:00:59                  185.6257 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-15 21:00:56                  185.8756 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 00:01:18                  186.0009 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 03:01:06                  186.1258 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 06:01:03                  186.2507 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 09:00:57                  186.3757 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 12:01:18                  186.5009 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 15:00:49                  186.6256 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 18:00:49                  186.7506 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-16 21:00:56                  186.8756 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 00:01:36                  187.0011 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 03:00:56                  187.1256 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 06:00:48                  187.2506 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 09:01:03                  187.3757 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 12:01:06                  187.5008 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 15:00:48                  187.6256 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-17 21:01:06                  187.8758 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 00:00:48                  188.0006 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 03:01:06                  188.1258 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 06:01:06                  188.2508 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 09:00:48                  188.3756 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 12:00:59                  188.5007 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 18:01:10                  188.7508 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-18 21:01:03                  188.8757 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 00:01:22                  189.0009 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 03:00:58                  189.1257 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 06:00:43                  189.2505 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 09:01:32                  189.3761 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 12:01:38                  189.5011 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 15:00:49                  189.6256 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 18:01:00                  189.7507 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-19 21:00:59                  189.8757 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-20 00:00:48                  190.0006 summer     Phase_2       NaN       NaN        NaN
     F30 2006-08-20 09:00:59                  190.3757 summer     Phase_3    7.3136  130.6011   426.1827
     F30 2006-08-20 12:00:48                  190.5006 summer     Phase_3    4.3585   66.5198   210.9868
     F30 2006-08-20 18:01:00                  190.7507 summer     Phase_3    3.4884   39.1085   116.6971
     F30 2006-08-21 00:01:18                  191.0009 summer     Phase_3    2.3249   21.3305    60.9092
     F30 2006-08-21 03:00:58                  191.1257 summer     Phase_3    2.3716    4.9612     8.4903
     F30 2006-08-21 06:00:54                  191.2506 summer     Phase_3    1.0789    2.8663     5.5114
     F30 2006-08-21 09:01:17                  191.3759 summer     Phase_3    0.7718    1.2572     1.8591
     F30 2006-08-21 12:01:19                  191.5009 summer     Phase_3    0.7355    1.2017     1.7805
     F30 2006-08-21 18:01:22                  191.7509 summer     Phase_3   54.6490   85.8424   123.9662
     F30 2006-08-21 21:01:22                  191.8759 summer     Phase_3   54.2209   84.9741   122.5250
     F30 2006-08-22 00:00:48                  192.0006 summer     Phase_3   54.4338   85.5569   123.6040
     F30 2006-08-22 03:00:59                  192.1257 summer     Phase_3   54.2983   85.3428   123.2935
     F30 2006-08-22 06:01:13                  192.2508 summer     Phase_3   58.4226   92.5707   134.4513
     F30 2006-08-22 09:01:18                  192.3759 summer     Phase_3    0.6368    1.0139     1.4774
     F30 2006-08-22 12:01:09                  192.5008 summer     Phase_3    0.6426    1.0300     1.5073
     F30 2006-08-22 18:01:21                  192.7509 summer     Phase_3    0.6372    1.0170     1.4841
     F30 2006-08-22 21:00:59                  192.8757 summer     Phase_3    0.6342    1.0100     1.4718
     F30 2006-08-23 00:00:57                  193.0007 summer     Phase_3    0.6296    1.0069     1.4713
     F30 2006-08-23 03:00:59                  193.1257 summer     Phase_3    0.6382    1.0370     1.5310
     F30 2006-08-23 06:01:14                  193.2509 summer     Phase_3   52.1820   95.6937   152.1804
     F30 2006-08-23 12:01:48                  193.5012 summer     Phase_3   57.7092   93.1322   136.8953
     F30 2006-08-23 18:00:54                  193.7506 summer     Phase_3   52.8759   95.1862   149.7284
     F30 2006-08-23 21:00:59                  193.8757 summer     Phase_3   56.0366   90.1497   132.2418
     F30 2006-08-24 00:00:44                  194.0005 summer     Phase_3   56.2630   89.9076   131.3107
     F30 2006-08-24 03:01:18                  194.1259 summer     Phase_3   56.0576   90.1368   132.1785
     F30 2006-08-24 06:01:18                  194.2509 summer     Phase_3   53.2834   99.2998   159.4017
     F30 2006-08-24 09:00:55                  194.3756 summer     Phase_3   53.3480   95.9831   150.9322
     F30 2006-08-24 12:01:12                  194.5008 summer     Phase_3   52.2264   92.2310   143.4242
     F30 2006-08-24 15:00:49                  194.6256 summer     Phase_3   51.0177   88.4360   135.9780
     F30 2006-08-24 18:01:20                  194.7509 summer     Phase_3   50.5117   86.1289   131.0945
     F30 2006-08-24 21:00:58                  194.8757 summer     Phase_3   47.9263   82.0331   125.1558
     F30 2006-08-25 00:00:58                  195.0007 summer     Phase_3   49.8552   77.9001   112.1032
     F30 2006-08-25 03:00:53                  195.1256 summer     Phase_3   50.4334   79.4173   114.8758
     F30 2006-08-25 06:01:51                  195.2513 summer     Phase_3   49.7509   78.0764   112.6822
     F30 2006-08-25 15:01:09                  195.6258 summer     Phase_3   46.7557   71.0475   100.3405
     F30 2006-08-25 18:00:57                  195.7507 summer     Phase_3   43.5784   65.5876    92.0244
     F30 2006-08-25 21:00:51                  195.8756 summer     Phase_3   42.2272   63.5319    89.1189
     F30 2006-08-26 09:01:22                  196.3759 summer     Phase_3   41.0465   60.3505    83.3162
     F30 2006-08-26 15:01:10                  196.6258 summer     Phase_3   40.1230   58.8996    81.2233
     F30 2006-08-26 18:00:49                  196.7506 summer     Phase_3   39.9966   58.8899    81.3801
     F30 2006-08-26 21:01:42                  196.8762 summer     Phase_3   39.9877   58.5301    80.5488
     F30 2006-08-27 00:01:25                  197.0010 summer     Phase_3   40.8315   59.8006    82.3316
     F30 2006-08-27 06:01:29                  197.2510 summer     Phase_3   41.3319   61.1511    84.7907
     F30 2006-08-27 09:00:51                  197.3756 summer     Phase_3   40.1889   59.1052    81.6122
     F30 2006-08-27 12:00:53                  197.5006 summer     Phase_3   37.3134   54.3091    74.4443
     F30 2006-08-27 15:00:56                  197.6256 summer     Phase_3   31.4500   45.2345    61.4848
     F30 2006-08-27 18:00:50                  197.7506 summer     Phase_3   31.7991   46.2087    63.2687
     F30 2006-08-27 21:00:49                  197.8756 summer     Phase_3   30.4798   44.0324    60.0386
     F30 2006-08-28 00:01:05                  198.0008 summer     Phase_3   30.1344   43.2893    58.7894
     F30 2006-08-28 03:01:18                  198.1259 summer     Phase_3   30.1261   43.2208    58.6415
     F30 2006-08-28 06:01:24                  198.2510 summer     Phase_3   30.0818   42.8687    57.8850
     F30 2006-08-28 09:01:32                  198.3761 summer     Phase_3   30.2331   43.0076    57.9980
     F30 2006-08-28 12:01:09                  198.5008 summer     Phase_3   30.4014   43.2653    58.3632
     F30 2006-08-28 15:01:09                  198.6258 summer     Phase_3   30.5664   43.5615    58.8225
     F30 2006-08-28 18:01:00                  198.7507 summer     Phase_3   30.6559   43.8318    59.3262
     F30 2006-08-28 21:00:59                  198.8757 summer     Phase_3   30.8880   44.0935    59.6122
     F30 2006-08-29 00:01:09                  199.0008 summer     Phase_3   31.2968   44.9494    61.0345
     F30 2006-08-29 03:01:00                  199.1257 summer     Phase_3   30.9802   44.5901    60.6391
     F30 2006-08-29 06:00:59                  199.2507 summer     Phase_3   31.6092   45.6507    62.2322
     F30 2006-08-29 09:00:59                  199.3757 summer     Phase_3   30.9458   44.3034    60.0198
     F30 2006-08-29 12:01:12                  199.5008 summer     Phase_3   29.1451   41.5620    56.1478
     F30 2006-08-29 15:00:59                  199.6257 summer     Phase_3   28.3062   40.2234    54.2016
     F30 2006-08-29 18:01:06                  199.7508 summer     Phase_3   27.9657   39.7226    53.5103
     F30 2006-08-29 21:00:48                  199.8756 summer     Phase_3   28.4341   40.3903    54.4120
     F30 2006-08-30 00:00:59                  200.0007 summer     Phase_3   28.4054   40.5898    54.9147
     F30 2006-08-30 03:01:13                  200.1258 summer     Phase_3   28.6434   41.0776    55.7183
     F30 2006-08-30 06:00:49                  200.2506 summer     Phase_3   27.6362   41.5844    58.3371
     F30 2006-08-30 09:01:09                  200.3758 summer     Phase_3   27.6957   39.2943    52.8897
     F30 2006-08-30 12:01:19                  200.5009 summer     Phase_3   28.6925   41.0959    55.6924
     F30 2006-08-30 15:00:46                  200.6255 summer     Phase_3   25.7947   36.7709    49.6623
     F30 2006-08-30 18:00:59                  200.7507 summer     Phase_3   25.4579   36.0127    48.3694
     F30 2006-08-30 21:01:18                  200.8759 summer     Phase_3   25.4138   36.2163    48.9020
     F30 2006-08-31 00:00:38                  201.0004 winter     Phase_3   25.1709   36.1425    49.0678
     F30 2006-08-31 03:01:25                  201.1260 winter     Phase_3   26.2035   38.1089    52.2088
     F30 2006-08-31 06:01:38                  201.2511 winter     Phase_3   25.5786   37.2514    51.0838
     F30 2006-08-31 09:01:50                  201.3763 winter     Phase_3   29.7368   43.7872    60.5132
     F30 2006-08-31 15:00:56                  201.6256 winter     Phase_3   30.5477   45.1303    62.5135
     F30 2006-08-31 18:00:59                  201.7507 winter     Phase_3   30.3361   44.7687    61.9653
     F30 2006-08-31 21:00:48                  201.8756 winter     Phase_3   30.4685   44.9740    62.2590
     F30 2006-09-01 00:00:48                  202.0006 winter     Phase_3   30.6354   45.4332    63.1008
     F30 2006-09-01 03:01:22                  202.1259 winter     Phase_3   31.5047   46.8858    65.2762
     F30 2006-09-01 06:00:49                  202.2506 winter     Phase_3   31.9008   48.1993    67.8076
     F30 2006-09-01 09:00:58                  202.3757 winter     Phase_3   33.1364   50.1435    70.6172
     F30 2006-09-01 12:00:49                  202.5006 winter     Phase_3   33.4129   50.7950    71.7596
     F30 2006-09-01 15:00:59                  202.6257 winter     Phase_3   35.5813   53.7825    75.6835
     F30 2006-09-01 18:00:30                  202.7503 winter     Phase_3   35.6170   54.1982    76.6178
     F30 2006-09-01 21:00:59                  202.8757 winter     Phase_3   34.8474   52.2413    73.1007
     F30 2006-09-02 00:00:46                  203.0005 winter     Phase_3   35.2207   53.1466    74.7016
     F30 2006-09-02 03:01:29                  203.1260 winter     Phase_3   34.7957   52.6053    74.0369
     F30 2006-09-02 06:00:49                  203.2506 winter     Phase_3   34.8249   52.8671    74.6153
     F30 2006-09-02 09:00:49                  203.3756 winter     Phase_3   36.3933   55.2488    77.9775
     F30 2006-09-02 12:00:56                  203.5006 winter     Phase_3   35.6985   53.3875    74.5798
     F30 2006-09-02 15:00:59                  203.6257 winter     Phase_3   35.8396   53.9528    75.7118
     F30 2006-09-02 18:01:14                  203.7509 winter     Phase_3   35.5091   53.2820    74.6040
     F30 2006-09-02 21:00:56                  203.8756 winter     Phase_3   35.7292   53.9183    75.7903
     F30 2006-09-03 00:00:49                  204.0006 winter     Phase_3   35.9134   54.2615    76.3357
     F30 2006-09-03 06:01:50                  204.2513 winter     Phase_3   37.9826   57.4364    80.8488
     F30 2006-09-03 09:01:09                  204.3758 winter     Phase_3   38.1010   57.8462    81.6481
     F30 2006-09-03 12:00:49                  204.5006 winter     Phase_3   37.0982   55.8043    78.2686
     F30 2006-09-03 15:00:57                  204.6257 winter     Phase_3   36.4915   55.0866    77.4498
     F30 2006-09-03 18:01:10                  204.7508 winter     Phase_3   35.3773   53.5683    75.4728
     F30 2006-09-03 21:01:19                  204.8759 winter     Phase_3   35.0464   52.8492    74.2504
     F30 2006-09-04 00:01:29                  205.0010 winter     Phase_3   35.6623   54.0154    76.1178
     F30 2006-09-04 03:01:14                  205.1259 winter     Phase_3   36.6963   56.6858    80.9521
     F30 2006-09-04 06:00:55                  205.2506 winter     Phase_3   36.8993   57.5957    82.8260
     F30 2006-09-04 09:00:59                  205.3757 winter     Phase_3   36.2467   56.1034    80.2279
     F30 2006-09-04 12:00:56                  205.5006 winter     Phase_3   36.3083   56.6248    81.3834
     F30 2006-09-04 15:01:00                  205.6257 winter     Phase_3   37.0071   58.5821    85.0323
     F30 2006-09-04 18:01:08                  205.7508 winter     Phase_3   35.3043   55.8307    80.9854
     F30 2006-09-04 21:01:21                  205.8759 winter     Phase_3   35.3417   55.2047    79.4261
     F30 2006-09-05 00:00:57                  206.0007 winter     Phase_3   35.5576   55.5094    79.8332
     F30 2006-09-05 03:01:02                  206.1257 winter     Phase_3   35.1533   55.6935    80.8835
     F30 2006-09-05 06:01:03                  206.2507 winter     Phase_3   35.3798   57.2233    84.2335
     F30 2006-09-05 09:00:59                  206.3757 winter     Phase_3   35.3042   57.4520    84.9052
     F30 2006-09-05 12:01:07                  206.5008 winter     Phase_3   37.5812   59.3998    86.1324
     F30 2006-09-05 15:00:59                  206.6257 winter     Phase_3   37.1150   57.0719    81.2539
     F30 2006-09-05 18:01:04                  206.7507 winter     Phase_3   37.0925   57.1993    81.5910
     F30 2006-09-05 21:00:48                  206.8756 winter     Phase_3   37.5872   58.3764    83.6687
     F30 2006-09-06 00:00:53                  207.0006 winter     Phase_3   37.3219   57.4805    81.9226
     F30 2006-09-06 03:01:05                  207.1258 winter     Phase_3   37.1083   57.3623    81.9566
     F30 2006-09-06 06:01:42                  207.2512 winter     Phase_3   48.0013   77.6764   114.3780
     F30 2006-09-06 09:00:51                  207.3756 winter     Phase_3   48.7393   79.3405   117.2768
     F30 2006-09-06 12:00:49                  207.5006 winter     Phase_3   49.4364   78.9048   115.1514
     F30 2006-09-06 18:00:49                  207.7506 winter     Phase_3   51.3613   82.3836   120.6169
     F30 2006-09-06 21:00:48                  207.8756 winter     Phase_3   51.3839   82.0407   119.7542
     F30 2006-09-07 00:00:59                  208.0007 winter     Phase_3   51.8219   81.8533   118.6384
     F30 2006-09-07 03:01:10                  208.1258 winter     Phase_3   55.9198   88.4940   128.4243
     F30 2006-09-07 06:01:49                  208.2513 winter     Phase_3   56.4261   89.7011   130.5644
     F30 2006-09-07 09:01:09                  208.3758 winter     Phase_3   59.7632   95.2285   138.8224
     F30 2006-09-07 15:00:54                  208.6256 winter     Phase_3   55.1841   88.9116   130.5529
     F30 2006-09-07 18:00:56                  208.7506 winter     Phase_3   55.5902   88.4944   128.9247
     F30 2006-09-07 21:00:48                  208.8756 winter     Phase_3   55.7454   88.4845   128.6652
     F30 2006-09-08 00:00:49                  209.0006 winter     Phase_3   59.5831   93.7999   135.6559
     F30 2006-09-08 03:01:00                  209.1257 winter     Phase_3   60.0749   95.0111   137.8260
     F30 2006-09-08 06:01:11                  209.2508 winter     Phase_3   60.3108   95.9039   139.6189
     F30 2006-09-08 09:00:49                  209.3756 winter     Phase_3   55.9300   89.8199   131.6073
     F30 2006-09-08 12:00:50                  209.5006 winter     Phase_3   56.4324   90.1915   131.7380
     F30 2006-09-08 15:00:55                  209.6256 winter     Phase_3   55.9433   89.6965   131.2889
     F30 2006-09-08 18:00:59                  209.7507 winter     Phase_3   55.3659   88.2462   128.6672
     F30 2006-09-08 21:00:59                  209.8757 winter     Phase_3   56.2016   89.3397   130.0339
     F30 2006-09-09 00:00:59                  210.0007 winter     Phase_3   56.7789   90.9643   133.0759
     F30 2006-09-09 03:01:06                  210.1258 winter     Phase_3   58.0339   93.8719   138.1881
     F30 2006-09-09 06:01:34                  210.2511 winter     Phase_3   59.1918   97.1696   144.4075
     F30 2006-09-09 09:00:56                  210.3756 winter     Phase_3   58.1993   94.3199   139.0197
     F30 2006-09-09 15:01:09                  210.6258 winter     Phase_3   57.5641   93.4121   137.7975
     F30 2006-09-09 21:00:59                  210.8757 winter     Phase_3   58.3915   95.0351   140.4587
     F30 2006-09-10 00:01:12                  211.0008 winter     Phase_3   58.2526   93.8959   137.9101
     F30 2006-09-10 03:01:24                  211.1260 winter     Phase_3   57.8593   92.5384   135.2292
     F30 2006-09-10 09:01:46                  211.3762 winter     Phase_3   57.6830   89.5136   128.2259
     F30 2006-09-10 12:01:07                  211.5008 winter     Phase_3   58.3997   90.9768   130.6590
     F30 2006-09-10 15:01:10                  211.6258 winter     Phase_3   61.0722   94.2647   134.5458
     F30 2006-09-10 18:01:39                  211.7511 winter     Phase_3   63.3504   97.7634   139.5227
     F30 2006-09-10 21:01:12                  211.8758 winter     Phase_3   62.8579   96.5225   137.2912
     F30 2006-09-11 00:00:59                  212.0007 winter     Phase_3   63.7361   98.4298   140.5418
     F30 2006-09-11 03:00:48                  212.1256 winter     Phase_3   62.6718   96.9751   138.6460
     F30 2006-09-11 06:01:15                  212.2509 winter     Phase_3   60.3804   91.5971   129.2149
     F30 2006-09-11 09:01:07                  212.3758 winter     Phase_3   60.5354   92.0423   130.0452
     F30 2006-09-11 15:00:53                  212.6256 winter     Phase_3   62.0013   94.8758   134.6314
     F30 2006-09-11 18:01:21                  212.7509 winter     Phase_3   61.0755   92.8455   131.1626
     F30 2006-09-11 21:00:48                  212.8756 winter     Phase_3   61.6342   94.1828   133.5222
     F30 2006-09-12 00:00:52                  213.0006 winter     Phase_3   60.4147   92.1220   130.4112
     F30 2006-09-12 03:00:49                  213.1256 winter     Phase_3   60.1586   91.4593   129.2118
     F30 2006-09-12 06:01:43                  213.2512 winter     Phase_3   61.2784   93.8791   133.3222
     F30 2006-09-12 09:01:32                  213.3761 winter     Phase_3   61.0019   94.5587   135.3520
     F30 2006-09-12 15:00:59                  213.6257 winter     Phase_3   59.3128   91.8767   131.4522
     F30 2006-09-12 21:01:06                  213.8758 winter     Phase_3   59.8674   93.2119   133.8198
     F30 2006-09-13 00:01:06                  214.0008 winter     Phase_3   59.5483   92.5286   132.6604
     F30 2006-09-13 06:01:07                  214.2508 winter     Phase_3   60.5208   95.4359   138.1747
     F30 2006-09-13 09:00:59                  214.3757 winter     Phase_3   53.8082   83.9258   120.6301
     F30 2006-09-13 12:00:59                  214.5007 winter     Phase_3   51.1306   79.0776   113.0205
     F30 2006-09-13 15:00:54                  214.6256 winter     Phase_3   48.3568   74.2356   105.5722
     F30 2006-09-13 18:00:57                  214.7507 winter     Phase_3   43.1014   65.6822    92.9436
     F30 2006-09-13 21:00:59                  214.8757 winter     Phase_3   40.1000   60.0035    83.8541
     F30 2006-09-14 00:00:48                  215.0006 winter     Phase_3   39.5510   59.3204    83.0331
     F30 2006-09-14 09:01:09                  215.3758 winter     Phase_3   41.1962   62.1203    87.2732
     F30 2006-09-14 15:01:09                  215.6258 winter     Phase_3   42.2766   64.1989    90.6273
     F30 2006-09-14 18:00:59                  215.7507 winter     Phase_3   42.6028   64.7423    91.4408
     F30 2006-09-14 21:00:48                  215.8756 winter     Phase_3   42.4938   65.0439    92.3173
     F30 2006-09-15 00:00:49                  216.0006 winter     Phase_3   40.8605   61.9017    87.2434
     F30 2006-09-15 09:01:21                  216.3759 winter     Phase_3   44.7463   67.4155    94.6567
     F30 2006-09-15 12:00:53                  216.5006 winter     Phase_3   44.7774   67.7607    95.4292
     F30 2006-09-15 15:01:42                  216.6262 winter     Phase_3   43.8895   65.8684    92.2382
     F30 2006-09-15 18:01:22                  216.7509 winter     Phase_3   43.7814   65.4441    91.3918
     F30 2006-09-15 21:00:59                  216.8757 winter     Phase_3   44.0263   66.1733    92.7611
     F30 2006-09-16 00:01:18                  217.0009 winter     Phase_3   43.7113   65.6003    91.8619
     F30 2006-09-16 03:01:26                  217.1260 winter     Phase_3   43.2349   65.3821    92.0366
     F30 2006-09-16 06:01:06                  217.2508 winter     Phase_3   42.7728   64.8908    91.5449
     F30 2006-09-16 09:00:49                  217.3756 winter     Phase_3   42.6473   64.8013    91.5158
     F30 2006-09-16 12:00:49                  217.5006 winter     Phase_3   42.3580   63.8863    89.7678
     F30 2006-09-16 15:00:59                  217.6257 winter     Phase_3   42.3693   63.7779    89.4948
     F30 2006-09-16 18:01:11                  217.7508 winter     Phase_3   42.7659   64.9481    91.6909
     F30 2006-09-17 00:00:59                  218.0007 winter     Phase_3   42.4656   64.6281    91.3702
     F30 2006-09-17 03:00:49                  218.1256 winter     Phase_3   41.8033   64.8806    92.9490
     F30 2006-09-17 06:01:00                  218.2507 winter     Phase_3   40.7805   63.2065    90.4676
     F30 2006-09-17 09:00:49                  218.3756 winter     Phase_3   43.9044   69.3138   100.4313
     F30 2006-09-17 12:01:20                  218.5009 winter     Phase_3   44.8411   70.0515   100.7951
     F30 2006-09-17 15:00:59                  218.6257 winter     Phase_3   45.0929   70.4583   101.3933
     F30 2006-09-17 18:01:18                  218.7509 winter     Phase_3   44.8376   70.4162   101.6750
     F30 2006-09-17 21:00:59                  218.8757 winter     Phase_3   43.8988   68.3217    98.0599
     F30 2006-09-18 00:01:12                  219.0008 winter     Phase_3   43.6557   67.8886    97.3856
     F30 2006-09-18 03:01:06                  219.1258 winter     Phase_3   44.2425   69.7656   101.0077
     F30 2006-09-18 06:00:57                  219.2507 winter     Phase_3   43.7115   69.1344   100.2911
     F30 2006-09-18 09:02:12                  219.3765 winter     Phase_3   43.5102   68.8118    99.8192
     F30 2006-09-18 12:01:04                  219.5007 winter     Phase_3   42.7971   67.5563    97.8759
     F30 2006-09-18 15:01:43                  219.6262 winter     Phase_3   42.8660   67.6743    98.0557
     F30 2006-09-18 18:01:18                  219.7509 winter     Phase_3   42.4117   67.2276    97.6676
     F30 2006-09-18 21:01:23                  219.8760 winter     Phase_3   42.4604   67.1752    97.4677
     F30 2006-09-19 00:00:56                  220.0006 winter     Phase_3   42.0591   66.3708    96.1389
     F30 2006-09-19 03:00:57                  220.1257 winter     Phase_3   41.0351   65.3709    95.2817
     F30 2006-09-19 06:01:37                  220.2511 winter     Phase_3   35.9129   64.6991   101.8183
     F30 2006-09-19 09:01:28                  220.3760 winter     Phase_3   39.1354   62.3025    90.7690
     F30 2006-09-19 12:00:49                  220.5006 winter     Phase_3   39.0114   61.6270    89.3301
     F30 2006-09-19 15:00:59                  220.6257 winter     Phase_3   38.4489   60.7772    88.1353
     F30 2006-09-19 18:01:06                  220.7508 winter     Phase_3   38.8081   61.0015    88.1332
     F30 2006-09-20 00:00:49                  221.0006 winter     Phase_3   39.0969   60.0452    85.4158
     F30 2006-09-20 03:00:49                  221.1256 winter     Phase_3   39.4834   60.9330    86.9618
     F30 2006-09-20 06:00:59                  221.2507 winter     Phase_3   40.1091   62.0789    88.7703
     F30 2006-09-20 09:01:04                  221.3757 winter     Phase_3   37.8512   69.1605   109.7500
     F30 2006-09-20 12:01:13                  221.5008 winter     Phase_3   37.0759   67.3250   106.4479
     F30 2006-09-20 15:00:44                  221.6255 winter     Phase_3   37.1230   66.9593   105.4501
     F30 2006-09-20 18:01:08                  221.7508 winter     Phase_3   37.7994   68.1173   107.2158
     F30 2006-09-20 21:00:52                  221.8756 winter     Phase_3   37.0887   64.4175    99.1666
     F30 2006-09-21 00:01:00                  222.0007 winter     Phase_3   37.0354   64.8117   100.2326
     F30 2006-09-21 03:01:38                  222.1261 winter     Phase_3   36.1166   66.1506   105.1223
     F30 2006-09-21 06:00:50                  222.2506 winter     Phase_3   35.7235   64.6423   101.9949
     F30 2006-09-21 09:00:59                  222.3757 winter     Phase_3   35.5994   63.6410    99.6937
     F30 2006-09-21 12:00:59                  222.5007 winter     Phase_3   40.6765   73.6256   116.1885
     F30 2006-09-21 15:00:49                  222.6256 winter     Phase_3   40.7957   74.1597   117.3290
     F30 2006-09-21 18:01:18                  222.7509 winter     Phase_3   39.1186   69.0087   107.2429
     F30 2006-09-21 21:01:14                  222.8759 winter     Phase_3   39.0299   68.2367   105.4682
     F30 2006-09-22 00:01:10                  223.0008 winter     Phase_3   39.2433   68.9037   106.7754
     F30 2006-09-22 03:01:29                  223.1260 winter     Phase_3   38.0732   74.5307   123.0276
     F30 2006-09-22 06:01:21                  223.2509 winter     Phase_3   37.3730   70.2867   113.4225
     F30 2006-09-22 09:01:09                  223.3758 winter     Phase_3   37.4116   71.3501   116.0618
     F30 2006-09-22 12:01:07                  223.5008 winter     Phase_3   39.7938   61.7114    88.3604
     F30 2006-09-22 15:00:52                  223.6256 winter     Phase_3   40.4041   63.3432    91.3567
     F30 2006-09-22 18:01:09                  223.7508 winter     Phase_3   40.6614   79.0627   130.0184
     F30 2006-09-22 21:01:09                  223.8758 winter     Phase_3   41.0044   79.1023   129.5077
     F30 2006-09-23 03:01:29                  224.1260 winter     Phase_3   45.4140   94.0367   160.0560
     F30 2006-09-23 09:01:09                  224.3758 winter     Phase_3   45.6695   94.0988   159.7379
     F30 2006-09-23 12:01:06                  224.5008 winter     Phase_3    0.4529    1.0591     1.9184
     F30 2006-09-23 15:00:59                  224.6257 winter     Phase_3   44.1440   99.5985   177.2283
     F30 2006-09-23 18:01:24                  224.7510 winter     Phase_3    0.4459    1.0578     1.9296
     F30 2006-09-23 21:00:59                  224.8757 winter     Phase_3    0.4459    1.0904     2.0182
     F30 2006-09-24 00:01:09                  225.0008 winter     Phase_3    0.4691    1.0558     1.8766
     F30 2006-09-24 03:01:28                  225.1260 winter     Phase_3    0.4730    1.0814     1.9369
     F30 2006-09-24 06:01:28                  225.2510 winter     Phase_3    0.4641    1.2255     2.3501
     F30 2006-09-24 09:01:08                  225.3758 winter     Phase_3    0.4290    1.0934     2.0632
     F30 2006-09-24 12:00:55                  225.5006 winter     Phase_3    0.4248    1.0975     2.0840
     F30 2006-09-24 15:00:49                  225.6256 winter     Phase_3    0.4210    1.0576     1.9822
     F30 2006-09-24 18:01:12                  225.7508 winter     Phase_3   40.2556   99.1203   184.0466
     F30 2006-09-24 21:00:48                  225.8756 winter     Phase_3   40.0920   98.1802   181.8308
     F30 2006-09-25 00:00:55                  226.0006 winter     Phase_3   40.4451   98.5606   182.1109
     F30 2006-09-25 06:01:25                  226.2510 winter     Phase_3    0.4265    1.0652     1.9912
     F30 2006-09-25 09:00:53                  226.3756 winter     Phase_3    0.4234    1.1526     2.2404
     F30 2006-09-25 12:01:20                  226.5009 winter     Phase_3    0.4220    1.1653     2.2796
     F30 2006-09-25 21:00:58                  226.8757 winter     Phase_3   39.5611   94.9009   174.0337
     F30 2006-09-26 00:00:55                  227.0006 winter     Phase_3    0.4156    1.0384     1.9416
     F30 2006-09-26 03:01:29                  227.1260 winter     Phase_3   42.4819   94.1269   165.9649
     F30 2006-09-26 06:00:49                  227.2506 winter     Phase_3   42.6798   97.6969   175.1045
     F30 2006-09-26 09:01:09                  227.3758 winter     Phase_3    0.4303    1.0174     1.8526
     F30 2006-09-26 12:00:49                  227.5006 winter     Phase_3   43.2682   97.7905   174.1615
     F30 2006-09-26 15:01:18                  227.6259 winter     Phase_3   43.7094   98.6087   175.4589
     F30 2006-09-26 18:01:18                  227.7509 winter     Phase_3    0.4167    1.0612     2.0018
     F30 2006-09-26 21:01:14                  227.8759 winter     Phase_3    0.4287    1.0284     1.8859
     F30 2006-09-27 00:00:49                  228.0006 winter     Phase_3    0.4259    1.0063     1.8319
     F30 2006-09-27 03:01:18                  228.1259 winter     Phase_3    0.4321    1.0007     1.8040
     F30 2006-09-27 06:00:59                  228.2507 winter     Phase_3    0.4280    1.0617     1.9782
     F30 2006-09-27 09:01:35                  228.3761 winter     Phase_3   46.4670   94.8163   160.1165
     F30 2006-09-27 12:00:55                  228.5006 winter     Phase_3   49.9490   82.4999   123.0854
     F30 2006-09-27 15:01:09                  228.6258 winter     Phase_3   49.1610   81.0701   120.8306
     F30 2006-09-27 18:01:20                  228.7509 winter     Phase_3   44.4846   89.3458   149.5866
     F30 2006-09-27 21:00:49                  228.8756 winter     Phase_3   44.3987   90.8892   153.7529
     F30 2006-09-28 00:00:59                  229.0007 winter     Phase_3   44.8895   90.2846   151.2734
     F30 2006-09-28 03:01:26                  229.1260 winter     Phase_3   49.5636   81.6770   121.6811
     F30 2006-09-28 06:00:49                  229.2506 winter     Phase_3   49.3236   81.6324   121.9482
     F30 2006-09-28 09:01:39                  229.3761 winter     Phase_3    0.4767    1.0051     1.7273
     F30 2006-09-28 12:01:10                  229.5008 winter     Phase_3    0.4836    1.0107     1.7287
     F30 2006-09-28 15:00:59                  229.6257 winter     Phase_3   53.3919   88.8237   133.1264
     F30 2006-09-28 18:00:57                  229.7507 winter     Phase_3    0.4944    1.0812     1.8938
     F30 2006-09-28 21:01:10                  229.8758 winter     Phase_3   55.8252   92.4885   138.2565
/var/folders/g5/99t7y6l16nld30m4l79m3pzw0000gn/T/ipykernel_7807/3090686660.py:292: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(
/var/folders/g5/99t7y6l16nld30m4l79m3pzw0000gn/T/ipykernel_7807/3090686660.py:328: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(
/var/folders/g5/99t7y6l16nld30m4l79m3pzw0000gn/T/ipykernel_7807/3090686660.py:393: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(

In [ ]:
# =============================================================================
# WHITE-TAILED DEER AGE / FAWN AUDIT
#
# Purpose
# -------
# 1. Read all sheets from the 2006-2010 CWD test workbooks.
# 2. Find age columns automatically.
# 3. Preserve raw ages exactly as recorded.
# 4. Correctly flag ages containing "+".
# 5. Parse numeric ages and age ranges.
# 6. Identify definite fawn records.
# 7. Identify records younger than 1.5 years.
# 8. Compare those deer against the exact two datasets used by the autoencoder.
#
# IMPORTANT
# ---------
# This script DOES NOT modify any source files.
# It only creates audit CSVs.
# =============================================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd


# =============================================================================
# PATHS
# =============================================================================

CWD_FILES = {

    2006:
        "<LOCAL_PATH>"
        "2006 Test Results.xls",

    2007:
        "<LOCAL_PATH>"
        "2007 Test Results.xls",

    2008:
        "<LOCAL_PATH>"
        "2008 Test Results.xls",

    2009:
        "<LOCAL_PATH>"
        "2009 Test Results.xlsx",

    2010:
        "<LOCAL_PATH>"
        "2010 Test Results.xlsx",
}


AE_FILES = {

    "CWD_negative_training":
        "<LOCAL_PATH>"
        "CWD_negative_training_with_seasons2.csv",

    "CWD_positive_testing":
        "<LOCAL_PATH>"
        "CWD_positive_testing_with_seasons2.csv",
}


OUTPUT_DIR = Path(
    "<LOCAL_PATH>"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# HELPERS
# =============================================================================

def normalize_column_name(name):
    """
    Convert column names into a simple comparable form.

    Examples
    --------
    'Deer ID'      -> 'deer_id'
    'Date Capture' -> 'date_capture'
    """

    name = str(name).strip().lower()

    name = re.sub(
        r"[^a-z0-9]+",
        "_",
        name,
    )

    return name.strip("_")


def normalize_deer_id(value):
    """
    Standardize deer IDs without changing their basic identity.
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    # Excel sometimes turns purely numeric IDs into "123.0"
    value = re.sub(
        r"\.0$",
        "",
        value,
    )

    # Remove internal whitespace only
    value = re.sub(
        r"\s+",
        "",
        value,
    )

    if value in {
        "",
        "NAN",
        "NONE",
        "NA",
    }:
        return np.nan

    return value


def find_best_column(columns, candidates):
    """
    Find a column using normalized exact matches first.
    """

    lookup = {
        normalize_column_name(col): col
        for col in columns
    }

    for candidate in candidates:

        if candidate in lookup:
            return lookup[candidate]

    return None


def find_age_columns(columns):
    """
    Find columns where 'age' is a true word/token.

    This avoids accidentally matching unrelated words.
    """

    age_columns = []

    for col in columns:

        normalized = normalize_column_name(col)

        if re.search(
            r"(^|_)age($|_)",
            normalized,
        ):
            age_columns.append(col)

    return age_columns


# =============================================================================
# AGE PARSER
# =============================================================================

def parse_age(value):
    """
    Parse values such as:

        0.5
        1.5
        5+
        3-5
        3–5
        Fawn
        Yearling

    '+' is treated as a lower-bound indicator.

    For example:

        5+

    means:

        minimum age = 5
        maximum age = unknown

    rather than pretending the exact age is 5.
    """

    result = {
        "age_raw": np.nan,
        "age_min_years": np.nan,
        "age_max_years": np.nan,
        "age_midpoint_years": np.nan,
        "age_has_plus": False,
        "age_is_range": False,
        "explicit_fawn_label": False,
        "explicit_yearling_label": False,
    }

    if pd.isna(value):
        return pd.Series(result)

    raw = str(value).strip()

    if raw == "":
        return pd.Series(result)

    result["age_raw"] = raw

    lower = raw.lower()

    result["age_has_plus"] = "+" in raw

    result["explicit_fawn_label"] = (
        "fawn" in lower
    )

    result["explicit_yearling_label"] = (
        "yearling" in lower
    )


    # Standardize long dashes
    standardized = (
        raw
        .replace("–", "-")
        .replace("—", "-")
    )


    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        standardized,
    )

    numbers = [
        float(x)
        for x in numbers
    ]


    # -------------------------------------------------------------------------
    # Numeric age range
    #
    # Example:
    #
    #     3-5
    #
    # -------------------------------------------------------------------------

    if (
        len(numbers) >= 2
        and re.search(
            r"\d(?:\.\d+)?\s*-\s*\d",
            standardized,
        )
    ):

        age_min = min(
            numbers[0],
            numbers[1],
        )

        age_max = max(
            numbers[0],
            numbers[1],
        )

        result["age_min_years"] = age_min
        result["age_max_years"] = age_max

        result["age_midpoint_years"] = (
            age_min + age_max
        ) / 2

        result["age_is_range"] = True


    # -------------------------------------------------------------------------
    # Single numeric value
    #
    # -------------------------------------------------------------------------

    elif len(numbers) >= 1:

        age_value = numbers[0]

        result["age_min_years"] = age_value

        # "5+" has no known upper bound.
        if result["age_has_plus"]:

            result["age_max_years"] = np.nan

        else:

            result["age_max_years"] = age_value

        result["age_midpoint_years"] = age_value


    return pd.Series(result)


# =============================================================================
# LIFE-STAGE CLASSIFICATION
# =============================================================================

def classify_age(row):
    """
    Conservative classifications.

    FAWN:
        Explicitly labelled fawn, OR
        known maximum age < 1 year.

    UNDER 1.5:
        Useful because Edmunds et al. defined adult/yearling
        home ranges as age >= 1.5 years.
    """

    explicit_fawn = (
        row["explicit_fawn_label"] is True
        or row["explicit_fawn_label"] == True
    )

    age_min = row["age_min_years"]
    age_max = row["age_max_years"]


    definitely_fawn = explicit_fawn


    if pd.notna(age_max):

        if age_max < 1.0:
            definitely_fawn = True


    # -------------------------------------------------------------------------
    # Definite >= 1.5
    # -------------------------------------------------------------------------

    definitely_1_5_plus = (

        pd.notna(age_min)

        and age_min >= 1.5
    )


    # -------------------------------------------------------------------------
    # Possible/known < 1.5
    # -------------------------------------------------------------------------

    possibly_under_1_5 = (

        pd.notna(age_min)

        and age_min < 1.5
    )


    return pd.Series({

        "definitely_fawn":
            definitely_fawn,

        "definitely_age_1_5_plus":
            definitely_1_5_plus,

        "possibly_under_1_5":
            possibly_under_1_5,
    })


# =============================================================================
# COLUMN CANDIDATES
# =============================================================================

ID_CANDIDATES = [

    "deer_id",
    "animal_id",
    "individual_id",
    "individual_local_identifier",
    "collar_id",
    "deer",
    "animal",
    "id",
]


DATE_CANDIDATES = [

    "date_capture",
    "capture_date",

    "date_tested",
    "test_date",

    "sample_date",

    "date",
]


CWD_CANDIDATES = [

    "cwd_status",
    "cwd_result",
    "test_result",
    "result",
    "cwd",
]


SEX_CANDIDATES = [

    "sex",
    "gender",
]


# =============================================================================
# READ ALL CWD WORKBOOKS
# =============================================================================

age_parts = []

inventory_parts = []


for year, file_path in CWD_FILES.items():

    path = Path(file_path)


    print(
        "\n",
        "=" * 100,
        sep="",
    )

    print(
        f"YEAR: {year}"
    )

    print(
        f"FILE: {path}"
    )

    print(
        "=" * 100
    )


    if not path.exists():

        print(
            "FILE NOT FOUND"
        )

        continue


    # -------------------------------------------------------------------------
    # Select correct Excel engine
    # -------------------------------------------------------------------------

    if path.suffix.lower() == ".xls":

        engine = "xlrd"

    else:

        engine = "openpyxl"


    # -------------------------------------------------------------------------
    # Read every sheet
    # -------------------------------------------------------------------------

    workbook = pd.read_excel(
        path,
        sheet_name=None,
        engine=engine,
    )


    print(
        "Sheets:",
        list(workbook.keys()),
    )


    for sheet_name, df in workbook.items():

        print(
            f"\nSheet: {sheet_name}"
        )

        print(
            f"Rows: {len(df):,}"
        )


        age_columns = find_age_columns(
            df.columns
        )


        id_col = find_best_column(
            df.columns,
            ID_CANDIDATES,
        )


        date_col = find_best_column(
            df.columns,
            DATE_CANDIDATES,
        )


        cwd_col = find_best_column(
            df.columns,
            CWD_CANDIDATES,
        )


        sex_col = find_best_column(
            df.columns,
            SEX_CANDIDATES,
        )


        print(
            "ID column:",
            id_col,
        )

        print(
            "Age columns:",
            age_columns,
        )

        print(
            "Date column:",
            date_col,
        )

        print(
            "CWD column:",
            cwd_col,
        )


        inventory_parts.append({

            "year":
                year,

            "file":
                str(path),

            "sheet":
                sheet_name,

            "n_rows":
                len(df),

            "id_column":
                id_col,

            "age_columns":
                " | ".join(
                    map(
                        str,
                        age_columns,
                    )
                ),

            "date_column":
                date_col,

            "cwd_column":
                cwd_col,

            "sex_column":
                sex_col,
        })


        # ---------------------------------------------------------------------
        # Skip sheets without age information
        # ---------------------------------------------------------------------

        if len(age_columns) == 0:
            continue


        for age_col in age_columns:

            out = pd.DataFrame()


            # -----------------------------------------------------------------
            # Deer ID
            # -----------------------------------------------------------------

            if id_col is not None:

                out["deer_id_raw"] = df[
                    id_col
                ]

                out["deer_id"] = df[
                    id_col
                ].map(
                    normalize_deer_id
                )

            else:

                out["deer_id_raw"] = np.nan
                out["deer_id"] = np.nan


            # -----------------------------------------------------------------
            # Date
            # -----------------------------------------------------------------

            if date_col is not None:

                out["record_date_raw"] = df[
                    date_col
                ]

                out["record_date"] = pd.to_datetime(
                    df[date_col],
                    errors="coerce",
                )

            else:

                out["record_date_raw"] = np.nan
                out["record_date"] = pd.NaT


            # -----------------------------------------------------------------
            # Other metadata
            # -----------------------------------------------------------------

            if cwd_col is not None:

                out["cwd_status_raw"] = df[
                    cwd_col
                ]

            else:

                out["cwd_status_raw"] = np.nan


            if sex_col is not None:

                out["sex_raw"] = df[
                    sex_col
                ]

            else:

                out["sex_raw"] = np.nan


            # -----------------------------------------------------------------
            # Age
            # -----------------------------------------------------------------

            parsed_age = df[
                age_col
            ].apply(
                parse_age
            )


            out = pd.concat(
                [
                    out.reset_index(
                        drop=True
                    ),

                    parsed_age.reset_index(
                        drop=True
                    ),
                ],
                axis=1,
            )


            classification = out.apply(
                classify_age,
                axis=1,
            )


            out = pd.concat(
                [
                    out,
                    classification,
                ],
                axis=1,
            )


            # -----------------------------------------------------------------
            # Provenance
            # -----------------------------------------------------------------

            out["source_year"] = year
            out["source_file"] = path.name
            out["source_sheet"] = sheet_name
            out["source_age_column"] = str(
                age_col
            )


            # -----------------------------------------------------------------
            # Drop completely empty age records
            # -----------------------------------------------------------------

            out = out.loc[
                out["age_raw"].notna()
            ].copy()


            age_parts.append(
                out
            )


# =============================================================================
# COMBINE AGE RECORDS
# =============================================================================

if len(age_parts) == 0:

    raise RuntimeError(
        "No age records were found in any workbook."
    )


age_records = pd.concat(
    age_parts,
    ignore_index=True,
)


# =============================================================================
# SAVE WORKBOOK INVENTORY
# =============================================================================

inventory = pd.DataFrame(
    inventory_parts
)


inventory.to_csv(
    OUTPUT_DIR /
    "workbook_sheet_inventory.csv",
    index=False,
)


# =============================================================================
# SAVE ALL AGE RECORDS
# =============================================================================

age_records.to_csv(
    OUTPUT_DIR /
    "all_age_records.csv",
    index=False,
)


# =============================================================================
# BASIC AGE AUDIT
# =============================================================================

print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "AGE AUDIT"
)

print(
    "=" * 100
)


print(
    "\nTotal age records:",
    f"{len(age_records):,}",
)


print(
    "Unique deer with age records:",
    age_records["deer_id"].nunique(
        dropna=True
    ),
)


print(
    "\nRaw age values:"
)


age_counts = (
    age_records["age_raw"]
    .astype(str)
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "age_raw"
    )
    .reset_index(
        name="n"
    )
)


print(
    age_counts.to_string(
        index=False
    )
)


age_counts.to_csv(
    OUTPUT_DIR /
    "raw_age_value_counts.csv",
    index=False,
)


# =============================================================================
# PLUS-SIGN AGES
# =============================================================================

plus_records = age_records.loc[
    age_records["age_has_plus"]
].copy()


print(
    "\nRecords containing '+':",
    len(plus_records),
)


if len(plus_records) > 0:

    print(
        "\nPlus-sign age values:"
    )

    print(
        plus_records[
            "age_raw"
        ].value_counts()
    )


plus_records.to_csv(
    OUTPUT_DIR /
    "plus_sign_age_records.csv",
    index=False,
)


# =============================================================================
# FAWN RECORDS
# =============================================================================

fawn_records = age_records.loc[
    age_records["definitely_fawn"]
].copy()


print(
    "\nDefinite fawn-age records:",
    len(fawn_records),
)


print(
    "Unique deer with a definite fawn record:",
    fawn_records["deer_id"].nunique(
        dropna=True
    ),
)


if len(fawn_records) > 0:

    print(
        "\nDeer with fawn records:"
    )

    print(
        sorted(
            fawn_records[
                "deer_id"
            ].dropna().unique()
        )
    )


fawn_records.to_csv(
    OUTPUT_DIR /
    "definite_fawn_records.csv",
    index=False,
)


# =============================================================================
# < 1.5 YEAR RECORDS
# =============================================================================

under_1_5 = age_records.loc[
    age_records[
        "possibly_under_1_5"
    ]
].copy()


under_1_5.to_csv(
    OUTPUT_DIR /
    "records_possibly_under_1_5_years.csv",
    index=False,
)


print(
    "\nRecords potentially younger than 1.5 years:",
    len(under_1_5),
)


print(
    "Unique deer potentially younger than 1.5:",
    under_1_5["deer_id"].nunique(
        dropna=True
    ),
)


# =============================================================================
# LOAD THE EXACT AUTOENCODER INPUT DATASETS
# =============================================================================

model_parts = []


for dataset_name, file_path in AE_FILES.items():

    path = Path(
        file_path
    )


    if not path.exists():

        print(
            "\nWARNING:",
            path,
            "does not exist."
        )

        continue


    df = pd.read_csv(
        path,
        low_memory=False,
    )


    id_col = find_best_column(
        df.columns,
        ID_CANDIDATES,
    )


    if id_col is None:

        raise RuntimeError(
            f"No deer ID column found in {path}"
        )


    model_ids = pd.DataFrame({

        "deer_id":
            df[id_col].map(
                normalize_deer_id
            ),

        "model_dataset":
            dataset_name,
    })


    model_ids = model_ids.dropna(
        subset=[
            "deer_id"
        ]
    )


    model_parts.append(
        model_ids
    )


model_id_rows = pd.concat(
    model_parts,
    ignore_index=True,
)


# =============================================================================
# MODEL ROW COUNTS BY DEER
# =============================================================================

model_counts = (

    model_id_rows

    .groupby(
        [
            "deer_id",
            "model_dataset",
        ]
    )

    .size()

    .reset_index(
        name="model_rows"
    )
)


model_ids = set(
    model_id_rows[
        "deer_id"
    ].unique()
)


print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "AUTOENCODER OVERLAP"
)

print(
    "=" * 100
)


print(
    "\nUnique deer in autoencoder datasets:",
    len(model_ids),
)


# =============================================================================
# KEEP AGE RECORDS FOR MODEL DEER ONLY
# =============================================================================

model_age_records = age_records.loc[
    age_records[
        "deer_id"
    ].isin(
        model_ids
    )
].copy()


model_age_records.to_csv(
    OUTPUT_DIR /
    "model_deer_age_records.csv",
    index=False,
)


print(
    "Model deer with at least one age record:",
    model_age_records[
        "deer_id"
    ].nunique(
        dropna=True
    ),
)


# =============================================================================
# SUMMARY PER MODEL DEER
# =============================================================================

def summarize_deer(group):

    raw_values = sorted(
        {
            str(x)
            for x in group[
                "age_raw"
            ].dropna()
        }
    )


    source_years = sorted(
        {
            int(x)
            for x in group[
                "source_year"
            ].dropna()
        }
    )


    return pd.Series({

        "n_age_records":
            len(group),

        "minimum_age_recorded":
            group[
                "age_min_years"
            ].min(),

        "maximum_exact_or_range_age":
            group[
                "age_max_years"
            ].max(),

        "any_plus_age":
            group[
                "age_has_plus"
            ].any(),

        "any_definite_fawn_record":
            group[
                "definitely_fawn"
            ].any(),

        "any_record_under_1_5":
            group[
                "possibly_under_1_5"
            ].any(),

        "all_raw_age_values":
            " | ".join(
                raw_values
            ),

        "source_years":
            " | ".join(
                map(
                    str,
                    source_years,
                )
            ),
    })


model_deer_summary = (

    model_age_records

    .groupby(
        "deer_id",
        dropna=False,
    )

    .apply(
        summarize_deer
    )

    .reset_index()
)


# =============================================================================
# ADD MODEL DATASET MEMBERSHIP
# =============================================================================

membership = (

    model_counts

    .pivot_table(
        index="deer_id",
        columns="model_dataset",
        values="model_rows",
        aggfunc="sum",
        fill_value=0,
    )

    .reset_index()
)


membership.columns.name = None


model_deer_summary = model_deer_summary.merge(
    membership,
    on="deer_id",
    how="outer",
)


model_deer_summary.to_csv(
    OUTPUT_DIR /
    "model_deer_age_summary.csv",
    index=False,
)


# =============================================================================
# MODEL DEER WITH FAWN RECORDS
# =============================================================================

model_fawn_records = model_age_records.loc[
    model_age_records[
        "definitely_fawn"
    ]
].copy()


model_fawn_records.to_csv(
    OUTPUT_DIR /
    "model_deer_with_fawn_records.csv",
    index=False,
)


model_fawn_ids = sorted(
    model_fawn_records[
        "deer_id"
    ].dropna().unique()
)


print(
    "\nModel deer with definite fawn records:",
    len(model_fawn_ids),
)


print(
    model_fawn_ids
)


# =============================================================================
# MODEL DEER WITH ANY RECORD BELOW 1.5
# =============================================================================

model_under_1_5 = model_age_records.loc[
    model_age_records[
        "possibly_under_1_5"
    ]
].copy()


model_under_1_5.to_csv(
    OUTPUT_DIR /
    "model_deer_records_under_1_5.csv",
    index=False,
)


under_1_5_ids = sorted(
    model_under_1_5[
        "deer_id"
    ].dropna().unique()
)


print(
    "\nModel deer with an age record below 1.5:",
    len(under_1_5_ids),
)


print(
    under_1_5_ids
)


# =============================================================================
# FINAL REPORT
# =============================================================================

print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "AUDIT COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)


print(
    "\nMain files:"
)

for filename in [

    "workbook_sheet_inventory.csv",
    "all_age_records.csv",
    "raw_age_value_counts.csv",
    "plus_sign_age_records.csv",
    "definite_fawn_records.csv",
    "records_possibly_under_1_5_years.csv",
    "model_deer_age_records.csv",
    "model_deer_age_summary.csv",
    "model_deer_with_fawn_records.csv",
    "model_deer_records_under_1_5.csv",
]:

    print(
        "  ",
        filename,
    )

In [ ]:
# =============================================================================
# DOUBLE-CHECK WHETHER FAWN-AGE TELEMETRY ACTUALLY ENTERED THE AUTOENCODER
#
# Key question:
#
#   Did any telemetry rows in the ACTUAL model datasets occur before
#   a deer reached approximately 1.5 years old?
#
# This script:
#
# 1. Loads the age audit we already created.
# 2. Loads the two exact AE input datasets.
# 3. Finds each deer's actual telemetry start/end.
# 4. Uses dated age records to estimate the date that deer reached age 1.5.
# 5. Counts telemetry rows occurring before that threshold.
# 6. Prints detailed records for the 12 deer of concern.
#
# NOTHING IS MODIFIED.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# PATHS
# =============================================================================

AGE_FILE = (
    "<LOCAL_PATH>"
    "Fawn_Age_Audit/all_age_records.csv"
)


AE_FILES = {

    "CWD_negative_training":
        "<LOCAL_PATH>"
        "CWD_negative_training_with_seasons2.csv",

    "CWD_positive_testing":
        "<LOCAL_PATH>"
        "CWD_positive_testing_with_seasons2.csv",
}


OUTPUT_DIR = Path(
    "<LOCAL_PATH>"
)


DEER_OF_CONCERN = [

    "A43",
    "A47",
    "A48",
    "A50",
    "F100",
    "F106",
    "F107",
    "F109",
    "F91",
    "F94",
    "F97",
    "F99",
]


# =============================================================================
# HELPERS
# =============================================================================

def normalize_id(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    if x.endswith(".0"):
        x = x[:-2]

    return x


def find_column(columns, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in columns
    }

    for candidate in candidates:

        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


# =============================================================================
# LOAD AGE RECORDS
# =============================================================================

age = pd.read_csv(
    AGE_FILE,
    low_memory=False,
)


age["deer_id"] = age[
    "deer_id"
].map(
    normalize_id
)


age["record_date"] = pd.to_datetime(
    age["record_date"],
    errors="coerce",
)


# =============================================================================
# KEEP RECORDS THAT CAN INFORM AGE TIMING
#
# For estimating age 1.5 date, use exact numeric ages.
#
# Exclude:
#   - "+" ages
#   - age ranges
#
# because those do not represent an exact age.
# =============================================================================

exact_age = age.loc[

    age["age_min_years"].notna()

    & age["age_max_years"].notna()

    & (
        age["age_min_years"]
        == age["age_max_years"]
    )

    & ~age["age_has_plus"].fillna(False)

    & ~age["age_is_range"].fillna(False)

    & age["record_date"].notna()

].copy()


# =============================================================================
# ESTIMATE DATE EACH DEER REACHED AGE 1.5
#
# Example:
#
# age = 0.5 on 2007-02-01
#
# estimated 1.5-year date:
#
#     2008-02-01
#
# Another age record, such as 1.5 in 2008, provides another independent
# estimate. We take the median of all exact-age estimates.
# =============================================================================

exact_age["estimated_age_1_5_date"] = (

    exact_age["record_date"]

    + pd.to_timedelta(
        (
            1.5
            - exact_age["age_min_years"]
        )
        * 365.25,
        unit="D",
    )
)


# =============================================================================
# SHOW ALL ESTIMATES FOR THE 12 DEER
# =============================================================================

concern_age_records = exact_age.loc[
    exact_age["deer_id"].isin(
        DEER_OF_CONCERN
    )
].copy()


print(
    "\n",
    "=" * 110,
    sep="",
)

print(
    "DATED AGE RECORDS FOR THE 12 DEER"
)

print(
    "=" * 110
)


show_cols = [

    "deer_id",
    "record_date",
    "age_raw",
    "age_min_years",
    "estimated_age_1_5_date",
    "source_year",
    "source_file",
    "source_sheet",
]


available_show_cols = [
    x
    for x in show_cols
    if x in concern_age_records.columns
]


print(
    concern_age_records[
        available_show_cols
    ]
    .sort_values(
        [
            "deer_id",
            "record_date",
        ]
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# SUMMARIZE ESTIMATED 1.5-YEAR DATE PER DEER
# =============================================================================

age_thresholds = (

    exact_age

    .groupby(
        "deer_id",
        as_index=False,
    )

    .agg(

        estimated_age_1_5_date=
            (
                "estimated_age_1_5_date",
                "median",
            ),

        earliest_age_1_5_estimate=
            (
                "estimated_age_1_5_date",
                "min",
            ),

        latest_age_1_5_estimate=
            (
                "estimated_age_1_5_date",
                "max",
            ),

        n_exact_age_records=
            (
                "age_min_years",
                "size",
            ),

        youngest_recorded_age=
            (
                "age_min_years",
                "min",
            ),

        oldest_recorded_age=
            (
                "age_min_years",
                "max",
            ),
    )
)


# =============================================================================
# LOAD ACTUAL AUTOENCODER DATA
# =============================================================================

ae_parts = []


for dataset_name, path in AE_FILES.items():

    df = pd.read_csv(
        path,
        low_memory=False,
    )


    # -------------------------------------------------------------------------
    # Find deer ID
    # -------------------------------------------------------------------------

    id_col = find_column(

        df.columns,

        [
            "id",
            "deer_id",
            "animal_id",
            "individual_id",
        ],
    )


    if id_col is None:

        raise RuntimeError(
            f"No ID column found in {path}"
        )


    # -------------------------------------------------------------------------
    # Find datetime
    # -------------------------------------------------------------------------

    datetime_col = find_column(

        df.columns,

        [
            "datetime",
            "date_time",
            "timestamp",
            "date",
        ],
    )


    if datetime_col is None:

        raise RuntimeError(
            f"No datetime column found in {path}"
        )


    print(
        f"\n{dataset_name}"
    )

    print(
        "ID column:",
        id_col,
    )

    print(
        "Datetime column:",
        datetime_col,
    )


    out = pd.DataFrame({

        "deer_id":
            df[id_col].map(
                normalize_id
            ),

        "telemetry_datetime":
            pd.to_datetime(
                df[datetime_col],
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None),

        "model_dataset":
            dataset_name,
    })


    out["source_row"] = np.arange(
        len(out)
    )


    ae_parts.append(
        out
    )


ae = pd.concat(
    ae_parts,
    ignore_index=True,
)


ae = ae.dropna(
    subset=[
        "deer_id",
        "telemetry_datetime",
    ]
)


# =============================================================================
# TELEMETRY RANGE FOR EACH DEER
# =============================================================================

telemetry_summary = (

    ae

    .groupby(
        "deer_id",
        as_index=False,
    )

    .agg(

        telemetry_start=
            (
                "telemetry_datetime",
                "min",
            ),

        telemetry_end=
            (
                "telemetry_datetime",
                "max",
            ),

        n_telemetry_rows=
            (
                "telemetry_datetime",
                "size",
            ),
    )
)


# =============================================================================
# MERGE AGE THRESHOLD WITH TELEMETRY
# =============================================================================

summary = telemetry_summary.merge(

    age_thresholds,

    on="deer_id",

    how="left",
)


# =============================================================================
# ESTIMATED AGE AT FIRST TELEMETRY
#
# Based on the median estimated date the deer reached 1.5 years.
# =============================================================================

summary["estimated_age_at_telemetry_start"] = (

    1.5

    + (
        summary["telemetry_start"]
        - summary["estimated_age_1_5_date"]
    ).dt.total_seconds()

    / (
        365.25
        * 24
        * 3600
    )
)


# =============================================================================
# CLASSIFY START OF TELEMETRY
# =============================================================================

summary["telemetry_started_before_age_1_5"] = (

    summary["telemetry_start"]
    <
    summary["estimated_age_1_5_date"]
)


# =============================================================================
# MERGE THRESHOLD ONTO EVERY TELEMETRY ROW
# =============================================================================

ae_with_age = ae.merge(

    age_thresholds[
        [
            "deer_id",
            "estimated_age_1_5_date",
            "earliest_age_1_5_estimate",
            "latest_age_1_5_estimate",
        ]
    ],

    on="deer_id",

    how="left",
)


# =============================================================================
# THREE-WAY CLASSIFICATION
#
# definitely_under_1_5:
# telemetry is earlier than EVERY estimated 1.5-year date.
#
# definitely_1_5_plus:
# telemetry is later than EVERY estimated 1.5-year date.
#
# uncertain:
# telemetry falls inside the disagreement range among age records.
# =============================================================================

ae_with_age["definitely_under_1_5"] = (

    ae_with_age["telemetry_datetime"]
    <
    ae_with_age["earliest_age_1_5_estimate"]
)


ae_with_age["definitely_1_5_plus"] = (

    ae_with_age["telemetry_datetime"]
    >=
    ae_with_age["latest_age_1_5_estimate"]
)


ae_with_age["age_timing_uncertain"] = (

    ~ae_with_age["definitely_under_1_5"]

    & ~ae_with_age["definitely_1_5_plus"]

    & ae_with_age[
        "estimated_age_1_5_date"
    ].notna()
)


# =============================================================================
# COUNT ACTUAL MODEL ROWS BY AGE CLASSIFICATION
# =============================================================================

row_age_summary = (

    ae_with_age

    .groupby(
        "deer_id",
        as_index=False,
    )

    .agg(

        n_rows_definitely_under_1_5=
            (
                "definitely_under_1_5",
                "sum",
            ),

        n_rows_age_timing_uncertain=
            (
                "age_timing_uncertain",
                "sum",
            ),

        n_rows_definitely_1_5_plus=
            (
                "definitely_1_5_plus",
                "sum",
            ),
    )
)


summary = summary.merge(

    row_age_summary,

    on="deer_id",

    how="left",
)


# =============================================================================
# ADD FIRST / LAST DEFINITE FAWN-AGE TELEMETRY DATE
# =============================================================================

under_rows = ae_with_age.loc[
    ae_with_age[
        "definitely_under_1_5"
    ]
].copy()


under_date_summary = (

    under_rows

    .groupby(
        "deer_id",
        as_index=False,
    )

    .agg(

        first_under_1_5_telemetry=
            (
                "telemetry_datetime",
                "min",
            ),

        last_under_1_5_telemetry=
            (
                "telemetry_datetime",
                "max",
            ),
    )
)


summary = summary.merge(

    under_date_summary,

    on="deer_id",

    how="left",
)


# =============================================================================
# ONLY THE 12 DEER OF CONCERN
# =============================================================================

concern_summary = summary.loc[
    summary["deer_id"].isin(
        DEER_OF_CONCERN
    )
].copy()


concern_summary = concern_summary.sort_values(
    "deer_id"
)


# =============================================================================
# PRINT RESULTS
# =============================================================================

print(
    "\n",
    "=" * 110,
    sep="",
)

print(
    "ACTUAL TELEMETRY VS AGE-1.5 THRESHOLD"
)

print(
    "=" * 110
)


cols = [

    "deer_id",

    "youngest_recorded_age",

    "telemetry_start",

    "estimated_age_1_5_date",

    "estimated_age_at_telemetry_start",

    "n_telemetry_rows",

    "n_rows_definitely_under_1_5",

    "n_rows_age_timing_uncertain",

    "n_rows_definitely_1_5_plus",
]


print(
    concern_summary[
        cols
    ].to_string(
        index=False
    )
)


# =============================================================================
# DEER THAT ACTUALLY HAVE PRE-1.5 MODEL TELEMETRY
# =============================================================================

actual_problem = concern_summary.loc[

    concern_summary[
        "n_rows_definitely_under_1_5"
    ].fillna(0)

    > 0

].copy()


print(
    "\n",
    "=" * 110,
    sep="",
)

print(
    "DEER WITH TELEMETRY ACTUALLY USED BEFORE AGE 1.5"
)

print(
    "=" * 110
)


if len(actual_problem) == 0:

    print(
        "\nNONE."
    )

    print(
        "\nThe 12 deer were captured/tested while young, "
        "but their model telemetry begins after they reached approximately 1.5 years."
    )

else:

    print(
        actual_problem[
            [
                "deer_id",
                "telemetry_start",
                "first_under_1_5_telemetry",
                "last_under_1_5_telemetry",
                "estimated_age_1_5_date",
                "n_rows_definitely_under_1_5",
                "n_telemetry_rows",
            ]
        ].to_string(
            index=False
        )
    )


# =============================================================================
# SAVE RESULTS
# =============================================================================

concern_summary.to_csv(

    OUTPUT_DIR /
    "fawn_telemetry_double_check_12_deer.csv",

    index=False,
)


ae_with_age.loc[
    ae_with_age["deer_id"].isin(
        DEER_OF_CONCERN
    )
].to_csv(

    OUTPUT_DIR /
    "fawn_telemetry_row_level_audit_12_deer.csv",

    index=False,
)


actual_problem.to_csv(

    OUTPUT_DIR /
    "deer_with_actual_pre_1_5_model_telemetry.csv",

    index=False,
)


print(
    "\nSaved:"
)

print(
    OUTPUT_DIR /
    "fawn_telemetry_double_check_12_deer.csv"
)

print(
    OUTPUT_DIR /
    "fawn_telemetry_row_level_audit_12_deer.csv"
)

print(
    OUTPUT_DIR /
    "deer_with_actual_pre_1_5_model_telemetry.csv"
)

In [ ]:
# =============================================================================
# STRICT FAWN AUDIT
#
# Question:
#   Did any deer used in the autoencoder have an EXPLICIT fawn-age record?
#
# IMPORTANT:
#   We DO NOT estimate birthdays.
#   We DO NOT treat age = 1 as a fawn.
#   We DO NOT extrapolate an age-1.5 date.
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# PATHS
# =============================================================================

AGE_FILE = (
    "<LOCAL_PATH>"
    "Fawn_Age_Audit/all_age_records.csv"
)

AE_FILES = {
    "CWD_negative_training":
        "<LOCAL_PATH>"
        "CWD_negative_training_with_seasons2.csv",

    "CWD_positive_testing":
        "<LOCAL_PATH>"
        "CWD_positive_testing_with_seasons2.csv",
}

OUT_DIR = Path(
    "<LOCAL_PATH>"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# HELPERS
# =============================================================================

def normalize_id(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()

    if x.endswith(".0"):
        x = x[:-2]

    return x


def find_column(columns, choices):

    lookup = {
        str(c).strip().lower(): c
        for c in columns
    }

    for choice in choices:

        if choice.lower() in lookup:
            return lookup[choice.lower()]

    return None


# =============================================================================
# LOAD AGE RECORDS
# =============================================================================

age = pd.read_csv(
    AGE_FILE,
    low_memory=False,
)


age["deer_id"] = age[
    "deer_id"
].map(
    normalize_id
)


age["record_date"] = pd.to_datetime(
    age["record_date"],
    errors="coerce",
)


# =============================================================================
# REMOVE DUPLICATE REPRESENTATIONS
#
# For example:
#
#   A43 | 2008-02-23 | age 1 | By ID
#   A43 | 2008-02-23 | age 1 | By Age
#
# represent the SAME age observation.
# =============================================================================

age_unique = (

    age

    .sort_values(
        [
            "deer_id",
            "record_date",
        ]
    )

    .drop_duplicates(
        subset=[
            "deer_id",
            "record_date",
            "age_raw",
            "source_year",
        ],
        keep="first",
    )

    .copy()
)


print(
    "\nRaw age rows:",
    len(age),
)

print(
    "Unique age observations after deduplication:",
    len(age_unique),
)


# =============================================================================
# STRICT AGE CLASSES
# =============================================================================

age_unique["age_raw_text"] = (

    age_unique["age_raw"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# -----------------------------------------------------------------------------
# Explicit fawn evidence
#
# We use:
#
#   - explicit word "fawn"
#   - numeric age < 1.0
#
# This is deliberately conservative.
#
# Age == 1 is NOT called a fawn.
# -----------------------------------------------------------------------------

age_unique["explicit_fawn"] = (

    age_unique[
        "age_raw_text"
    ].str.contains(
        "fawn",
        na=False,
    )

    |

    (
        age_unique[
            "age_min_years"
        ].notna()

        &

        (
            age_unique[
                "age_min_years"
            ] < 1.0
        )
    )
)


# -----------------------------------------------------------------------------
# Age 1 is retained as its own category.
# -----------------------------------------------------------------------------

age_unique["age_one_record"] = (

    age_unique[
        "age_min_years"
    ].notna()

    &

    np.isclose(
        age_unique[
            "age_min_years"
        ],
        1.0,
    )
)


# -----------------------------------------------------------------------------
# Clearly 1.5+
# -----------------------------------------------------------------------------

age_unique["age_1_5_plus"] = (

    age_unique[
        "age_min_years"
    ].notna()

    &

    (
        age_unique[
            "age_min_years"
        ] >= 1.5
    )
)


# =============================================================================
# LOAD AUTOENCODER DATA
# =============================================================================

telemetry_parts = []


for dataset_name, file_path in AE_FILES.items():

    df = pd.read_csv(
        file_path,
        low_memory=False,
    )


    id_col = find_column(
        df.columns,
        [
            "id",
            "deer_id",
        ],
    )


    date_col = find_column(
        df.columns,
        [
            "datetime",
            "date_time",
            "timestamp",
            "date",
        ],
    )


    if id_col is None:
        raise RuntimeError(
            f"No ID column found in {file_path}"
        )


    if date_col is None:
        raise RuntimeError(
            f"No datetime column found in {file_path}"
        )


    temp = pd.DataFrame({

        "deer_id":
            df[id_col].map(
                normalize_id
            ),

        "telemetry_datetime":
            pd.to_datetime(
                df[date_col],
                errors="coerce",
                utc=True,
            ).dt.tz_localize(None),

        "dataset":
            dataset_name,
    })


    telemetry_parts.append(
        temp
    )


telemetry = pd.concat(
    telemetry_parts,
    ignore_index=True,
)


telemetry = telemetry.dropna(
    subset=[
        "deer_id",
        "telemetry_datetime",
    ]
)


model_ids = set(
    telemetry[
        "deer_id"
    ].unique()
)


# =============================================================================
# KEEP AGE RECORDS FOR MODEL DEER
# =============================================================================

model_age = age_unique.loc[
    age_unique[
        "deer_id"
    ].isin(
        model_ids
    )
].copy()


# =============================================================================
# 1. EXPLICIT FAWN RECORDS
# =============================================================================

model_fawn = model_age.loc[
    model_age[
        "explicit_fawn"
    ]
].copy()


print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "EXPLICIT FAWN RECORDS AMONG MODEL DEER"
)

print(
    "=" * 100
)


if len(model_fawn) == 0:

    print(
        "\nNONE FOUND."
    )

else:

    print(
        model_fawn[
            [
                "deer_id",
                "record_date",
                "age_raw",
                "source_year",
                "source_file",
            ]
        ]
        .sort_values(
            [
                "deer_id",
                "record_date",
            ]
        )
        .to_string(
            index=False
        )
    )


# =============================================================================
# 2. AGE-1 RECORDS
#
# Keep these separate.
# These are NOT automatically classified as fawns.
# =============================================================================

model_age_one = model_age.loc[
    model_age[
        "age_one_record"
    ]
].copy()


print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "MODEL DEER WITH AN AGE-1 RECORD"
)

print(
    "=" * 100
)


print(
    model_age_one[
        [
            "deer_id",
            "record_date",
            "age_raw",
            "source_year",
        ]
    ]
    .sort_values(
        [
            "deer_id",
            "record_date",
        ]
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# TELEMETRY START / END
# =============================================================================

tel_summary = (

    telemetry

    .groupby(
        "deer_id",
        as_index=False,
    )

    .agg(

        telemetry_start=(
            "telemetry_datetime",
            "min",
        ),

        telemetry_end=(
            "telemetry_datetime",
            "max",
        ),

        n_telemetry_rows=(
            "telemetry_datetime",
            "size",
        ),
    )
)


# =============================================================================
# FIND AGE RECORD CLOSEST TO TELEMETRY START
#
# This is much more defensible than estimating a birthday.
# =============================================================================

nearest_rows = []


for _, tel_row in tel_summary.iterrows():

    deer_id = tel_row[
        "deer_id"
    ]

    tel_start = tel_row[
        "telemetry_start"
    ]


    deer_age = model_age.loc[
        model_age[
            "deer_id"
        ] == deer_id
    ].copy()


    if len(deer_age) == 0:

        nearest_rows.append({

            "deer_id":
                deer_id,

            "telemetry_start":
                tel_start,

            "nearest_age_date":
                pd.NaT,

            "days_between":
                np.nan,

            "nearest_age_raw":
                np.nan,

            "nearest_explicit_fawn":
                False,
        })

        continue


    deer_age["days_between"] = (

        deer_age[
            "record_date"
        ]

        - tel_start

    ).abs().dt.total_seconds() / 86400


    nearest = (

        deer_age

        .sort_values(
            "days_between"
        )

        .iloc[0]
    )


    nearest_rows.append({

        "deer_id":
            deer_id,

        "telemetry_start":
            tel_start,

        "nearest_age_date":
            nearest[
                "record_date"
            ],

        "days_between":
            nearest[
                "days_between"
            ],

        "nearest_age_raw":
            nearest[
                "age_raw"
            ],

        "nearest_explicit_fawn":
            nearest[
                "explicit_fawn"
            ],

        "nearest_age_one":
            nearest[
                "age_one_record"
            ],
    })


nearest_age = pd.DataFrame(
    nearest_rows
)


# =============================================================================
# SHOW THE DEER WE PREVIOUSLY FLAGGED
# =============================================================================

previously_flagged = [

    "A43",
    "A47",
    "A48",
    "A50",

    "F100",
    "F106",
    "F107",
    "F109",

    "F91",
    "F94",
    "F97",
    "F99",
]


check_12 = nearest_age.loc[
    nearest_age[
        "deer_id"
    ].isin(
        previously_flagged
    )
].copy()


print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "AGE RECORD NEAREST ACTUAL TELEMETRY START"
)

print(
    "=" * 100
)


print(
    check_12
    .sort_values(
        "deer_id"
    )
    .to_string(
        index=False
    )
)


# =============================================================================
# FINAL COUNTS
# =============================================================================

explicit_fawn_ids = sorted(
    model_fawn[
        "deer_id"
    ].dropna().unique()
)


age_one_ids = sorted(
    model_age_one[
        "deer_id"
    ].dropna().unique()
)


print(
    "\n",
    "=" * 100,
    sep="",
)

print(
    "FINAL FAWN AUDIT"
)

print(
    "=" * 100
)


print(
    "\nModel deer with explicit fawn evidence:",
    len(explicit_fawn_ids),
)

print(
    explicit_fawn_ids
)


print(
    "\nModel deer with age = 1 records:",
    len(age_one_ids),
)

print(
    age_one_ids
)


if len(explicit_fawn_ids) == 0:

    print(
        "\nRESULT: No explicit fawn-age records were found "
        "for deer present in the autoencoder datasets."
    )


# =============================================================================
# SAVE
# =============================================================================

model_fawn.to_csv(
    OUT_DIR /
    "STRICT_model_deer_explicit_fawn_records.csv",
    index=False,
)


model_age_one.to_csv(
    OUT_DIR /
    "STRICT_model_deer_age_one_records.csv",
    index=False,
)


nearest_age.to_csv(
    OUT_DIR /
    "STRICT_nearest_age_to_telemetry_start.csv",
    index=False,
)

### Checking if MinMax Scaler is the issue

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler


# =============================================================================
# PATH
# =============================================================================

TRAIN_FILE = (
    "<LOCAL_PATH>"
    "CWD_negative_training_with_seasons2_AKDE_FIXED.csv"
)


# =============================================================================
# LOAD
# =============================================================================

df = pd.read_csv(TRAIN_FILE)

AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high",
]


# =============================================================================
# KEEP ROWS WITH COMPLETE AKDE FOR THIS DIAGNOSTIC
# =============================================================================

akde = df[AKDE_FEATURES].dropna().copy()

print("Rows used:", len(akde))


# =============================================================================
# 1. CURRENT APPROACH:
# RAW AKDE -> MINMAX
# =============================================================================

raw_scaler = MinMaxScaler()

raw_scaled = raw_scaler.fit_transform(
    akde[AKDE_FEATURES]
)

raw_scaled = pd.DataFrame(
    raw_scaled,
    columns=[
        f"{c}_raw_minmax"
        for c in AKDE_FEATURES
    ],
    index=akde.index,
)


# =============================================================================
# 2. ALTERNATIVE:
# LOG1P AKDE -> MINMAX
# =============================================================================

log_akde = np.log1p(
    akde[AKDE_FEATURES]
)

log_scaler = MinMaxScaler()

log_scaled = log_scaler.fit_transform(
    log_akde
)

log_scaled = pd.DataFrame(
    log_scaled,
    columns=[
        f"{c}_log_minmax"
        for c in AKDE_FEATURES
    ],
    index=akde.index,
)


# =============================================================================
# 3. QUANTILE SUMMARY
# =============================================================================

def summarize(series):
    return pd.Series({
        "min": series.min(),
        "p01": series.quantile(0.01),
        "p05": series.quantile(0.05),
        "p25": series.quantile(0.25),
        "median": series.median(),
        "p75": series.quantile(0.75),
        "p95": series.quantile(0.95),
        "p99": series.quantile(0.99),
        "p999": series.quantile(0.999),
        "max": series.max(),
    })


for feature in AKDE_FEATURES:

    print("\n" + "=" * 100)
    print(f"{feature}: RAW -> MINMAX")
    print("=" * 100)

    print(
        summarize(
            raw_scaled[
                f"{feature}_raw_minmax"
            ]
        )
    )

    print("\n" + "=" * 100)
    print(f"{feature}: LOG1P -> MINMAX")
    print("=" * 100)

    print(
        summarize(
            log_scaled[
                f"{feature}_log_minmax"
            ]
        )
    )


# =============================================================================
# 4. DIRECTLY SHOW HOW COMPRESSED THE ORDINARY VALUES ARE
# =============================================================================

comparison = pd.DataFrame({
    "akde_est_raw":
        akde["akde_est"],

    "raw_minmax":
        raw_scaled["akde_est_raw_minmax"],

    "log1p_minmax":
        log_scaled["akde_est_log_minmax"],
})


print("\n" + "=" * 100)
print("AKDE_EST SCALING COMPARISON")
print("=" * 100)

print(
    comparison.describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.999,
        ]
    ).T
)


# =============================================================================
# 5. SHOW REPRESENTATIVE RAW AKDE_EST VALUES
# =============================================================================

representative_values = np.array([
    0.001,
    0.01,
    0.1,
    0.5,
    1,
    2,
    5,
    10,
    25,
    50,
    100,
    500,
    1000,
    2066,
], dtype=float)


raw_demo = (
    representative_values
    - akde["akde_est"].min()
) / (
    akde["akde_est"].max()
    - akde["akde_est"].min()
)


log_min = np.log1p(
    akde["akde_est"].min()
)

log_max = np.log1p(
    akde["akde_est"].max()
)

log_demo = (
    np.log1p(
        representative_values
    )
    - log_min
) / (
    log_max
    - log_min
)


demo = pd.DataFrame({
    "akde_est_km2":
        representative_values,

    "raw_minmax":
        raw_demo,

    "log1p_minmax":
        log_demo,
})


print("\n" + "=" * 100)
print("REPRESENTATIVE AKDE_EST VALUES")
print("=" * 100)

print(
    demo.to_string(
        index=False
    )
)

## Removing unstable AKDEs

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler


# =============================================================================
# SETTINGS
# =============================================================================

TRAIN_FILE = (
    "<LOCAL_PATH>"
    "CWD_negative_training_with_seasons2_AKDE_FIXED.csv"
)

TEST_FILE = (
    "<LOCAL_PATH>"
    "CWD_positive_testing_with_seasons2_AKDE_FIXED.csv"
)

AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high",
]

HIGH_AKDE_THRESHOLD = 150.0


# =============================================================================
# LOAD CORRECTED TRAINING + TESTING DATA
# =============================================================================

train_df = pd.read_csv(
    TRAIN_FILE,
    low_memory=False,
)

test_df = pd.read_csv(
    TEST_FILE,
    low_memory=False,
)


# Add temporary split labels for QC summaries only

train_df_qc = train_df.copy()
train_df_qc["model_split"] = "CWD-negative training"

test_df_qc = test_df.copy()
test_df_qc["model_split"] = "CWD-positive testing"


combined_df = pd.concat(
    [
        train_df_qc,
        test_df_qc,
    ],
    ignore_index=True,
)


# =============================================================================
# ORIGINAL DATASET TOTALS
# =============================================================================

print("=" * 100)
print("ORIGINAL CORRECTED DATASETS")
print("=" * 100)

print(
    f"Training rows:          {len(train_df):,}"
)

print(
    f"Testing rows:           {len(test_df):,}"
)

print(
    f"TOTAL rows:             {len(combined_df):,}"
)

print()

print(
    f"Training unique deer:   {train_df['id'].nunique():,}"
)

print(
    f"Testing unique deer:    {test_df['id'].nunique():,}"
)

print(
    f"Combined unique deer:   {combined_df['id'].nunique():,}"
)


# =============================================================================
# IDENTIFY AKDE_EST > 150 KM^2
# =============================================================================

train_high_mask = (
    train_df["akde_est"].notna()
    & (
        train_df["akde_est"]
        > HIGH_AKDE_THRESHOLD
    )
)


test_high_mask = (
    test_df["akde_est"].notna()
    & (
        test_df["akde_est"]
        > HIGH_AKDE_THRESHOLD
    )
)


combined_high_mask = (
    combined_df["akde_est"].notna()
    & (
        combined_df["akde_est"]
        > HIGH_AKDE_THRESHOLD
    )
)


train_high = (
    train_df_qc.loc[
        train_high_mask
    ]
    .copy()
)


test_high = (
    test_df_qc.loc[
        test_high_mask
    ]
    .copy()
)


combined_high = (
    combined_df.loc[
        combined_high_mask
    ]
    .copy()
)


# =============================================================================
# TRUE TOTAL HIGH-AKDE PREVALENCE
# =============================================================================

print("\n" + "=" * 100)
print("AKDE > 150 KM^2 — TRAINING + TESTING")
print("=" * 100)

summary = pd.DataFrame({

    "dataset": [
        "CWD-negative training",
        "CWD-positive testing",
        "TOTAL",
    ],

    "original_rows": [
        len(train_df),
        len(test_df),
        len(combined_df),
    ],

    "rows_over_150": [
        len(train_high),
        len(test_high),
        len(combined_high),
    ],
})


summary["percent_removed"] = (
    100
    * summary["rows_over_150"]
    / summary["original_rows"]
)


summary["rows_retained"] = (
    summary["original_rows"]
    - summary["rows_over_150"]
)


summary["percent_retained"] = (
    100
    * summary["rows_retained"]
    / summary["original_rows"]
)


print(
    summary.to_string(
        index=False,
        formatters={
            "percent_removed":
                lambda x: f"{x:.6f}%",

            "percent_retained":
                lambda x: f"{x:.6f}%",
        },
    )
)


# =============================================================================
# SIMPLE TRUE-TOTAL STATEMENT
# =============================================================================

total_original = len(combined_df)

total_removed = len(combined_high)

total_retained = (
    total_original
    - total_removed
)


print("\n" + "=" * 100)
print("TRUE COMBINED QC TOTAL")
print("=" * 100)

print(
    f"Original training + testing rows: {total_original:,}"
)

print(
    f"Rows removed (>150 km²):         {total_removed:,}"
)

print(
    f"Rows retained:                    {total_retained:,}"
)

print(
    f"Percent removed:                  "
    f"{100 * total_removed / total_original:.6f}%"
)

print(
    f"Percent retained:                 "
    f"{100 * total_retained / total_original:.6f}%"
)


# =============================================================================
# HIGH AKDE BY MODEL SPLIT / DEER / PHASE
# =============================================================================

group_cols = [
    c
    for c in [
        "model_split",
        "dataset",
        "id",
        "id_phase",
    ]
    if c in combined_high.columns
]


flag_summary = (
    combined_high
    .groupby(
        group_cols,
        dropna=False,
    )
    .agg(

        n_removed=(
            "akde_est",
            "size",
        ),

        min_est=(
            "akde_est",
            "min",
        ),

        median_est=(
            "akde_est",
            "median",
        ),

        max_est=(
            "akde_est",
            "max",
        ),

        min_low=(
            "akde_low",
            "min",
        ),

        max_high=(
            "akde_high",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        "n_removed",
        ascending=False,
    )
)


print("\n" + "=" * 100)
print("AKDE > 150 KM^2 BY TRAINING / TESTING DEER")
print("=" * 100)

print(
    flag_summary.to_string(
        index=False
    )
)


# =============================================================================
# FILTER BOTH DATASETS
# =============================================================================

train_filtered = (
    train_df.loc[
        ~train_high_mask
    ]
    .copy()
)


test_filtered = (
    test_df.loc[
        ~test_high_mask
    ]
    .copy()
)


# =============================================================================
# VERIFY FILTERING
# =============================================================================

assert not (
    train_filtered["akde_est"]
    > HIGH_AKDE_THRESHOLD
).any()

assert not (
    test_filtered["akde_est"]
    > HIGH_AKDE_THRESHOLD
).any()


print("\n" + "=" * 100)
print("FILTERED DATASET TOTALS")
print("=" * 100)

print(
    f"Training before:  {len(train_df):,}"
)

print(
    f"Training after:   {len(train_filtered):,}"
)

print(
    f"Training removed: {len(train_df) - len(train_filtered):,}"
)

print()

print(
    f"Testing before:   {len(test_df):,}"
)

print(
    f"Testing after:    {len(test_filtered):,}"
)

print(
    f"Testing removed:  {len(test_df) - len(test_filtered):,}"
)

print()

print(
    f"TOTAL before:     "
    f"{len(train_df) + len(test_df):,}"
)

print(
    f"TOTAL after:      "
    f"{len(train_filtered) + len(test_filtered):,}"
)

print(
    f"TOTAL removed:    "
    f"{(len(train_df) + len(test_df)) - (len(train_filtered) + len(test_filtered)):,}"
)


# =============================================================================
# FUNCTION: MINMAX QUANTILE SUMMARY
#
# IMPORTANT:
# MinMaxScaler assessment remains TRAINING ONLY.
# The test data must never determine the scaler.
# =============================================================================

QUANTILES = [
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.95,
    0.99,
    0.999,
]


def minmax_summary(
    data,
    feature,
):

    values = (
        data[
            [feature]
        ]
        .dropna()
        .astype(float)
    )

    scaler = MinMaxScaler()

    scaled = scaler.fit_transform(
        values
    ).ravel()

    scaled = pd.Series(
        scaled,
        index=values.index,
    )

    result = {

        "feature":
            feature,

        "n":
            len(values),

        "raw_min":
            values[
                feature
            ].min(),

        "raw_max":
            values[
                feature
            ].max(),

        "scaled_min":
            scaled.min(),
    }


    for q in QUANTILES:

        name = (
            f"p{q * 100:g}"
            .replace(
                ".",
                "_",
            )
        )

        result[name] = (
            scaled.quantile(q)
        )


    result["scaled_max"] = (
        scaled.max()
    )

    return result


# =============================================================================
# TRAINING MINMAX BEFORE FILTERING
# =============================================================================

before_results = []


for feature in AKDE_FEATURES:

    result = minmax_summary(
        train_df,
        feature,
    )

    result["dataset_version"] = (
        "All corrected training AKDE"
    )

    before_results.append(
        result
    )


# =============================================================================
# TRAINING MINMAX AFTER >150 FILTER
# =============================================================================

after_results = []


for feature in AKDE_FEATURES:

    result = minmax_summary(
        train_filtered,
        feature,
    )

    result["dataset_version"] = (
        "Training AKDE <=150 km2"
    )

    after_results.append(
        result
    )


# =============================================================================
# COMBINE MINMAX RESULTS
# =============================================================================

scale_comparison = pd.DataFrame(
    before_results
    + after_results
)


column_order = [

    "dataset_version",
    "feature",
    "n",
    "raw_min",
    "raw_max",
    "scaled_min",
    "p1",
    "p5",
    "p25",
    "p50",
    "p75",
    "p95",
    "p99",
    "p99_9",
    "scaled_max",
]


scale_comparison = (
    scale_comparison[
        column_order
    ]
)


print("\n" + "=" * 100)
print("TRAINING MINMAX: BEFORE VS AFTER >150 KM^2 QC")
print("=" * 100)

print(
    scale_comparison.to_string(
        index=False
    )
)


# =============================================================================
# DIRECT AKDE_EST COMPARISON
# =============================================================================

est_comparison = (
    scale_comparison.loc[
        scale_comparison[
            "feature"
        ]
        == "akde_est"
    ]
    .copy()
)


print("\n" + "=" * 100)
print("AKDE_EST MINMAX COMPARISON — TRAINING ONLY")
print("=" * 100)

print(
    est_comparison.to_string(
        index=False
    )
)


# =============================================================================
# HOW MUCH DID THE TRAINING SCALE EXPAND?
# =============================================================================

before = (
    est_comparison
    .loc[
        est_comparison[
            "dataset_version"
        ]
        == "All corrected training AKDE"
    ]
    .iloc[0]
)


after = (
    est_comparison
    .loc[
        est_comparison[
            "dataset_version"
        ]
        == "Training AKDE <=150 km2"
    ]
    .iloc[0]
)


expansion = pd.DataFrame({

    "quantile": [
        "25th",
        "median",
        "75th",
        "95th",
        "99th",
        "99.9th",
    ],

    "before": [
        before["p25"],
        before["p50"],
        before["p75"],
        before["p95"],
        before["p99"],
        before["p99_9"],
    ],

    "after": [
        after["p25"],
        after["p50"],
        after["p75"],
        after["p95"],
        after["p99"],
        after["p99_9"],
    ],
})


expansion[
    "scale_expansion_factor"
] = (
    expansion["after"]
    / expansion["before"]
)


print("\n" + "=" * 100)
print("HOW MUCH MORE OF THE TRAINING MINMAX RANGE DOES AKDE_EST USE?")
print("=" * 100)

print(
    expansion.to_string(
        index=False
    )
)

In [ ]:
# =============================================================================
# DEER-LEVEL CWD+ VS CWD- COMPARISON
#
# No sex separation.
# No subsampling.
# No forced equal sample sizes.
#
# Every deer contributes exactly ONE value per feature.
# =============================================================================


# =============================================================================
# Create one feature vector per deer
# =============================================================================

def make_deer_level_data(
    df,
    features,
    ids
):
    """
    Collapse all GPS observations to one mean value per deer.

    Every deer receives equal weight regardless of the number
    of GPS observations.
    """

    subset = df[
        df["id_clean"].isin(ids)
    ][
        ["id_clean"] + features
    ].copy()


    if subset.empty:
        raise ValueError(
            "No observations were found for the requested deer IDs."
        )


    deer_level = (
        subset
        .groupby(
            "id_clean",
            as_index=False
        )
        [features]
        .mean()
    )


    return deer_level


# =============================================================================
# Create deer-level datasets
# =============================================================================

neg_deer = make_deer_level_data(
    all_raw,
    FEATURES,
    neg_ids_present
)


pos_deer = make_deer_level_data(
    all_raw,
    FEATURES,
    pos_ids_present
)


print(
    "\n"
    + "=" * 80
)

print(
    "DEER-LEVEL CWD+ VS CWD- COMPARISON"
)

print(
    "=" * 80
)


print(
    f"\nCWD- deer: {len(neg_deer)}"
)

print(
    f"CWD+ deer: {len(pos_deer)}"
)


# =============================================================================
# Group means
#
# IMPORTANT:
# These are means of DEER, not means of GPS observations.
# =============================================================================

mu_neg_mean = (
    neg_deer[
        FEATURES
    ]
    .mean()
)


mu_pos_mean = (
    pos_deer[
        FEATURES
    ]
    .mean()
)


# =============================================================================
# Group medians
#
# Useful because AKDE/home-range estimates can be highly right-skewed.
# =============================================================================

mu_neg_median = (
    neg_deer[
        FEATURES
    ]
    .median()
)


mu_pos_median = (
    pos_deer[
        FEATURES
    ]
    .median()
)


# =============================================================================
# Percentage difference using deer-level MEANS
# =============================================================================

mean_pct_difference = pct_diff(
    mu_pos_mean.values,
    mu_neg_mean.values
)


# =============================================================================
# Percentage difference using deer-level MEDIANS
# =============================================================================

median_pct_difference = pct_diff(
    mu_pos_median.values,
    mu_neg_median.values
)


# =============================================================================
# Results table
# =============================================================================

comparison_results = pd.DataFrame(
    {
        "feature":
            FEATURES,

        "CWD-_deer_mean":
            mu_neg_mean.values,

        "CWD+_deer_mean":
            mu_pos_mean.values,

        "mean_pct_difference":
            mean_pct_difference,

        "CWD-_deer_median":
            mu_neg_median.values,

        "CWD+_deer_median":
            mu_pos_median.values,

        "median_pct_difference":
            median_pct_difference
    }
)


# =============================================================================
# Print all results
# =============================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DEER-LEVEL RESULTS"
)

print(
    "=" * 100
)


print(
    comparison_results
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# Specifically inspect AKDE
# =============================================================================

AKDE_FEATURES = [
    "akde_low",
    "akde_est",
    "akde_high"
]


akde_results = comparison_results[
    comparison_results[
        "feature"
    ].isin(
        AKDE_FEATURES
    )
].copy()


print(
    "\n"
    + "=" * 100
)

print(
    "AKDE / HOME-RANGE RESULTS"
)

print(
    "=" * 100
)


print(
    akde_results
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# Print INDIVIDUAL AKDE values
#
# This is extremely important for determining whether a few CWD+ deer
# are driving the apparent difference.
# =============================================================================

available_akde = [
    feature
    for feature in AKDE_FEATURES
    if feature in FEATURES
]


print(
    "\n"
    + "=" * 100
)

print(
    "INDIVIDUAL CWD- AKDE VALUES"
)

print(
    "=" * 100
)


print(
    neg_deer[
        [
            "id_clean",
            *available_akde
        ]
    ]
    .sort_values(
        "akde_est"
        if "akde_est" in available_akde
        else available_akde[0]
    )
    .round(3)
    .to_string(
        index=False
    )
)


print(
    "\n"
    + "=" * 100
)

print(
    "INDIVIDUAL CWD+ AKDE VALUES"
)

print(
    "=" * 100
)


print(
    pos_deer[
        [
            "id_clean",
            *available_akde
        ]
    ]
    .sort_values(
        "akde_est"
        if "akde_est" in available_akde
        else available_akde[0]
    )
    .round(3)
    .to_string(
        index=False
    )
)


# =============================================================================
# BUILD HEATMAP
#
# I recommend using DEER-LEVEL MEDIANS for the AKDE-skew issue.
#
# But if you want the entire heatmap to remain mean-based,
# change median_pct_difference to mean_pct_difference below.
# =============================================================================

comparison_columns = {

    "feature":
        FEATURES,

    "CWD$^{+}$ vs. CWD$^{-}$":
        median_pct_difference
}


all_df = pd.DataFrame(
    comparison_columns
)


comp_order = [
    "CWD$^{+}$ vs. CWD$^{-}$"
]


heatmap_df = (
    all_df
    .set_index(
        "feature"
    )
    .loc[
        :,
        comp_order
    ]
)


# =============================================================================
# Save numerical results
# =============================================================================

output_path = (
    OUT_DIR /
    "CWD+_vs_CWD-_DEER_LEVEL_RESULTS.csv"
)


comparison_results.to_csv(
    output_path,
    index=False
)


print(
    "\nResults saved to:"
)

print(
    output_path
)

In [ ]:
# make_ae_feature_contributions_heatmap_BLUE.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    r"<LOCAL_PATH>"
)

CSV_PATH = (
    BASE
    / "ae_holdout_results"
    / "ae_holdout_feature_contributions_by_feature.csv"
)

OUT_DIR = (
    BASE
    / "ae_holdout_results"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = (
    OUT_DIR
    / "ae_feature_contributions_heatmap_single_file"
)


# =============================================================================
# Model labels
#
# MODEL_LABEL is retained internally.
# MODEL_DISPLAY_LABEL controls only how the column heading appears in the figure.
# =============================================================================

MODEL_LABEL = (
    "White-Tailed Deer Autoencoder Model"
)

MODEL_DISPLAY_LABEL = (
    "Full Holdout"
)


# =============================================================================
# Load feature-contribution data
# =============================================================================

if not CSV_PATH.exists():

    raise FileNotFoundError(
        f"Input file not foun<LOCAL_PATH>"
        f"{CSV_PATH}"
    )


df = pd.read_csv(
    CSV_PATH
)


required_columns = {
    "feature",
    "All_%"
}


missing_columns = required_columns.difference(
    df.columns
)


if missing_columns:

    raise ValueError(
        f"Missing columns: "
        f"{sorted(missing_columns)}\n"
        f"Available columns: "
        f"{list(df.columns)}"
    )


df = df[
    [
        "feature",
        "All_%"
    ]
].copy()


df["All_%"] = pd.to_numeric(
    df["All_%"],
    errors="coerce"
)


df["model"] = MODEL_LABEL


# =============================================================================
# Convert to heatmap format
# =============================================================================

heatmap_df = df.pivot(
    index="feature",
    columns="model",
    values="All_%"
)


# =============================================================================
# Feature ordering and display labels
# =============================================================================

pretty_order = [

    # Velocity metrics
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction and turning-angle metrics
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path-shape metrics
    "sinuosity",
    "sinuosity_28pt",
    "sinuosity_56pt",

    # Space-use metrics
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "sinuosity_28pt":
        "Sinuosity",

    "sinuosity_56pt":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [
    feature
    for feature in pretty_order
    if feature in heatmap_df.index
]


heatmap_df = heatmap_df.reindex(
    available_pretty_order
)


heatmap_df.index = [
    pretty_map.get(
        feature,
        feature
    )
    for feature in heatmap_df.index
]


heatmap_df = heatmap_df.apply(
    pd.to_numeric,
    errors="coerce"
)


# =============================================================================
# Replace model column heading with publication-friendly display label
# =============================================================================

heatmap_df.columns = [
    MODEL_DISPLAY_LABEL
]


# =============================================================================
# Print numerical values for verification
# =============================================================================

print(
    "\nFeature-contribution table:"
)


print(
    heatmap_df
    .round(1)
    .to_string()
)


# =============================================================================
# Determine color scale
# =============================================================================

data_values = heatmap_df.to_numpy(
    dtype=float
)


if not np.isfinite(
    data_values
).any():

    raise ValueError(
        "The heatmap contains no finite "
        "feature-contribution values."
    )


true_min = float(
    np.nanmin(
        data_values
    )
)


true_max = float(
    np.nanmax(
        data_values
    )
)


vmin = 0.0


vmax = float(
    np.ceil(
        true_max
    )
)


if vmax <= vmin:

    vmax = 5.0


print(
    f"\nData range: "
    f"{true_min:.1f}% to "
    f"{true_max:.1f}%"
)


print(
    f"Plotting range: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Build discrete blue color boundaries
#
# 0% to 5% receives a light blue.
# Larger feature contributions become progressively darker.
# =============================================================================

candidate_bounds = [
    vmin,
    5,
    10,
    15,
    20,
    30,
    40,
    50,
    vmax
]


bounds = [
    float(bound)
    for bound in candidate_bounds
    if vmin <= bound <= vmax
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(bounds)
)


if len(bounds) < 2:

    bounds = [
        vmin,
        vmax
    ]


print(
    f"Color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Create one blue shade for each discrete interval
#
# Avoid the nearly white lower portion of the Blues colormap.
# The highest interval receives the darkest blue.
# =============================================================================

base_cmap = mpl.colormaps[
    "Blues"
]


bin_midpoints = np.array(
    [
        (
            lower +
            upper
        ) / 2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


MIN_CMAP_POSITION = 0.15

MAX_CMAP_POSITION = 1.00


if vmax > 0:

    relative_positions = (
        bin_midpoints /
        vmax
    )

else:

    relative_positions = np.zeros_like(
        bin_midpoints
    )


relative_positions = np.clip(
    relative_positions,
    0,
    1
)


color_positions = (
    MIN_CMAP_POSITION
    +
    relative_positions
    *
    (
        MAX_CMAP_POSITION
        -
        MIN_CMAP_POSITION
    )
)


# =============================================================================
# Force first and last intervals to intended endpoints
# =============================================================================

if len(color_positions) > 0:

    color_positions[
        0
    ] = MIN_CMAP_POSITION

    color_positions[
        -1
    ] = MAX_CMAP_POSITION


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_blues"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


print(
    "\nColor intervals:"
)


for index, (
    lower,
    upper
) in enumerate(
    zip(
        bounds[:-1],
        bounds[1:]
    )
):

    print(
        f"  {lower:>6.1f}% to "
        f"{upper:>6.1f}% "
        f"-> colormap position "
        f"{color_positions[index]:.3f}"
    )


# =============================================================================
# Figure dimensions
#
# Match the proportions of the CWD+ vs. CWD- heatmap.
# The overall figure remains wide enough for long feature names while the
# actual colored data column stays compact.
# =============================================================================

n_rows, n_cols = heatmap_df.shape


FIG_WIDTH = 8.0

ROW_HEIGHT = 0.62


FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.0,
    8.5,
    12.0
)


# =============================================================================
# Font sizes
# =============================================================================

AX_LABEL_FS = 26

X_TICK_FS = 19

Y_TICK_FS = 20

ANNOT_FS = 20

CBAR_LABEL_FS = 24

CBAR_TICK_FS = 18


# =============================================================================
# Create figure
#
# Fixed axes are used instead of constrained_layout so that both heatmaps
# have consistent column and colorbar proportions.
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# =============================================================================
# Main heatmap axis
#
# [left, bottom, width, height]
# =============================================================================

ax = fig.add_axes(
    [
        0.52,   # left
        0.12,   # bottom
        0.18,   # width
        0.80    # height
    ]
)


# =============================================================================
# Colorbar axis
# =============================================================================

cax = fig.add_axes(
    [
        0.76,   # left
        0.12,   # bottom
        0.035,  # width
        0.80    # height
    ]
)


# =============================================================================
# Create heatmap
# =============================================================================

hm = sns.heatmap(
    heatmap_df,

    cmap=discrete_cmap,
    norm=norm,

    annot=True,
    fmt=".1f",

    annot_kws={
        "fontsize": ANNOT_FS,
        "fontfamily": "Times New Roman",
        "fontweight": "bold"
    },

    linewidths=1.0,
    linecolor="white",

    ax=ax,

    cbar=True,
    cbar_ax=cax
)


# =============================================================================
# Allow cell to use controlled axis width
# =============================================================================

ax.set_aspect(
    "auto"
)


# =============================================================================
# Determine darkest blue interval
# =============================================================================

darkest_threshold = (
    bounds[-2]
    if len(bounds) >= 2
    else vmax
)


# =============================================================================
# Force ALL annotations to the same Times New Roman bold style
# =============================================================================

flat_values = (
    heatmap_df
    .to_numpy()
    .flatten()
)


for text, value in zip(
    ax.texts,
    flat_values
):

    text.set_fontfamily(
        "Times New Roman"
    )

    text.set_fontsize(
        ANNOT_FS
    )

    text.set_fontweight(
        "bold"
    )


    if not np.isfinite(
        value
    ):

        continue


    # White text only on the darkest blue interval
    if value >= darkest_threshold:

        text.set_color(
            "white"
        )

    else:

        text.set_color(
            "black"
        )


# =============================================================================
# Axis labels
# =============================================================================

ax.set_xlabel(
    "Model Reconstruction",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=16
)


ax.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Tick positions
# =============================================================================

ax.tick_params(
    axis="x",
    rotation=0,
    length=0
)


ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)


# =============================================================================
# Force x-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_xticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        X_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )

    label.set_horizontalalignment(
        "center"
    )

    label.set_multialignment(
        "center"
    )


# =============================================================================
# Force y-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        Y_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        0
    )


# =============================================================================
# Colorbar formatting
# =============================================================================

cbar = hm.collections[
    0
].colorbar


cbar.set_label(
    "% of Reconstruction Error",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


# =============================================================================
# Colorbar ticks
# =============================================================================

candidate_ticks = np.array(
    [
        vmin,
        5,
        10,
        15,
        20,
        30,
        40,
        50,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [
        f"{tick:.0f}%"
        for tick in nice_ticks
    ]
)


# =============================================================================
# Force all colorbar tick labels to same font and weight
# =============================================================================

for label in cbar.ax.get_yticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        CBAR_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cax.yaxis.set_label_position(
    "right"
)


cax.yaxis.set_ticks_position(
    "right"
)


# =============================================================================
# Strengthen colorbar outline
# =============================================================================

cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Save and display
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG to: "
    f"{png_path}"
)


print(
    f"Saved PDF to: "
    f"{pdf_path}"
)


plt.show()


plt.close(
    fig
)

In [ ]:
# =============================================================================
# Confirm and print reported Δ values from long_df
# =============================================================================

def get_delta_table(
    model_name=None,
    classes=None,
    features=None,
    exclude_zero=False
):
    """
    Return a formatted table of Delta_pp values from long_df.

    Parameters
    ----------
    model_name : str or None
        Model to retain, such as "Full Holdout".
        If None, all models are retained.

    classes : str, list, or None
        Class or classes to retain, such as "TP", "FN", or ["TP", "FN"].
        If None, all classes are retained.

    features : str, list, or None
        Feature or features to retain.
        If None, all features are retained.

    exclude_zero : bool
        If True, rows with Delta_pp equal to zero are removed.
    """
    sub = long_df.copy()

    if model_name is not None:
        sub = sub[sub["model"] == model_name].copy()

    if classes is not None:
        if isinstance(classes, str):
            classes = [classes]

        sub = sub[sub["class"].isin(classes)].copy()

    if features is not None:
        if isinstance(features, str):
            features = [features]

        sub = sub[sub["feature"].isin(features)].copy()

    sub["Delta_pp"] = pd.to_numeric(sub["Delta_pp"], errors="coerce")
    sub = sub.dropna(subset=["Delta_pp"]).copy()

    if exclude_zero:
        sub = sub[~np.isclose(sub["Delta_pp"], 0.0)].copy()

    if sub.empty:
        print(
            "⚠️ No matching rows found for "
            f"model={model_name}, classes={classes}, features={features}"
        )
        return pd.DataFrame()

    sub["feature_num"] = sub["feature"].map(feat_nums)
    sub["pretty_feature"] = sub["feature"].map(pretty_map)
    sub["Delta_pp_rounded"] = sub["Delta_pp"].round(1)

    sub = sub[
        [
            "model",
            "class",
            "feature_num",
            "feature",
            "pretty_feature",
            "Delta_pp",
            "Delta_pp_rounded"
        ]
    ].sort_values(
        ["model", "class", "feature_num"],
        ascending=[True, True, True]
    )

    return sub.reset_index(drop=True)


def print_delta_range(
    label,
    model_name=None,
    classes=None,
    features=None,
    exclude_zero=False
):
    """
    Print the minimum and maximum Delta_pp values for a selected subset,
    followed by all feature-level values included in that range.
    """
    sub = get_delta_table(
        model_name=model_name,
        classes=classes,
        features=features,
        exclude_zero=exclude_zero
    )

    if sub.empty:
        return

    min_idx = sub["Delta_pp"].idxmin()
    max_idx = sub["Delta_pp"].idxmax()

    min_row = sub.loc[min_idx]
    max_row = sub.loc[max_idx]

    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    if model_name is not None:
        print(f"Model: {model_name}")

    if classes is not None:
        class_text = classes if isinstance(classes, str) else ", ".join(classes)
        print(f"Class(es): {class_text}")

    print(f"Number of values: {len(sub)}")
    print(
        f"Overall range: "
        f"{min_row['Delta_pp']:.1f} to {max_row['Delta_pp']:.1f} "
        "percentage points"
    )

    print(
        "Minimum: "
        f"{min_row['Delta_pp']:.1f} pp — "
        f"{min_row['pretty_feature']} "
        f"(Feature {int(min_row['feature_num'])}, "
        f"{min_row['class']})"
    )

    print(
        "Maximum: "
        f"{max_row['Delta_pp']:.1f} pp — "
        f"{max_row['pretty_feature']} "
        f"(Feature {int(max_row['feature_num'])}, "
        f"{max_row['class']})"
    )

    print("-" * 100)

    display_cols = [
        "class",
        "feature_num",
        "feature",
        "pretty_feature",
        "Delta_pp_rounded"
    ]

    print(
        sub[display_cols]
        .rename(columns={"Delta_pp_rounded": "Delta_pp"})
        .to_string(index=False)
    )


def print_positive_negative_ranges(
    model_name,
    cls,
    exclude_zero=True
):
    """
    Print separate ranges for positive and negative Delta_pp values
    for one model and class.
    """
    sub = get_delta_table(
        model_name=model_name,
        classes=cls,
        exclude_zero=exclude_zero
    )

    if sub.empty:
        return

    positive = sub[sub["Delta_pp"] > 0].copy()
    negative = sub[sub["Delta_pp"] < 0].copy()

    print("\n" + "=" * 100)
    print(f"{model_name}: {cls} positive and negative Δ ranges")
    print("=" * 100)

    if not positive.empty:
        positive_min = positive.loc[positive["Delta_pp"].idxmin()]
        positive_max = positive.loc[positive["Delta_pp"].idxmax()]

        print(
            f"Positive range: "
            f"{positive_min['Delta_pp']:.1f} to "
            f"{positive_max['Delta_pp']:.1f} percentage points"
        )
        print(
            f"  Lowest positive: {positive_min['pretty_feature']} "
            f"({positive_min['Delta_pp']:.1f} pp)"
        )
        print(
            f"  Highest positive: {positive_max['pretty_feature']} "
            f"({positive_max['Delta_pp']:.1f} pp)"
        )
    else:
        print("Positive range: no positive values")

    if not negative.empty:
        negative_min = negative.loc[negative["Delta_pp"].idxmin()]
        negative_max = negative.loc[negative["Delta_pp"].idxmax()]

        print(
            f"Negative range: "
            f"{negative_min['Delta_pp']:.1f} to "
            f"{negative_max['Delta_pp']:.1f} percentage points"
        )
        print(
            f"  Most negative: {negative_min['pretty_feature']} "
            f"({negative_min['Delta_pp']:.1f} pp)"
        )
        print(
            f"  Closest to zero: {negative_max['pretty_feature']} "
            f"({negative_max['Delta_pp']:.1f} pp)"
        )
    else:
        print("Negative range: no negative values")


def print_tp_fn_comparison(
    model_name="Full Holdout",
    exclude_zero=False
):
    """
    Print TP and FN Delta_pp values side by side for every feature.

    Also calculates:
        TP_minus_FN_pp = TP Delta_pp - FN Delta_pp
    """
    sub = get_delta_table(
        model_name=model_name,
        classes=["TP", "FN"],
        exclude_zero=exclude_zero
    )

    if sub.empty:
        return pd.DataFrame()

    comparison = (
        sub.pivot_table(
            index=["feature_num", "feature", "pretty_feature"],
            columns="class",
            values="Delta_pp",
            aggfunc="first"
        )
        .reset_index()
        .sort_values("feature_num")
    )

    comparison.columns.name = None

    for cls in ["TP", "FN"]:
        if cls not in comparison.columns:
            comparison[cls] = np.nan

    comparison["TP_minus_FN_pp"] = comparison["TP"] - comparison["FN"]

    comparison = comparison[
        [
            "feature_num",
            "feature",
            "pretty_feature",
            "TP",
            "FN",
            "TP_minus_FN_pp"
        ]
    ]

    numeric_cols = ["TP", "FN", "TP_minus_FN_pp"]
    comparison[numeric_cols] = comparison[numeric_cols].round(1)

    print("\n" + "=" * 100)
    print(f"{model_name}: TP versus FN feature-level Δ values")
    print("=" * 100)
    print(comparison.to_string(index=False))

    return comparison


# =============================================================================
# Print Full Holdout ranges
# =============================================================================

# Overall TP range across all features
print_delta_range(
    label="Full Holdout TP Δ values across all features",
    model_name="Full Holdout",
    classes="TP",
    exclude_zero=False
)

# Overall FN range across all features
print_delta_range(
    label="Full Holdout FN Δ values across all features",
    model_name="Full Holdout",
    classes="FN",
    exclude_zero=False
)

# Combined range across both TP and FN
print_delta_range(
    label="Full Holdout combined TP and FN Δ values",
    model_name="Full Holdout",
    classes=["TP", "FN"],
    exclude_zero=False
)

# Separate positive and negative ranges
print_positive_negative_ranges(
    model_name="Full Holdout",
    cls="TP",
    exclude_zero=True
)

print_positive_negative_ranges(
    model_name="Full Holdout",
    cls="FN",
    exclude_zero=True
)

# Side-by-side feature comparison
tp_fn_comparison = print_tp_fn_comparison(
    model_name="Full Holdout",
    exclude_zero=False
)

In [ ]:
# make_early_late_vs_neg_pct_heatmap_RAW.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path
import joblib


# =============================================================================
# Global font settings
# =============================================================================

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
})


# =============================================================================
# Paths
# =============================================================================

BASE = Path(
    r"<LOCAL_PATH>"
)

TRAIN_PATH = (
    BASE
    / "train_processed_100.csv"
)

VAL_PATH = (
    BASE
    / "val_processed_100.csv"
)

TEST_PATH = (
    BASE
    / "test_processed_100.csv"
)

SCALER_PATH = (
    BASE
    / "scaler_100.pkl"
)

OUT_DIR = (
    BASE
    / "figures_early_late"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SAVE_BASE = (
    OUT_DIR
    / "early_late_vs_neg_pct_heatmap_RAW_New"
)


# =============================================================================
# CWD-positive deer IDs
# =============================================================================

ACTUAL_POS_IDS = [
    "F87", "F26", "F30", "A41", "F107", "F94",
    "A56", "F73", "F48", "F66", "F83", "F79",
    "A52", "F106", "F27", "F62", "A60", "F78"
]


# =============================================================================
# Deer that initially tested negative and later tested positive
# =============================================================================

BECAME_POS_IDS = [
    "F48", "F66", "F83", "F79", "A52",
    "F106", "F27", "F62", "A60", "F78"
]


# =============================================================================
# Heatmap comparison order
# =============================================================================

comp_order = [
    "Early vs. CWD$^{-}$",
    "Late vs. CWD$^{-}$",
    "Late vs. Early"
]


# =============================================================================
# Load scaled datasets
# =============================================================================

train = pd.read_csv(
    TRAIN_PATH
)

val = pd.read_csv(
    VAL_PATH
)

test = pd.read_csv(
    TEST_PATH
)


print(
    f"Training rows:   "
    f"{len(train):,}"
)

print(
    f"Validation rows: "
    f"{len(val):,}"
)

print(
    f"Testing rows:    "
    f"{len(test):,}"
)


# =============================================================================
# Load scaler and obtain exact feature order
# =============================================================================

scaler = joblib.load(
    SCALER_PATH
)


if not hasattr(
    scaler,
    "feature_names_in_"
):

    raise AttributeError(
        "The loaded scaler does not contain feature_names_in_. "
        "Confirm that the scaler was fitted using a pandas DataFrame."
    )


FEATURES = list(
    scaler.feature_names_in_
)


print(
    f"\nNumber of model features: "
    f"{len(FEATURES)}"
)

print(
    "Features:"
)


for feature in FEATURES:

    print(
        f"  - {feature}"
    )


# =============================================================================
# Validate required columns
# =============================================================================

def validate_columns(
    df,
    dataset_name
):
    """
    Confirm that each dataset contains ID and all model features.
    """

    required_columns = {
        "id",
        *FEATURES
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:

        raise ValueError(
            f"{dataset_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )


validate_columns(
    train,
    "Training dataset"
)

validate_columns(
    val,
    "Validation dataset"
)

validate_columns(
    test,
    "Testing dataset"
)


# =============================================================================
# Keep only ID and model-feature columns
# =============================================================================

def keep_cols(df):
    """
    Keep only deer ID and model-feature columns.
    """

    return df[
        ["id"] + FEATURES
    ].copy()


train = keep_cols(
    train
)

val = keep_cols(
    val
)

test = keep_cols(
    test
)


# =============================================================================
# Inverse-transform scaled features back to raw units
# =============================================================================

def inverse_transform_features(df):
    """
    Convert scaled model features back to their raw units.
    """

    transformed_values = scaler.inverse_transform(
        df[
            FEATURES
        ].to_numpy()
    )

    output = df[
        ["id"]
    ].copy()

    output[
        FEATURES
    ] = transformed_values

    return output


train_raw = inverse_transform_features(
    train
)

val_raw = inverse_transform_features(
    val
)

test_raw = inverse_transform_features(
    test
)


# =============================================================================
# Define negative, early-stage, and late-stage groups
# =============================================================================

# CWD-negative observations consist of training and validation observations.

neg_raw = pd.concat(
    [
        train_raw,
        val_raw
    ],
    ignore_index=True
)


test_ids = set(
    test_raw[
        "id"
    ]
    .astype(str)
    .unique()
)


# =============================================================================
# Early-stage
#
# Deer initially CWD-negative that subsequently became CWD-positive.
# =============================================================================

early_ids = sorted(
    set(
        BECAME_POS_IDS
    ).intersection(
        test_ids
    )
)


# =============================================================================
# Late-stage
#
# Deer already CWD-positive at first capture.
# =============================================================================

late_ids = sorted(
    set(
        ACTUAL_POS_IDS
    )
    .intersection(
        test_ids
    )
    .difference(
        early_ids
    )
)


# =============================================================================
# Negative deer
# =============================================================================

negative_ids = sorted(
    neg_raw[
        "id"
    ]
    .astype(str)
    .unique()
)


if not early_ids:

    raise ValueError(
        "No early-stage IDs were found in the testing dataset."
    )


if not late_ids:

    raise ValueError(
        "No late-stage IDs were found in the testing dataset."
    )


if not negative_ids:

    raise ValueError(
        "No CWD-negative IDs were found in the training "
        "or validation data."
    )


print(
    "\nGroup composition:"
)

print(
    f"Negative IDs "
    f"({len(negative_ids)}): "
    f"{negative_ids}"
)

print(
    f"Early-stage IDs "
    f"({len(early_ids)}): "
    f"{early_ids}"
)

print(
    f"Late-stage IDs "
    f"({len(late_ids)}): "
    f"{late_ids}"
)


# =============================================================================
# Calculate group means in raw units
# =============================================================================

def group_mean(
    df,
    ids
):
    """
    Calculate observation-level feature means for selected deer IDs.
    """

    subset = df[
        df[
            "id"
        ]
        .astype(str)
        .isin(
            ids
        )
    ]


    if subset.empty:

        raise ValueError(
            f"No rows were found for the supplied IDs: "
            f"{ids}"
        )


    return subset[
        FEATURES
    ].mean(
        numeric_only=True
    )


mu_neg = group_mean(
    neg_raw,
    negative_ids
)


mu_early = group_mean(
    test_raw,
    early_ids
)


mu_late = group_mean(
    test_raw,
    late_ids
)


# =============================================================================
# Calculate percentage differences in raw units
#
# Formula:
#
#     100 × (A − B) / max(|B|, floor)
#
# A = comparison group
# B = reference group
# =============================================================================

BASE_FLOOR = 0.0000005


def pct_diff(
    A,
    B,
    floor=BASE_FLOOR
):
    """
    Calculate percentage difference between two groups.
    """

    A = np.asarray(
        A,
        dtype=float
    )

    B = np.asarray(
        B,
        dtype=float
    )


    denominator = np.maximum(
        np.abs(
            B
        ),
        floor
    )


    return (
        100.0
        *
        (
            A - B
        )
        /
        denominator
    )


# =============================================================================
# Build percentage-difference table
# =============================================================================

all_df = pd.DataFrame({

    "feature":
        FEATURES,

    "Early vs. CWD$^{-}$":
        pct_diff(
            mu_early.values,
            mu_neg.values
        ),

    "Late vs. CWD$^{-}$":
        pct_diff(
            mu_late.values,
            mu_neg.values
        ),

    "Late vs. Early":
        pct_diff(
            mu_late.values,
            mu_early.values
        ),
})


# =============================================================================
# Convert to feature × comparison heatmap matrix
# =============================================================================

heatmap_df = (
    all_df
    .set_index(
        "feature"
    )
    .loc[
        :,
        comp_order
    ]
)


# =============================================================================
# Feature ordering and display labels
# =============================================================================

pretty_order = [

    # Velocity metrics
    "velo",
    "rolling_velo_mean",
    "rolling_velo_sd",
    "rolling_velo_skew",
    "rolling_velo_kurt",

    # Direction and turning-angle metrics
    "abs_rel_angle",
    "rolling_abs_rel_angle_mean",
    "rolling_abs_rel_angle_sd",
    "rolling_abs_rel_angle_skew",
    "rolling_abs_rel_angle_kurt",

    # Path-shape metric
    "sinuosity",

    # Space-use metrics
    "akde_est",
    "akde_low",
    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity S.D.",

    "rolling_velo_skew":
        "Rolling Velocity Skew",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Abs. Rel. Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Abs. Rel. Angle S.D.",

    "rolling_abs_rel_angle_skew":
        "Rolling Abs. Rel. Angle Skew",

    "rolling_abs_rel_angle_kurt":
        "Rolling Abs. Rel. Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "akde_est":
        "Median Space Usage",

    "akde_low":
        "Low C.I. Space Usage",

    "akde_high":
        "High C.I. Space Usage"
}


available_pretty_order = [
    feature
    for feature in pretty_order
    if feature in heatmap_df.index
]


heatmap_df = heatmap_df.reindex(
    available_pretty_order
)


heatmap_df.index = [
    pretty_map.get(
        feature,
        feature
    )
    for feature in heatmap_df.index
]


# =============================================================================
# Ensure all heatmap values are numeric
# =============================================================================

heatmap_df = heatmap_df.apply(
    pd.to_numeric,
    errors="coerce"
)


# =============================================================================
# Print numerical values for verification
# =============================================================================

print(
    "\nPercentage-difference table:"
)


print(
    heatmap_df
    .round(1)
    .to_string()
)


# =============================================================================
# Determine color scale
# =============================================================================

data_values = heatmap_df.to_numpy(
    dtype=float
)


if not np.isfinite(
    data_values
).any():

    raise ValueError(
        "The heatmap contains no finite values."
    )


data_min = float(
    np.nanmin(
        data_values
    )
)


data_max = float(
    np.nanmax(
        data_values
    )
)


print(
    f"\nData range: "
    f"{data_min:.1f}% to "
    f"{data_max:.1f}%"
)


# =============================================================================
# Rounded plotting limits
# =============================================================================

vmin = float(
    np.floor(
        data_min
    )
)


vmax = float(
    np.ceil(
        data_max
    )
)


# Ensure both negative and positive sides appear
if vmin >= 0:

    vmin = -5.0


if vmax <= 0:

    vmax = 5.0


print(
    f"Rounded scale range: "
    f"{vmin:.0f}% to "
    f"{vmax:.0f}%"
)


# =============================================================================
# Build custom discrete color boundaries
# =============================================================================

candidate_bounds = [
    vmin,
    -30,
    -15,
    -10,
    -5,
    0,
    5,
    10,
    15,
    30,
    vmax
]


bounds = [
    float(bound)
    for bound in candidate_bounds
    if vmin <= bound <= vmax
]


bounds.extend(
    [
        float(vmin),
        float(vmax)
    ]
)


bounds = sorted(
    set(
        bounds
    )
)


if len(bounds) < 2:

    bounds = [
        vmin - 1,
        vmax + 1
    ]


print(
    f"Color boundaries: "
    f"{bounds}"
)


# =============================================================================
# Build one blue/red color for each discrete interval
#
# Negative values:
#     stronger negative = darker blue
#
# Positive values:
#     stronger positive = darker red
# =============================================================================

base_cmap = mpl.colormaps[
    "RdBu_r"
]


bin_midpoints = np.array(
    [
        (
            lower
            +
            upper
        )
        /
        2.0

        for lower, upper in zip(
            bounds[:-1],
            bounds[1:]
        )
    ],
    dtype=float
)


negative_extent = max(
    abs(
        vmin
    ),
    1.0
)


positive_extent = max(
    abs(
        vmax
    ),
    1.0
)


color_positions = []


for midpoint in bin_midpoints:


    if midpoint < 0:

        relative_strength = min(
            abs(
                midpoint
            )
            /
            negative_extent,
            1.0
        )


        cmap_position = (
            0.5
            -
            0.5
            *
            relative_strength
        )


    elif midpoint > 0:

        relative_strength = min(
            midpoint
            /
            positive_extent,
            1.0
        )


        cmap_position = (
            0.5
            +
            0.5
            *
            relative_strength
        )


    else:

        cmap_position = 0.5


    color_positions.append(
        cmap_position
    )


color_positions = np.asarray(
    color_positions,
    dtype=float
)


# =============================================================================
# Force extreme intervals to use darkest endpoint colors
# =============================================================================

if len(
    color_positions
) > 0:


    if bin_midpoints[
        0
    ] < 0:

        color_positions[
            0
        ] = 0.0


    if bin_midpoints[
        -1
    ] > 0:

        color_positions[
            -1
        ] = 1.0


discrete_colors = base_cmap(
    color_positions
)


discrete_cmap = mpl.colors.ListedColormap(
    discrete_colors,
    name="custom_discrete_RdBu"
)


norm = mpl.colors.BoundaryNorm(
    boundaries=bounds,
    ncolors=discrete_cmap.N,
    clip=True
)


print(
    "\nColor intervals:"
)


for index, (
    lower,
    upper
) in enumerate(
    zip(
        bounds[:-1],
        bounds[1:]
    )
):

    print(
        f"  {lower:>6.1f}% to "
        f"{upper:>6.1f}% "
        f"-> colormap position "
        f"{color_positions[index]:.3f}"
    )


# =============================================================================
# Figure dimensions
#
# Same figure family as the other heatmaps.
#
# This plot contains three comparison columns instead of one, so the heatmap
# itself needs to be wider while maintaining the same approximate width
# per individual data column.
# =============================================================================

n_rows, n_cols = heatmap_df.shape


FIG_WIDTH = 7.5

ROW_HEIGHT = 0.62


FIG_HEIGHT = np.clip(
    ROW_HEIGHT * n_rows + 2.0,
    8.5,
    12.0
)


# =============================================================================
# Font sizes
# =============================================================================

AX_LABEL_FS = 26

X_TICK_FS = 18

Y_TICK_FS = 20

ANNOT_FS = 18

CBAR_LABEL_FS = 24

CBAR_TICK_FS = 18


# =============================================================================
# Create figure
#
# Fixed positions are used instead of constrained_layout.
# This provides consistent control across all related heatmaps.
# =============================================================================

fig = plt.figure(
    figsize=(
        FIG_WIDTH,
        FIG_HEIGHT
    )
)


# =============================================================================
# Main heatmap axis
#
# [left, bottom, width, height]
#
# There are three comparison columns, so this panel is wider than the
# one-column heatmaps while preserving similar per-column proportions.
# =============================================================================

ax = fig.add_axes(
    [
        -0.30,   # left 19
        0.12,   # bottom
        0.325,   # width
        0.80    # height
    ]
)


# =============================================================================
# Colorbar axis
# =============================================================================

#cax = fig.add_axes(
#    [
#        0.865,  # left
#        0.12,   # bottom
#        0.026,  # width
#        0.80    # height
#    ]
#)

cax = fig.add_axes(
    [
        0.06,
        0.12,
        0.026,
        0.80
    ]
)


# =============================================================================
# Create heatmap
# =============================================================================

hm = sns.heatmap(
    heatmap_df,

    cmap=discrete_cmap,
    norm=norm,

    annot=True,
    fmt=".1f",

    annot_kws={
        "fontsize": ANNOT_FS,
        "fontfamily": "Times New Roman",
        "fontweight": "bold"
    },

    linewidths=1.0,
    linecolor="white",

    ax=ax,

    cbar=True,
    cbar_ax=cax
)


# =============================================================================
# Allow controlled heatmap width
# =============================================================================

ax.set_aspect(
    "auto"
)


# =============================================================================
# Force ALL cell annotations to same bold Times New Roman style
# =============================================================================

flat_values = (
    heatmap_df
    .to_numpy()
    .flatten()
)


for text, value in zip(
    ax.texts,
    flat_values
):


    text.set_fontfamily(
        "Times New Roman"
    )


    text.set_fontsize(
        ANNOT_FS
    )


    text.set_fontweight(
        "bold"
    )


    if not np.isfinite(
        value
    ):

        continue


    # White text only on darkest blue/red intervals
    if (
        value <= bounds[1]
        or
        value >= bounds[-2]
    ):

        text.set_color(
            "white"
        )


    else:

        text.set_color(
            "black"
        )


# =============================================================================
# Axis labels
# =============================================================================

ax.set_xlabel(
    "Comparison",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=16
)


ax.set_ylabel(
    "Feature",
    fontsize=AX_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold",
    labelpad=18
)


# =============================================================================
# Tick positions
# =============================================================================

ax.tick_params(
    axis="x",
    rotation=0,
    length=0
)


ax.tick_params(
    axis="y",
    rotation=0,
    length=0
)


# =============================================================================
# Force x-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_xticklabels():

    label.set_fontfamily(
        "Times New Roman"
    )

    label.set_fontsize(
        X_TICK_FS
    )

    label.set_fontweight(
        "bold"
    )

    label.set_rotation(
        45
    )

    label.set_horizontalalignment(
        "right"
    )

    label.set_rotation_mode(
        "anchor"
    )


# =============================================================================
# Force y-axis labels to Times New Roman + bold
# =============================================================================

for label in ax.get_yticklabels():


    label.set_fontfamily(
        "Times New Roman"
    )


    label.set_fontsize(
        Y_TICK_FS
    )


    label.set_fontweight(
        "bold"
    )


    label.set_rotation(
        0
    )


# =============================================================================
# Colorbar formatting
# =============================================================================

cbar = hm.collections[
    0
].colorbar


cbar.set_label(
    "% Difference (Raw Units)",
    rotation=90,
    labelpad=18,
    fontsize=CBAR_LABEL_FS,
    fontfamily="Times New Roman",
    fontweight="bold"
)


# =============================================================================
# Colorbar tick positions
# =============================================================================

candidate_ticks = np.array(
    [
        vmin,
        -30,
        -15,
        -10,
        -5,
        0,
        5,
        10,
        15,
        30,
        vmax
    ],
    dtype=float
)


nice_ticks = candidate_ticks[
    (
        candidate_ticks >= vmin
    )
    &
    (
        candidate_ticks <= vmax
    )
]


nice_ticks = np.unique(
    nice_ticks
)


cbar.set_ticks(
    nice_ticks
)


cbar.set_ticklabels(
    [
        f"{tick:.0f}%"
        for tick in nice_ticks
    ]
)


# =============================================================================
# Force colorbar tick labels to same font and weight
# =============================================================================

for label in cbar.ax.get_yticklabels():


    label.set_fontfamily(
        "Times New Roman"
    )


    label.set_fontsize(
        CBAR_TICK_FS
    )


    label.set_fontweight(
        "bold"
    )


cbar.ax.tick_params(
    axis="y",
    length=5,
    width=1.3
)


cax.yaxis.set_label_position(
    "right"
)


cax.yaxis.set_ticks_position(
    "right"
)


# =============================================================================
# Strengthen colorbar outline
# =============================================================================

cbar.outline.set_linewidth(
    1.2
)


# =============================================================================
# Save and display
# =============================================================================

png_path = SAVE_BASE.with_suffix(
    ".png"
)


pdf_path = SAVE_BASE.with_suffix(
    ".pdf"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    pdf_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)


print(
    f"\nSaved PNG to: "
    f"{png_path}"
)


print(
    f"Saved PDF to: "
    f"{pdf_path}"
)


plt.show()


plt.close(
    fig
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from pathlib import Path


# =============================================================================
# Configuration
# =============================================================================

CLASS_ORDER = [
    "TP",
    "FN"
]

USE_NUMERIC_Y = True


CLASS_COLORS = {
    "TP": "#001AFF",
    "FN": "#FF7B00"
}


MARKER = "o"


SAVE_BASE = Path(
    "<LOCAL_PATH>"
)


SAVE_BASE.mkdir(
    parents=True,
    exist_ok=True
)


files = {

    "Full Holdout": {

        "TP": (
            "<LOCAL_PATH>"
            "ae_holdout_results/ae_all_features_TP.csv"
        ),

        "FN": (
            "<LOCAL_PATH>"
            "ae_holdout_results/ae_all_features_FN.csv"
        ),
    }
}


# =============================================================================
# Feature order and display names
# =============================================================================

pretty_order = [

    "velo",

    "rolling_velo_mean",

    "rolling_velo_sd",

    "rolling_velo_skew",

    "rolling_velo_kurt",

    "abs_rel_angle",

    "rolling_abs_rel_angle_mean",

    "rolling_abs_rel_angle_sd",

    "rolling_abs_rel_angle_skew",

    "rolling_abs_rel_angle_kurt",

    "sinuosity",

    "akde_est",

    "akde_low",

    "akde_high"
]


pretty_map = {

    "velo":
        "Velocity",

    "rolling_velo_mean":
        "Rolling Velocity Mean",

    "rolling_velo_sd":
        "Rolling Velocity SD",

    "rolling_velo_skew":
        "Rolling Velocity Skewness",

    "rolling_velo_kurt":
        "Rolling Velocity Kurtosis",

    "abs_rel_angle":
        "Absolute Relative Turn Angle",

    "rolling_abs_rel_angle_mean":
        "Rolling Absolute Relative Turn Angle Mean",

    "rolling_abs_rel_angle_sd":
        "Rolling Absolute Relative Turn Angle SD",

    "rolling_abs_rel_angle_skew":
        "Rolling Absolute Relative Turn Angle Skewness",

    "rolling_abs_rel_angle_kurt":
        "Rolling Absolute Relative Turn Angle Kurtosis",

    "sinuosity":
        "Sinuosity",

    "akde_est":
        "Median Space Use",

    "akde_low":
        "Lower C.I. Space Use",

    "akde_high":
        "Upper C.I. Space Use",
}


# =============================================================================
# Global Matplotlib settings
#
# Match the same publication theme used in the heatmap figures:
#
#     - Times New Roman
#     - Bold throughout
#     - Large axis labels
#     - Large readable tick labels
#     - Embedded TrueType fonts in PDF output
# =============================================================================

mpl.rcParams.update({

    "font.family":
        "Times New Roman",

    "font.weight":
        "bold",

    "axes.labelweight":
        "bold",

    "axes.titleweight":
        "bold",

    "axes.titlesize":
        24,

    "axes.labelsize":
        26,

    "xtick.labelsize":
        20,

    "ytick.labelsize":
        20,

    "legend.fontsize":
        18,

    "figure.dpi":
        160,

    "pdf.fonttype":
        42,

    "ps.fonttype":
        42,
})


# =============================================================================
# Load one feature contribution file
# =============================================================================

def load_one(
    path,
    class_label
):
    """
    Load one TP or FN feature-contribution file and create a standardized
    Delta_pp column.

    The function first looks for:

        Delta_TP_minus_All_pp
        Delta_FN_minus_All_pp

    If that column does not exist, it calculates the difference from:

        TP_% or FN_%
        All_%
    """

    path = Path(
        path
    )


    if not path.exists():

        raise FileNotFoundError(
            f"File not foun<LOCAL_PATH>"
            f"{path}"
        )


    df = pd.read_csv(
        path
    )


    if "feature" not in df.columns:

        raise KeyError(
            f"'feature' column was not found i<LOCAL_PATH>"
            f"{path}\n\n"
            f"Available column<LOCAL_PATH>"
            f"{df.columns.tolist()}"
        )


    expected_delta = (
        f"Delta_{class_label}_minus_All_pp"
    )


    if expected_delta in df.columns:

        df[
            "Delta_pp"
        ] = pd.to_numeric(
            df[
                expected_delta
            ],
            errors="coerce"
        )


    elif (
        f"{class_label}_%" in df.columns
        and
        "All_%" in df.columns
    ):

        df[
            "Delta_pp"
        ] = (

            pd.to_numeric(
                df[
                    f"{class_label}_%"
                ],
                errors="coerce"
            )

            -

            pd.to_numeric(
                df[
                    "All_%"
                ],
                errors="coerce"
            )
        )


    else:

        raise KeyError(
            f"Could not identify the expected delta columns i<LOCAL_PATH>"
            f"{path}\n\n"
            f"Expected eithe<LOCAL_PATH>"
            f"  {expected_delta}\n\n"
            f"or bot<LOCAL_PATH>"
            f"  {class_label}_%\n"
            f"  All_%\n\n"
            f"Available column<LOCAL_PATH>"
            f"{df.columns.tolist()}"
        )


    output = df[
        [
            "feature",
            "Delta_pp"
        ]
    ].copy()


    output[
        "feature"
    ] = (
        output[
            "feature"
        ]
        .astype(str)
        .str.strip()
    )


    output = output.dropna(
        subset=[
            "feature",
            "Delta_pp"
        ]
    )


    return output


# =============================================================================
# Build combined long-format dataframe
# =============================================================================

long_dfs = []


for model, class_dict in files.items():

    for cls, path in class_dict.items():

        df = load_one(
            path=path,
            class_label=cls
        )


        df[
            "model"
        ] = model


        df[
            "class"
        ] = cls


        # ---------------------------------------------------------------------
        # Add missing expected features as zero so all features appear.
        # ---------------------------------------------------------------------

        missing_features = (

            set(
                pretty_order
            )

            -

            set(
                df[
                    "feature"
                ]
            )
        )


        if missing_features:

            print(
                f"\n{model} {cls}: "
                f"adding missing features as 0.0:\n"
                f"{sorted(missing_features)}"
            )


            df_missing = pd.DataFrame({

                "feature":
                    sorted(
                        missing_features
                    ),

                "Delta_pp":
                    0.0,

                "model":
                    model,

                "class":
                    cls
            })


            df = pd.concat(
                [
                    df,
                    df_missing
                ],
                ignore_index=True
            )


        long_dfs.append(
            df
        )


if not long_dfs:

    raise ValueError(
        "No data were loaded."
    )


long_df = pd.concat(
    long_dfs,
    ignore_index=True
)


# =============================================================================
# Keep only features used in the figure
# =============================================================================

long_df = long_df[
    long_df[
        "feature"
    ].isin(
        pretty_order
    )
].copy()


# =============================================================================
# Ensure numeric values
# =============================================================================

long_df[
    "Delta_pp"
] = pd.to_numeric(
    long_df[
        "Delta_pp"
    ],
    errors="coerce"
)


long_df = long_df.dropna(
    subset=[
        "Delta_pp"
    ]
).copy()


# =============================================================================
# Create feature numbering and plotting positions
# =============================================================================

feat_order = [
    feature
    for feature in pretty_order
    if feature in long_df[
        "feature"
    ].unique()
]


if not feat_order:

    raise ValueError(
        "None of the expected features "
        "were found in the input files."
    )


# =============================================================================
# Feature numbers follow pretty_order from top to bottom
# =============================================================================

feat_nums = {

    feature:
        number

    for number, feature
    in enumerate(
        feat_order,
        start=1
    )
}


# =============================================================================
# Reverse plotting coordinates so Feature 1 appears at the top
# =============================================================================

y_pos = {

    feature:
        position

    for position, feature
    in enumerate(
        feat_order[::-1]
    )
}


y_ticks = [
    y_pos[
        feature
    ]
    for feature in feat_order
]


if USE_NUMERIC_Y:

    y_ticklabels = [

        str(
            feat_nums[
                feature
            ]
        )

        for feature in feat_order
    ]


else:

    y_ticklabels = [

        pretty_map.get(
            feature,
            feature
        )

        for feature in feat_order
    ]


# =============================================================================
# Create combined FN and TP output table
# =============================================================================

def build_tp_fn_delta_table(
    model_name="Full Holdout"
):
    """
    Create one row per feature with the columns:

        feature_num
        feature
        pretty_feature
        FN_minus_All_pp
        TP_minus_All_pp
    """

    sub = long_df[
        long_df[
            "model"
        ] == model_name
    ].copy()


    if sub.empty:

        raise ValueError(
            f"No rows were found for model: "
            f"{model_name}"
        )


    sub[
        "feature_num"
    ] = (
        sub[
            "feature"
        ].map(
            feat_nums
        )
    )


    sub[
        "pretty_feature"
    ] = (
        sub[
            "feature"
        ].map(
            pretty_map
        )
    )


    # =========================================================================
    # Check for duplicate feature/class combinations
    # =========================================================================

    duplicate_mask = sub.duplicated(
        subset=[
            "feature",
            "class"
        ],
        keep=False
    )


    if duplicate_mask.any():

        duplicate_rows = sub.loc[
            duplicate_mask,
            [
                "feature",
                "class",
                "Delta_pp"
            ]
        ].sort_values(
            [
                "feature",
                "class"
            ]
        )


        print(
            "\nDuplicate feature/class rows "
            "were found. "
            "The first value will be used:"
        )


        print(
            duplicate_rows.to_string(
                index=False
            )
        )


    # =========================================================================
    # Convert TP and FN into separate columns
    # =========================================================================

    output_table = (

        sub

        .pivot_table(
            index=[
                "feature_num",
                "feature",
                "pretty_feature"
            ],
            columns="class",
            values="Delta_pp",
            aggfunc="first"
        )

        .reset_index()
    )


    output_table.columns.name = None


    if "FN" not in output_table.columns:

        output_table[
            "FN"
        ] = np.nan


    if "TP" not in output_table.columns:

        output_table[
            "TP"
        ] = np.nan


    output_table = output_table.rename(
        columns={
            "FN":
                "FN_minus_All_pp",

            "TP":
                "TP_minus_All_pp"
        }
    )


    output_table = output_table[
        [
            "feature_num",
            "feature",
            "pretty_feature",
            "FN_minus_All_pp",
            "TP_minus_All_pp"
        ]
    ]


    output_table = (

        output_table

        .sort_values(
            "feature_num"
        )

        .reset_index(
            drop=True
        )
    )


    output_table[
        "FN_minus_All_pp"
    ] = (
        output_table[
            "FN_minus_All_pp"
        ]
        .round(
            1
        )
    )


    output_table[
        "TP_minus_All_pp"
    ] = (
        output_table[
            "TP_minus_All_pp"
        ]
        .round(
            1
        )
    )


    return output_table


# =============================================================================
# Print and save combined output table
# =============================================================================

tp_fn_delta_table = (
    build_tp_fn_delta_table(
        model_name="Full Holdout"
    )
)


print(
    "\n"
    + "=" * 125
)


print(
    "Full Holdout: FN and TP "
    "Δ values by feature"
)


print(
    "=" * 125
)


print(
    tp_fn_delta_table.to_string(
        index=False,
        na_rep="NA"
    )
)


output_csv = (
    SAVE_BASE
    /
    "Full_Holdout_FN_TP_Delta_Values.csv"
)


tp_fn_delta_table.to_csv(
    output_csv,
    index=False
)


print(
    f"\nSaved combined table t<LOCAL_PATH>"
    f"{output_csv}"
)


# =============================================================================
# Print overall FN and TP ranges
# =============================================================================

def print_class_range(
    delta_table,
    value_column,
    class_label
):
    """
    Print minimum and maximum feature-level delta values
    for one class.
    """

    valid = delta_table.dropna(
        subset=[
            value_column
        ]
    ).copy()


    if valid.empty:

        print(
            f"\nNo values were found "
            f"for {class_label}."
        )

        return


    min_idx = valid[
        value_column
    ].idxmin()


    max_idx = valid[
        value_column
    ].idxmax()


    min_row = valid.loc[
        min_idx
    ]


    max_row = valid.loc[
        max_idx
    ]


    print(
        "\n"
        + "=" * 100
    )


    print(
        f"{class_label} Δ range"
    )


    print(
        "=" * 100
    )


    print(
        f"Range: "
        f"{min_row[value_column]:.1f} to "
        f"{max_row[value_column]:.1f} "
        f"percentage points"
    )


    print(
        f"Minimum: "
        f"Feature {int(min_row['feature_num'])}, "
        f"{min_row['pretty_feature']} "
        f"({min_row[value_column]:.1f} pp)"
    )


    print(
        f"Maximum: "
        f"Feature {int(max_row['feature_num'])}, "
        f"{max_row['pretty_feature']} "
        f"({max_row[value_column]:.1f} pp)"
    )


print_class_range(
    delta_table=tp_fn_delta_table,
    value_column="FN_minus_All_pp",
    class_label="FN minus All"
)


print_class_range(
    delta_table=tp_fn_delta_table,
    value_column="TP_minus_All_pp",
    class_label="TP minus All"
)


# =============================================================================
# Plot a single model
# =============================================================================

def plot_single_model(
    model_name,
    out_stub
):
    """
    Plot TP and FN feature-level delta values.

    Figure styling:
        - Times New Roman
        - Bold text throughout
        - Large publication-readable fonts
        - Full black border
        - No interior grid lines
        - Manually positioned x-axis title
        - TP/FN legend below x-axis title
        - Larger two-column Feature Key below TP/FN legend
    """

    # =========================================================================
    # Figure size
    #
    # Extra vertical space is reserved beneath the plotting panel for:
    #
    #   Δ (percentage points)
    #   TP / FN legend
    #   Feature Key
    # =========================================================================

    fig, ax = plt.subplots(
        figsize=(
            9.5,
            10.5
        )
    )


    # =========================================================================
    # Select model
    # =========================================================================

    sub_df = long_df[
        long_df[
            "model"
        ] == model_name
    ].copy()


    if sub_df.empty:

        plt.close(
            fig
        )

        raise ValueError(
            f"No plotting data were found "
            f"for model: {model_name}"
        )


    # =========================================================================
    # Zero-reference line
    # =========================================================================

    ax.axvline(
        0,
        color="black",
        linewidth=1.4,
        linestyle="--",
        alpha=0.80,
        zorder=1
    )


    # =========================================================================
    # No interior grid lines
    # =========================================================================

    ax.grid(
        False
    )


    # =========================================================================
    # Plot TP and FN points
    # =========================================================================

    for cls in CLASS_ORDER:

        sub = sub_df[
            sub_df[
                "class"
            ] == cls
        ].copy()


        if sub.empty:

            print(
                f"No plotting rows found for "
                f"model={model_name}, "
                f"class={cls}"
            )

            continue


        sub = (
            sub
            .drop_duplicates(
                subset=[
                    "feature"
                ],
                keep="first"
            )
            .set_index(
                "feature"
            )
            .reindex(
                feat_order
            )
            .reset_index()
        )


        y_values = [
            y_pos[
                feature
            ]
            for feature in sub[
                "feature"
            ]
        ]


        x_values = (
            sub[
                "Delta_pp"
            ]
            .to_numpy()
        )


        ax.scatter(
            x_values,
            y_values,
            s=115,
            marker=MARKER,
            facecolor=CLASS_COLORS[
                cls
            ],
            edgecolor="black",
            linewidth=1.1,
            zorder=3
        )


    # =========================================================================
    # Title
    # =========================================================================

    ax.set_title(
        model_name,
        loc="left",
        pad=18,
        fontsize=24,
        fontfamily="Times New Roman",
        fontweight="bold"
    )


    # =========================================================================
    # Axis labels
    #
    # IMPORTANT:
    #
    # The x-axis title is NOT positioned automatically.
    # It is placed later with fig.text() so it cannot overlap the TP/FN legend.
    # =========================================================================

    ax.set_xlabel(
        ""
    )


    ax.set_ylabel(
        "Feature",
        fontsize=26,
        fontfamily="Times New Roman",
        fontweight="bold",
        labelpad=16
    )


    # =========================================================================
    # Y-axis ticks
    # =========================================================================

    ax.set_yticks(
        y_ticks
    )


    ax.set_yticklabels(
        y_ticklabels
    )


    # =========================================================================
    # Tick styling
    # =========================================================================

    ax.tick_params(
        axis="x",
        labelsize=20,
        width=1.2,
        length=5
    )


    ax.tick_params(
        axis="y",
        labelsize=20,
        width=1.2,
        length=5
    )


    # =========================================================================
    # Force x-axis tick labels to Times New Roman + bold
    # =========================================================================

    for label in ax.get_xticklabels():

        label.set_fontfamily(
            "Times New Roman"
        )

        label.set_fontsize(
            20
        )

        label.set_fontweight(
            "bold"
        )


    # =========================================================================
    # Force y-axis tick labels to Times New Roman + bold
    # =========================================================================

    for label in ax.get_yticklabels():

        label.set_fontfamily(
            "Times New Roman"
        )

        label.set_fontsize(
            20
        )

        label.set_fontweight(
            "bold"
        )


    # =========================================================================
    # Y-axis limits
    # =========================================================================

    ax.set_ylim(
        -0.6,
        len(
            feat_order
        ) - 0.4
    )


    # =========================================================================
    # Full plot border
    # =========================================================================

    for spine in ax.spines.values():

        spine.set_visible(
            True
        )

        spine.set_color(
            "black"
        )

        spine.set_linewidth(
            1.2
        )


    # =========================================================================
    # Reserve bottom space BEFORE placing figure-level elements
    #
    # Vertical layout:
    #
    #       plotting panel
    #
    #       Δ (percentage points)
    #
    #       TP / FN legend
    #
    #       Feature Key
    # =========================================================================

    fig.subplots_adjust(
        top=0.95,
        left=0.13,
        right=0.97,
        bottom=0.43
    )


    # =========================================================================
    # Manually positioned x-axis title
    #
    # Because this is figure-level text, its position is independent of the
    # plotting axes and cannot collide with the TP/FN legend.
    # =========================================================================

    fig.text(
        0.50,
        0.385,
        "Δ (percentage points)",
        ha="center",
        va="center",
        fontsize=26,
        fontfamily="Times New Roman",
        fontweight="bold"
    )


    # =========================================================================
    # TP / FN legend handles
    # =========================================================================

    legend_handles = [

        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=10,
            markerfacecolor=CLASS_COLORS[
                "TP"
            ],
            markeredgecolor="black",
            markeredgewidth=1.1,
            label="True Positive (TP)"
        ),

        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markersize=10,
            markerfacecolor=CLASS_COLORS[
                "FN"
            ],
            markeredgecolor="black",
            markeredgewidth=1.1,
            label="False Negative (FN)"
        )
    ]


    # =========================================================================
    # TP / FN legend
    #
    # Explicitly positioned below the x-axis title.
    # =========================================================================

    legend = fig.legend(
        handles=legend_handles,
        loc="center",
        bbox_to_anchor=(
            0.50,
            0.335
        ),
        ncol=2,
        frameon=False,
        fontsize=18,
        handlelength=1.0,
        handletextpad=0.7,
        columnspacing=3.2
    )


    # =========================================================================
    # Force TP / FN legend text to Times New Roman + bold
    # =========================================================================

    for text in legend.get_texts():

        text.set_fontfamily(
            "Times New Roman"
        )

        text.set_fontsize(
            18
        )

        text.set_fontweight(
            "bold"
        )


    # =========================================================================
    # Feature Key
    #
    # Larger text for publication readability.
    # =========================================================================

    feature_key_left = (
        "1   Velocity\n"
        "2   Rolling Velocity Mean\n"
        "3   Rolling Velocity SD\n"
        "4   Rolling Velocity Skewness\n"
        "5   Rolling Velocity Kurtosis\n"
        "6   Absolute Relative Turn Angle\n"
        "7   Rolling Abs. Rel. Turn Angle Mean"
    )


    feature_key_right = (
        "8   Rolling Abs. Rel. Turn Angle SD\n"
        "9   Rolling Abs. Rel. Turn Angle Skewness\n"
        "10  Rolling Abs. Rel. Turn Angle Kurtosis\n"
        "11  Sinuosity\n"
        "12  Median Space Use\n"
        "13  Lower C.I. Space Use\n"
        "14  Upper C.I. Space Use"
    )


    # -------------------------------------------------------------------------
    # Feature-key heading
    # -------------------------------------------------------------------------

    fig.text(
        0.50,
        0.255,
        "Feature Key",
        ha="center",
        va="top",
        fontsize=20,
        fontfamily="Times New Roman",
        fontweight="bold"
    )


    # -------------------------------------------------------------------------
    # Left feature-key column
    # -------------------------------------------------------------------------

    fig.text(
        0.10,
        0.215,
        feature_key_left,
        ha="left",
        va="top",
        fontsize=15,
        fontfamily="Times New Roman",
        fontweight="bold",
        linespacing=1.25
    )


    # -------------------------------------------------------------------------
    # Right feature-key column
    # -------------------------------------------------------------------------

    fig.text(
        0.54,
        0.215,
        feature_key_right,
        ha="left",
        va="top",
        fontsize=15,
        fontfamily="Times New Roman",
        fontweight="bold",
        linespacing=1.25
    )


    # =========================================================================
    # Save figure
    # =========================================================================

    save_path = (
        SAVE_BASE
        /
        out_stub
    )


    fig.savefig(
        f"{save_path}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )


    fig.savefig(
        f"{save_path}.pdf",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )


    print(
        f"\nSaved figur<LOCAL_PATH>"
        f"{save_path}.png\n"
        f"{save_path}.pdf"
    )


    plt.show()


    plt.close(
        fig
    )


# =============================================================================
# Run plot
# =============================================================================

plot_single_model(
    model_name="Full Holdout",
    out_stub="Figure_TP_vs_FN_Full_Holdout"
)

# =============================================================================
# Run plot
# =============================================================================

plot_single_model(
    model_name="Full Holdout",
    out_stub="Figure_TP_vs_FN_Full_Holdout"
)


# =============================================================================
# Figure caption / feature key for manuscript
# =============================================================================

FIGURE_CAPTION = (

    "Figure X. Feature-level differences in autoencoder classification "

    "outcomes for true-positive (TP) and false-negative (FN) individuals "

    "in the full holdout dataset. Values represent the difference (Δ) in "

    "feature contribution, expressed in percentage points, relative to all "

    "holdout individuals. The dashed vertical line indicates no difference. "

    "Feature key: 1, velocity; 2, rolling velocity mean; "

    "3, rolling velocity standard deviation; "

    "4, rolling velocity skewness; "

    "5, rolling velocity kurtosis; "

    "6, absolute relative turn angle; "

    "7, rolling absolute relative turn angle mean; "

    "8, rolling absolute relative turn angle standard deviation; "

    "9, rolling absolute relative turn angle skewness; "

    "10, rolling absolute relative turn angle kurtosis; "

    "11, sinuosity; "

    "12, median space use; "

    "13, lower confidence interval of space use; and "

    "14, upper confidence interval of space use."
)


print(
    "\n"
    + "=" * 100
)


print(
    "FIGURE CAPTION"
)


print(
    "=" * 100
)


print(
    FIGURE_CAPTION
)

In [ ]:
import json
import os

def count_code_lines_in_ipynb(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    line_count = 0
    for cell in data.get("cells", []):
        if cell.get("cell_type") == "code":
            # Count non-empty lines inside the code cells
            for line in cell.get("source", []):
                if line.strip():  # skip empty or whitespace-only lines
                    line_count += 1
    return line_count

# Update this list with your own file paths
files = [
    "<LOCAL_PATH>",
    "<LOCAL_PATH>"
]

total_lines = 0

for f in files:
    if os.path.exists(f):
        count = count_code_lines_in_ipynb(f)
        total_lines += count
        print(f"{f}: {count} lines of code")
    else:
        print(f"{f}: File not found")

print(f"\nTotal lines of code across all notebooks: {total_lines}")
